In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
git clone https://github.com/halitdsgn-debug/MHproject.git

SyntaxError: invalid syntax (1651061183.py, line 1)

In [3]:
import os
if not os.path.exists("/kaggle/working/Unique3D"):
    !git clone https://github.com/AiuniAI/Unique3D.git /kaggle/working/Unique3D

%cd /kaggle/working/Unique3D
!pip install setuptools wheel ninja fire rembg
!pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"
!pip install -r requirements.txt

Cloning into '/kaggle/working/Unique3D'...
remote: Enumerating objects: 277, done.
remote: Total 277 (delta 0), reused 0 (delta 0), pack-reused 277 (from 1)
Receiving objects: 100% (277/277), 9.86 MiB | 27.66 MiB/s, done.
Resolving deltas: 100% (131/131), done.
/kaggle/working/Unique3D
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 88.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 92.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 90.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 34.1 MB/s eta 0:

In [4]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.0 FINAL)
# ══════════════════════════════════════════════════════════════════
# Her yeni oturumda bu tek hücreyi çalıştırın.
# Ön koşul: Kaggle → Settings → Accelerator → GPU T4 x2 + Restart
# ══════════════════════════════════════════════════════════════════

import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════════════
# 0) GPU KONTROLÜ
# ══════════════════════════════════════════════════════════════════
print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ══════════════════════════════════════════════════════════════════
# 1) DİSK TEMİZLİĞİ
# ══════════════════════════════════════════════════════════════════
print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")
os.system("df -h /kaggle/working")

# ══════════════════════════════════════════════════════════════════
# 2) PROJE KLONLAMA
# ══════════════════════════════════════════════════════════════════
print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")
print(f"   ✅ Proje dizini: {os.getcwd()}")

# ══════════════════════════════════════════════════════════════════
# 3) BAĞIMLILIKLAR (TÜMÜ)
# ══════════════════════════════════════════════════════════════════
print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
# Temel
os.system('pip install -q setuptools==69.5.1 wheel ninja 2>&1 | tail -1')
os.system('pip install -q fire omegaconf jaxtyping typeguard 2>&1 | tail -1')
os.system('pip install -q rembg 2>&1 | tail -1')
os.system('pip install -q pymeshlab trimesh pygltflib xatlas 2>&1 | tail -1')
os.system('pip install -q scikit-image opencv-python-headless 2>&1 | tail -1')
os.system('pip install -q onnxruntime hf_transfer 2>&1 | tail -1')
# ML
os.system('pip install -q huggingface_hub accelerate 2>&1 | tail -1')
os.system('pip install -q diffusers transformers safetensors 2>&1 | tail -1')
# Sürüm sabitleme
os.system('pip install -q "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" 2>&1 | tail -1')

# pytorch3d
if importlib.util.find_spec("pytorch3d") is None:
    print("   ⏳ pytorch3d derleniyor (2-4 dk)...")
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" 2>&1 | tail -3')

# nvdiffrast
if importlib.util.find_spec("nvdiffrast") is None:
    print("   ⏳ nvdiffrast derleniyor (2-3 dk)...")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" 2>&1 | tail -3')

# torch_scatter
if importlib.util.find_spec("torch_scatter") is None:
    print("   ⏳ torch_scatter kuruluyor...")
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if importlib.util.find_spec("torch_scatter") is None:
        print("   ⚠️ Wheel yok, kaynaktan derleniyor...")
        os.system('pip install -q --no-build-isolation "git+https://github.com/rusty1s/pytorch_scatter.git" 2>&1 | tail -2')

# Requirements (Unique3D'nin kendi listesi)
os.system("pip install -q -r requirements.txt 2>&1 | tail -1")
print("   ✅ Bağımlılıklar tamam.")

# ══════════════════════════════════════════════════════════════════
# 4) MODEL İNDİRME (KONTROLLÜ)
# ══════════════════════════════════════════════════════════════════
print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"

if os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10:
    print("   ✅ Model zaten mevcut, indirme atlanıyor.")
else:
    print("   ⏳ Model indiriliyor (18GB+, uzun sürebilir)...")
    os.system("rm -rf ckpt temp_download")
    
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Wuvin/Unique3D",
        repo_type="space",
        local_dir="temp_download",
        allow_patterns=["ckpt/**"],
        max_workers=2
    )
    
    if os.path.exists("temp_download/ckpt"):
        shutil.move("temp_download/ckpt", "ckpt")
    else:
        for item in os.listdir("temp_download"):
            if os.path.exists(item):
                shutil.rmtree(item) if os.path.isdir(item) else os.remove(item)
            shutil.move(f"temp_download/{item}", item)
    os.system("rm -rf temp_download")
    
    if not os.path.exists(MODEL_CHECK):
        print(f"❌ KRİTİK: {MODEL_CHECK} hala yok! Disk kontrol:")
        os.system("df -h /kaggle/working")
        os.system("ls -la ckpt/ 2>/dev/null")
        sys.exit(1)
    print("   ✅ Model indirildi.")

# ══════════════════════════════════════════════════════════════════
# 5) MODEL YAMALARI
# ══════════════════════════════════════════════════════════════════
print("\n🩹 5/8: Model yamaları uygulanıyor...")

# in_channels=4
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} -> 4 kanal")

# sd-variations
os.system("rm -rf ckpt/sd-variations")
os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    src = f"ckpt/img2mvimg/{comp}"
    dst = f"ckpt/sd-variations/{comp}"
    if os.path.exists(src):
        os.system(f"cp -r {src} {dst}")

with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({
        "_class_name": "StableDiffusionImage2MVCustomPipeline",
        "_diffusers_version": "0.10.0",
        "feature_extractor": ["transformers", "CLIPImageProcessor"],
        "image_encoder": ["transformers", "CLIPVisionModelWithProjection"],
        "safety_checker": [None, None],
        "scheduler": ["diffusers", "PNDMScheduler"],
        "unet": ["diffusers", "UNet2DConditionModel"],
        "vae": ["diffusers", "AutoencoderKL"]
    }, f, indent=2)

os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet")
os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

# YAML güncelle
for yaml_path in glob.glob("**/*.yaml", recursive=True):
    if os.path.exists(yaml_path):
        with open(yaml_path, "r") as f: c = f.read()
        c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations")
        c = c.replace("matttrent/sd-image-variations-diffusers", "./ckpt/sd-variations")
        c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
        c = c.replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
        with open(yaml_path, "w") as f: f.write(c)

# Çözünürlük 256
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    with open(mvi_path, "w") as f: f.write(mvi_code)

# VRAM optimizasyonu
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code += """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Uyarı: {e}")
    return pipe
"""
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

# rembg CPU
utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

# Gradio 6 parametreleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w") as f: f.write(content)

# class_embedding koruması
unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"^(\s*)self\.cls_embedding_param_dict\s*=\s*\{id\(param\):\s*param\s+for\s+param\s+in\s+self\.unet\.class_embedding\.parameters\(\)\}",
        r"\1self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code, flags=re.MULTILINE)
    with open(unifield_path, "w") as f: f.write(uf_code)

# Tüm dosyalarda runwayml
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(('.py', '.yaml', '.yml', '.json', '.txt')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()
                if "runwayml/stable-diffusion-v1-5" in content:
                    content = content.replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
                    with open(filepath, "w", encoding="utf-8") as f:
                        f.write(content)
            except: pass
print("   ✅ Model yamaları tamam.")

# ══════════════════════════════════════════════════════════════════
# 6) KRİTİK KÜTÜPHANE YAMALARI
# ══════════════════════════════════════════════════════════════════
print("\n🩹 6/8: Kritik kütüphane yamaları...")
with open("gradio_app.py", "r") as f: gc_code = f.read()

patches = '''
import os as _os
import json as _json

import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    if isinstance(pretrained_model_name_or_path, str):
        p = pretrained_model_name_or_path
        if p.endswith('.json') and _os.path.isfile(p):
            try:
                with open(p, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs:
                    if return_commit_hash: return config_dict, {}, None
                    return config_dict, {}
                if return_commit_hash: return config_dict, None
                return config_dict
            except Exception: pass
        is_hf_cache = 'huggingface' in p or '.cache/hub' in p or 'hub/models--' in p
        if p.startswith(('./', '/')) and not is_hf_cache and _os.path.isdir(p):
            subfolder = kwargs.get("subfolder", None) or ""
            config_name = getattr(cls, "config_name", "config.json")
            config_file = _os.path.join(p, subfolder, config_name) if subfolder else _os.path.join(p, config_name)
            if _os.path.isfile(config_file):
                with open(config_file, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs:
                    if return_commit_hash: return config_dict, {}, None
                    return config_dict, {}
                if return_commit_hash: return config_dict, None
                return config_dict
            model_index = _os.path.join(p, "model_index.json")
            if _os.path.isfile(model_index):
                with open(model_index, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs:
                    if return_commit_hash: return config_dict, {}, None
                    return config_dict, {}
                if return_commit_hash: return config_dict, None
                return config_dict
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)

_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"):
    _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = None
        if args: target_device = args[0]
        elif 'device' in kwargs: target_device = kwargs['device']
        if target_device is None or (hasattr(target_device, '__str__') and 'meta' in str(target_device)):
            target_device = 'cpu'
        print(f"⚠️ {meta_count} meta tensor -> to_empty({target_device})")
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

import diffusers.models.model_loading_utils as _mlu
_orig_load_into_meta = _mlu.load_model_dict_into_meta
def _patched_load_into_meta(model, state_dict, **kwargs):
    try: own_state = model.state_dict()
    except Exception: own_state = {}
    for name, param in list(state_dict.items()):
        if name in own_state and hasattr(param, 'shape') and hasattr(own_state[name], 'shape'):
            target_shape = own_state[name].shape
            if param.shape != target_shape and not param.is_meta:
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    state_dict[name] = param[slices].contiguous()
                except Exception: pass
    return _orig_load_into_meta(model, state_dict, **kwargs)
_mlu.load_model_dict_into_meta = _patched_load_into_meta

import torch.nn as _nn
_orig_load_state_dict = _nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    own_state = self.state_dict()
    filtered = {}
    for name, param in state_dict.items():
        if name in own_state:
            target_shape = own_state[name].shape
            if param.shape == target_shape:
                filtered[name] = param
            elif not param.is_meta:
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    filtered[name] = param[slices].contiguous()
                except Exception: pass
        else:
            filtered[name] = param
    try: return _orig_load_state_dict(self, filtered, strict=False, assign=assign)
    except Exception: return None
_nn.Module.load_state_dict = _patched_load_state_dict

import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache()
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _sdpa

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)
'''

if "Akıllı load_config" not in gc_code:
    gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)
print("   ✅ Kritik yamalar uygulandı.")

# ══════════════════════════════════════════════════════════════════
# 7) TÜM PAKETLERİN IMPORT DOĞRULAMASI
# ══════════════════════════════════════════════════════════════════
print("\n🔍 7/8: Tüm paketler import edilerek doğrulanıyor...")
packages_to_check = {
    "torch": "PyTorch", "pytorch3d": "PyTorch3D", "nvdiffrast": "Nvdiffrast",
    "torch_scatter": "Torch Scatter", "diffusers": "Diffusers",
    "transformers": "Transformers", "accelerate": "Accelerate",
    "rembg": "Rembg", "pymeshlab": "PyMeshLab", "trimesh": "Trimesh",
    "jaxtyping": "Jaxtyping", "onnxruntime": "ONNX Runtime",
    "huggingface_hub": "HuggingFace Hub", "safetensors": "Safetensors",
    "cv2": "OpenCV", "skimage": "Scikit-Image",
}
all_ok = True
for import_name, display_name in packages_to_check.items():
    try:
        __import__(import_name)
        print(f"   ✅ {display_name}")
    except ImportError as e:
        all_ok = False
        print(f"   ❌ {display_name} - HATA: {e}")

print("\n" + "="*70)
if all_ok:
    print("✅ TÜM BAĞIMLILIKLAR BAŞARIYLA KURULDU VE DOĞRULANDI!")
else:
    print("❌ BAZI BAĞIMLILIKLAR EKSİK VEYA HATALI! (yukarıya bakın)")
print("="*70)

# ══════════════════════════════════════════════════════════════════
# 8) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════════════
print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env,
                        stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)
public_url = None
last_size = 0

for _ in range(300):
    time.sleep(3)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            if satir.strip() and "_load_from_state_dict" not in satir and "UserWarning" not in satir and "warnings.warn" not in satir:
                print(f"👉 {satir[:140]}")
    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
            print("="*70)
            print("\n⏳ ÖNEMLİ: Linke tıklamadan önce 40-60 saniye bekleyin!")
            print("   Modeller arka planda hâlâ yükleniyor olabilir.\n")
            break
    if proc.poll() is not None:
        print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
        print("\nSon 30 log satırı:")
        os.system(f"tail -30 {LOG}")
        break

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   22M   20G   1% /kaggle/working

📦 2/8: Proje hazırlanıyor...
   ✅ Proje dizini: /kaggle/working/Unique3D

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.0 MB/s eta 0:00:00
cucim-cu12 26.2.1 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
   ⏳ nvdiffrast derleniyor (2-3 dk)...


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


   ⏳ torch_scatter kuruluyor...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 91.5 MB/s eta 0:00:00
note: This error originates from a subprocess, and is likely not a problem with pip.
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...
   ⏳ Model indiriliyor (18GB+, uzun sürebilir)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

   ✅ Model indirildi.

🩹 5/8: Model yamaları uygulanıyor...
   ✅ ckpt/img2mvimg/unet/config.json -> 4 kanal
   ✅ ckpt/image2normal/unet/config.json -> 4 kanal
   ✅ Model yamaları tamam.

🩹 6/8: Kritik kütüphane yamaları...
   ✅ Kritik yamalar uygulandı.

🔍 7/8: Tüm paketler import edilerek doğrulanıyor...
   ✅ PyTorch
   ✅ PyTorch3D
   ✅ Nvdiffrast
   ✅ Torch Scatter
   ✅ Diffusers
   ✅ Transformers
   ✅ Accelerate
   ✅ Rembg
   ✅ PyMeshLab
   ✅ Trimesh
   ✅ Jaxtyping
   ✅ ONNX Runtime
   ✅ HuggingFace Hub
   ✅ Safetensors
   ✅ OpenCV
   ✅ Scikit-Image

✅ TÜM BAĞIMLILIKLAR BAŞARIYLA KURULDU VE DOĞRULANDI!

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE LİNK TAKİBİ
👉 Downloading data from 'https://github.com/danielgatis/rembg/releases/download/v0.0.0/bria-rmbg-2.0.onnx' to file '/root/.rembg/models/bria-r
👉   0%|                                              | 0.00/1.02G [00:00<?, ?B/s]
👉   0%|                                      | 786k/1.02G [00:00<02:10, 7.86MB/s]
👉   0%|            

In [5]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.0 FINAL)
# ══════════════════════════════════════════════════════════════════
# Her yeni oturumda bu tek hücreyi çalıştırın.
# Ön koşul: Kaggle → Settings → Accelerator → GPU T4 x2 + Restart
# ══════════════════════════════════════════════════════════════════

import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════════════
# 0) GPU KONTROLÜ
# ══════════════════════════════════════════════════════════════════
print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ══════════════════════════════════════════════════════════════════
# 1) DİSK TEMİZLİĞİ
# ══════════════════════════════════════════════════════════════════
print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")
os.system("df -h /kaggle/working")

# ══════════════════════════════════════════════════════════════════
# 2) PROJE KLONLAMA
# ══════════════════════════════════════════════════════════════════
print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")
print(f"   ✅ Proje dizini: {os.getcwd()}")

# ══════════════════════════════════════════════════════════════════
# 3) BAĞIMLILIKLAR (TÜMÜ)
# ══════════════════════════════════════════════════════════════════
print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
# Temel
os.system('pip install -q setuptools==69.5.1 wheel ninja 2>&1 | tail -1')
os.system('pip install -q fire omegaconf jaxtyping typeguard 2>&1 | tail -1')
os.system('pip install -q rembg 2>&1 | tail -1')
os.system('pip install -q pymeshlab trimesh pygltflib xatlas 2>&1 | tail -1')
os.system('pip install -q scikit-image opencv-python-headless 2>&1 | tail -1')
os.system('pip install -q onnxruntime hf_transfer 2>&1 | tail -1')
# ML
os.system('pip install -q huggingface_hub accelerate 2>&1 | tail -1')
os.system('pip install -q diffusers transformers safetensors 2>&1 | tail -1')
# Sürüm sabitleme
os.system('pip install -q "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" 2>&1 | tail -1')

# pytorch3d
if importlib.util.find_spec("pytorch3d") is None:
    print("   ⏳ pytorch3d derleniyor (2-4 dk)...")
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" 2>&1 | tail -3')

# nvdiffrast
if importlib.util.find_spec("nvdiffrast") is None:
    print("   ⏳ nvdiffrast derleniyor (2-3 dk)...")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" 2>&1 | tail -3')

# torch_scatter
if importlib.util.find_spec("torch_scatter") is None:
    print("   ⏳ torch_scatter kuruluyor...")
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if importlib.util.find_spec("torch_scatter") is None:
        print("   ⚠️ Wheel yok, kaynaktan derleniyor...")
        os.system('pip install -q --no-build-isolation "git+https://github.com/rusty1s/pytorch_scatter.git" 2>&1 | tail -2')

# Requirements (Unique3D'nin kendi listesi)
os.system("pip install -q -r requirements.txt 2>&1 | tail -1")
print("   ✅ Bağımlılıklar tamam.")

# ══════════════════════════════════════════════════════════════════
# 4) MODEL İNDİRME (KONTROLLÜ)
# ══════════════════════════════════════════════════════════════════
print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"

if os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10:
    print("   ✅ Model zaten mevcut, indirme atlanıyor.")
else:
    print("   ⏳ Model indiriliyor (18GB+, uzun sürebilir)...")
    os.system("rm -rf ckpt temp_download")
    
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Wuvin/Unique3D",
        repo_type="space",
        local_dir="temp_download",
        allow_patterns=["ckpt/**"],
        max_workers=2
    )
    
    if os.path.exists("temp_download/ckpt"):
        shutil.move("temp_download/ckpt", "ckpt")
    else:
        for item in os.listdir("temp_download"):
            if os.path.exists(item):
                shutil.rmtree(item) if os.path.isdir(item) else os.remove(item)
            shutil.move(f"temp_download/{item}", item)
    os.system("rm -rf temp_download")
    
    if not os.path.exists(MODEL_CHECK):
        print(f"❌ KRİTİK: {MODEL_CHECK} hala yok! Disk kontrol:")
        os.system("df -h /kaggle/working")
        os.system("ls -la ckpt/ 2>/dev/null")
        sys.exit(1)
    print("   ✅ Model indirildi.")

# ══════════════════════════════════════════════════════════════════
# 5) MODEL YAMALARI
# ══════════════════════════════════════════════════════════════════
print("\n🩹 5/8: Model yamaları uygulanıyor...")

# in_channels=4
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} -> 4 kanal")

# sd-variations
os.system("rm -rf ckpt/sd-variations")
os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    src = f"ckpt/img2mvimg/{comp}"
    dst = f"ckpt/sd-variations/{comp}"
    if os.path.exists(src):
        os.system(f"cp -r {src} {dst}")

with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({
        "_class_name": "StableDiffusionImage2MVCustomPipeline",
        "_diffusers_version": "0.10.0",
        "feature_extractor": ["transformers", "CLIPImageProcessor"],
        "image_encoder": ["transformers", "CLIPVisionModelWithProjection"],
        "safety_checker": [None, None],
        "scheduler": ["diffusers", "PNDMScheduler"],
        "unet": ["diffusers", "UNet2DConditionModel"],
        "vae": ["diffusers", "AutoencoderKL"]
    }, f, indent=2)

os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet")
os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

# YAML güncelle
for yaml_path in glob.glob("**/*.yaml", recursive=True):
    if os.path.exists(yaml_path):
        with open(yaml_path, "r") as f: c = f.read()
        c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations")
        c = c.replace("matttrent/sd-image-variations-diffusers", "./ckpt/sd-variations")
        c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
        c = c.replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
        with open(yaml_path, "w") as f: f.write(c)

# Çözünürlük 256 ve Görünmez OOM Yaması
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    # Hata Göstergesini mesh refine içine yerleştiriyoruz
    mvi_code = re.sub(
        r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)',
        r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata veya VRAM taşması oldu!\\nDETAY: {e}\\n")\n\1    raise e',
        mvi_code
    )
    with open(mvi_path, "w") as f: f.write(mvi_code)

# VRAM optimizasyonu
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code += """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Uyarı: {e}")
    return pipe
"""
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

# rembg CPU
utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

# Gradio 6 parametreleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w") as f: f.write(content)

# class_embedding koruması
unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"^(\s*)self\.cls_embedding_param_dict\s*=\s*\{id\(param\):\s*param\s+for\s+param\s+in\s+self\.unet\.class_embedding\.parameters\(\)\}",
        r"\1self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code, flags=re.MULTILINE)
    with open(unifield_path, "w") as f: f.write(uf_code)

# Tüm dosyalarda runwayml
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(('.py', '.yaml', '.yml', '.json', '.txt')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()
                if "runwayml/stable-diffusion-v1-5" in content:
                    content = content.replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
                    with open(filepath, "w", encoding="utf-8") as f:
                        f.write(content)
            except: pass
print("   ✅ Model yamaları tamam.")

# ══════════════════════════════════════════════════════════════════
# 6) KRİTİK KÜTÜPHANE YAMALARI
# ══════════════════════════════════════════════════════════════════
print("\n🩹 6/8: Kritik kütüphane yamaları...")
with open("gradio_app.py", "r") as f: gc_code = f.read()

patches = '''
import os as _os
import json as _json

import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    if isinstance(pretrained_model_name_or_path, str):
        p = pretrained_model_name_or_path
        if p.endswith('.json') and _os.path.isfile(p):
            try:
                with open(p, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs:
                    if return_commit_hash: return config_dict, {}, None
                    return config_dict, {}
                if return_commit_hash: return config_dict, None
                return config_dict
            except Exception: pass
        is_hf_cache = 'huggingface' in p or '.cache/hub' in p or 'hub/models--' in p
        if p.startswith(('./', '/')) and not is_hf_cache and _os.path.isdir(p):
            subfolder = kwargs.get("subfolder", None) or ""
            config_name = getattr(cls, "config_name", "config.json")
            config_file = _os.path.join(p, subfolder, config_name) if subfolder else _os.path.join(p, config_name)
            if _os.path.isfile(config_file):
                with open(config_file, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs:
                    if return_commit_hash: return config_dict, {}, None
                    return config_dict, {}
                if return_commit_hash: return config_dict, None
                return config_dict
            model_index = _os.path.join(p, "model_index.json")
            if _os.path.isfile(model_index):
                with open(model_index, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs:
                    if return_commit_hash: return config_dict, {}, None
                    return config_dict, {}
                if return_commit_hash: return config_dict, None
                return config_dict
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)

_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"):
    _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = None
        if args: target_device = args[0]
        elif 'device' in kwargs: target_device = kwargs['device']
        if target_device is None or (hasattr(target_device, '__str__') and 'meta' in str(target_device)):
            target_device = 'cpu'
        print(f"⚠️ {meta_count} meta tensor -> to_empty({target_device})")
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

import diffusers.models.model_loading_utils as _mlu
_orig_load_into_meta = _mlu.load_model_dict_into_meta
def _patched_load_into_meta(model, state_dict, **kwargs):
    try: own_state = model.state_dict()
    except Exception: own_state = {}
    for name, param in list(state_dict.items()):
        if name in own_state and hasattr(param, 'shape') and hasattr(own_state[name], 'shape'):
            target_shape = own_state[name].shape
            if param.shape != target_shape and not param.is_meta:
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    state_dict[name] = param[slices].contiguous()
                except Exception: pass
    return _orig_load_into_meta(model, state_dict, **kwargs)
_mlu.load_model_dict_into_meta = _patched_load_into_meta

import torch.nn as _nn
_orig_load_state_dict = _nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    own_state = self.state_dict()
    filtered = {}
    for name, param in state_dict.items():
        if name in own_state:
            target_shape = own_state[name].shape
            if param.shape == target_shape:
                filtered[name] = param
            elif not param.is_meta:
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    filtered[name] = param[slices].contiguous()
                except Exception: pass
        else:
            filtered[name] = param
    try: return _orig_load_state_dict(self, filtered, strict=False, assign=assign)
    except Exception: return None
_nn.Module.load_state_dict = _patched_load_state_dict

import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache()
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _sdpa

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)
'''

if "Akıllı load_config" not in gc_code:
    gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)
print("   ✅ Kritik yamalar uygulandı.")

# ══════════════════════════════════════════════════════════════════
# 7) TÜM PAKETLERİN IMPORT DOĞRULAMASI
# ══════════════════════════════════════════════════════════════════
print("\n🔍 7/8: Tüm paketler import edilerek doğrulanıyor...")
packages_to_check = {
    "torch": "PyTorch", "pytorch3d": "PyTorch3D", "nvdiffrast": "Nvdiffrast",
    "torch_scatter": "Torch Scatter", "diffusers": "Diffusers",
    "transformers": "Transformers", "accelerate": "Accelerate",
    "rembg": "Rembg", "pymeshlab": "PyMeshLab", "trimesh": "Trimesh",
    "jaxtyping": "Jaxtyping", "onnxruntime": "ONNX Runtime",
    "huggingface_hub": "HuggingFace Hub", "safetensors": "Safetensors",
    "cv2": "OpenCV", "skimage": "Scikit-Image",
}
all_ok = True
for import_name, display_name in packages_to_check.items():
    try:
        __import__(import_name)
        print(f"   ✅ {display_name}")
    except ImportError as e:
        all_ok = False
        print(f"   ❌ {display_name} - HATA: {e}")

print("\n" + "="*70)
if all_ok:
    print("✅ TÜM BAĞIMLILIKLAR BAŞARIYLA KURULDU VE DOĞRULANDI!")
else:
    print("❌ BAZI BAĞIMLILIKLAR EKSİK VEYA HATALI! (yukarıya bakın)")
print("="*70)

# ══════════════════════════════════════════════════════════════════
# 8) SUNUCUYU BAŞLAT (HATA GÖSTERGELİ YENİ SİSTEM)
# ══════════════════════════════════════════════════════════════════
print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env,
                        stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE HATA TAKİBİ (BU HÜCRE HİÇ KAPANMAYACAK!)")
print("="*70)
public_url = None
last_size = 0
link_printed = False

# ARTIK DÖNGÜYÜ "FOR" İLE DEĞİL "WHILE" İLE SINIRSIZ YAPIYORUZ
while proc.poll() is None:
    time.sleep(1.5) # Hızlı tepki süresi
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir: continue
            if "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            
            # 🚨 GÖSTERGE SİSTEMİ BURADA: Hata içeren cümleleri yakalayıp vurguluyor
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA GÖSTERGESİ: {satir}")
            else:
                print(f"👉 {satir[:140]}")
                
    if not link_printed:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
            print("="*70)
            print("\n⏳ ÖNEMLİ: Linke tıklamadan önce 40-60 saniye bekleyin!")
            print("   Modeller arka planda hâlâ yükleniyor olabilir.")
            print("🚨 DİKKAT: İşlem yaparken hata alırsan artık gizli kalmayacak. Loglar direkt buraya (Kaggle terminaline) düşecek! 🚨\n")
            link_printed = True
            # ESKİ KODDAKİ BREAK KOMUTU SİLİNDİ, LOG AKMAYA DEVAM EDECEK!

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    print("\nSon 30 log satırı:")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   18G  1.8G  91% /kaggle/working

📦 2/8: Proje hazırlanıyor...
   ✅ Proje dizini: /kaggle/working/Unique3D

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
cucim-cu12 26.2.1 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
cucim-cu12 26.2.1 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
note: This error originates from a subprocess, and is likely not a problem with pip.
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...
   ✅ Model zaten mevcut, indirme atlanıyor.

🩹 5/8: Model yamaları uygulanıyor...
   ✅ ckpt/img2mvimg/unet/config.json -> 4 kanal
   ✅ ckpt/image2normal/unet/config.json -> 4 kanal
   ✅ Model yamaları tamam.

🩹 6/8: Kritik kütüphane yamaları...
   ✅ Kritik yamalar uygulandı.

🔍 7/8: Tüm paketler import edilerek doğrulanıyor...
   ✅ PyTorch
  

KeyboardInterrupt: 

In [6]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.1 FINAL - Açı Düzeltmeli)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

# 👉 KRİTİK YAMA: "AssertionError: The view direction is not correct" Hatasını Esneten Düzeltme
proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    # AssertionError fırlatan satırı bul ve sadece uyarı verip ters açıları (normal) negatife çevirecek şekilde değiştir
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); normals = -normals; cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")
with open("gradio_app.py", "r") as f: gc_code = f.read()
gc_code = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", gc_code)
gc_code = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", gc_code)

patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    for p_path in [p, _os.path.join(p, kwargs.get("subfolder", ""), getattr(cls, "config_name", "config.json")), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Downloading data from 'https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx' to file '/root/.u2net/u2net.onnx'.
👉 0%|                                               | 0.00/176M [00:00<?, ?B/s]
👉 0%|▏                                      | 740k/176M [00:00<00:23, 7.40MB/s]
👉 1%|▍                                     | 1.98M/176M [00:00<00:16, 10.2MB/s]
👉 2%|▊                                     | 3.79M/176M [00:00<00:12, 13.8MB/s]
👉 4%|█▍                                    | 6.61M/176M [00:00<00:08, 19.5MB/s]
👉 6%|██▎                                   | 10.8M/176M [00:00<00:06, 27.4MB/s]
👉 10%|███▋                                  | 17.0M/

In [7]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.2 FINAL - Parametre Yamalı)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); normals = -normals; cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

# 👉 KRİTİK YAMA: "AttributeError: 'NoneType' object has no attribute 'parameters'" çözen boşluktan bağımsız kod.
unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"self\.cls_embedding_param_dict\s*=\s*\{[^\}]*self\.unet\.class_embedding\.parameters\(\)\}",
        "self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code
    )
    with open(unifield_path, "w") as f: f.write(uf_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")
with open("gradio_app.py", "r") as f: gc_code = f.read()
gc_code = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", gc_code)
gc_code = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", gc_code)

patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    for p_path in [p, _os.path.join(p, kwargs.get("subfolder", ""), getattr(cls, "config_name", "config.json")), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Some weights of the model checkpoint at ./ckpt/img2mvimg were not used when initializing UnifieldWrappedUNet:
👉 ['class_embedding.weight']
👉 Some weights of UnifieldWrappedUNet were not initialized from the model checkpoint at ./ckpt/img2mvimg and are newly initialized because the
👉 - conv_in.weight: found shape torch.Size([320, 8, 3, 3]) in the checkpoint and torch.Size([320, 4, 3, 3]) in the model instantiated
👉 You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
👉 There are modules in AutoencoderKL that should be kept in float32

In [8]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.3 FINAL - NoneType Path Yamalı)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); normals = -normals; cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"self\.cls_embedding_param_dict\s*=\s*\{[^\}]*self\.unet\.class_embedding\.parameters\(\)\}",
        "self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code
    )
    with open(unifield_path, "w") as f: f.write(uf_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")
with open("gradio_app.py", "r") as f: gc_code = f.read()
gc_code = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", gc_code)
gc_code = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", gc_code)

patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    # 👉 NoneType hatası için düzeltme: sub ve cname değişkenlerini güvenli hale getiriyoruz
    sub = kwargs.get("subfolder") or ""
    cname = getattr(cls, "config_name", "config.json") or "config.json"
    
    for p_path in [p, _os.path.join(p, sub, cname), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Some weights of the model checkpoint at ./ckpt/img2mvimg were not used when initializing UnifieldWrappedUNet:
👉 ['class_embedding.weight']
👉 Some weights of UnifieldWrappedUNet were not initialized from the model checkpoint at ./ckpt/img2mvimg and are newly initialized because the
👉 - conv_in.weight: found shape torch.Size([320, 8, 3, 3]) in the checkpoint and torch.Size([320, 4, 3, 3]) in the model instantiated
👉 You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
👉 There are modules in AutoencoderKL that should be kept in float32

In [9]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.4 FINAL - Tensör Kırpma Yamalı)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); normals = -normals; cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"self\.cls_embedding_param_dict\s*=\s*\{[^\}]*self\.unet\.class_embedding\.parameters\(\)\}",
        "self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code
    )
    with open(unifield_path, "w") as f: f.write(uf_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")
with open("gradio_app.py", "r") as f: gc_code = f.read()
gc_code = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", gc_code)
gc_code = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", gc_code)

patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    sub = kwargs.get("subfolder") or ""
    cname = getattr(cls, "config_name", "config.json") or "config.json"
    for p_path in [p, _os.path.join(p, sub, cname), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

# 👉 KRİTİK YAMA: 8 kanallı tensörü 4 kanala kırpan (slice) bellek içi şekil düzenleyici
import diffusers.models.model_loading_utils as _mlu
_orig_load_into_meta = _mlu.load_model_dict_into_meta
def _patched_load_into_meta(model, state_dict, **kwargs):
    try: own_state = model.state_dict()
    except Exception: own_state = {}
    for name, param in list(state_dict.items()):
        if name in own_state and hasattr(param, 'shape') and hasattr(own_state[name], 'shape'):
            target_shape = own_state[name].shape
            if param.shape != target_shape and not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    state_dict[name] = param[slices].contiguous()
                except Exception: pass
    return _orig_load_into_meta(model, state_dict, **kwargs)
_mlu.load_model_dict_into_meta = _patched_load_into_meta

import torch.nn as _nn
_orig_load_state_dict = _nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    own_state = self.state_dict()
    filtered = {}
    for name, param in state_dict.items():
        if name in own_state:
            target_shape = own_state[name].shape
            if param.shape == target_shape:
                filtered[name] = param
            elif not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    filtered[name] = param[slices].contiguous()
                except Exception: pass
        else:
            filtered[name] = param
    try: return _orig_load_state_dict(self, filtered, strict=False, assign=assign)
    except Exception: return None
_nn.Module.load_state_dict = _patched_load_state_dict

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Some weights of the model checkpoint at ./ckpt/img2mvimg were not used when initializing UnifieldWrappedUNet:
👉 ['class_embedding.weight']
👉 Some weights of UnifieldWrappedUNet were not initialized from the model checkpoint at ./ckpt/img2mvimg and are newly initialized because the
👉 - conv_in.weight: found shape torch.Size([320, 8, 3, 3]) in the checkpoint and torch.Size([320, 4, 3, 3]) in the model instantiated
👉 You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
👉 There are modules in AutoencoderKL that should be kept in float32

In [10]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.5 FINAL - Gradio Share Yamalı)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); normals = -normals; cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"self\.cls_embedding_param_dict\s*=\s*\{[^\}]*self\.unet\.class_embedding\.parameters\(\)\}",
        "self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code
    )
    with open(unifield_path, "w") as f: f.write(uf_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")

# 👉 KRİTİK YAMA: Yeni Gradio sürümlerinde desteklenmeyen show_share_button argümanı
# Sadece gradio_app.py'den değil, tüm python dosyalarından (özellikle app altındakilerden) siliyoruz.
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    c = f.read()
                if "show_share_button" in c:
                    c = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", c)
                    c = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", c)
                    with open(filepath, "w", encoding="utf-8") as f:
                        f.write(c)
            except: pass

with open("gradio_app.py", "r") as f: gc_code = f.read()
patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    sub = kwargs.get("subfolder") or ""
    cname = getattr(cls, "config_name", "config.json") or "config.json"
    for p_path in [p, _os.path.join(p, sub, cname), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.model_loading_utils as _mlu
_orig_load_into_meta = _mlu.load_model_dict_into_meta
def _patched_load_into_meta(model, state_dict, **kwargs):
    try: own_state = model.state_dict()
    except Exception: own_state = {}
    for name, param in list(state_dict.items()):
        if name in own_state and hasattr(param, 'shape') and hasattr(own_state[name], 'shape'):
            target_shape = own_state[name].shape
            if param.shape != target_shape and not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    state_dict[name] = param[slices].contiguous()
                except Exception: pass
    return _orig_load_into_meta(model, state_dict, **kwargs)
_mlu.load_model_dict_into_meta = _patched_load_into_meta

import torch.nn as _nn
_orig_load_state_dict = _nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    own_state = self.state_dict()
    filtered = {}
    for name, param in state_dict.items():
        if name in own_state:
            target_shape = own_state[name].shape
            if param.shape == target_shape:
                filtered[name] = param
            elif not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    filtered[name] = param[slices].contiguous()
                except Exception: pass
        else:
            filtered[name] = param
    try: return _orig_load_state_dict(self, filtered, strict=False, assign=assign)
    except Exception: return None
_nn.Module.load_state_dict = _patched_load_state_dict

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Some weights of the model checkpoint at ./ckpt/img2mvimg were not used when initializing UnifieldWrappedUNet:
👉 ['class_embedding.weight']
👉 Some weights of UnifieldWrappedUNet were not initialized from the model checkpoint at ./ckpt/img2mvimg and are newly initialized because the
👉 - conv_in.weight: found shape torch.Size([320, 8, 3, 3]) in the checkpoint and torch.Size([320, 4, 3, 3]) in the model instantiated
👉 You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
👉 There are modules in AutoencoderKL that should be kept in float32

KeyboardInterrupt: 

In [11]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.6 FINAL - OOM & VRAM Yamalı)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); normals = -normals; cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"self\.cls_embedding_param_dict\s*=\s*\{[^\}]*self\.unet\.class_embedding\.parameters\(\)\}",
        "self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code
    )
    with open(unifield_path, "w") as f: f.write(uf_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")

# 👉 KRİTİK YAMA 1: Gradio Arayüzünde show_share_button temizliği
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f: c = f.read()
                if "show_share_button" in c:
                    c = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", c)
                    c = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", c)
                    with open(filepath, "w", encoding="utf-8") as f: f.write(c)
            except: pass

# 👉 KRİTİK YAMA 2: Aşama geçişlerinde GPU belleğini (VRAM) agresif olarak boşalt (Çöp toplama)
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r") as f: g_code = f.read()
    if "import gc, torch" not in g_code: g_code = "import gc, torch\n" + g_code
    g_code = g_code.replace("new_meshes = geo_reconstruct", "gc.collect(); torch.cuda.empty_cache()\n    new_meshes = geo_reconstruct")
    g_code = g_code.replace("mv_images = run_mvprediction", "gc.collect(); torch.cuda.empty_cache()\n    mv_images = run_mvprediction")
    with open(gen_path, "w") as f: f.write(g_code)

with open("gradio_app.py", "r") as f: gc_code = f.read()

patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    sub = kwargs.get("subfolder") or ""
    cname = getattr(cls, "config_name", "config.json") or "config.json"
    for p_path in [p, _os.path.join(p, sub, cname), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.model_loading_utils as _mlu
_orig_load_into_meta = _mlu.load_model_dict_into_meta
def _patched_load_into_meta(model, state_dict, **kwargs):
    try: own_state = model.state_dict()
    except Exception: own_state = {}
    for name, param in list(state_dict.items()):
        if name in own_state and hasattr(param, 'shape') and hasattr(own_state[name], 'shape'):
            target_shape = own_state[name].shape
            if param.shape != target_shape and not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    state_dict[name] = param[slices].contiguous()
                except Exception: pass
    return _orig_load_into_meta(model, state_dict, **kwargs)
_mlu.load_model_dict_into_meta = _patched_load_into_meta

import torch.nn as _nn
_orig_load_state_dict = _nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    own_state = self.state_dict()
    filtered = {}
    for name, param in state_dict.items():
        if name in own_state:
            target_shape = own_state[name].shape
            if param.shape == target_shape:
                filtered[name] = param
            elif not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    filtered[name] = param[slices].contiguous()
                except Exception: pass
        else:
            filtered[name] = param
    try: return _orig_load_state_dict(self, filtered, strict=False, assign=assign)
    except Exception: return None
_nn.Module.load_state_dict = _patched_load_state_dict

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)

# 👉 KRİTİK YAMA 3: 4 GB VRAM Taşıran Devasa Matrisleri 256MB Lokmalara Bölme
import torch.nn.functional as _F
_orig_sdpa = _F.scaled_dot_product_attention
def _patched_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    try:
        if query.size(2) > 256 and attn_mask is None:
            out = []
            for i in range(0, query.size(2), 256):
                out.append(_orig_sdpa(query[:, :, i:i+256, :], key, value, None, dropout_p, is_causal, scale))
            return torch.cat(out, dim=2)
    except Exception: pass
    return _orig_sdpa(query, key, value, attn_mask, dropout_p, is_causal, scale)
_F.scaled_dot_product_attention = _patched_sdpa
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Some weights of the model checkpoint at ./ckpt/img2mvimg were not used when initializing UnifieldWrappedUNet:
👉 ['class_embedding.weight']
👉 Some weights of UnifieldWrappedUNet were not initialized from the model checkpoint at ./ckpt/img2mvimg and are newly initialized because the
👉 - conv_in.weight: found shape torch.Size([320, 8, 3, 3]) in the checkpoint and torch.Size([320, 4, 3, 3]) in the model instantiated
👉 You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
👉 There are modules in AutoencoderKL that should be kept in float32

KeyboardInterrupt: 

In [12]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.7 FINAL - PyTorch SDPA Yamalı)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); normals = -normals; cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"self\.cls_embedding_param_dict\s*=\s*\{[^\}]*self\.unet\.class_embedding\.parameters\(\)\}",
        "self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code
    )
    with open(unifield_path, "w") as f: f.write(uf_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f: c = f.read()
                if "show_share_button" in c:
                    c = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", c)
                    c = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", c)
                    with open(filepath, "w", encoding="utf-8") as f: f.write(c)
            except: pass

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r") as f: g_code = f.read()
    if "import gc, torch" not in g_code: g_code = "import gc, torch\n" + g_code
    g_code = g_code.replace("new_meshes = geo_reconstruct", "gc.collect(); torch.cuda.empty_cache()\n    new_meshes = geo_reconstruct")
    g_code = g_code.replace("mv_images = run_mvprediction", "gc.collect(); torch.cuda.empty_cache()\n    mv_images = run_mvprediction")
    with open(gen_path, "w") as f: f.write(g_code)

with open("gradio_app.py", "r") as f: gc_code = f.read()

patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    sub = kwargs.get("subfolder") or ""
    cname = getattr(cls, "config_name", "config.json") or "config.json"
    for p_path in [p, _os.path.join(p, sub, cname), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.model_loading_utils as _mlu
_orig_load_into_meta = _mlu.load_model_dict_into_meta
def _patched_load_into_meta(model, state_dict, **kwargs):
    try: own_state = model.state_dict()
    except Exception: own_state = {}
    for name, param in list(state_dict.items()):
        if name in own_state and hasattr(param, 'shape') and hasattr(own_state[name], 'shape'):
            target_shape = own_state[name].shape
            if param.shape != target_shape and not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    state_dict[name] = param[slices].contiguous()
                except Exception: pass
    return _orig_load_into_meta(model, state_dict, **kwargs)
_mlu.load_model_dict_into_meta = _patched_load_into_meta

import torch.nn as _nn
_orig_load_state_dict = _nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    own_state = self.state_dict()
    filtered = {}
    for name, param in state_dict.items():
        if name in own_state:
            target_shape = own_state[name].shape
            if param.shape == target_shape:
                filtered[name] = param
            elif not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    filtered[name] = param[slices].contiguous()
                except Exception: pass
        else:
            filtered[name] = param
    try: return _orig_load_state_dict(self, filtered, strict=False, assign=assign)
    except Exception: return None
_nn.Module.load_state_dict = _patched_load_state_dict

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)

# 👉 KRİTİK YAMA 3 v2: VRAM Taşıran Devasa Matrisleri modern PyTorch (keyword-arg) yapısına uygun bölen yamamız.
import torch.nn.functional as _F
_orig_sdpa = _F.scaled_dot_product_attention
def _patched_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    try:
        if query.size(2) > 256 and attn_mask is None:
            out = []
            for i in range(0, query.size(2), 256):
                out.append(_orig_sdpa(
                    query[:, :, i:i+256, :], key, value, 
                    attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, scale=scale
                ))
            return torch.cat(out, dim=2)
    except Exception: pass
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
_F.scaled_dot_product_attention = _patched_sdpa
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Some weights of the model checkpoint at ./ckpt/img2mvimg were not used when initializing UnifieldWrappedUNet:
👉 ['class_embedding.weight']
👉 Some weights of UnifieldWrappedUNet were not initialized from the model checkpoint at ./ckpt/img2mvimg and are newly initialized because the
👉 - conv_in.weight: found shape torch.Size([320, 8, 3, 3]) in the checkpoint and torch.Size([320, 4, 3, 3]) in the model instantiated
👉 You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
👉 There are modules in AutoencoderKL that should be kept in float32

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 🚀 UNIQUE3D ANAHTAR KODU (v41.8 FINAL - UnboundLocalError Yamalı)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, re, json, glob, shutil, importlib
import torch

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🔍 GPU kontrolü...")
if not torch.cuda.is_available():
    print("❌ GPU YOK! Kaggle → Settings → Accelerator → GPU T4 x2 seçin ve Restart Session yapın.")
    sys.exit(1)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

print("\n🧹 1/8: Disk temizleniyor...")
os.system("pip cache purge >/dev/null 2>&1")
os.system("rm -rf /tmp/* >/dev/null 2>&1")
os.system("rm -rf /root/.cache/huggingface/hub >/dev/null 2>&1")
os.system(f"rm -rf {u3d_dir}/temp_download >/dev/null 2>&1")
os.system("find /kaggle/working -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null")

print("\n📦 2/8: Proje hazırlanıyor...")
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
os.chdir(u3d_dir)
os.system("git checkout . >/dev/null 2>&1")

print("\n📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...")
os.system('pip install -q setuptools==69.5.1 wheel ninja fire omegaconf jaxtyping typeguard rembg pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless onnxruntime hf_transfer huggingface_hub accelerate diffusers transformers safetensors "numpy==2.1.3" "scipy==1.14.1" "numba==0.61.0" >/dev/null 2>&1')
if importlib.util.find_spec("pytorch3d") is None:
    os.system('pip install -q --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git" >/dev/null 2>&1')
if importlib.util.find_spec("nvdiffrast") is None:
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system('pip install -q --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git" >/dev/null 2>&1')
if importlib.util.find_spec("torch_scatter") is None:
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html >/dev/null 2>&1")
os.system("pip install -q -r requirements.txt >/dev/null 2>&1")
print("   ✅ Bağımlılıklar tamam.")

print("\n📥 4/8: Model kontrol ediliyor...")
MODEL_CHECK = "ckpt/img2mvimg/unet/config.json"
if not (os.path.exists(MODEL_CHECK) and os.path.getsize(MODEL_CHECK) > 10):
    os.system("rm -rf ckpt temp_download")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", local_dir="temp_download", allow_patterns=["ckpt/**"], max_workers=2)
    shutil.move("temp_download/ckpt", "ckpt") if os.path.exists("temp_download/ckpt") else None
    os.system("rm -rf temp_download")

print("\n🩹 5/8: Model yamaları uygulanıyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/image2normal/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

os.system("rm -rf ckpt/sd-variations"); os.makedirs("ckpt/sd-variations", exist_ok=True)
for comp in ["image_encoder", "feature_extractor", "scheduler"]:
    if os.path.exists(f"ckpt/img2mvimg/{comp}"): os.system(f"cp -r ckpt/img2mvimg/{comp} ckpt/sd-variations/{comp}")
with open("ckpt/sd-variations/model_index.json", "w") as f:
    json.dump({"_class_name": "StableDiffusionImage2MVCustomPipeline", "_diffusers_version": "0.10.0", "feature_extractor": ["transformers", "CLIPImageProcessor"], "image_encoder": ["transformers", "CLIPVisionModelWithProjection"], "safety_checker": [None, None], "scheduler": ["diffusers", "PNDMScheduler"], "unet": ["diffusers", "UNet2DConditionModel"], "vae": ["diffusers", "AutoencoderKL"]}, f, indent=2)
os.system("rm -f ckpt/sd-variations/unet ckpt/sd-variations/vae")
os.system("ln -s ../img2mvimg/unet ckpt/sd-variations/unet"); os.system("ln -s ../img2mvimg/vae ckpt/sd-variations/vae")

for yaml_path in glob.glob("**/*.yaml", recursive=True):
    with open(yaml_path, "r") as f: c = f.read()
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "./ckpt/sd-variations").replace("runwayml/stable-diffusion-v1-5", "stable-diffusion-v1-5/stable-diffusion-v1-5")
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    with open(yaml_path, "w") as f: f.write(c)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    mvi_code = re.sub(r'([ \t]+)vertices, faces = run_mesh_refine\(.*?\)', r'\1try:\n\1    \g<0>\n\1except Exception as e:\n\1    print(f"\\n🚨 HATA GÖSTERGESİ: 3D Mesh (Örgü) hesaplanırken hata oldu!\\nDETAY: {e}\\n")\n\1    raise e', mvi_code)
    with open(mvi_path, "w") as f: f.write(mvi_code)

# 👉 KRİTİK YAMA: "normals" hatasını çözen güvenli açı düzeltme kodumuz
proj_path = "scripts/project_mesh.py"
if os.path.exists(proj_path):
    with open(proj_path, "r") as f: proj_code = f.read()
    proj_code = re.sub(
        r'assert cos_angles\.mean\(\) < 0, f"The view direction is not correct.*?"', 
        r'if cos_angles.mean() > 0:\n        print(f"⚠️ Yüzey açıları ters hesaplandı ({cos_angles.mean()}), otomatik düzeltiliyor..."); cos_angles = -cos_angles', 
        proj_code
    )
    with open(proj_path, "w") as f: f.write(proj_code)

unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r") as f: uf_code = f.read()
    uf_code = re.sub(
        r"self\.cls_embedding_param_dict\s*=\s*\{[^\}]*self\.unet\.class_embedding\.parameters\(\)\}",
        "self.cls_embedding_param_dict = {id(param): param for param in self.unet.class_embedding.parameters()} if getattr(self.unet, 'class_embedding', None) is not None else {}",
        uf_code
    )
    with open(unifield_path, "w") as f: f.write(uf_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r") as f: u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code = u_code.replace("return pipeline", "try:\n        if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing('max')\n        if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\n    except Exception as e: print(e)\n    return pipeline")
        with open(utils_path, "w") as f: f.write(u_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CPUExecutionProvider']")
    with open(utils2_path, "w") as f: f.write(u2_code)

print("\n🩹 6/8: Kritik kütüphane yamaları...")

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f: c = f.read()
                if "show_share_button" in c:
                    c = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", c)
                    c = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", c)
                    with open(filepath, "w", encoding="utf-8") as f: f.write(c)
            except: pass

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r") as f: g_code = f.read()
    if "import gc, torch" not in g_code: g_code = "import gc, torch\n" + g_code
    g_code = g_code.replace("new_meshes = geo_reconstruct", "gc.collect(); torch.cuda.empty_cache()\n    new_meshes = geo_reconstruct")
    g_code = g_code.replace("mv_images = run_mvprediction", "gc.collect(); torch.cuda.empty_cache()\n    mv_images = run_mvprediction")
    with open(gen_path, "w") as f: f.write(g_code)

with open("gradio_app.py", "r") as f: gc_code = f.read()

patches = '''
import os as _os, json as _json, torch
import huggingface_hub.utils._validators as _validators
_validators.validate_repo_id = lambda *a, **k: None

import diffusers.configuration_utils as _cu
_orig_load_config = _cu.ConfigMixin.load_config
@classmethod
def _patched_load_config(cls, pretrained_model_name_or_path, return_unused_kwargs=False, return_commit_hash=False, **kwargs):
    p = str(pretrained_model_name_or_path)
    sub = kwargs.get("subfolder") or ""
    cname = getattr(cls, "config_name", "config.json") or "config.json"
    for p_path in [p, _os.path.join(p, sub, cname), _os.path.join(p, "model_index.json")]:
        if _os.path.isfile(p_path):
            try:
                with open(p_path, 'r') as f: config_dict = _json.load(f)
                if return_unused_kwargs: return (config_dict, {}, None) if return_commit_hash else (config_dict, {})
                return (config_dict, None) if return_commit_hash else config_dict
            except Exception: pass
    return _orig_load_config.__func__(cls, pretrained_model_name_or_path, return_unused_kwargs=return_unused_kwargs, return_commit_hash=return_commit_hash, **kwargs)
_cu.ConfigMixin.load_config = _patched_load_config
if hasattr(_cu, "ModelMixin"): _cu.ModelMixin.load_config = _patched_load_config

import diffusers.models.model_loading_utils as _mlu
_orig_load_into_meta = _mlu.load_model_dict_into_meta
def _patched_load_into_meta(model, state_dict, **kwargs):
    try: own_state = model.state_dict()
    except Exception: own_state = {}
    for name, param in list(state_dict.items()):
        if name in own_state and hasattr(param, 'shape') and hasattr(own_state[name], 'shape'):
            target_shape = own_state[name].shape
            if param.shape != target_shape and not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    state_dict[name] = param[slices].contiguous()
                except Exception: pass
    return _orig_load_into_meta(model, state_dict, **kwargs)
_mlu.load_model_dict_into_meta = _patched_load_into_meta

import torch.nn as _nn
_orig_load_state_dict = _nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    own_state = self.state_dict()
    filtered = {}
    for name, param in state_dict.items():
        if name in own_state:
            target_shape = own_state[name].shape
            if param.shape == target_shape:
                filtered[name] = param
            elif not getattr(param, 'is_meta', False):
                try:
                    slices = tuple(slice(0, min(p, t)) for p, t in zip(param.shape, target_shape))
                    filtered[name] = param[slices].contiguous()
                except Exception: pass
        else:
            filtered[name] = param
    try: return _orig_load_state_dict(self, filtered, strict=False, assign=assign)
    except Exception: return None
_nn.Module.load_state_dict = _patched_load_state_dict

import diffusers.models.modeling_utils as _mu
_orig_model_to = _mu.ModelMixin.to
def _patched_model_to(self, *args, **kwargs):
    meta_count = sum(1 for p in self.parameters() if p.is_meta)
    if meta_count > 0:
        target_device = args[0] if args else kwargs.get('device', 'cpu')
        target_device = 'cpu' if target_device is None or 'meta' in str(target_device) else target_device
        try: self.to_empty(device=target_device)
        except Exception: self.to_empty(device='cpu')
    return _orig_model_to(self, *args, **kwargs)
_mu.ModelMixin.to = _patched_model_to

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

import diffusers.pipelines.pipeline_utils as _pu
_orig = _pu.DiffusionPipeline.from_pretrained.__func__
def _patched(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto": del kwargs["device_map"]
    return _orig(cls, *args, **kwargs)
_pu.DiffusionPipeline.from_pretrained = classmethod(_patched)

import torch.nn.functional as _F
_orig_sdpa = _F.scaled_dot_product_attention
def _patched_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    try:
        if query.size(2) > 256 and attn_mask is None:
            out = []
            for i in range(0, query.size(2), 256):
                out.append(_orig_sdpa(
                    query[:, :, i:i+256, :], key, value, 
                    attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, scale=scale
                ))
            return torch.cat(out, dim=2)
    except Exception: pass
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
_F.scaled_dot_product_attention = _patched_sdpa
'''
if "Akıllı load_config" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w") as f: f.write(gc_code)

print("\n🚀 8/8: Sunucu başlatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("\n" + "="*70 + "\n📺 CANLI İŞLEM VE HATA TAKİBİ\n" + "="*70)
public_url = None; last_size = 0; link_printed = False

while proc.poll() is None:
    time.sleep(1.5)
    if not os.path.exists(LOG): continue
    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size); yeni = f.read(); last_size = boyut
        for satir in yeni.strip().split('\n'):
            satir = satir.strip()
            if not satir or "_load_from_state_dict" in satir or "UserWarning" in satir or "warnings.warn" in satir: continue
            if "Error" in satir or "Traceback" in satir or "Exception" in satir or "out of memory" in satir.lower():
                print(f"🚨 HATA: {satir}")
            else: print(f"👉 {satir[:140]}")
    if not link_printed and (m := re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", open(LOG, errors="ignore").read())):
        public_url = m.group(0)
        print("\n" + "="*70 + f"\n🌐 LİNKİNİZ: {public_url}\n" + "="*70)
        print("⏳ ÖNEMLİ: Linke tıklamadan 40 saniye bekleyin!\n")
        link_printed = True

if proc.poll() is not None:
    print(f"\n❌ Süreç öldü (kod: {proc.returncode})")
    os.system(f"tail -30 {LOG}")

🔍 GPU kontrolü...
✅ GPU: Tesla T4

🧹 1/8: Disk temizleniyor...

📦 2/8: Proje hazırlanıyor...

📦 3/8: Bağımlılıklar kuruluyor (5-10 dk)...
   ✅ Bağımlılıklar tamam.

📥 4/8: Model kontrol ediliyor...

🩹 5/8: Model yamaları uygulanıyor...

🩹 6/8: Kritik kütüphane yamaları...

🚀 8/8: Sunucu başlatılıyor...

📺 CANLI İŞLEM VE HATA TAKİBİ
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Some weights of the model checkpoint at ./ckpt/img2mvimg were not used when initializing UnifieldWrappedUNet:
👉 ['class_embedding.weight']
👉 Some weights of UnifieldWrappedUNet were not initialized from the model checkpoint at ./ckpt/img2mvimg and are newly initialized because the
👉 - conv_in.weight: found shape torch.Size([320, 8, 3, 3]) in the checkpoint and torch.Size([320, 4, 3, 3]) in the model instantiated
👉 You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
👉 There are modules in AutoencoderKL that should be kept in float32

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) KLONLAMA VE DİZİN KONTROLÜ
# ══════════════════════════════════════════════════════════
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    print("📦 Unique3D reposu bulunamadı veya eksik. Sıfırdan indiriliyor...")
    os.system(f"rm -rf {u3d_dir}")
    os.chdir(WORK_DIR)
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")
    if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
        print("❌ HATA: Klonlama başarısız oldu.")
        sys.exit(1)
    print("✅ Repo başarıyla indirildi.")

os.chdir(u3d_dir)
print(f"✅ Çalışma dizini: {os.getcwd()}")

# ══════════════════════════════════════════════════════════
# 2) KÜTÜPHANELER
# ══════════════════════════════════════════════════════════
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q fire omegaconf jaxtyping typeguard pymeshlab trimesh pygltflib xatlas scikit-image opencv-python-headless")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q onnxruntime hf_transfer")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62'")

if os.system("python -c 'import pytorch3d' 2>/dev/null") != 0:
    import torch
    TV = torch.__version__.split('+')[0]
    CV = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
    os.system(f"pip install -q pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py313_cu{CV}_pyt{TV}/download.html")
    if os.system("python -c 'import pytorch3d' 2>/dev/null") != 0:
        os.system("pip install -q 'git+https://github.com/facebookresearch/pytorch3d.git@stable'")

# ══════════════════════════════════════════════════════════
# 3) MODELLERİ İNDİR
# ══════════════════════════════════════════════════════════
print("📥 Model ağırlıkları kontrol ediliyor (Zaten indiyse atlar)...")
os.makedirs("ckpt", exist_ok=True)
from huggingface_hub import snapshot_download
try:
    snapshot_download(
        repo_id="Wuvin/Unique3D", repo_type="space", local_dir="ckpt",
        ignore_patterns=["*.md", ".git*", "*.png", "*.jpg", "*.gif", "*.mp4", "*.ipynb"]
    )
    print("✅ Modeller hazır.")
except Exception as e:
    print(f"⚠️ Model indirme atlandı: {e}")

# ══════════════════════════════════════════════════════════
# 4) YAMALAR (VRAM, YEREL YOL, GRADIO, DEVICE_MAP)
# ══════════════════════════════════════════════════════════
print("⚙️ Yamalar uygulanıyor...")

# Kanallar
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# Çözünürlük
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(r"^(\s*)rm_normals = predict_normals\(", r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(", mvi_code, flags=re.MULTILINE)
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

# Attention Slicing
attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f: attn_code = f.read()
    attn_code = re.sub(r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)", "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)", attn_code)
    with open(attn_path, "w", encoding="utf-8") as f: f.write(attn_code)

# CPU Offload
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

# Gradio temizliği
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

# unifield_processor local_files_only
unifield_path = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(unifield_path):
    with open(unifield_path, "r", encoding="utf-8") as f: uf_code = f.read()
    if "ignore_mismatched_sizes=True," in uf_code:
        uf_code = uf_code.replace("ignore_mismatched_sizes=True,", "ignore_mismatched_sizes=True, local_files_only=True,")
    elif "**unet_kwargs" in uf_code:
        uf_code = uf_code.replace("**unet_kwargs", '**{"local_files_only": True, **unet_kwargs}')
    with open(unifield_path, "w", encoding="utf-8") as f: f.write(uf_code)

# gradio_app.py içine KRİTİK YAMALAR (SDPA, cached_download, device_map fix)
with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download
import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

# KRİTİK YAMA: device_map="auto" parametresini sil
import diffusers.pipelines.pipeline_utils as _pipeline_utils
_orig_from_pretrained = _pipeline_utils.DiffusionPipeline.from_pretrained.__func__
def _patched_from_pretrained(cls, *args, **kwargs):
    if kwargs.get("device_map") == "auto":
        del kwargs["device_map"]
    return _orig_from_pretrained(cls, *args, **kwargs)
_pipeline_utils.DiffusionPipeline.from_pretrained = classmethod(_patched_from_pretrained)
"""
if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 5) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
    "HF_HUB_OFFLINE": "1", # Yerel dosyaları zorla
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size); yeni = f.read(); last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [12]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🩹 Bozulan dosyalar git ile orijinal haline döndürülüyor...")
os.system("git checkout scripts/multiview_inference.py app/custom_models/utils.py custum_3d_diffusion/custum_modules/attention_processors.py")
time.sleep(1)

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Çözünürlük 256x256'ya düşürülüyor ve VRAM temizliği ekleniyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(
        r"^(\s*)rm_normals = predict_normals\(",
        r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(",
        mvi_code,
        flags=re.MULTILINE
    )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)

# ══════════════════════════════════════════════════════════
# 🚨 DİLİM (CHUNK) BOYUTUNU ARTIR VE BELLEK TEMİZLE
# ══════════════════════════════════════════════════════════
print("🩹 Agresif Attention Slicing (128'lik dilimler) ve Bellek Temizliği aktif ediliyor...")

# 1. Unique3D'nin kendi attention dosyasını değiştir
attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f:
        attn_code = f.read()
    # Dosyadaki F.scaled_dot_product_attention'ı bul ve döngüye al
    attn_code = re.sub(
        r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)",
        "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)",
        attn_code
    )
    with open(attn_path, "w", encoding="utf-8") as f:
        f.write(attn_code)
    print("   ✅ attention_processors.py içine 128'lik dilimleme eklendi")

# 2. Ana dosyaya (gradio_app.py) PyTorch seviyesinde en agresif yamayı ekle
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
import torch.nn.functional as F

# PyTorch'un SDPA fonksiyonunu 128'lik dilimlerle ve zorla bellek temizleyerek çalıştır
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    # Her zaman dilimle (VRAM patlamasını önlemek için)
    chunk_size = 128
    outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out)
        # Her dilimden sonra GPU belleğini zorla boşalt
        torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)

F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)
with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# 3. Model Yükleme Ayarları
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'):
            pipe.enable_model_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
    print("   ✅ Model CPU Offload aktif edildi")

# Gradio Hata Düzeltmeleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

# ══════════════════════════════════════════════════════════
# SUNUCUYU BAŞLAT (Maksimum Bellek Tasarrufu Ayarları)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...
🩹 Bozulan dosyalar git ile orijinal haline döndürülüyor...


Updated 3 paths from the index


⚙️ Model kanal yapılandırmaları kontrol ediliyor...
🩹 Çözünürlük 256x256'ya düşürülüyor ve VRAM temizliği ekleniyor...
🩹 Agresif Attention Slicing (128'lik dilimler) ve Bellek Temizliği aktif ediliyor...
   ✅ attention_processors.py içine 128'lik dilimleme eklendi
   ✅ Model CPU Offload aktif edildi
🚀 Unique3D Stüdyo başlatılıyor...

📺 CANLI İŞLEM VE LİNK TAKİBİ
👉 No onnxruntime backend found.
👉 Please install rembg with CPU or GPU support:
👉     pip install "rembg[cpu]"  # for CPU
👉     pip install "rembg[gpu]"  # for NVIDIA/CUDA GPU
👉 For more information, see: https://github.com/danielgatis/rembg#installation


In [13]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"

print("📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...")
if not os.path.exists(u3d_dir):
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
os.system('pip install setuptools wheel ninja fire "rembg[gpu]"')
os.system('pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"')
os.system("pip install -r requirements.txt")

print("\n🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")

for cfg_path, ch in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = ch
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("\n🩹 3/4: VRAM Optimizasyonları uygulanıyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\n[ \t]+)return pipeline", patch, u_code)
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    patch2 = r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward("
    norm_code = re.sub(r"(\n[ \t]+)images = trainer\.pipeline_forward\(", patch2, norm_code)
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

if os.path.exists("gradio_app.py"):
    with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
    patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda a, **k: _old_load(*a, *{**k, 'weights_only': False})
"""
    if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
    gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

print("\n🚀 4/4: Unique3D Stüdyo başlatılıyor...")
LOG = os.path.join(WORK_DIR, "gradio.log")
env = os.environ.copy()
env.update({"PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl"})
open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

public_url = None
last_size = 0
while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print(f"\n🌐 LİNKİNİZ HAZIR: {public_url}\n")

📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.6/255.6 MB 5.5 MB/s eta 0:00:00
  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-shtfwg75


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-shtfwg75


  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-7bzhjv2d


  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-7bzhjv2d


  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


  Getting requirements to build wheel: finished with status 'error'

🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...

🩹 3/4: VRAM Optimizasyonları uygulanıyor...

🚀 4/4: Unique3D Stüdyo başlatılıyor...
👉 Traceback (most recent call last):
👉   File "/kaggle/working/Unique3D/gradio_app.py", line 21, in <module>
👉     from app.gradio_3dgen import create_ui as create_3d_ui
👉   File "/kaggle/working/Unique3D/app/gradio_3dgen.py", line 6, in <module>
👉     from app.custom_models.mvimg_prediction import run_mvprediction
👉   File "/kaggle/working/Unique3D/app/custom_models/mvimg_prediction.py", line 9, in <module>
👉     from scripts.all_typing import *
👉   File "/kaggle/working/Unique3D/scripts/all_typing.py", line 33, in <module>
👉     from jaxtyping import Bool, Complex, Float, Inexact, Int, Integer, Num, Shaped, UInt
👉 ModuleNotFoundError: No module named 'jaxtyping'


In [14]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"

print("📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...")
if not os.path.exists(u3d_dir):
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
os.system('pip install setuptools wheel ninja fire "rembg[gpu]"')
os.system('pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"')
os.system("pip install -r requirements.txt")

print("\n🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")

for cfg_path, ch in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = ch
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("\n🩹 3/4: VRAM Optimizasyonları uygulanıyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\n[ \t]+)return pipeline", patch, u_code)
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    patch2 = r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward("
    norm_code = re.sub(r"(\n[ \t]+)images = trainer\.pipeline_forward\(", patch2, norm_code)
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

if os.path.exists("gradio_app.py"):
    with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
    patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda a, **k: _old_load(*a, *{**k, 'weights_only': False})
"""
    if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
    gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

print("\n🚀 4/4: Unique3D Stüdyo başlatılıyor...")
LOG = os.path.join(WORK_DIR, "gradio.log")
env = os.environ.copy()
env.update({"PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl"})
open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

public_url = None
last_size = 0
while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print(f"\n🌐 LİNKİNİZ HAZIR: {public_url}\n")

📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...
  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-da7dc5dd


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-da7dc5dd


  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-h9gz91k7


  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-h9gz91k7


  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.



🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...

🩹 3/4: VRAM Optimizasyonları uygulanıyor...

🚀 4/4: Unique3D Stüdyo başlatılıyor...
👉 Traceback (most recent call last):
👉   File "/kaggle/working/Unique3D/gradio_app.py", line 21, in <module>
👉     from app.gradio_3dgen import create_ui as create_3d_ui
👉   File "/kaggle/working/Unique3D/app/gradio_3dgen.py", line 6, in <module>
👉     from app.custom_models.mvimg_prediction import run_mvprediction
👉   File "/kaggle/working/Unique3D/app/custom_models/mvimg_prediction.py", line 9, in <module>
👉     from scripts.all_typing import *
👉   File "/kaggle/working/Unique3D/scripts/all_typing.py", line 33, in <module>
👉     from jaxtyping import Bool, Complex, Float, Inexact, Int, Integer, Num, Shaped, UInt
👉 ModuleNotFoundError: No module named 'jaxtyping'


In [15]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"

print("📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...")
if not os.path.exists(u3d_dir):
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
# İŞTE EKSİK OLAN TÜM PAKETLER BURAYA EKLENDİ (jaxtyping ve rembg[gpu] dahil)
os.system('pip install setuptools wheel ninja fire "rembg[gpu]" jaxtyping')
os.system('pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"')
os.system("pip install -r requirements.txt")

print("\n🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")

for cfg_path, ch in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = ch
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("\n🩹 3/4: VRAM Optimizasyonları uygulanıyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\n[ \t]+)return pipeline", patch, u_code)
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    patch2 = r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward("
    norm_code = re.sub(r"(\n[ \t]+)images = trainer\.pipeline_forward\(", patch2, norm_code)
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

if os.path.exists("gradio_app.py"):
    with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
    patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
    if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
    gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

print("\n🚀 4/4: Unique3D Stüdyo başlatılıyor...")
LOG = os.path.join(WORK_DIR, "gradio.log")
env = os.environ.copy()
env.update({"PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl"})
open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

public_url = None
last_size = 0
while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print(f"\n🌐 LİNKİNİZ HAZIR: {public_url}\n")

📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.7 MB/s eta 0:00:00
  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-hbe5emb_


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-hbe5emb_


  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-ev0tdtph


  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-ev0tdtph


  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.



🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...

🩹 3/4: VRAM Optimizasyonları uygulanıyor...

🚀 4/4: Unique3D Stüdyo başlatılıyor...
👉 Traceback (most recent call last):
👉   File "/kaggle/working/Unique3D/gradio_app.py", line 21, in <module>
👉     from app.gradio_3dgen import create_ui as create_3d_ui
👉   File "/kaggle/working/Unique3D/app/gradio_3dgen.py", line 6, in <module>
👉     from app.custom_models.mvimg_prediction import run_mvprediction
👉   File "/kaggle/working/Unique3D/app/custom_models/mvimg_prediction.py", line 10, in <module>
👉     from scripts.utils import session, simple_preprocess
👉   File "/kaggle/working/Unique3D/scripts/utils.py", line 4, in <module>
👉     import pymeshlab
👉 ModuleNotFoundError: No module named 'pymeshlab'


In [16]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"

print("📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...")
if not os.path.exists(u3d_dir):
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
# pymeshlab doğrudan ana kurulum satırına eklendi
os.system('pip install setuptools wheel ninja fire "rembg[gpu]" jaxtyping pymeshlab')
os.system('pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"')
os.system("pip install -r requirements.txt")

print("\n🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")

for cfg_path, ch in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = ch
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("\n🩹 3/4: VRAM Optimizasyonları uygulanıyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\n[ \t]+)return pipeline", patch, u_code)
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    patch2 = r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward("
    norm_code = re.sub(r"(\n[ \t]+)images = trainer\.pipeline_forward\(", patch2, norm_code)
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

if os.path.exists("gradio_app.py"):
    with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
    patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda a, **k: _old_load(*a, *{**k, 'weights_only': False})
"""
    if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
    gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

print("\n🚀 4/4: Unique3D Stüdyo başlatılıyor...")
LOG = os.path.join(WORK_DIR, "gradio.log")
env = os.environ.copy()
env.update({"PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl"})
open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

public_url = None
last_size = 0
while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print(f"\n🌐 LİNKİNİZ HAZIR: {public_url}\n")

📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 MB 17.3 MB/s eta 0:00:00
  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-xcr9apcc


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-xcr9apcc


  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-q2823mfr


  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-q2823mfr


  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.



🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...

🩹 3/4: VRAM Optimizasyonları uygulanıyor...

🚀 4/4: Unique3D Stüdyo başlatılıyor...
👉 Traceback (most recent call last):
👉   File "/kaggle/working/Unique3D/gradio_app.py", line 21, in <module>
👉     from app.gradio_3dgen import create_ui as create_3d_ui
👉   File "/kaggle/working/Unique3D/app/gradio_3dgen.py", line 6, in <module>
👉     from app.custom_models.mvimg_prediction import run_mvprediction
👉   File "/kaggle/working/Unique3D/app/custom_models/mvimg_prediction.py", line 10, in <module>
👉     from scripts.utils import session, simple_preprocess
👉   File "/kaggle/working/Unique3D/scripts/utils.py", line 14, in <module>
👉     import trimesh
👉 ModuleNotFoundError: No module named 'trimesh'


In [17]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"

print("📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...")
if not os.path.exists(u3d_dir):
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
# trimesh paketi manuel listeye eklendi
os.system('pip install setuptools wheel ninja fire "rembg[gpu]" jaxtyping pymeshlab trimesh')
os.system('pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"')
os.system("pip install -r requirements.txt")

print("\n🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")

for cfg_path, ch in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = ch
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("\n🩹 3/4: VRAM Optimizasyonları uygulanıyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\n[ \t]+)return pipeline", patch, u_code)
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    patch2 = r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward("
    norm_code = re.sub(r"(\n[ \t]+)images = trainer\.pipeline_forward\(", patch2, norm_code)
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

if os.path.exists("gradio_app.py"):
    with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
    patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
    if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
    gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

print("\n🚀 4/4: Unique3D Stüdyo başlatılıyor...")
LOG = os.path.join(WORK_DIR, "gradio.log")
env = os.environ.copy()
env.update({"PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl"})
open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

public_url = None
last_size = 0
while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print(f"\n🌐 LİNKİNİZ HAZIR: {public_url}\n")

📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.5/751.5 kB 12.2 MB/s eta 0:00:00
  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-bj4qmy9h


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-bj4qmy9h


  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-2wynj3wr


  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-2wynj3wr


  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.



🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...

🩹 3/4: VRAM Optimizasyonları uygulanıyor...

🚀 4/4: Unique3D Stüdyo başlatılıyor...
👉 Downloading data from 'https://github.com/danielgatis/rembg/releases/download/v0.0.0/bria-rmbg-2.0.onnx' to file '/root/.rembg/models/bria-r
👉   0%|                                              | 0.00/1.02G [00:00<?, ?B/s]
👉   0%|                                      | 857k/1.02G [00:00<01:59, 8.53MB/s]
👉   0%|                                     | 2.33M/1.02G [00:00<01:24, 12.2MB/s]
👉   0%|▏                                    | 4.75M/1.02G [00:00<00:57, 17.6MB/s]
👉   1%|▎                                    | 8.56M/1.02G [00:00<00:39, 25.7MB/s]
👉   1%|▌                                    | 14.1M/1.02G [00:00<00:27, 36.3MB/s]
👉   2%|▋                                    | 20.4M/1.02G [00:00<00:22, 45.6MB/s]
👉   3%|▉                                    | 27.0M/1.02G [00:00<00:19, 52.1MB/s]
👉   3%|█▏                                   | 34.0M/1.

In [18]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"

print("📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...")
if not os.path.exists(u3d_dir):
    os.system(f"git clone https://github.com/AiuniAI/Unique3D.git {u3d_dir}")

os.chdir(u3d_dir)
os.system('pip install setuptools wheel ninja fire "rembg[gpu]" jaxtyping pymeshlab trimesh')
os.system('pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"')
os.system("pip install -r requirements.txt")

print("\n🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")

for cfg_path, ch in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = ch
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("\n🩹 3/4: VRAM Optimizasyonları ve Hata Yamaları uygulanıyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("(512, 512)", "(256, 256)")
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\n[ \t]+)return pipeline", patch, u_code)
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    patch2 = r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward("
    norm_code = re.sub(r"(\n[ \t]+)images = trainer\.pipeline_forward\(", patch2, norm_code)
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)

utils2_path = "scripts/utils.py"
if os.path.exists(utils2_path):
    with open(utils2_path, "r", encoding="utf-8") as f: u2_code = f.read()
    u2_code = u2_code.replace("providers=providers", "providers=['CUDAExecutionProvider', 'CPUExecutionProvider']")
    with open(utils2_path, "w", encoding="utf-8") as f: f.write(u2_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

if os.path.exists("gradio_app.py"):
    with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
    patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
    if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
    gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

print("\n🚀 4/4: Unique3D Stüdyo başlatılıyor...")
LOG = os.path.join(WORK_DIR, "gradio.log")
env = os.environ.copy()
env.update({"PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl"})
open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

public_url = None
last_size = 0
while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print(f"\n🌐 LİNKİNİZ HAZIR: {public_url}\n")

📦 1/4: Dosyalar indiriliyor ve kütüphaneler kuruluyor (3-4 dakika sürebilir)...
  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-7kkrwkzs


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-7kkrwkzs


  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-9sn_80p1


  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-9sn_80p1


  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.



🧹 2/4: Dosyalar onarılıyor ve model kanalları ayarlanıyor...

🩹 3/4: VRAM Optimizasyonları ve Hata Yamaları uygulanıyor...

🚀 4/4: Unique3D Stüdyo başlatılıyor...
👉 2026-10-26 23:17:54.841627131 [E:onnxruntime:Default, provider_bridge_ort.cc:2458 operator()] /onnxruntime_src/onnxruntime/core/sessi
👉 
👉 /usr/local/lib/python3.13/dist-packages/rembg/session_factory.py:56: RuntimeWarning: CUDAExecutionProvider was requested but the session is 
👉   return session_class(model_name, sess_opts, *args, **kwargs)
👉 Warning! extra parameter in cli is not verified, may cause erros.
👉 Traceback (most recent call last):
👉   File "/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py", line 423, in load_config
👉     config_file = hf_hub_download(
👉         pretrained_model_name_or_path,
👉     ...<10 lines>...
👉         local_dir_use_symlinks=local_dir_use_symlinks,
👉     )
👉   File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py", line 84, in _inner_

In [10]:
%cd /kaggle/working/Unique3D
!pip install setuptools wheel ninja fire rembg
!pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"
!pip install -r requirements.txt

/kaggle/working
  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-jehm8ct1
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-jehm8ct1
  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-ctxdmuqs
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-ctxdmuqs
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requi

In [2]:
!git clone https://github.com/AiuniAI/Unique3D.git /kaggle/working/Unique3D

Cloning into '/kaggle/working/Unique3D'...
remote: Enumerating objects: 277, done.
remote: Total 277 (delta 0), reused 0 (delta 0), pack-reused 277 (from 1)
Receiving objects: 100% (277/277), 9.87 MiB | 28.22 MiB/s, done.
Resolving deltas: 100% (128/128), done.


In [5]:
!pip install -r /kaggle/working/Unique3D/requirements.txt

  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-kxrwrm6l
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-kxrwrm6l
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [4]:
!pip install fire

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 2.8 MB/s eta 0:00:00a 0:00:01


In [4]:
!git clone https://github.com/AiuniAI/Unique3D.git /kaggle/working/Unique3D

Cloning into '/kaggle/working/Unique3D'...
remote: Enumerating objects: 277, done.
remote: Total 277 (delta 0), reused 0 (delta 0), pack-reused 277 (from 1)
Receiving objects: 100% (277/277), 9.87 MiB | 31.47 MiB/s, done.
Resolving deltas: 100% (128/128), done.


In [7]:
!pip install setuptools wheel ninja
!pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git"

  Cloning https://github.com/facebookresearch/pytorch3d.git to /tmp/pip-req-build-wf_t8wcj
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-wf_t8wcj
  Resolved https://github.com/facebookresearch/pytorch3d.git to commit 204fe5831b3a5bbe079f09d72afb4515178f8892
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 5.0 MB/s eta 0:00:00
  Created wheel for pytorch3d: filename=pytorch3d-0.7.9-cp313-cp313-linux_x86_64.whl size=62063443 sha256=ef56b33ad11352c2ccf303d46457645891eb495f1a58b95c8e83deabf126f39d
  Stored in directory: /tmp/pip-ephem-wheel-cache-ed8h04bv/wheels/57/18/de/2618f6a546ad0687ec78e6dc401c6f4f461947b6fc7a5b1508
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31596 sha256=502a275505ff0843156db390ddb45940bf02cfe

In [10]:
import os, sys, subprocess, time, re, json, tarfile

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) REPO KONTROLÜ VE GEREKLİ PAKETLERİN KURULUMU
# ══════════════════════════════════════════════════════════
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    print("❌ HATA: gradio_app.py bulunamadı. Lütfen önce Unique3D reposunu klonlayın.")
    sys.exit(1)

os.chdir(u3d_dir)
print(f"✅ Çalışma dizini: {os.getcwd()}")

print("📦 Temel bağımlılıklar ve onnxruntime kuruluyor...")
os.system("pip install -q fire omegaconf")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q onnxruntime")  # EKSİK OLAN PAKET
os.system("pip install -q hf_transfer")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62'")

# ══════════════════════════════════════════════════════════
# 2) PYTORCH3D KURULUMU
# ══════════════════════════════════════════════════════════
print("📦 pytorch3d kontrol ediliyor...")
if os.system("python -c 'import pytorch3d' 2>/dev/null") != 0:
    import torch
    TORCH_VERSION = torch.__version__.split('+')[0]
    CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
    print(f"   🔍 PyTorch: {TORCH_VERSION}, CUDA: {CUDA_VERSION}")
    
    install_cmd = (
        f"pip install -q pytorch3d "
        f"-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py313_cu{CUDA_VERSION}_pyt{TORCH_VERSION}/download.html"
    )
    os.system(install_cmd)
    
    if os.system("python -c 'import pytorch3d' 2>/dev/null") != 0:
        print("   ⚠️  Tekerlek bulunamadı, kaynaktan derleniyor...")
        os.system("pip install -q 'git+https://github.com/facebookresearch/pytorch3d.git@stable'")
    if os.system("python -c 'import pytorch3d' 2>/dev/null") == 0:
        print("   ✅ pytorch3d hazır")
    else:
        print("   ❌ pytorch3d kurulumu başarısız!")
else:
    print("   ✓ pytorch3d zaten kurulu")

# ══════════════════════════════════════════════════════════
# 3) MODEL AĞIRLIKLARINI İNDİRME (Wuvin/Unique3D Space)
# ══════════════════════════════════════════════════════════
print("📥 Model ağırlıkları 'Wuvin/Unique3D' Space'inden indiriliyor...")
os.makedirs("ckpt", exist_ok=True)
from huggingface_hub import snapshot_download
REPO_ID = "Wuvin/Unique3D"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    snapshot_download(
        repo_id=REPO_ID, repo_type="space", local_dir="ckpt",
        token=HF_TOKEN if HF_TOKEN else None,
        ignore_patterns=["*.md", ".git*", "*.png", "*.jpg", "*.gif", "*.mp4", "*.ipynb"]
    )
    print("✅ Modeller başarıyla indirildi.")
except Exception as e:
    print(f"⚠️ Space üzerinden indirme başarısız: {e}")
    print("🔄 Alternatif olarak doğrudan ckpt.zip indiriliyor...")
    os.system("wget -q https://huggingface.co/spaces/Wuvin/Unique3D/resolve/main/ckpt.zip -O ckpt.zip")
    if os.path.exists("ckpt.zip") and os.path.getsize("ckpt.zip") > 1000000:
        with tarfile.open("ckpt.zip", "r:gz") as tar: tar.extractall(".")
        os.remove("ckpt.zip")
        print("✅ ckpt.zip indirildi ve açıldı.")
    else:
        print("❌ Otomatik indirme başarısız oldu.")

# ══════════════════════════════════════════════════════════
# 4) VRAM VE GRADIO YAMALARI
# ══════════════════════════════════════════════════════════
print("⚙️ VRAM ve Gradio yamaları uygulanıyor...")

for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(r"^(\s*)rm_normals = predict_normals\(", r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(", mvi_code, flags=re.MULTILINE)
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f: attn_code = f.read()
    attn_code = re.sub(r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)", "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)", attn_code)
    with open(attn_path, "w", encoding="utf-8") as f: f.write(attn_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download
import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 5) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve GPU belleği temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size); yeni = f.read(); last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

✅ Çalışma dizini: /kaggle/working/Unique3D
📦 Temel bağımlılıklar ve onnxruntime kuruluyor...


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.8 MB/s eta 0:00:00
📦 pytorch3d kontrol ediliyor...
   ✓ pytorch3d zaten kurulu
📥 Model ağırlıkları 'Wuvin/Unique3D' Space'inden indiriliyor...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 68 files:   0%|          | 0/68 [00:00<?, ?it/s]

✅ Modeller başarıyla indirildi.
⚙️ VRAM ve Gradio yamaları uygulanıyor...
🧹 Eski süreçler kapatılıyor ve GPU belleği temizleniyor...
🚀 Unique3D Stüdyo başlatılıyor...

📺 CANLI İŞLEM VE LİNK TAKİBİ (Link çıkana kadar bekleyin)
👉 Traceback (most recent call last):
👉   File "/kaggle/working/Unique3D/gradio_app.py", line 37, in <module>
👉     from app.gradio_3dgen import create_ui as create_3d_ui
👉   File "/kaggle/working/Unique3D/app/gradio_3dgen.py", line 6, in <module>
👉     from app.custom_models.mvimg_prediction import run_mvprediction
👉   File "/kaggle/working/Unique3D/app/custom_models/mvimg_prediction.py", line 9, in <module>
👉     from scripts.all_typing import *
👉   File "/kaggle/working/Unique3D/scripts/all_typing.py", line 33, in <module>
👉     from jaxtyping import Bool, Complex, Float, Inexact, Int, Integer, Num, Shaped, UInt
👉 ModuleNotFoundError: No module named 'jaxtyping'


In [ ]:
!rm -rf /kaggle/working/Unique3D
!git clone https://github.com/AiuniAI/Unique3D.git /kaggle/working/Unique3D
!ls -la /kaggle/working/Unique3D | head -20

In [3]:
import os, sys, subprocess, time, re, json, tarfile

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) REPO KONTROLÜ VE GEREKLİ PAKETLERİN KURULUMU
# ══════════════════════════════════════════════════════════
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    print("❌ HATA: gradio_app.py bulunamadı. Lütfen önce Unique3D reposunu klonlayın.")
    sys.exit(1)

os.chdir(u3d_dir)
print(f"✅ Çalışma dizini: {os.getcwd()}")

print("📦 Temel bağımlılıklar kuruluyor (fire, omegaconf ve diğerleri)...")
# fire ve omegaconf'ı ayrıca ve öncelikli olarak kur
os.system("pip install -q fire omegaconf")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q hf_transfer")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62'")

# nvdiffrast ve torch_scatter
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git")

if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TV = torch._version_.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git")

# ══════════════════════════════════════════════════════════
# 2) MODEL AĞIRLIKLARINI DOĞRU ADRESTEN İNDİRME
# ══════════════════════════════════════════════════════════
print("📥 Model ağırlıkları 'Wuvin/Unique3D' Space'inden indiriliyor (18GB+, uzun sürebilir)...")
os.makedirs("ckpt", exist_ok=True)

from huggingface_hub import snapshot_download

# Doğru repo ve repo tipi: Wuvin/Unique3D (Space)
REPO_ID = "Wuvin/Unique3D"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

try:
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="space",  # Space olduğunu belirtmek kritik
        local_dir="ckpt",
        token=HF_TOKEN if HF_TOKEN else None,
        ignore_patterns=[".md", ".git", ".png", ".jpg", ".gif", ".mp4", "*.ipynb"]
    )
    print("✅ Modeller başarıyla indirildi.")
except Exception as e:
    print(f"⚠️ Space üzerinden indirme başarısız: {e}")
    print("🔄 Alternatif olarak doğrudan ckpt.zip indiriliyor...")
    # Space'teki doğrudan dosya indirme linki
    os.system("wget -q https://huggingface.co/spaces/Wuvin/Unique3D/resolve/main/ckpt.zip -O ckpt.zip")
    if os.path.exists("ckpt.zip") and os.path.getsize("ckpt.zip") > 1000000:
        with tarfile.open("ckpt.zip", "r:gz") as tar:
            tar.extractall(".")
        os.remove("ckpt.zip")
        print("✅ ckpt.zip indirildi ve açıldı.")
    else:
        print("❌ Otomatik indirme başarısız oldu. Lütfen HF_TOKEN ayarlayın veya manuel indirin.")

# ══════════════════════════════════════════════════════════
# 3) VRAM VE GRADIO YAMALARI (Değişiklik yok)
# ══════════════════════════════════════════════════════════
print("⚙️ VRAM ve Gradio yamaları uygulanıyor...")

# Kanallar (img2mvimg: 8, unet_4ch: 4)
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# Çözünürlük düşürme ve VRAM temizliği
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(r"^(\s*)rm_normals = predict_normals\(", r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(", mvi_code, flags=re.MULTILINE)
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

# Attention dilimleme (Unique3D içi)
attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f: attn_code = f.read()
    attn_code = re.sub(r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)", "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)", attn_code)
    with open(attn_path, "w", encoding="utf-8") as f: f.write(attn_code)

# Model CPU Offload
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

# Gradio parametre temizliği
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

# Gradio_app.py içine SDPA ve cached_download yamaları
with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download
import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa
_old_load = torch.load
torch.load = lambda a, **k: _old_load(*a, *{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve GPU belleği temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size); yeni = f.read(); last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

❌ HATA: gradio_app.py bulunamadı. Lütfen önce Unique3D reposunu klonlayın.


SystemExit: 1

In [ ]:
import os, sys, subprocess, time, re, json, tarfile

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) REPO KONTROLÜ VE GEREKLİ PAKETLERİN KURULUMU
# ══════════════════════════════════════════════════════════
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    print("❌ HATA: gradio_app.py bulunamadı. Lütfen önce Unique3D reposunu klonlayın.")
    sys.exit(1)

os.chdir(u3d_dir)
print(f"✅ Çalışma dizini: {os.getcwd()}")

print("📦 Temel bağımlılıklar kuruluyor (fire, omegaconf ve diğerleri)...")
# fire ve omegaconf'ı ayrıca ve öncelikli olarak kur
os.system("pip install -q fire omegaconf")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q hf_transfer")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62'")

# nvdiffrast ve torch_scatter
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git")

if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git")

# ══════════════════════════════════════════════════════════
# 2) MODEL AĞIRLIKLARINI DOĞRU ADRESTEN İNDİRME
# ══════════════════════════════════════════════════════════
print("📥 Model ağırlıkları 'Wuvin/Unique3D' Space'inden indiriliyor (18GB+, uzun sürebilir)...")
os.makedirs("ckpt", exist_ok=True)

from huggingface_hub import snapshot_download

# Doğru repo ve repo tipi: Wuvin/Unique3D (Space)
REPO_ID = "Wuvin/Unique3D"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

try:
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="space",  # Space olduğunu belirtmek kritik
        local_dir="ckpt",
        token=HF_TOKEN if HF_TOKEN else None,
        ignore_patterns=["*.md", ".git*", "*.png", "*.jpg", "*.gif", "*.mp4", "*.ipynb"]
    )
    print("✅ Modeller başarıyla indirildi.")
except Exception as e:
    print(f"⚠️ Space üzerinden indirme başarısız: {e}")
    print("🔄 Alternatif olarak doğrudan ckpt.zip indiriliyor...")
    # Space'teki doğrudan dosya indirme linki
    os.system("wget -q https://huggingface.co/spaces/Wuvin/Unique3D/resolve/main/ckpt.zip -O ckpt.zip")
    if os.path.exists("ckpt.zip") and os.path.getsize("ckpt.zip") > 1000000:
        with tarfile.open("ckpt.zip", "r:gz") as tar:
            tar.extractall(".")
        os.remove("ckpt.zip")
        print("✅ ckpt.zip indirildi ve açıldı.")
    else:
        print("❌ Otomatik indirme başarısız oldu. Lütfen HF_TOKEN ayarlayın veya manuel indirin.")

# ══════════════════════════════════════════════════════════
# 3) VRAM VE GRADIO YAMALARI (Değişiklik yok)
# ══════════════════════════════════════════════════════════
print("⚙️ VRAM ve Gradio yamaları uygulanıyor...")

# Kanallar (img2mvimg: 8, unet_4ch: 4)
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# Çözünürlük düşürme ve VRAM temizliği
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(r"^(\s*)rm_normals = predict_normals\(", r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(", mvi_code, flags=re.MULTILINE)
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

# Attention dilimleme (Unique3D içi)
attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f: attn_code = f.read()
    attn_code = re.sub(r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)", "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)", attn_code)
    with open(attn_path, "w", encoding="utf-8") as f: f.write(attn_code)

# Model CPU Offload
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

# Gradio parametre temizliği
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

# Gradio_app.py içine SDPA ve cached_download yamaları
with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download
import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve GPU belleği temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size); yeni = f.read(); last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, tarfile

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) REPO KONTROLÜ VE GEREKLİ PAKETLERİN KURULUMU
# ══════════════════════════════════════════════════════════
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    print("❌ HATA: gradio_app.py bulunamadı. Lütfen önce Unique3D reposunu klonlayın.")
    sys.exit(1)

os.chdir(u3d_dir)
print(f"✅ Çalışma dizini: {os.getcwd()}")

print("📦 Temel bağımlılıklar kuruluyor...")
os.system("pip install -q fire omegaconf")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q hf_transfer")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62'")

# ══════════════════════════════════════════════════════════
# 2) PYTORCH3D KURULUMU (KRİTİK)
# ══════════════════════════════════════════════════════════
print("📦 pytorch3d kuruluyor (PyTorch ve CUDA sürümüne uygun)...")

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"   🔍 PyTorch: {TORCH_VERSION}, CUDA: {CUDA_VERSION}")

# Yöntem 1: Resmi tekerlek indeksi (en hızlı ve güvenilir)
install_cmd = (
    f"pip install -q pytorch3d "
    f"-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py313_cu{CUDA_VERSION}_pyt{TORCH_VERSION}/download.html"
)
print(f"   ⏳ Deneniyor: {install_cmd}")
os.system(install_cmd)

# Doğrulama
if os.system("python -c 'import pytorch3d; print(\"pytorch3d OK\")' 2>&1") == 0:
    print("   ✅ pytorch3d tekerlek ile kuruldu")
else:
    print("   ⚠️  Tekerlek bulunamadı, kaynaktan derleniyor (bu birkaç dakika sürebilir)...")
    # Yöntem 2: Kaynaktan derleme
    os.system("pip install -q 'git+https://github.com/facebookresearch/pytorch3d.git@stable'")
    if os.system("python -c 'import pytorch3d' 2>/dev/null") == 0:
        print("   ✅ pytorch3d kaynaktan derlendi")
    else:
        print("   ❌ pytorch3d kurulumu başarısız! Manuel kontrol gerekli.")

# ══════════════════════════════════════════════════════════
# 3) MODEL AĞIRLIKLARINI İNDİRME (Wuvin/Unique3D Space)
# ══════════════════════════════════════════════════════════
print("📥 Model ağırlıkları 'Wuvin/Unique3D' Space'inden indiriliyor (18GB+, uzun sürebilir)...")
os.makedirs("ckpt", exist_ok=True)

from huggingface_hub import snapshot_download

REPO_ID = "Wuvin/Unique3D"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

try:
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="space",
        local_dir="ckpt",
        token=HF_TOKEN if HF_TOKEN else None,
        ignore_patterns=["*.md", ".git*", "*.png", "*.jpg", "*.gif", "*.mp4", "*.ipynb"]
    )
    print("✅ Modeller başarıyla indirildi.")
except Exception as e:
    print(f"⚠️ Space üzerinden indirme başarısız: {e}")
    print("🔄 Alternatif olarak doğrudan ckpt.zip indiriliyor...")
    os.system("wget -q https://huggingface.co/spaces/Wuvin/Unique3D/resolve/main/ckpt.zip -O ckpt.zip")
    if os.path.exists("ckpt.zip") and os.path.getsize("ckpt.zip") > 1000000:
        with tarfile.open("ckpt.zip", "r:gz") as tar:
            tar.extractall(".")
        os.remove("ckpt.zip")
        print("✅ ckpt.zip indirildi ve açıldı.")
    else:
        print("❌ Otomatik indirme başarısız oldu. Lütfen HF_TOKEN ayarlayın veya manuel indirin.")

# ══════════════════════════════════════════════════════════
# 4) VRAM VE GRADIO YAMALARI
# ══════════════════════════════════════════════════════════
print("⚙️ VRAM ve Gradio yamaları uygulanıyor...")

for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(r"^(\s*)rm_normals = predict_normals\(", r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(", mvi_code, flags=re.MULTILINE)
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f: attn_code = f.read()
    attn_code = re.sub(r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)", "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)", attn_code)
    with open(attn_path, "w", encoding="utf-8") as f: f.write(attn_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download
import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 5) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve GPU belleği temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size); yeni = f.read(); last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, shutil, tarfile, tempfile

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) REPO KONTROLÜ VE GEREKLİ PAKETLER
# ══════════════════════════════════════════════════════════
if not os.path.exists(os.path.join(u3d_dir, "gradio_app.py")):
    print("❌ HATA: gradio_app.py bulunamadı. Lütfen önce Unique3D reposunu klonlayın.")
    sys.exit(1)

os.chdir(u3d_dir)
print(f"✅ Çalışma dizini: {os.getcwd()}")

print("📦 Temel bağımlılıklar kuruluyor...")
os.system("pip install -q fire omegaconf")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q hf_transfer")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62'")

if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git")

if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git")

# ══════════════════════════════════════════════════════════
# 2) MODEL AĞIRLIKLARINI İNDİRME (Wuvin/Unique3D Space)
# ══════════════════════════════════════════════════════════
print("📥 Model ağırlıkları HuggingFace Space'ten indiriliyor...")
os.makedirs("ckpt", exist_ok=True)

from huggingface_hub import snapshot_download

# Doğru repo: Wuvin/Unique3D (resmi Space)
REPO_ID = "Wuvin/Unique3D"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

try:
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="space",
        local_dir="ckpt",
        token=HF_TOKEN if HF_TOKEN else None,
        ignore_patterns=["*.md", ".git*", "*.png", "*.jpg", "*.gif", "*.mp4"]
    )
    print("✅ Modeller başarıyla indirildi.")
except Exception as e:
    print(f"⚠️ Space üzerinden indirme başarısız: {e}")
    print("🔄 Alternatif olarak doğrudan ckpt.zip indiriliyor...")
    os.system("wget -q https://huggingface.co/spaces/Wuvin/Unique3D/resolve/main/ckpt.zip -O ckpt.zip")
    if os.path.exists("ckpt.zip") and os.path.getsize("ckpt.zip") > 1000000:
        with tarfile.open("ckpt.zip", "r:gz") as tar:
            tar.extractall(".")
        os.remove("ckpt.zip")
        print("✅ ckpt.zip indirildi ve açıldı.")
    else:
        print("❌ Otomatik indirme başarısız oldu. Lütfen Hugging Face token'ınızı ayarlayın (HF_TOKEN) veya manuel indirin.")

# ══════════════════════════════════════════════════════════
# 3) VRAM VE GRADIO YAMALARI
# ══════════════════════════════════════════════════════════
print("⚙️ VRAM ve Gradio yamaları uygulanıyor...")

for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f: mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(r"^(\s*)rm_normals = predict_normals\(", r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(", mvi_code, flags=re.MULTILINE)
    with open(mvi_path, "w", encoding="utf-8") as f: f.write(mvi_code)

attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f: attn_code = f.read()
    attn_code = re.sub(r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)", "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)", attn_code)
    with open(attn_path, "w", encoding="utf-8") as f: f.write(attn_code)

utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'): pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'): pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'): pipe.enable_model_cpu_offload()
    except Exception as e: print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)

gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'): _hf.cached_download = _hf.hf_hub_download
import torch, gc
import torch.nn.functional as F
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    chunk_size = 128; outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out); torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)
F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code: gc_code = patches + "\n" + gc_code
gc_code = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)", r"\1.queue().launch(share=True, inbrowser=False)", gc_code)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve GPU belleği temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1", "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0", "USE_FLAX": "0", "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size); yeni = f.read(); last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip(): print(f"👉 {satir[:140]}")
        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ VE BOZULAN DOSYALARI SIFIRLA
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve dosyalar orijinal haline getiriliyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("git checkout scripts/multiview_inference.py app/custom_models/normal_prediction.py app/custom_models/utils.py gradio_app.py app/gradio_3dgen.py >/dev/null 2>&1")
time.sleep(2)

# ══════════════════════════════════════════════════════════
# 2) MODEL KANALLARINI SABİTLE
# ══════════════════════════════════════════════════════════
print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 3) %100 GÜVENLİ VRAM VE ATTENTION SLICING YAMALARI (REGEX)
# ══════════════════════════════════════════════════════════
print("🩹 VRAM temizleme ve Attention Slicing yamaları dinamik girintiyle uygulanıyor...")

# normal_prediction.py içine tam doğru yere temizleyici koyuyoruz
norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    # \1 mevcut satırın girintisini (boşluklarını) temsil eder
    norm_code = re.sub(
        r"(\s+)images = trainer\.pipeline_forward\(",
        r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward(",
        norm_code
    )
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)
    print("   ✅ normal_prediction.py için VRAM temizleyici başarıyla eklendi")

# utils.py içine Attention Slicing
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    u_code = re.sub(
        r"(\s+)return pipeline",
        r"\1if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1return pipeline",
        u_code
    )
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)
    print("   ✅ utils.py içine VRAM optimizasyonları eklendi")

# Gradio arayüz ve import yamaları
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:64",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

# ══════════════════════════════════════════════════════════
# 5) CANLI LOG VE ÜRETİM TAKİBİ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ VE BOZULAN DOSYALARI SIFIRLA
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve dosyalar orijinal haline getiriliyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("git checkout scripts/multiview_inference.py app/custom_models/normal_prediction.py app/custom_models/utils.py gradio_app.py app/gradio_3dgen.py >/dev/null 2>&1")
time.sleep(2)

# ══════════════════════════════════════════════════════════
# 2) MODEL KANALLARINI SABİTLE
# ══════════════════════════════════════════════════════════
print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 3) -4 GB VRAM KAZANDIRAN CPU OFFLOAD YAMASI
# ══════════════════════════════════════════════════════════
print("🩹 CPU Offload ve Attention Slicing yamaları ile VRAM boşaltılıyor...")

# utils.py içine CPU Offload ekliyoruz (Kullanılmayan modeli GPU'dan RAM'e atar)
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    
    # "return pipeline" bulduğumuzda güvenli bir şekilde CPU Offload komutlarını ekle
    vram_patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\s+)return pipeline", vram_patch, u_code)
    
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)
    print("   ✅ Modeller yüklendiğinde otomatik VRAM boşaltma aktif edildi (-4 GB tasarruf)")

# normal_prediction.py içine Aşamalar arası manuel GPU çöp toplayıcı
norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    norm_code = re.sub(
        r"(\s+)images = trainer\.pipeline_forward\(",
        r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward(",
        norm_code
    )
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)
    print("   ✅ Aşamalar arası VRAM temizliği (gc.collect) eklendi")

# Gradio arayüz yamaları
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

# ══════════════════════════════════════════════════════════
# 5) CANLI LOG VE ÜRETİM TAKİBİ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve VRAM temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 🚨 OOM HATASINI ÇÖZEN KRİTİK YAMALAR
# ══════════════════════════════════════════════════════════
print("🩹 Derin VRAM yamaları ve Çözünürlük Düşürme (512->320) uygulanıyor...")

# 1. Normal Tahmini çözünürlüğünü 512'den 320'ye düşür (VRAM'i ~3 kat azaltır)
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    
    # Çözünürlüğü düşür
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((320, 320)")
    
    # Bellek temizliği ekle
    if "gc.collect()" not in mvi_code:
        mvi_code = mvi_code.replace(
            "rm_normals = predict_normals(",
            "import gc, torch\ngc.collect()\ntorch.cuda.empty_cache()\nrm_normals = predict_normals("
        )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)
    print("   ✅ Multiview çözünürlüğü 320x320'ye düşürüldü")

# 2. Model Yükleme Yardımcısına Sıralı CPU Offload (Sequential) Ekle
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    
    if "enable_attention_slicing" not in u_code:
        vram_opt_code = """

def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        # En agresif VRAM boşaltma yöntemi (Sıralı CPU Offload)
        if hasattr(pipe, 'enable_sequential_cpu_offload'):
            pipe.enable_sequential_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
        u_code += vram_opt_code
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
        print("   ✅ Sıralı CPU Offload (Sequential) aktif edildi")

# 3. Gradio 6 Parametre Temizliği
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

# 4. Gradio Ana Dosya Yamaları
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# SUNUCUYU BAŞLAT (Agresif Bellek Yönetimi)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    # Bellek parçalanmasını önlemek için ÇOK DAHA agresif ayar
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, shutil, subprocess, time, re, json, gc

# Çalışma dizinini otomatik algıla (Kaggle veya Colab)
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(WORK_DIR)

# CUDA Derleme ve Bellek Optimizasyon Değişkenleri (T4 GPU = 7.5)
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0;8.6"
os.environ["FORCE_CUDA"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYOPENGL_PLATFORM"] = "egl"

print("=== [1/9] Eski Süreçler ve Bellek Temizleniyor ===")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
gc.collect()

os.makedirs(f"{WORK_DIR}/uretilen_3d_modeller", exist_ok=True)
os.makedirs(f"{WORK_DIR}/uretilen_2d_gorseller", exist_ok=True)

print("=== [2/9] Linux Sistem ve OpenGL/EGL Derleme Paketleri Kuruluyor ===")
os.system("apt-get update -qq && apt-get install -y -qq libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev libglib2.0-0 build-essential ninja-build psmisc git wget >/dev/null 2>&1")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "ninja", "setuptools", "wheel"])

print("=== [3/9] Unique3D Projesi ve Temel Python Paketleri Kuruluyor ===")
if not os.path.exists(os.path.join(u3d_dir, ".git")):
    shutil.rmtree(u3d_dir, ignore_errors=True)
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git", u3d_dir], check=True)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "fire", "rembg", "onnxruntime", "trimesh", "einops", "omegaconf",
    "transformers", "diffusers", "peft", "accelerate", "huggingface_hub",
    "gradio", "gradio-client", "pydantic", "jaxtyping", "typeguard",
    "imageio", "scikit-image", "opencv-python-headless", "matplotlib", "plyfile", "pymeshlab"
])

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("=== [4/9] nvdiffrast ve PyTorch3D Kuruluyor ===")
try:
    import nvdiffrast
    print("✅ nvdiffrast zaten kurulu.")
except ImportError:
    print("⚙️ nvdiffrast kuruluyor...")
    os.chdir(WORK_DIR)
    shutil.rmtree("nvdiffrast", ignore_errors=True)
    subprocess.run(["git", "clone", "https://github.com/NVlabs/nvdiffrast.git"], check=True)
    os.chdir("nvdiffrast")
    res = subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "."], capture_output=True, text=True)
    if res.returncode != 0:
        print("⚠️ Standart C++ derlemesi atlandı, nvdiffrast JIT modunda kuruluyor...")
        site_pkg = [p for p in sys.path if "dist-packages" in p or "site-packages" in p][0]
        shutil.copytree("nvdiffrast", os.path.join(site_pkg, "nvdiffrast"), dirs_exist_ok=True)
    os.chdir(WORK_DIR)
    import nvdiffrast
    print("✅ nvdiffrast başarıyla aktif edildi!")

try:
    import pytorch3d
    print("✅ PyTorch3D zaten kurulu.")
except ImportError:
    print("⚙️ PyTorch3D derlenip kuruluyor (2-3 dakika sürebilir)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
        "git+https://github.com/facebookresearch/pytorch3d.git"
    ])
    print("✅ PyTorch3D başarıyla kuruldu!")

os.chdir(u3d_dir)

print("=== [5/9] Model Ağırlıkları ve 4-Kanallı UNet Yapılandırması ===")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    print("📥 Unique3D model ağırlıkları indiriliyor...")
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

img2mvimg_yolu = f"{u3d_dir}/ckpt/img2mvimg"
unet_4ch_yolu = f"{u3d_dir}/ckpt/unet_4ch"
if os.path.exists(img2mvimg_yolu):
    shutil.rmtree(unet_4ch_yolu, ignore_errors=True)
    os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)
    ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
    if os.path.exists(ana_cfg_path):
        ana_cfg = json.loads(open(ana_cfg_path, "r").read())
        ana_cfg["in_channels"] = 4
        ana_cfg.pop("num_class_embeds", None)
        ana_cfg["addition_embed_type"] = None
        open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet":
            os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)
    for d in os.listdir(img2mvimg_yolu + "/unet"):
        if d != "config.json":
            os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("=== [6/9] Tüm Python 3.13, CUDA ve Bellek Yamaları Uygulanıyor ===")
try:
    import gradio_client.utils as client_utils
    utils_path = client_utils.__file__
    code_client = open(utils_path, "r", encoding="utf-8").read()
    if "if not isinstance(schema, dict): return 'Any'" not in code_client:
        code_client = code_client.replace("def get_type(schema):", "def get_type(schema):\n    if not isinstance(schema, dict): return 'Any'")
        code_client = code_client.replace("def _json_schema_to_python_type(schema, defs):", "def _json_schema_to_python_type(schema, defs):\n    if not isinstance(schema, dict): return 'Any'")
        open(utils_path, "w", encoding="utf-8").write(code_client)
except Exception:
    pass

for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            txt = open(path, "r", errors="ignore").read()
            new_txt = txt
            new_txt = new_txt.replace("C:\\\\Users\\\\admin\\\\Desktop\\\\Unique3D\\\\", "")
            new_txt = new_txt.replace("RasterizeGLContext", "RasterizeCudaContext")
            new_txt = new_txt.replace("output_db=False,", "").replace(", output_db=False", "").replace("output_db=False", "")
            new_txt = new_txt.replace("pymeshlab.PercentageValue", "pymeshlab.PureValue").replace("pymeshlab.Percentage", "pymeshlab.PureValue")
            if "torch.load(" in new_txt and "weights_only" not in new_txt:
                new_txt = re.sub(r"torch\.load\(([^)]+)\)", r"torch.load(\1, weights_only=False)", new_txt)
            if txt != new_txt:
                open(path, "w", encoding="utf-8").write(new_txt)

norm_p = os.path.join(u3d_dir, "app/custom_models/normal_prediction.py")
if os.path.exists(norm_p):
    ntxt = open(norm_p, "r", encoding="utf-8").read()
    ntxt = ntxt.replace("ckpt/img2mvimg", "ckpt/unet_4ch").replace("ckpt/normal", "ckpt/unet_4ch")
    open(norm_p, "w", encoding="utf-8").write(ntxt)

# scripts/utils.py dosyasını doğrudan hatasız ve temiz bir kodla yeniden yazıyoruz
utils_p = os.path.join(u3d_dir, "scripts/utils.py")
clean_utils_code = '''
import rembg
import numpy as np
from PIL import Image

try:
    session = rembg.new_session(providers=['CPUExecutionProvider'])
except Exception:
    session = rembg.new_session()

def simple_preprocess(image, background_color=(255, 255, 255)):
    if isinstance(image, Image.Image):
        image = np.array(image)
    return image
'''
open(utils_p, "w", encoding="utf-8").write(clean_utils_code)

recon_path = os.path.join(u3d_dir, "mesh_reconstruction/recon.py")
if os.path.exists(recon_path):
    code_recon = open(recon_path, "r", encoding="utf-8").read()
    code_recon = code_recon.replace("res=512", "res=256").replace("resolution=512", "resolution=256")
    code_recon = code_recon.replace("512,", "256,").replace("512)", "256)")
    if "torch.cuda.empty_cache()" not in code_recon:
        code_recon = "import gc, torch\n" + code_recon + "\ngc.collect()\ntorch.cuda.empty_cache()\n"
    open(recon_path, "w", encoding="utf-8").write(code_recon)

print("=== [7/9] 2D Stüdyo Sekmesi ve 3D Kayıt Yaması Yazılıyor ===")
patch_path = os.path.join(u3d_dir, "studio_2d_patch.py")
studio_2d_lines = [
    "import os, sys, gc, time, random, shutil, torch, rembg, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import gradio as gr",
    "from diffusers import AutoPipelineForText2Image",
    "",
    "def _get_rembg_session():",
    "    from scripts.utils import session as _s",
    "    return _s",
    "",
    "def uret_2d_gorsel(prompt, neg_prompt, steps, cfg, bg_sil):",
    "    if not prompt or not prompt.strip():",
    "        return None, '⚠️ Lütfen İngilizce nesne tarifi yazın (Örn: a red sports car, front view)'",
    "    try:",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        Moved = []",
    "        for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction']:",
    "            mod = sys.modules.get(m_adi)",
    "            if mod and hasattr(mod, 'pipeline') and hasattr(mod.pipeline, 'unet'):",
    "                try:",
    "                    mod.pipeline.unet.to('cpu')",
    "                    Moved.append(mod.pipeline.unet)",
    "                except Exception: pass",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        pipe = AutoPipelineForText2Image.from_pretrained(",
    "            'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16', low_cpu_mem_usage=True",
    "        ).to('cuda')",
    "        seed = random.randint(1, 99999999)",
    "        gen = torch.Generator(device='cuda').manual_seed(seed)",
    "        with torch.inference_mode():",
    "            img = pipe(",
    "                prompt=prompt + ', isometric, front view, centered, clean background, high detail',",
    "                negative_prompt=neg_prompt,",
    "                num_inference_steps=int(steps),",
    "                guidance_scale=float(cfg),",
    "                generator=gen",
    "            ).images[0]",
    "        del pipe, gen",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        for u in Moved:",
    "            try: u.to('cuda')",
    "            except Exception: pass",
    "        if bg_sil:",
    "            img = rembg.remove(img, session=_get_rembg_session())",
    f"        kayit_yolu = f'{WORK_DIR}/uretilen_2d_gorseller/gorsel_{{int(time.time())}}_{{seed}}.png'",
    "        img.save(kayit_yolu)",
    "        return img, f'✅ Başarılı (Seed: {seed}) -> {kayit_yolu}'",
    "    except Exception as e:",
    "        import traceback",
    "        traceback.print_exc()",
    "        return None, f'❌ Hata: {str(e)}'",
    "",
    "def sadece_arkaplan_sil(img):",
    "    if img is None: return None, '⚠️ Görsel yüklenmedi'",
    "    try:",
    "        out = rembg.remove(img, session=_get_rembg_session())",
    f"        kayit_yolu = f'{WORK_DIR}/uretilen_2d_gorseller/temiz_{{int(time.time())}}.png'",
    "        out.save(kayit_yolu)",
    "        return out, f'✅ Arka plan silindi -> {kayit_yolu}'",
    "    except Exception as e:",
    "        return None, f'❌ Hata: {str(e)}'",
    "",
    "def create_2d_tab():",
    "    with gr.Tab('🎨 2D Görsel Üret & Hazırla (3D İçin)'):",
    "        gr.Markdown('### 1. Metinden 3D Uyumlu 2D Nesne Üret veya Arka Planını Sil')",
    "        with gr.Row():",
    "            with gr.Column():",
    "                prompt_in = gr.Textbox(label='2D Nesne Tarifi (İngilizce)', value='', placeholder='Örn: a wooden medieval shield, front view')",
    "                neg_in = gr.Textbox(label='Negatif Prompt', value='blurry, cropped, low quality, multiple objects')",
    "                with gr.Row():",
    "                    steps_in = gr.Slider(1, 8, value=4, step=1, label='Adım Sayısı')",
    "                    cfg_in = gr.Slider(0.0, 3.0, value=0.0, step=0.5, label='CFG Scale')",
    "                bg_chk = gr.Checkbox(value=True, label='Otomatik Arka Plan Sil')",
    "                btn_uret = gr.Button('🎨 2D Görsel Üret', variant='primary')",
    "                gr.Markdown('---')",
    "                yukle_in = gr.Image(type='pil', label='Veya Kendi Fotoğrafını Yükle')",
    "                btn_sil = gr.Button('✂️ Sadece Arka Planı Sil')",
    "            with gr.Column():",
    "                out_img = gr.Image(type='pil', label='Hazırlanan 2D Görsel')",
    "                out_path = gr.Textbox(label='Kayıt Yolu', interactive=False)",
    "        btn_uret.click(uret_2d_gorsel, inputs=[prompt_in, neg_in, steps_in, cfg_in, bg_chk], outputs=[out_img, out_path])",
    "        btn_sil.click(sadece_arkaplan_sil, inputs=[yukle_in], outputs=[out_img, out_path])"
]
open(patch_path, "w", encoding="utf-8").write("\n".join(studio_2d_lines) + "\n")

app_path = os.path.join(u3d_dir, "gradio_app.py")
if os.path.exists(app_path):
    acode = open(app_path, "r", encoding="utf-8").read()
    if "huggingface_hub.cached_download" not in acode:
        acode = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + acode
    if "studio_2d_patch" not in acode:
        acode = acode.replace(
            "create_3d_ui()",
            "from studio_2d_patch import create_2d_tab\n        create_2d_tab()\n        create_3d_ui()"
        )
    acode = re.sub(
        r"demo(?:\.queue\(\))*\.launch\([^\)]*\)",
        "demo.queue().launch(share=True, server_name='0.0.0.0', server_port=7860)",
        acode
    )
    open(app_path, "w", encoding="utf-8").write(acode)

print("=== [8/9] Cloudflare Tüneli Başlatılıyor ===")
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin], check=True)
    subprocess.run(["chmod", "+x", cf_bin], check=True)

cf_log_path = f"{WORK_DIR}/cf.log"
cf_log = open(cf_log_path, "w")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(3)

print("=== [9/9] Unique3D Stüdyo Sunucusu Başlatılıyor ===")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
gradio_log = open(gradio_log_path, "w")
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("⏳ Modeller GPU belleğine yükleniyor, linkler hazırlanıyor (20-30 sn)...")
basladi = False
for _ in range(60):
    time.sleep(2)
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        if "Running on local URL" in icerik or "7860" in icerik:
            basladi = True
            break
    if proc.poll() is not None:
        print("❌ Sunucu başlatılırken hata oluştu! Log detayı:")
        print(open(gradio_log_path, "r", errors="ignore").read()[-2000:])
        break

if basladi:
    time.sleep(4)
    cf_icerik = open(cf_log_path, "r", errors="ignore").read()
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n" + "="*70)
    print("🚀 UNIQUE3D 2D+3D STÜDYO TAMAMEN HAZIR VE ARKA PLANDA ÇALIŞIYOR!")
    if cf_linkler:
        print(f"🌐 1. CLOUDFLARE LİNKİNİZ (Önerilen) : {cf_linkler[0]}")
    if gr_linkler:
        print(f"🌐 2. GRADIO LIVE LİNKİNİZ            : {gr_linkler[0]}")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ VE BOZULAN DOSYALARI SIFIRLA
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve dosyalar orijinal haline getiriliyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("git checkout scripts/multiview_inference.py app/custom_models/normal_prediction.py app/custom_models/utils.py gradio_app.py app/gradio_3dgen.py >/dev/null 2>&1")
time.sleep(2)

# ══════════════════════════════════════════════════════════
# 2) MODEL KANALLARINI SABİTLE
# ══════════════════════════════════════════════════════════
print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 3) -4 GB VRAM KAZANDIRAN CPU OFFLOAD YAMASI
# ══════════════════════════════════════════════════════════
print("🩹 CPU Offload ve Attention Slicing yamaları ile VRAM boşaltılıyor...")

# utils.py içine CPU Offload ekliyoruz (Kullanılmayan modeli GPU'dan RAM'e atar)
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f: u_code = f.read()
    
    vram_patch = r"\1try:\1    if hasattr(pipeline, 'enable_model_cpu_offload'): pipeline.enable_model_cpu_offload()\1    if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing(1)\1except: pass\1return pipeline"
    u_code = re.sub(r"(\s+)return pipeline", vram_patch, u_code)
    
    with open(utils_path, "w", encoding="utf-8") as f: f.write(u_code)
    print("   ✅ Modeller yüklendiğinde otomatik VRAM boşaltma aktif edildi (-4 GB tasarruf)")

# normal_prediction.py içine Aşamalar arası manuel GPU çöp toplayıcı
norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f: norm_code = f.read()
    norm_code = re.sub(
        r"(\s+)images = trainer\.pipeline_forward\(",
        r"\1import gc, torch; gc.collect(); torch.cuda.empty_cache()\1images = trainer.pipeline_forward(",
        norm_code
    )
    with open(norm_path, "w", encoding="utf-8") as f: f.write(norm_code)
    print("   ✅ Aşamalar arası VRAM temizliği (gc.collect) eklendi")

# Gradio arayüz yamaları
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f: f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f: gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)
with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

# ══════════════════════════════════════════════════════════
# 5) CANLI LOG VE ÜRETİM TAKİBİ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Çözünürlük KESİN olarak 256x256'ya düşürülüyor ve VRAM yamaları yapılıyor...")

# ══════════════════════════════════════════════════════════
# 1) ÇÖZÜNÜRLÜĞÜ REGEX İLE KESİN DÜŞÜR (512 veya 320 farketmez)
# ══════════════════════════════════════════════════════════
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    
    # Dosyadaki resize((512, 512) veya resize((320, 320) gibi tüm kalıpları bul ve resize((256, 256) yap
    mvi_code = re.sub(r"resize\(\(\d+,\s*\d+\)", "resize((256, 256)", mvi_code)
    
    # Tahmin öncesi VRAM temizliği ekle
    if "empty_cache()" not in mvi_code:
        mvi_code = mvi_code.replace(
            "rm_normals = predict_normals(",
            "import gc, torch\ngc.collect()\ntorch.cuda.empty_cache()\nrm_normals = predict_normals("
        )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)
    print("   ✅ Normal tahmini çözünürlüğü zorla 256x256'ya düşürüldü")

# ══════════════════════════════════════════════════════════
# 2) MODEL YÜKLEMEYE AGRESİF CPU OFFLOAD EKLE
# ══════════════════════════════════════════════════════════
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        vram_opt = """

def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        # Sıralı CPU Offload (En agresif bellek tasarrufu, VRAM'i neredeyse sıfırlar)
        if hasattr(pipe, 'enable_sequential_cpu_offload'):
            pipe.enable_sequential_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
        print("   ✅ Gelişmiş VRAM Optimizasyonu (Sequential Offload) uygulandı")

# ══════════════════════════════════════════════════════════
# 3) GRADIO HATA DÜZELTMELERİ
# ══════════════════════════════════════════════════════════
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT (En Agresif Bellek Yönetimi)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    # Bellek parçalanmasını önlemek için en agresif ayar
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:16,garbage_collection_threshold:0.8", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Çözünürlük KESİN olarak 256x256'ya düşürülüyor ve VRAM yamaları yapılıyor...")

# 1) Çözünürlüğü Regex ile 256x256 yap
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    
    mvi_code = re.sub(r"resize\(\(\d+,\s*\d+\)", "resize((256, 256)", mvi_code)
    
    if "empty_cache()" not in mvi_code:
        mvi_code = mvi_code.replace(
            "rm_normals = predict_normals(",
            "import gc, torch\ngc.collect()\ntorch.cuda.empty_cache()\nrm_normals = predict_normals("
        )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)
    print("   ✅ Normal tahmini çözünürlüğü zorla 256x256'ya düşürüldü")

# 2) Model Yüklemeye Agresif CPU Offload Ekle
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        vram_opt = """

def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_sequential_cpu_offload'):
            pipe.enable_sequential_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
        print("   ✅ Gelişmiş VRAM Optimizasyonu (Sequential Offload) uygulandı")

# 3) Gradio Hata Düzeltmeleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT (Düzeltilmiş Bellek Yönetimi)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    # HATA ÇÖZÜMÜ: 16 yerine 32 yapıldı (PyTorch en az 20 istiyor)
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32,garbage_collection_threshold:0.8", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🩹 Bozulan multiview dosyası orijinal haline döndürülüyor...")
# Bozulan dosyayı git ile sıfırla
os.system("git checkout scripts/multiview_inference.py")
time.sleep(1)

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Çözünürlük KESİN olarak 256x256'ya düşürülüyor...")
# Sadece çözünürlüğü güvenli bir şekilde değiştir
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    
    # Orijinal koddaki 512x512'yi 256x256 yap
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)
    print("   ✅ Normal tahmini çözünürlüğü 256x256'ya düşürüldü")

# Model Yüklemeye Agresif CPU Offload Ekle
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        vram_opt = """

def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_sequential_cpu_offload'):
            pipe.enable_sequential_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
        print("   ✅ Gelişmiş VRAM Optimizasyonu (Sequential Offload) uygulandı")

# Gradio Hata Düzeltmeleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# SUNUCUYU BAŞLAT (Düzeltilmiş Bellek Yönetimi)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32,garbage_collection_threshold:0.8", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🩹 Bozulan dosyalar git ile orijinal haline döndürülüyor...")
# Hem multiview hem utils dosyasını sıfırla ki temiz yamalar yapalım
os.system("git checkout scripts/multiview_inference.py app/custom_models/utils.py")
time.sleep(1)

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Çözünürlük 256x256'ya düşürülüyor ve GÜVENLİ VRAM temizliği ekleniyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    
    # 1. Çözünürlüğü 256 yap
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    
    # 2. OOM hatasını önlemek için normal tahmininden hemen önce RAM/VRAM temizliği ekle (Girintiyi koruyarak)
    mvi_code = re.sub(
        r"^(\s*)rm_normals = predict_normals\(",
        r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(",
        mvi_code,
        flags=re.MULTILINE
    )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)
    print("   ✅ Çözünürlük 256x256 yapıldı ve VRAM temizliği eklendi")

# 3. Model Yüklemeye Daha İyi CPU Offload (Sequential yerine Model Offload) Ekle
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    
    vram_opt = """

def apply_vram_opt(pipe):
    try:
        # VAE ve Attention ayarları
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        
        # Model CPU Offload: Tüm modeli RAM'de tutar, sadece işlem sırasında GPU'ya atar.
        # Sequential offload'dan daha hızlıdır ve float16 hatasını önler.
        if hasattr(pipe, 'enable_model_cpu_offload'):
            pipe.enable_model_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    u_code += vram_opt
    u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
    with open(utils_path, "w", encoding="utf-8") as f:
        f.write(u_code)
    print("   ✅ Gelişmiş Model CPU Offload aktif edildi")

# Gradio Hata Düzeltmeleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🩹 Bozulan dosyalar git ile orijinal haline döndürülüyor...")
# Tüm değiştirilmiş dosyaları sıfırla
os.system("git checkout scripts/multiview_inference.py app/custom_models/utils.py custum_3d_diffusion/custum_modules/attention_processors.py")
time.sleep(1)

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Attention Slicing ve VRAM yamaları uygulanıyor...")

# 1. Çözünürlüğü 256 yap
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    
    # Çözünürlüğü 256x256 yap
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    
    # Normal tahmininden hemen önce güvenli VRAM temizliği ekle
    mvi_code = re.sub(
        r"^(\s*)rm_normals = predict_normals\(",
        r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(",
        mvi_code,
        flags=re.MULTILINE
    )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)
    print("   ✅ Çözünürlük 256x256 yapıldı ve VRAM temizliği eklendi")

# 2. Attention Processors içindeki scaled_dot_product_attention'ı sliced_attention ile değiştir
attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f:
        attn_code = f.read()
    
    # F.scaled_dot_product_attention çağrısını bul ve attn.sliced_attention ile değiştir
    # Bu, 4GB'lık dev tensor tahsisini önler ve VRAM'i parçalara böler.
    attn_code = re.sub(
        r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)",
        "hidden_states = attn.sliced_attention(query, key, value, query, attention_mask, query.shape[1], slice_size=1)",
        attn_code
    )
    with open(attn_path, "w", encoding="utf-8") as f:
        f.write(attn_code)
    print("   ✅ Attention Slicing (sliced_attention) aktif edildi")

# 3. Model Yüklemeye Model CPU Offload Ekle
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    
    vram_opt = """

def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'):
            pipe.enable_model_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    u_code += vram_opt
    u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
    with open(utils_path, "w", encoding="utf-8") as f:
        f.write(u_code)
    print("   ✅ Model CPU Offload aktif edildi")

# Gradio Hata Düzeltmeleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# SUNUCUYU BAŞLAT (Optimize Edilmiş Bellek Ayarları)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    # HATA ÇÖZÜMÜ: max_split_size_mb:128 yapıldı (4GB'lık tensorler için daha uygun)
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🩹 Bozulan dosyalar git ile orijinal haline döndürülüyor...")
os.system("git checkout scripts/multiview_inference.py app/custom_models/utils.py custum_3d_diffusion/custum_modules/attention_processors.py")
time.sleep(1)

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Çözünürlük 256x256'ya düşürülüyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    
    mvi_code = re.sub(
        r"^(\s*)rm_normals = predict_normals\(",
        r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(",
        mvi_code,
        flags=re.MULTILINE
    )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)
    print("   ✅ Çözünürlük 256x256 yapıldı ve VRAM temizliği eklendi")

# ══════════════════════════════════════════════════════════
# 🚨 KRİTİK YAMA: PyTorch Attention Slicing (Dev Tensor Engelleme)
# ══════════════════════════════════════════════════════════
print("🩹 PyTorch Attention Slicing (Query Dilimleme) aktif ediliyor...")
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    
    vram_opt = """

def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'):
            pipe.enable_model_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    u_code += vram_opt
    u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
    with open(utils_path, "w", encoding="utf-8") as f:
        f.write(u_code)
    print("   ✅ Model CPU Offload aktif edildi")

# Gradio Hata Düzeltmeleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

# 🔥 EN KRİTİK YAMA BURADA: F.scaled_dot_product_attention'ı dilimleyerek çalıştır
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
import torch.nn.functional as F

# PyTorch'un SDPA fonksiyonunu VRAM dostu hale getir (Query dilimleme)
_original_sdpa = F.scaled_dot_product_attention
def _memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    # Eğer Query matrisi çok büyükse (örneğin 1024'ten fazla), parçalara böl
    if query.shape[2] > 1024:
        chunk_size = 512
        outputs = []
        for i in range(0, query.shape[2], chunk_size):
            q_chunk = query[:, :, i:i+chunk_size, :]
            mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
            out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
            outputs.append(out)
        return torch.cat(outputs, dim=2)
    return _original_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale)

F.scaled_dot_product_attention = _memory_efficient_sdpa

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor ve GPU belleği (VRAM) tamamen boşaltılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

print("🩹 Bozulan dosyalar git ile orijinal haline döndürülüyor...")
os.system("git checkout scripts/multiview_inference.py app/custom_models/utils.py custum_3d_diffusion/custum_modules/attention_processors.py")
time.sleep(1)

print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
for cfg_path, channels in [("ckpt/img2mvimg/unet/config.json", 8), ("ckpt/unet_4ch/unet/config.json", 4)]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = channels
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🩹 Çözünürlük 256x256'ya düşürülüyor ve VRAM temizliği ekleniyor...")
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    mvi_code = mvi_code.replace("img.resize((512, 512)", "img.resize((256, 256)")
    mvi_code = re.sub(
        r"^(\s*)rm_normals = predict_normals\(",
        r"\1import gc, torch\n\1gc.collect()\n\1torch.cuda.empty_cache()\n\1rm_normals = predict_normals(",
        mvi_code,
        flags=re.MULTILINE
    )
    with open(mvi_path, "w", encoding="utf-8") as f:
        f.write(mvi_code)

# ══════════════════════════════════════════════════════════
# 🚨 DİLİM (CHUNK) BOYUTUNU ARTIR VE BELLEK TEMİZLE
# ══════════════════════════════════════════════════════════
print("🩹 Agresif Attention Slicing (128'lik dilimler) ve Bellek Temizliği aktif ediliyor...")

# 1. Unique3D'nin kendi attention dosyasını değiştir
attn_path = "custum_3d_diffusion/custum_modules/attention_processors.py"
if os.path.exists(attn_path):
    with open(attn_path, "r", encoding="utf-8") as f:
        attn_code = f.read()
    # Dosyadaki F.scaled_dot_product_attention'ı bul ve döngüye al
    attn_code = re.sub(
        r"hidden_states = F\.scaled_dot_product_attention\(\s*query,\s*key,\s*value,\s*attn_mask=attention_mask,\s*dropout_p=0\.0,\s*is_causal=False\s*\)",
        "hidden_states = attn.sliced_attention(query, key, value, query.shape[1], attention_mask, slice_size=128)",
        attn_code
    )
    with open(attn_path, "w", encoding="utf-8") as f:
        f.write(attn_code)
    print("   ✅ attention_processors.py içine 128'lik dilimleme eklendi")

# 2. Ana dosyaya (gradio_app.py) PyTorch seviyesinde en agresif yamayı ekle
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
import torch.nn.functional as F

# PyTorch'un SDPA fonksiyonunu 128'lik dilimlerle ve zorla bellek temizleyerek çalıştır
_original_sdpa = F.scaled_dot_product_attention
def _extreme_memory_efficient_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    # Her zaman dilimle (VRAM patlamasını önlemek için)
    chunk_size = 128
    outputs = []
    for i in range(0, query.shape[2], chunk_size):
        q_chunk = query[:, :, i:i+chunk_size, :]
        mask_chunk = attn_mask[:, :, i:i+chunk_size, :] if attn_mask is not None else None
        out = _original_sdpa(q_chunk, key, value, attn_mask=mask_chunk, dropout_p=dropout_p, is_causal=is_causal, scale=scale)
        outputs.append(out)
        # Her dilimden sonra GPU belleğini zorla boşalt
        torch.cuda.empty_cache() 
    return torch.cat(outputs, dim=2)

F.scaled_dot_product_attention = _extreme_memory_efficient_sdpa

_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)
with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# 3. Model Yükleme Ayarları
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    vram_opt = """
def apply_vram_opt(pipe):
    try:
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        if hasattr(pipe, 'enable_model_cpu_offload'):
            pipe.enable_model_cpu_offload()
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
    if "enable_attention_slicing" not in u_code:
        u_code += vram_opt
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
    print("   ✅ Model CPU Offload aktif edildi")

# Gradio Hata Düzeltmeleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

# ══════════════════════════════════════════════════════════
# SUNUCUYU BAŞLAT (Maksimum Bellek Tasarrufu Ayarları)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:32", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
gradio_log_path = f"{WORK_DIR}/gradio.log"

if os.path.exists(gradio_log_path):
    print("📜 CANLI LOG ÇIKTISI (Son 20 satır):")
    print("="*60)
    print("\n".join(open(gradio_log_path, "r", errors="ignore").read().strip().split('\n')[-20:]))
    print("="*60)
else:
    print("❌ Log dosyası henüz oluşmamış.")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 Eksik olan 'rotate_normals_torch' fonksiyonu utils.py dosyasına ekleniyor...")

# Güncel ve eksiksiz utils.py içeriği (Orijinal utils + rotate_normals_torch fonksiyonu)
tam_utils_kodu = """
import os
import torch
import onnxruntime as ort
from rembg import new_session

providers = ["CPUExecutionProvider"]

try:
    session = new_session(providers=providers)
except Exception as e:
    print(f"Rembg session initialization warning: {e}")
    session = None

def simple_preprocess(img):
    return img

def rotate_normals_torch(normals, rot_mat):
    # Normal haritalarını döndürmek için gerekli yardımcı fonksiyon
    original_shape = normals.shape
    normals = normals.reshape(-1, 3)
    normals = torch.matmul(normals, rot_mat.T)
    return normals.reshape(original_shape)
"""

open("scripts/utils.py", "w", encoding="utf-8").write(tam_utils_kodu)

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Link birazdan ekranda olacak)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

if proc.poll() is None:
    time.sleep(3)
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n" + "="*70)
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: İnternetten model çekme isteği yerel klasöre yönlendiriliyor...")

# Eğer yerel klasör eksikse doğrudan indirip garantiye alıyoruz
lambda_dir = f"{u3d_dir}/lambdalabs_local"
if not os.path.exists(lambda_dir) or not os.path.exists(os.path.join(lambda_dir, "config.json")):
    print("⏳ Lambdalabs yerel modeli güvenli şekilde indiriliyor...")
    from huggingface_hub import snapshot_download
    snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir=lambda_dir)

# Kod içindeki tüm "lambdalabs/sd-image-variations-diffusers" referanslarını yerel klasöre sabitliyoruz
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            try:
                content = open(path, "r", encoding="utf-8").read()
                new_content = content.replace("lambdalabs/sd-image-variations-diffusers", lambda_dir)
                if content != new_content:
                    open(path, "w", encoding="utf-8").write(new_content)
            except:
                pass

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Artık internete ihtiyaç duymayacak)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

if proc.poll() is None:
    time.sleep(3)
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n" + "="*70)
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: Hugging Face hub fonksiyonu seviyesinde küresel yönlendirme yapılıyor...")

# Yerel klasör yolunu garantiye alıyoruz
lambda_dir = f"{u3d_dir}/lambdalabs_local"
if not os.path.exists(os.path.join(lambda_dir, "config.json")):
    from huggingface_hub import snapshot_download
    snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir=lambda_dir)

# gradio_app.py'nin en başına küresel bir yönlendirme yaması enjekte ediyoruz
global_patch = f"""
import huggingface_hub
_old_hf_hub_download = huggingface_hub.hf_hub_download
_old_snapshot_download = huggingface_hub.snapshot_download

def _new_hf_hub_download(repo_id, *args, **kwargs):
    if repo_id and "lambdalabs" in str(repo_id):
        return r"{lambda_dir}"
    return _old_hf_hub_download(repo_id, *args, **kwargs)

def _new_snapshot_download(repo_id, *args, **kwargs):
    if repo_id and "lambdalabs" in str(repo_id):
        return r"{lambda_dir}"
    return _old_snapshot_download(repo_id, *args, **kwargs)

huggingface_hub.hf_hub_download = _new_hf_hub_download
huggingface_hub.snapshot_download = _new_snapshot_download
"""

app_path = "gradio_app.py"
if os.path.exists(app_path):
    code = open(app_path, "r", encoding="utf-8").read()
    if "_new_hf_hub_download" not in code:
        code = global_patch + "\n" + code
        open(app_path, "w", encoding="utf-8").write(code)
        print("✅ Küresel Hub yönlendirmesi başarıyla eklendi!")

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

if proc.poll() is None:
    time.sleep(3)
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n" + "="*70)
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: Hub yönlendirmesi doğrudan dosya yoluna (config.json) ayarlanıyor...")

lambda_dir = f"{u3d_dir}/lambdalabs_local"
config_file_path = os.path.join(lambda_dir, "config.json")

# Eğer config.json yoksa indir
if not os.path.exists(config_file_path):
    from huggingface_hub import snapshot_download
    snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir=lambda_dir)

# gradio_app.py başındaki yamayı güncelliyoruz (Dosya yolu döndürecek şekilde)
app_path = "gradio_app.py"
if os.path.exists(app_path):
    code = open(app_path, "r", encoding="utf-8").read()
    
    # Eski hub yamasını temizle
    import re
    code = re.sub(r'import huggingface_hub.*?\n\ndef _new_hf_hub_download.*?\n\nhuggingface_hub\.snapshot_download = _new_snapshot_download\n', '', code, flags=re.DOTALL)

    # Yeni ve kusursuz yama
    perfect_patch = f"""
import huggingface_hub
_old_hf_hub_download = huggingface_hub.hf_hub_download

def _new_hf_hub_download(repo_id, *args, **kwargs):
    if repo_id and "lambdalabs" in str(repo_id):
        filename = kwargs.get("filename", "")
        if filename:
            return r"{lambda_dir}/" + filename
        return r"{config_file_path}"
    return _old_hf_hub_download(repo_id, *args, **kwargs)

huggingface_hub.hf_hub_download = _new_hf_hub_download
"""
    code = perfect_patch + "\n" + code
    open(app_path, "w", encoding="utf-8").write(code)
    print("✅ Mükemmel Hub yönlendirmesi güncellendi!")

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Arayüz artık kesin açılıyor)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

if proc.poll() is None:
    time.sleep(3)
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n" + "="*70)
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("="*70 + "\n")

In [ ]:
import os, sys, time, re, gc, subprocess, torch

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working/Unique3D") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

# 1. Eski süreçleri temizle
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
time.sleep(2)

# 2. studio_2d_patch.py dosyasındaki varsayılan promptu boş olarak yeniden yaz
patch_path = os.path.join(u3d_dir, "studio_2d_patch.py")
studio_2d_lines = [
    "import os, sys, gc, time, random, torch, rembg, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import gradio as gr",
    "from diffusers import AutoPipelineForText2Image",
    "",
    "def _get_rembg_session():",
    "    from scripts.utils import session as _s",
    "    return _s",
    "",
    "def uret_2d_gorsel(prompt, neg_prompt, steps, cfg, bg_sil):",
    "    try:",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        Moved = []",
    "        for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction']:",
    "            mod = sys.modules.get(m_adi)",
    "            if mod and hasattr(mod, 'pipeline') and hasattr(mod.pipeline, 'unet'):",
    "                try:",
    "                    mod.pipeline.unet.to('cpu')",
    "                    Moved.append(mod.pipeline.unet)",
    "                except Exception: pass",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        pipe = AutoPipelineForText2Image.from_pretrained(",
    "            'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16', low_cpu_mem_usage=True",
    "        ).to('cuda')",
    "        seed = random.randint(1, 99999999)",
    "        gen = torch.Generator(device='cuda').manual_seed(seed)",
    "        with torch.inference_mode():",
    "            img = pipe(",
    "                prompt=prompt + ', isometric, front view, centered, clean background, high detail',",
    "                negative_prompt=neg_prompt,",
    "                num_inference_steps=int(steps),",
    "                guidance_scale=float(cfg),",
    "                generator=gen",
    "            ).images[0]",
    "        del pipe, gen",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        for u in Moved:",
    "            try: u.to('cuda')",
    "            except Exception: pass",
    "        if bg_sil:",
    "            img = rembg.remove(img, session=_get_rembg_session())",
    f"        kayit_yolu = f'{WORK_DIR}/uretilen_2d_gorseller/gorsel_{{int(time.time())}}_{{seed}}.png'",
    "        img.save(kayit_yolu)",
    "        return img, f'Basarili (Seed: {seed}) -> {kayit_yolu}'",
    "    except Exception as e:",
    "        import traceback",
    "        traceback.print_exc()",
    "        return None, f'Hata: {str(e)}'",
    "",
    "def sadece_arkaplan_sil(img):",
    "    if img is None: return None, 'Gorsel yuklenmedi'",
    "    try:",
    "        out = rembg.remove(img, session=_get_rembg_session())",
    f"        kayit_yolu = f'{WORK_DIR}/uretilen_2d_gorseller/temiz_{{int(time.time())}}.png'",
    "        out.save(kayit_yolu)",
    "        return out, kayit_yolu",
    "    except Exception as e:",
    "        return None, f'Hata: {str(e)}'",
    "",
    "def create_2d_tab():",
    "    with gr.Tab('🎨 2D Görsel Üret & Hazırla (3D İçin)'):",
    "        gr.Markdown('### 1. Metinden 3D Uyumlu 2D Nesne Üret veya Hazır Fotoğrafın Arka Planını Sil')",
    "        with gr.Row():",
    "            with gr.Column():",
    "                # Varsayılan prompt tamamen boş bırakıldı",
    "                prompt_in = gr.Textbox(label='2D Nesne Tarifi (İngilizce)', value='', placeholder='Örn: a red sports car, front view')",
    "                neg_in = gr.Textbox(label='Negatif Prompt', value='blurry, cropped, low quality, multiple objects, complex background')",
    "                with gr.Row():",
    "                    steps_in = gr.Slider(1, 8, value=4, step=1, label='Adım Sayısı (2-4 idealdir)')",
    "                    cfg_in = gr.Slider(0.0, 3.0, value=0.0, step=0.5, label='CFG Scale (Turbo için 0.0)')",
    "                bg_chk = gr.Checkbox(value=True, label='Üretilen Görselin Arka Planını Otomatik Sil (Rembg)')",
    "                btn_uret = gr.Button('🎨 2D Görsel Üret', variant='primary')",
    "                gr.Markdown('---')",
    "                yukle_in = gr.Image(type='pil', label='Veya Kendi 2D Fotoğrafını Yükle (Arka Plan Silmek İçin)')",
    "                btn_sil = gr.Button('✂️ Sadece Arka Planı Sil')",
    "            with gr.Column():",
    "                out_img = gr.Image(type='pil', label='Hazırlanan 2D Görsel')",
    "                out_path = gr.Textbox(label='Durum ve Diskteki Kayıt Yolu', interactive=False)",
    "        btn_uret.click(uret_2d_gorsel, inputs=[prompt_in, neg_in, steps_in, cfg_in, bg_chk], outputs=[out_img, out_path])",
    "        btn_sil.click(sadece_arkaplan_sil, inputs=[yukle_in], outputs=[out_img, out_path])"
]
open(patch_path, "w", encoding="utf-8").write("\n".join(studio_2d_lines) + "\n")

# 3. Gradio uygulamasını başlat
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"

print("🚀 Stüdyo güncellenmiş ve boş prompt ayarıyla başlatılıyor...")
!python -u gradio_app.py --share

In [ ]:
import os, sys, shutil, subprocess, time, re, json, gc, torch

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(WORK_DIR)

# CUDA Derleme ve Bellek Optimizasyon Değişkenleri (T4 GPU = 7.5)
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0;8.6"
os.environ["FORCE_CUDA"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:16"
os.environ["PYOPENGL_PLATFORM"] = "egl"

print("=== [1/8] Eski Süreçler ve Bellek Temizleniyor ===")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

os.makedirs(f"{WORK_DIR}/uretilen_3d_modeller", exist_ok=True)
os.makedirs(f"{WORK_DIR}/uretilen_2d_gorseller", exist_ok=True)

print("=== [2/8] Linux Sistem ve OpenGL/EGL Derleme Paketleri Kuruluyor ===")
os.system("apt-get update -qq && apt-get install -y -qq libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev build-essential ninja-build psmisc git wget >/dev/null 2>&1")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "setuptools", "wheel"])

print("=== [3/8] Unique3D Projesi ve Temel Python Paketleri Kontrol Ediliyor ===")
if not os.path.exists(os.path.join(u3d_dir, ".git")):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git", u3d_dir], check=True)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "fire", "rembg", "onnxruntime", "trimesh", "einops", "omegaconf",
    "transformers", "diffusers", "peft", "accelerate", "huggingface_hub",
    "gradio", "gradio-client", "pydantic", "jaxtyping", "typeguard",
    "imageio", "scikit-image", "opencv-python-headless", "matplotlib", "plyfile", "pymeshlab"
])

print("=== [4/8] nvdiffrast ve PyTorch3D Kuruluyor ===")
# 1. nvdiffrast Kurulumu (Çift Korumalı: C++ Derleme + JIT Yedekleme)
try:
    import nvdiffrast
    print("✅ nvdiffrast zaten kurulu.")
except ImportError:
    print("⚙️ nvdiffrast kuruluyor...")
    os.chdir(WORK_DIR)
    shutil.rmtree("nvdiffrast", ignore_errors=True)
    subprocess.run(["git", "clone", "https://github.com/NVlabs/nvdiffrast.git"], check=True)
    os.chdir("nvdiffrast")
    res = subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "."], capture_output=True, text=True)
    if res.returncode != 0:
        print("⚠️ Standart C++ derlemesi atlandı, nvdiffrast JIT (Runtime) modunda kuruluyor...")
        site_pkg = [p for p in sys.path if "dist-packages" in p or "site-packages" in p][0]
        shutil.copytree("nvdiffrast", os.path.join(site_pkg, "nvdiffrast"), dirs_exist_ok=True)
    os.chdir(WORK_DIR)
    import nvdiffrast
    print("✅ nvdiffrast başarıyla aktif edildi!")

# 2. PyTorch3D Kurulumu
try:
    import pytorch3d
    print("✅ PyTorch3D zaten kurulu.")
except ImportError:
    print("⚙️ PyTorch3D derlenip kuruluyor (약 2-3 dakika sürebilir, lütfen bekleyin)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
        "git+https://github.com/facebookresearch/pytorch3d.git"
    ])
    print("✅ PyTorch3D başarıyla kuruldu!")

os.chdir(u3d_dir)

print("=== [5/8] Model Ağırlıkları ve 4-Kanallı UNet Yapılandırması ===")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    print("📥 Model ağırlıkları indiriliyor...")
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

img2mvimg_yolu = f"{u3d_dir}/ckpt/img2mvimg"
unet_4ch_yolu = f"{u3d_dir}/ckpt/unet_4ch"
if os.path.exists(img2mvimg_yolu) and not os.path.exists(unet_4ch_yolu):
    os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)
    ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
    if os.path.exists(ana_cfg_path):
        ana_cfg = json.loads(open(ana_cfg_path, "r").read())
        ana_cfg["in_channels"] = 4
        ana_cfg.pop("num_class_embeds", None)
        ana_cfg["addition_embed_type"] = None
        open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet" and not os.path.exists(unet_4ch_yolu + "/" + d):
            os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)
    for d in os.listdir(img2mvimg_yolu + "/unet"):
        if d != "config.json" and not os.path.exists(unet_4ch_yolu + "/unet/" + d):
            os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("=== [6/8] Kod, Bellek (256 Çözünürlük) ve Gradio Yamaları Uygulanıyor ===")
# 1. Gradio Client Bool Şema Yaması
try:
    import gradio_client.utils as client_utils
    utils_path = client_utils.__file__
    code_client = open(utils_path, "r", encoding="utf-8").read()
    if "if not isinstance(schema, dict): return 'Any'" not in code_client:
        code_client = code_client.replace("def get_type(schema):", "def get_type(schema):\n    if not isinstance(schema, dict): return 'Any'")
        code_client = code_client.replace("def _json_schema_to_python_type(schema, defs):", "def _json_schema_to_python_type(schema, defs):\n    if not isinstance(schema, dict): return 'Any'")
        open(utils_path, "w", encoding="utf-8").write(code_client)
except Exception:
    pass

# 2. RasterizeCudaContext ve output_db Temizliği
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            txt = open(path, "r", errors="ignore").read()
            new_txt = txt.replace("output_db=False,", "").replace(", output_db=False", "").replace("output_db=False", "")
            new_txt = new_txt.replace("RasterizeGLContext", "RasterizeCudaContext")
            if txt != new_txt:
                open(path, "w", encoding="utf-8").write(new_txt)

# 3. Mesh Reconstruction Çözünürlüğünü 256'ya Düşürme (T4 VRAM Çökmesini Önler)
recon_path = os.path.join(u3d_dir, "mesh_reconstruction/recon.py")
if os.path.exists(recon_path):
    code_recon = open(recon_path, "r", encoding="utf-8").read()
    code_recon = code_recon.replace("res=512", "res=256").replace("resolution=512", "resolution=256")
    code_recon = code_recon.replace("512,", "256,").replace("512)", "256)")
    open(recon_path, "w", encoding="utf-8").write(code_recon)

# 4. 2D Stüdyo Yaması (Varsayılan Robot Kaskı Promptu Temizlenmiş Halde)
patch_path = os.path.join(u3d_dir, "studio_2d_patch.py")
if os.path.exists(patch_path):
    ptxt = open(patch_path, "r", encoding="utf-8").read()
    ptxt = re.sub(r"value=['\"]a futuristic helmet[^'\"]*['\"]", "value=''", ptxt)
    open(patch_path, "w", encoding="utf-8").write(ptxt)

# 5. gradio_app.py içinde share=True garantisi
app_path = os.path.join(u3d_dir, "gradio_app.py")
if os.path.exists(app_path):
    acode = open(app_path, "r", encoding="utf-8").read()
    acode = re.sub(
        r"demo(?:\.queue\(\))*\.launch\([^\)]*\)",
        "demo.queue().launch(share=True, server_name='0.0.0.0', server_port=7860)",
        acode
    )
    open(app_path, "w", encoding="utf-8").write(acode)

print("=== [7/8] Cloudflare Tüneli Başlatılıyor ===")
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin], check=True)
    subprocess.run(["chmod", "+x", cf_bin], check=True)

cf_log_path = f"{WORK_DIR}/cf.log"
cf_log = open(cf_log_path, "w")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(3)

print("=== [8/8] Unique3D Stüdyo Sunucusu Başlatılıyor ===")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:16"

gradio_log_path = f"{WORK_DIR}/gradio.log"
gradio_log = open(gradio_log_path, "w")
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("⏳ Modeller GPU belleğine yükleniyor, linkler hazırlanıyor (15-25 sn)...")
basladi = False
for _ in range(45):
    time.sleep(2)
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        if "Running on local URL" in icerik or "7860" in icerik:
            basladi = True
            break
    if proc.poll() is not None:
        print("❌ Sunucu başlatılırken hata oluştu! Log detayı:")
        print(open(gradio_log_path, "r", errors="ignore").read()[-1500:])
        break

if basladi:
    time.sleep(3)
    cf_icerik = open(cf_log_path, "r", errors="ignore").read()
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n" + "="*70)
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR VE ARKA PLANDA ÇALIŞIYOR!")
    if cf_linkler:
        print(f"🌐 1. CLOUDFLARE LİNKİNİZ (Kopmaz) : {cf_linkler[0]}")
    if gr_linkler:
        print(f"🌐 2. GRADIO LIVE LİNKİNİZ         : {gr_linkler[0]}")
    print("="*70 + "\n")

In [ ]:
import os, sys, time, re, gc, subprocess, torch

# Ortamı otomatik tespit et (/kaggle/working veya /content)
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working/Unique3D") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print(f"📂 Çalışma Dizini: {u3d_dir}")

# 1. Eski süreçleri temizle
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
time.sleep(2)

# 2. gradio_app.py içinde Gradio 6.0 uyumluluğu ve share=True garantisi
app_path = os.path.join(u3d_dir, "gradio_app.py")
if os.path.exists(app_path):
    code = open(app_path, "r", encoding="utf-8").read()
    code = re.sub(
        r"demo(?:\.queue\(\))*\.launch\([^\)]*\)",
        "demo.queue().launch(share=True, server_name='0.0.0.0', server_port=7860)",
        code
    )
    open(app_path, "w", encoding="utf-8").write(code)

# 3. Cloudflare Tunnel'i arka planda başlat (Yedek garantili link)
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin], check=True)
    subprocess.run(["chmod", "+x", cf_bin], check=True)

cf_log_path = f"{WORK_DIR}/cf.log"
cf_log = open(cf_log_path, "w")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)

# 4. Gradio sunucusunu arka planda (-u: tamponsuz anlık log ile) başlat
gradio_log_path = f"{WORK_DIR}/gradio.log"
gradio_log = open(gradio_log_path, "w")

env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("⏳ Modeller yükleniyor ve port 7860 açılıyor (약 15-25 saniye)...")

# 5. Sunucu ayağa kalkana kadar logu canlı takip et
basladi = False
for i in range(60):
    time.sleep(2)
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        if "Running on local URL" in icerik or "7860" in icerik:
            basladi = True
            break
    if proc.poll() is not None:
        print("❌ Sunucu beklenmedik şekilde kapandı! Son loglar:")
        print(open(gradio_log_path, "r", errors="ignore").read()[-1500:])
        break

if basladi:
    time.sleep(3)
    cf_icerik = open(cf_log_path, "r", errors="ignore").read()
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)
    
    print("\n" + "="*70)
    print("✅ STÜDYO ARKA PLANDA AKTİF! (Bu hücre tamamlandı, durdurmana gerek yok)")
    if cf_linkler:
        print(f"🌐 1. CLOUDFLARE LİNKİN (Önerilen): {cf_linkler[0]}")
    if gr_linkler:
        print(f"🌐 2. GRADIO LIVE LİNKİN          : {gr_linkler[0]}")
    print("="*70 + "\n")

In [ ]:
import os, sys, time, gc, torch

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working/Unique3D") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

# Bellek ve süreç temizliği
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Bellek tüketimini kısıtlayan ve çökme önleyen ortam değişkenleri
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
# PyTorch bellek parçaalanmasını ve sızıntısını engellemek için:
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:32"

print("🚀 Stüdyo bellek optimize edilmiş modda başlatılıyor...")
!python -u gradio_app.py --share

In [ ]:
import os, sys, time, re, gc, subprocess, torch

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working/Unique3D") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

# 1. Tüm eski süreçleri ve portları temizle
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
time.sleep(2)

# 2. VRAM optimizasyon ortam değişkenlerini en katı seviyeye getir
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
# PyTorch bellek parçaalanmasını sıfırlamak için agresif allocator ayarları:
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:16"

print("🛡️ Kaggle VRAM Koruma Modu Aktif: Agresif Bellek Yönetimi ile Başlatılıyor...")

# 3. Gradio uygulamasını başlat
subprocess.run([sys.executable, "-u", "gradio_app.py", "--share"], cwd=u3d_dir, env=env_vars)

In [ ]:
import os

u3d_dir = "/kaggle/working/Unique3D" if os.path.exists("/kaggle/working/Unique3D") else "/content/Unique3D"
recon_path = os.path.join(u3d_dir, "mesh_reconstruction/recon.py")

if os.path.exists(recon_path):
    code = open(recon_path, "r", encoding="utf-8").read()
    # Çözünürlük 512 ise 256'ya düşürerek VRAM tüketimini 4 kat azaltalım
    code = code.replace("res=512", "res=256").replace("resolution=512", "resolution=256")
    code = code.replace("512,", "256,").replace("512)", "256)")
    open(recon_path, "w", encoding="utf-8").write(code)
    print("✅ Mesh reconstruction çözünürlüğü Kaggle/T4 için 256'ya optimize edildi!")
else:
    print("⚠️ recon.py bulunamadı.")

In [ ]:
import os, sys, time, gc, torch

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working/Unique3D") else "/content"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

# Eski süreçleri temizle
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
time.sleep(2)

env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:16"

print("🚀 Stüdyo optimize edilmiş ayarlarla başlatılıyor...")
subprocess.run([sys.executable, "-u", "gradio_app.py", "--share"], cwd=u3d_dir, env=env_vars)

In [ ]:
import os, gc, torch

# 1. gradio_client içindeki TÜM bool şema çökmelerini kökten engelle
import gradio_client.utils as client_utils
utils_path = client_utils.__file__

code = open(utils_path, "r", encoding="utf-8").read()

# get_type ve _json_schema_to_python_type fonksiyonlarına bool koruması ekle
code = code.replace(
    "def get_type(schema):",
    "def get_type(schema):\n    if not isinstance(schema, dict): return 'Any'"
)
code = code.replace(
    "def _json_schema_to_python_type(schema, defs):",
    "def _json_schema_to_python_type(schema, defs):\n    if not isinstance(schema, dict): return 'Any'"
)
open(utils_path, "w", encoding="utf-8").write(code)
print("✅ gradio_client bool hatası tamamen yamandı!")

# 2. Portu temizle ve stüdyoyu yeniden başlat
os.chdir("/content/Unique3D")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
gc.collect()
torch.cuda.empty_cache()

os.environ["PYTHONPATH"] = "/content/Unique3D"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"

print("🚀 Sunucu hatasız modda başlatılıyor... Yeni çıkan gradio.live linkine tıkla (Hücreyi durdurma!):")
!python -u gradio_app.py --share

In [ ]:
import os, sys, re, time, json, shutil, socket, subprocess

# 0. INTERNET KONTROLU VE ORTAM TESPITI (COLAB / KAGGLE OTOMATIK)
try:
    socket.create_connection(("github.com", 443), timeout=5)
except OSError:
    raise RuntimeError("❌ HATA: Internet baglantisi yok!")

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0;8.6;8.9;9.0"
os.makedirs(f"{WORK_DIR}/uretilen_3d_modeller", exist_ok=True)
os.makedirs(f"{WORK_DIR}/uretilen_2d_gorseller", exist_ok=True)

print("0.5. Sistem Paketleri (OpenGL/EGL) Kuruluyor...")
os.system("apt-get update -qq && apt-get install -y -qq libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev build-essential ninja-build psmisc >/dev/null 2>&1")

print("1. Unique3D Projesi Hazirlaniyor...")
u3d_dir = f"{WORK_DIR}/Unique3D"
if not os.path.exists(os.path.join(u3d_dir, ".git")):
    if os.path.exists(os.path.join(u3d_dir, "ckpt")):
        shutil.move(os.path.join(u3d_dir, "ckpt"), f"{WORK_DIR}/_ckpt_yedek")
    shutil.rmtree(u3d_dir, ignore_errors=True)
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git", u3d_dir], check=True)
    if os.path.exists(f"{WORK_DIR}/_ckpt_yedek"):
        shutil.rmtree(os.path.join(u3d_dir, "ckpt"), ignore_errors=True)
        shutil.move(f"{WORK_DIR}/_ckpt_yedek", os.path.join(u3d_dir, "ckpt"))

os.chdir(u3d_dir)

try:
    import nvdiffrast, pytorch3d
    print("2-3-4. PyTorch ve C++ 3D Motorlari zaten kurulu, derleme atlaniyor...")
except ImportError:
    print("2. Temel Python Kutuphaneleri Yukleniyor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "setuptools", "wheel", "fire", "rembg", "onnxruntime", "trimesh", "einops", "omegaconf", "transformers", "diffusers", "peft", "accelerate", "pydantic", "jaxtyping", "typeguard", "huggingface_hub", "gradio", "imageio", "scikit-image", "opencv-python-headless", "matplotlib", "plyfile"])

    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "pymeshlab"])
    try:
        import torch
        tv = torch.__version__.split("+")[0]
        cu = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
        subprocess.call([sys.executable, "-m", "pip", "install", "-q", "torch-scatter", "-f", f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"])
    except Exception:
        subprocess.call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "torch-scatter"])

    print("3. MANUEL KURULUM: Nvdiffrast Klonlaniyor ve Standart Derleniyor...")
    os.chdir(WORK_DIR)
    shutil.rmtree("nvdiffrast", ignore_errors=True)
    subprocess.run(["git", "clone", "https://github.com/NVlabs/nvdiffrast.git"], check=True)
    os.chdir("nvdiffrast")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-build-isolation", "."])

    print("4. MANUEL KURULUM: PyTorch3D Kuruluyor...")
    os.chdir(WORK_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

os.chdir(u3d_dir)

print("5. Model Agirliklari Indiriliyor...")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    shutil.rmtree("ckpt/img2mvimg", ignore_errors=True)
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

print("6. 4 Kanalli U-Net (unet_4ch) Mimarisi Kuruluyor...")
img2mvimg_yolu = f"{u3d_dir}/ckpt/img2mvimg"
unet_4ch_yolu = f"{u3d_dir}/ckpt/unet_4ch"
shutil.rmtree(unet_4ch_yolu, ignore_errors=True)
os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)

ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
if os.path.exists(ana_cfg_path):
    ana_cfg = json.loads(open(ana_cfg_path, "r").read())
    ana_cfg["in_channels"] = 4
    ana_cfg.pop("num_class_embeds", None)
    ana_cfg["addition_embed_type"] = None
    open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))

for d in os.listdir(img2mvimg_yolu):
    if d != "unet" and not os.path.exists(unet_4ch_yolu + "/" + d):
        os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)

for d in os.listdir(img2mvimg_yolu + "/unet"):
    if d != "config.json" and not os.path.exists(unet_4ch_yolu + "/unet/" + d):
        os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("7. Kod Agaci Sifirlaniyor ve Tum 3D + 2D Zirhlar Isleniyor...")
for klasor in ["app", "scripts", "mesh_reconstruction", "custum_3d_diffusion"]:
    shutil.rmtree(klasor, ignore_errors=True)
os.system("git reset --hard HEAD")
os.system("git clean -fd -e ckpt -e ckpt/*")

oom_lines = [
    "import os, gc, torch, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    u_kod = open("scripts/utils.py", "r", encoding="utf-8").read()
    u_kod = u_kod.replace("providers=providers", "providers=['CPUExecutionProvider']")
    u_ek = [
        "",
        "import rembg",
        "try:",
        "    session = rembg.new_session(providers=['CPUExecutionProvider'])",
        "except Exception:",
        "    session = rembg.new_session()",
        "",
        "def simple_preprocess(input_image, rembg_session=None, background_color=255):",
        "    if 'remove_backgroud' in globals():",
        "        return remove_backgroud(input_image, rembg_session=rembg_session or session, background_color=background_color)",
        "    return input_image"
    ]
    if "def simple_preprocess" not in u_kod:
        u_kod = u_kod + "\n" + "\n".join(u_ek) + "\n"
    open("scripts/utils.py", "w", encoding="utf-8").write(u_kod)

if os.path.exists("scripts/project_mesh.py"):
    pm_satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_pm = []
    for s in pm_satirlar:
        if "assert " in s and "cos_angles" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_pm.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_pm.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_pm)

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale', 'studio_2d_patch']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer', 'pipe_2d']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
mvc = mvc.replace("_ = multiview_color_projection(", "# _ = multiview_color_projection(")
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    mod_2d = sys.modules.get('studio_2d_patch')",
    "    if mod_2d and getattr(mod_2d, 'pipe_2d', None) is not None:",
    "        try: mod_2d.pipe_2d.to('cpu')",
    "        except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

if os.path.exists("app/gradio_3dgen.py"):
    g3d = open("app/gradio_3dgen.py", "r", encoding="utf-8").read()
    g3d = g3d.replace(
        "def generate3dv2(",
        "def generate3dv2(*args, **kwargs):\n    import sys\n    m = sys.modules.get('scripts.multiview_inference')\n    if m and hasattr(m, '_yukle_vram'): m._yukle_vram()\n    return _orig_generate3dv2(*args, **kwargs)\ndef _orig_generate3dv2("
    )
    kayit_kodu = (
        "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)\n"
        "    try:\n"
        "        import time as _t\n"
        f"        _cikti = f'{WORK_DIR}/uretilen_3d_modeller/model_{{int(_t.time())}}.glb'\n"
        "        new_meshes.export(_cikti)\n"
        "        print(f'\\n✅ 3D MODEL DISKE YEDEKLENDI: {_cikti}\\n')\n"
        "    except Exception as _e:\n"
        "        print('Yedekleme uyarisi:', _e)"
    )
    g3d = g3d.replace(
        "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)",
        kayit_kodu
    )
    open("app/gradio_3dgen.py", "w", encoding="utf-8").write(g3d)

studio_2d_lines = [
    "import os, sys, gc, time, random, torch, rembg, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import gradio as gr",
    "from diffusers import AutoPipelineForText2Image",
    "",
    "pipe_2d = None",
    "rembg_sess = None",
    "",
    "def _get_rembg_session():",
    "    global rembg_sess",
    "    if rembg_sess is None:",
    "        try:",
    "            from scripts.utils import session as _s",
    "            if _s is not None: rembg_sess = _s",
    "        except Exception:",
    "            pass",
    "    if rembg_sess is None:",
    "        try: rembg_sess = rembg.new_session('u2net', providers=['CPUExecutionProvider'])",
    "        except Exception: rembg_sess = rembg.new_session('u2net')",
    "    return rembg_sess",
    "",
    "def _3d_modelleri_cpuya_al():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except Exception: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()",
    "",
    "def uret_2d_gorsel(prompt, neg_prompt, steps, cfg, bg_sil):",
    "    global pipe_2d",
    "    try:",
    "        _3d_modelleri_cpuya_al()",
    "        if pipe_2d is None:",
    "            pipe_2d = AutoPipelineForText2Image.from_pretrained(",
    "                'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16'",
    "            )",
    "        pipe_2d.to('cuda')",
    "        seed = random.randint(1, 99999999)",
    "        gen = torch.Generator(device='cuda').manual_seed(seed)",
    "        with torch.inference_mode():",
    "            img = pipe_2d(",
    "                prompt=prompt + ', 3d asset, isometric, front view, centered, clean background, high detail',",
    "                negative_prompt=neg_prompt,",
    "                num_inference_steps=int(steps),",
    "                guidance_scale=float(cfg),",
    "                generator=gen",
    "            ).images[0]",
    "        pipe_2d.to('cpu')",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        if bg_sil:",
    "            img = rembg.remove(img, session=_get_rembg_session())",
    f"        kayit_yolu = f'{WORK_DIR}/uretilen_2d_gorseller/gorsel_{{int(time.time())}}_{{seed}}.png'",
    "        img.save(kayit_yolu)",
    "        return img, f'Basarili (Seed: {seed}) -> {kayit_yolu}'",
    "    except Exception as e:",
    "        import traceback",
    "        traceback.print_exc()",
    "        return None, f'Hata: {str(e)}'",
    "",
    "def sadece_arkaplan_sil(img):",
    "    if img is None: return None, 'Gorsel yuklenmedi'",
    "    try:",
    "        out = rembg.remove(img, session=_get_rembg_session())",
    f"        kayit_yolu = f'{WORK_DIR}/uretilen_2d_gorseller/temiz_{{int(time.time())}}.png'",
    "        out.save(kayit_yolu)",
    "        return out, kayit_yolu",
    "    except Exception as e:",
    "        return None, f'Hata: {str(e)}'",
    "",
    "def create_2d_tab():",
    "    with gr.Tab('🎨 2D Görsel Üret & Hazırla (3D İçin)'):",
    "        gr.Markdown('### 1. Metinden 3D Uyumlu 2D Nesne Üret veya Hazır Fotoğrafın Arka Planını Sil')",
    "        with gr.Row():",
    "            with gr.Column():",
    "                prompt_in = gr.Textbox(label='2D Nesne Tarifi (İngilizce)', value='a futuristic helmet, game asset, front view')",
    "                neg_in = gr.Textbox(label='Negatif Prompt', value='blurry, cropped, low quality, multiple objects, complex background')",
    "                with gr.Row():",
    "                    steps_in = gr.Slider(1, 8, value=4, step=1, label='Adım Sayısı (2-4 idealdir)')",
    "                    cfg_in = gr.Slider(0.0, 3.0, value=0.0, step=0.5, label='CFG Scale (Turbo için 0.0)')",
    "                bg_chk = gr.Checkbox(value=True, label='Üretilen Görselin Arka Planını Otomatik Sil (Rembg)')",
    "                btn_uret = gr.Button('🎨 2D Görsel Üret (Her Tıklamada Yeni Varyasyon)', variant='primary')",
    "                gr.Markdown('---')",
    "                yukle_in = gr.Image(type='pil', label='Veya Kendi 2D Fotoğrafını Yükle (Arka Plan Silmek İçin)')",
    "                btn_sil = gr.Button('✂️ Sadece Arka Planı Sil')",
    "            with gr.Column():",
    "                out_img = gr.Image(type='pil', label='Hazırlanan 2D Görsel (Sağ üstten indirip 3D sekmesine sürükleyebilirsin)')",
    "                out_path = gr.Textbox(label='Durum ve Diskteki Kayıt Yolu', interactive=False)",
    "        btn_uret.click(uret_2d_gorsel, inputs=[prompt_in, neg_in, steps_in, cfg_in, bg_chk], outputs=[out_img, out_path])",
    "        btn_sil.click(sadece_arkaplan_sil, inputs=[yukle_in], outputs=[out_img, out_path])"
]
open("studio_2d_patch.py", "w", encoding="utf-8").write("\n".join(studio_2d_lines) + "\n")

if os.path.exists("gradio_app.py"):
    ga = open("gradio_app.py", "r", encoding="utf-8").read()
    ga = ga.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
    if "studio_2d_patch" not in ga:
        ga = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\nimport studio_2d_patch\n" + ga
        ga = ga.replace(
            "create_3d_ui(",
            "studio_2d_patch.create_2d_tab()\n        create_3d_ui("
        )
    ga = re.sub(
        r"demo(?:\.queue\(\))*\.launch\([^\)]*\)",
        "demo.queue().launch(share=True, server_name='127.0.0.1', server_port=7860)",
        ga
    )
    open("gradio_app.py", "w", encoding="utf-8").write(ga)

if os.path.exists("app/gradio_3dgen.py"):
    kh = open("app/gradio_3dgen.py", "r", encoding="utf-8").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
    if "oom_patch" not in kh:
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
    open("app/gradio_3dgen.py", "w", encoding="utf-8").write(kh)

print("8. Port 7860 Temizleniyor ve Kopmayan Cloudflare Tunnel Kuruluyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin], check=True)
    subprocess.run(["chmod", "+x", cf_bin], check=True)

cf_log = open(f"{WORK_DIR}/cf.log", "w")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(5)
cf_icerik = open(f"{WORK_DIR}/cf.log", "r").read()
linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
if linkler:
    print("\n" + "="*70)
    print(f"🌐 KOPMAYAN CLOUDFLARE LINKIN (2D + 3D STUDIO): {linkler[0]}")
    print("⚠️ DIKKAT: Linke tiklamadan once asagida 'Running on local URL' yazmasini (20 sn) bekle!")
    print("="*70 + "\n")

print("🚀 9. 2D + 3D BIRLESIK STUDYO CANLI LOGLARLA BASLATILIYOR...")
!cd {u3d_dir} && PYTHONPATH={u3d_dir} PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python -u gradio_app.py

In [ ]:
import os, sys, re, time, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.makedirs("/kaggle/working/uretilen_3d_modeller", exist_ok=True)
os.makedirs("/kaggle/working/uretilen_2d_gorseller", exist_ok=True)

print("1. Unique3D Projesi Hazirlaniyor...")
if not os.path.exists("/kaggle/working/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir("/kaggle/working/Unique3D")

try:
    import nvdiffrast, pytorch3d
    print("2-3-4. PyTorch ve C++ 3D Motorlari zaten kurulu, derleme atlaniyor...")
except ImportError:
    print("2. Orijinal GPU PyTorch (CUDA 12.4) Kuruluyor...")
    os.system("pip uninstall -y xformers torch torchvision torchaudio pytorch3d nvdiffrast")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

    print("3. Gerekli Kutuphaneler Yukleniyor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel", "fire", "rembg", "onnxruntime", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard", "huggingface_hub"])

    print("4. C++ 3D Motorlari (Nvdiffrast ve PyTorch3D) Derleniyor...")
    shutil.rmtree("nvdiffrast", ignore_errors=True)
    shutil.rmtree("nvdiffrast_repo", ignore_errors=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Indiriliyor...")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    shutil.rmtree("ckpt/img2mvimg", ignore_errors=True)
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

print("6. 4 Kanalli U-Net (unet_4ch) Mimarisi Kuruluyor...")
img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"
shutil.rmtree(unet_4ch_yolu, ignore_errors=True)
os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)

ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
if os.path.exists(ana_cfg_path):
    ana_cfg = json.loads(open(ana_cfg_path, "r").read())
    ana_cfg["in_channels"] = 4
    ana_cfg.pop("num_class_embeds", None)
    ana_cfg["addition_embed_type"] = None
    open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))

for d in os.listdir(img2mvimg_yolu):
    if d != "unet" and not os.path.exists(unet_4ch_yolu + "/" + d):
        os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)

for d in os.listdir(img2mvimg_yolu + "/unet"):
    if d != "config.json" and not os.path.exists(unet_4ch_yolu + "/unet/" + d):
        os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("7. Kod Agaci Sifirlaniyor ve Tum 3D + 2D Zirhlar Isleniyor...")
for klasor in ["app", "scripts", "mesh_reconstruction", "custum_3d_diffusion"]:
    shutil.rmtree(klasor, ignore_errors=True)
os.system("git reset --hard HEAD")
os.system("git clean -fd -e ckpt -e ckpt/*")

oom_lines = [
    "import os, gc, torch, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    u_kod = open("scripts/utils.py", "r", encoding="utf-8").read()
    u_kod = u_kod.replace("providers=providers", "providers=['CPUExecutionProvider']")
    u_ek = [
        "",
        "import rembg",
        "try:",
        "    session = rembg.new_session(providers=['CPUExecutionProvider'])",
        "except Exception:",
        "    session = rembg.new_session()",
        "",
        "def simple_preprocess(input_image, rembg_session=None, background_color=255):",
        "    if 'remove_backgroud' in globals():",
        "        return remove_backgroud(input_image, rembg_session=rembg_session or session, background_color=background_color)",
        "    return input_image"
    ]
    if "def simple_preprocess" not in u_kod:
        u_kod = u_kod + "\n" + "\n".join(u_ek) + "\n"
    open("scripts/utils.py", "w", encoding="utf-8").write(u_kod)

if os.path.exists("scripts/project_mesh.py"):
    pm_satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_pm = []
    for s in pm_satirlar:
        if "assert " in s and "cos_angles" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_pm.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_pm.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_pm)

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale', 'studio_2d_patch']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer', 'pipe_2d']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
mvc = mvc.replace("_ = multiview_color_projection(", "# _ = multiview_color_projection(")
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    mod_2d = sys.modules.get('studio_2d_patch')",
    "    if mod_2d and getattr(mod_2d, 'pipe_2d', None) is not None:",
    "        try: mod_2d.pipe_2d.to('cpu')",
    "        except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

if os.path.exists("app/gradio_3dgen.py"):
    g3d = open("app/gradio_3dgen.py", "r", encoding="utf-8").read()
    # 3D uretim baslamadan once modellerin GPU'da oldugundan emin ol:
    g3d = g3d.replace(
        "def generate3dv2(",
        "def generate3dv2(*args, **kwargs):\n    import sys\n    m = sys.modules.get('scripts.multiview_inference')\n    if m and hasattr(m, '_yukle_vram'): m._yukle_vram()\n    return _orig_generate3dv2(*args, **kwargs)\ndef _orig_generate3dv2("
    )
    kayit_kodu = (
        "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)\n"
        "    try:\n"
        "        import time as _t\n"
        "        _cikti = f'/kaggle/working/uretilen_3d_modeller/model_{int(_t.time())}.glb'\n"
        "        new_meshes.export(_cikti)\n"
        "        print(f'\\n✅ 3D MODEL DISKE YEDEKLENDI: {_cikti}\\n')\n"
        "    except Exception as _e:\n"
        "        print('Yedekleme uyarisi:', _e)"
    )
    g3d = g3d.replace(
        "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)",
        kayit_kodu
    )
    open("app/gradio_3dgen.py", "w", encoding="utf-8").write(g3d)

# 8. KILITLENMEYEN 2D GORSEL URETIM VE ARKA PLAN SILME MODULU
studio_2d_lines = [
    "import os, sys, gc, time, random, torch, rembg, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import gradio as gr",
    "from diffusers import AutoPipelineForText2Image",
    "",
    "pipe_2d = None",
    "rembg_sess = None",
    "",
    "def _get_rembg_session():",
    "    global rembg_sess",
    "    if rembg_sess is None:",
    "        try:",
    "            from scripts.utils import session as _s",
    "            if _s is not None: rembg_sess = _s",
    "        except Exception:",
    "            pass",
    "    if rembg_sess is None:",
    "        try: rembg_sess = rembg.new_session('u2net', providers=['CPUExecutionProvider'])",
    "        except Exception: rembg_sess = rembg.new_session('u2net')",
    "    return rembg_sess",
    "",
    "def _3d_modelleri_cpuya_al():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except Exception: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()",
    "",
    "def uret_2d_gorsel(prompt, neg_prompt, steps, cfg, bg_sil):",
    "    global pipe_2d",
    "    try:",
    "        _3d_modelleri_cpuya_al()",
    "        if pipe_2d is None:",
    "            pipe_2d = AutoPipelineForText2Image.from_pretrained(",
    "                'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16'",
    "            )",
    "        pipe_2d.to('cuda')",
    "        seed = random.randint(1, 99999999)",
    "        gen = torch.Generator(device='cuda').manual_seed(seed)",
    "        with torch.inference_mode():",
    "            img = pipe_2d(",
    "                prompt=prompt + ', 3d asset, isometric, front view, centered, clean background, high detail',",
    "                negative_prompt=neg_prompt,",
    "                num_inference_steps=int(steps),",
    "                guidance_scale=float(cfg),",
    "                generator=gen",
    "            ).images[0]",
    "        pipe_2d.to('cpu')",
    "        gc.collect()",
    "        torch.cuda.empty_cache()",
    "        if bg_sil:",
    "            img = rembg.remove(img, session=_get_rembg_session())",
    "        kayit_yolu = f'/kaggle/working/uretilen_2d_gorseller/gorsel_{int(time.time())}_{seed}.png'",
    "        img.save(kayit_yolu)",
    "        return img, f'Basarili (Seed: {seed}) -> {kayit_yolu}'",
    "    except Exception as e:",
    "        import traceback",
    "        traceback.print_exc()",
    "        return None, f'Hata: {str(e)}'",
    "",
    "def sadece_arkaplan_sil(img):",
    "    if img is None: return None, 'Gorsel yuklenmedi'",
    "    try:",
    "        out = rembg.remove(img, session=_get_rembg_session())",
    "        kayit_yolu = f'/kaggle/working/uretilen_2d_gorseller/temiz_{int(time.time())}.png'",
    "        out.save(kayit_yolu)",
    "        return out, kayit_yolu",
    "    except Exception as e:",
    "        return None, f'Hata: {str(e)}'",
    "",
    "def create_2d_tab():",
    "    with gr.Tab('🎨 2D Görsel Üret & Hazırla (3D İçin)'):",
    "        gr.Markdown('### 1. Metinden 3D Uyumlu 2D Nesne Üret veya Hazır Fotoğrafın Arka Planını Sil')",
    "        with gr.Row():",
    "            with gr.Column():",
    "                prompt_in = gr.Textbox(label='2D Nesne Tarifi (İngilizce)', value='a futuristic helmet, game asset, front view')",
    "                neg_in = gr.Textbox(label='Negatif Prompt', value='blurry, cropped, low quality, multiple objects, complex background')",
    "                with gr.Row():",
    "                    steps_in = gr.Slider(1, 8, value=4, step=1, label='Adım Sayısı (2-4 idealdir)')",
    "                    cfg_in = gr.Slider(0.0, 3.0, value=0.0, step=0.5, label='CFG Scale (Turbo için 0.0)')",
    "                bg_chk = gr.Checkbox(value=True, label='Üretilen Görselin Arka Planını Otomatik Sil (Rembg)')",
    "                btn_uret = gr.Button('🎨 2D Görsel Üret (Her Tıklamada Yeni Varyasyon)', variant='primary')",
    "                gr.Markdown('---')",
    "                yukle_in = gr.Image(type='pil', label='Veya Kendi 2D Fotoğrafını Yükle (Arka Plan Silmek İçin)')",
    "                btn_sil = gr.Button('✂️ Sadece Arka Planı Sil')",
    "            with gr.Column():",
    "                out_img = gr.Image(type='pil', label='Hazırlanan 2D Görsel (Sağ üstten indirip 3D sekmesine sürükleyebilirsin)')",
    "                out_path = gr.Textbox(label='Durum ve Diskteki Kayıt Yolu', interactive=False)",
    "        btn_uret.click(uret_2d_gorsel, inputs=[prompt_in, neg_in, steps_in, cfg_in, bg_chk], outputs=[out_img, out_path])",
    "        btn_sil.click(sadece_arkaplan_sil, inputs=[yukle_in], outputs=[out_img, out_path])"
]
open("studio_2d_patch.py", "w", encoding="utf-8").write("\n".join(studio_2d_lines) + "\n")

if os.path.exists("gradio_app.py"):
    ga = open("gradio_app.py", "r", encoding="utf-8").read()
    ga = ga.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
    if "studio_2d_patch" not in ga:
        ga = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\nimport studio_2d_patch\n" + ga
        ga = ga.replace(
            "create_3d_ui(",
            "studio_2d_patch.create_2d_tab()\n        create_3d_ui("
        )
    ga = re.sub(
        r"demo(?:\.queue\(\))*\.launch\([^\)]*\)",
        "demo.queue().launch(share=True, server_name='127.0.0.1', server_port=7860)",
        ga
    )
    open("gradio_app.py", "w", encoding="utf-8").write(ga)

if os.path.exists("app/gradio_3dgen.py"):
    kh = open("app/gradio_3dgen.py", "r", encoding="utf-8").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
    if "oom_patch" not in kh:
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
    open("app/gradio_3dgen.py", "w", encoding="utf-8").write(kh)

print("8. Port 7860 Temizleniyor ve Kopmayan Cloudflare Tunnel Kuruluyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/kaggle/working/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/kaggle/working/cloudflared"], check=True)

cf_log = open("/kaggle/working/cf.log", "w")
subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(5)
cf_icerik = open("/kaggle/working/cf.log", "r").read()
linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
if linkler:
    print("\n" + "="*70)
    print(f"🌐 KOPMAYAN CLOUDFLARE LINKIN (2D + 3D STUDIO): {linkler[0]}")
    print("⚠️ DIKKAT: Linke tiklamadan once asagida 'Running on local URL' yazmasini (20 sn) bekle!")
    print("="*70 + "\n")

print("🚀 9. 2D + 3D BIRLESIK STUDYO TAM ZIRHLI BASLATILIYOR...")
os.chdir("/kaggle/working/Unique3D")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python /kaggle/working/Unique3D/gradio_app.py

In [ ]:
import os, sys, re, time, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.makedirs("/kaggle/working/uretilen_3d_modeller", exist_ok=True)

print("1. Unique3D Projesi Hazirlaniyor...")
if not os.path.exists("/kaggle/working/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir("/kaggle/working/Unique3D")

try:
    import nvdiffrast, pytorch3d
    print("2-3-4. PyTorch ve C++ 3D Motorlari zaten kurulu, derleme atlaniyor...")
except ImportError:
    print("2. Orijinal GPU PyTorch (CUDA 12.4) Kuruluyor...")
    os.system("pip uninstall -y xformers torch torchvision torchaudio pytorch3d nvdiffrast")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

    print("3. Gerekli Kutuphaneler Yukleniyor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel", "fire", "rembg", "onnxruntime", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard", "huggingface_hub"])

    print("4. C++ 3D Motorlari (Nvdiffrast ve PyTorch3D) Derleniyor...")
    shutil.rmtree("nvdiffrast", ignore_errors=True)
    shutil.rmtree("nvdiffrast_repo", ignore_errors=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Indiriliyor...")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    shutil.rmtree("ckpt/img2mvimg", ignore_errors=True)
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

print("6. 4 Kanalli U-Net (unet_4ch) Mimarisi Kuruluyor...")
img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"
shutil.rmtree(unet_4ch_yolu, ignore_errors=True)
os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)

ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
if os.path.exists(ana_cfg_path):
    ana_cfg = json.loads(open(ana_cfg_path, "r").read())
    ana_cfg["in_channels"] = 4
    ana_cfg.pop("num_class_embeds", None)
    ana_cfg["addition_embed_type"] = None
    open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))

for d in os.listdir(img2mvimg_yolu):
    if d != "unet" and not os.path.exists(unet_4ch_yolu + "/" + d):
        os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)

for d in os.listdir(img2mvimg_yolu + "/unet"):
    if d != "config.json" and not os.path.exists(unet_4ch_yolu + "/unet/" + d):
        os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("7. Kod Agaci Sifirlaniyor ve Tum Zirhlar + Zaman Asimi Korumasi Isleniyor...")
for klasor in ["app", "scripts", "mesh_reconstruction", "custum_3d_diffusion"]:
    shutil.rmtree(klasor, ignore_errors=True)
os.system("git reset --hard HEAD")
os.system("git clean -fd -e ckpt -e ckpt/*")

oom_lines = [
    "import os, gc, torch",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    u_kod = open("scripts/utils.py", "r", encoding="utf-8").read()
    u_kod = u_kod.replace("providers=providers", "providers=['CPUExecutionProvider']")
    u_ek = [
        "",
        "import rembg",
        "try:",
        "    session = rembg.new_session(providers=['CPUExecutionProvider'])",
        "except Exception:",
        "    session = rembg.new_session()",
        "",
        "def simple_preprocess(input_image, rembg_session=None, background_color=255):",
        "    if 'remove_backgroud' in globals():",
        "        return remove_backgroud(input_image, rembg_session=rembg_session or session, background_color=background_color)",
        "    return input_image"
    ]
    if "def simple_preprocess" not in u_kod:
        u_kod = u_kod + "\n" + "\n".join(u_ek) + "\n"
    open("scripts/utils.py", "w", encoding="utf-8").write(u_kod)

if os.path.exists("scripts/project_mesh.py"):
    pm_satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_pm = []
    for s in pm_satirlar:
        if "assert " in s and "cos_angles" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_pm.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_pm.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_pm)

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
mvc = mvc.replace("_ = multiview_color_projection(", "# _ = multiview_color_projection(")
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

# Otomatik .glb yedekleyici (baglanti kopsa bile model kaybolmaz)
if os.path.exists("app/gradio_3dgen.py"):
    g3d = open("app/gradio_3dgen.py", "r", encoding="utf-8").read()
    kayit_kodu = (
        "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)\n"
        "    try:\n"
        "        import time as _t\n"
        "        _cikti = f'/kaggle/working/uretilen_3d_modeller/model_{int(_t.time())}.glb'\n"
        "        new_meshes.export(_cikti)\n"
        "        print(f'\\n✅ 3D MODEL DISKE YEDEKLENDI: {_cikti}\\n')\n"
        "    except Exception as _e:\n"
        "        print('Yedekleme uyarisi:', _e)"
    )
    g3d = g3d.replace(
        "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)",
        kayit_kodu
    )
    open("app/gradio_3dgen.py", "w", encoding="utf-8").write(g3d)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = re.sub(
            r"demo\.launch\([^\)]*\)",
            "demo.queue().launch(share=True, server_name='0.0.0.0', server_port=7860, max_threads=40)",
            kh
        )
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("8. Kopmayan Cloudflare Tunnel Kuruluyor...")
if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/kaggle/working/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/kaggle/working/cloudflared"], check=True)

os.system("pkill -f cloudflared")
cf_log = open("/kaggle/working/cf.log", "w")
subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(5)
cf_icerik = open("/kaggle/working/cf.log", "r").read()
linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
if linkler:
    print("\n" + "="*65)
    print(f"🌐 KOPMAYAN CLOUDFLARE LINKIN (BUNA TIKLA): {linkler[0]}")
    print("="*65 + "\n")

print("🚀 9. STUDYO TAM ZIRHLI VE ZAMAN ASIMI KORUMALI BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, re, time, subprocess

os.chdir("/kaggle/working/Unique3D")

print("1. cached_download import sirasi duzeltiliyor...")
# studio_2d_patch.py dosyasinin en tepesine diffusers'tan ONCE yamayi ekle:
if os.path.exists("studio_2d_patch.py"):
    s2d = open("studio_2d_patch.py", "r", encoding="utf-8").read()
    yama = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n"
    if "cached_download" not in s2d:
        s2d = yama + s2d
        open("studio_2d_patch.py", "w", encoding="utf-8").write(s2d)

# gradio_app.py icinde de once yama, sonra studio_2d_patch import edilsin:
if os.path.exists("gradio_app.py"):
    ga = open("gradio_app.py", "r", encoding="utf-8").read()
    ga = ga.replace(
        "import oom_patch, huggingface_hub, studio_2d_patch\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download",
        "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\nimport studio_2d_patch"
    )
    ga = re.sub(
        r"demo(?:\.queue\(\))*\.launch\([^\)]*\)",
        "demo.queue().launch(share=True, server_name='127.0.0.1', server_port=7860)",
        ga
    )
    open("gradio_app.py", "w", encoding="utf-8").write(ga)

os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
cf_log = open("/kaggle/working/cf.log", "w")
subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(5)
linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/kaggle/working/cf.log", "r").read())
if linkler:
    print("\n" + "="*70)
    print(f"🌐 KOPMAYAN CLOUDFLARE LINKIN (2D + 3D STUDIO): {linkler[0]}")
    print("⚠️ DIKKAT: Linke tiklamadan once asagida 'Running on local URL' yazmasini (20 sn) bekle!")
    print("="*70 + "\n")

print("🚀 2. Studyo Baslatiliyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

print("1. Unique3D Projesi Hazirlaniyor...")
if not os.path.exists("/kaggle/working/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir("/kaggle/working/Unique3D")

try:
    import nvdiffrast, pytorch3d
    print("2-3-4. PyTorch ve C++ 3D Motorlari zaten kurulu, derleme atlaniyor...")
except ImportError:
    print("2. Orijinal GPU PyTorch (CUDA 12.4) Kuruluyor...")
    os.system("pip uninstall -y xformers torch torchvision torchaudio pytorch3d nvdiffrast")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

    print("3. Gerekli Kutuphaneler Yukleniyor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel", "fire", "rembg", "onnxruntime", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard", "huggingface_hub"])

    print("4. C++ 3D Motorlari (Nvdiffrast ve PyTorch3D) Derleniyor...")
    shutil.rmtree("nvdiffrast", ignore_errors=True)
    shutil.rmtree("nvdiffrast_repo", ignore_errors=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Indiriliyor...")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    shutil.rmtree("ckpt/img2mvimg", ignore_errors=True)
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

print("6. 4 Kanalli U-Net (unet_4ch) Mimarisi Kuruluyor...")
img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"
shutil.rmtree(unet_4ch_yolu, ignore_errors=True)
os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)

ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
if os.path.exists(ana_cfg_path):
    ana_cfg = json.loads(open(ana_cfg_path, "r").read())
    ana_cfg["in_channels"] = 4
    ana_cfg.pop("num_class_embeds", None)
    ana_cfg["addition_embed_type"] = None
    open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))

for d in os.listdir(img2mvimg_yolu):
    if d != "unet" and not os.path.exists(unet_4ch_yolu + "/" + d):
        os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)

for d in os.listdir(img2mvimg_yolu + "/unet"):
    if d != "config.json" and not os.path.exists(unet_4ch_yolu + "/unet/" + d):
        os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("7. Kod Agaci Temizleniyor ve Tum Zirhlar Isleniyor...")
for klasor in ["app", "scripts", "mesh_reconstruction", "custum_3d_diffusion"]:
    shutil.rmtree(klasor, ignore_errors=True)
os.system("git reset --hard HEAD")
os.system("git clean -fd -e ckpt -e ckpt/*")

oom_lines = [
    "import os, gc, torch",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    u_kod = open("scripts/utils.py", "r", encoding="utf-8").read()
    u_kod = u_kod.replace("providers=providers", "providers=['CPUExecutionProvider']")
    u_ek = [
        "",
        "import rembg",
        "try:",
        "    session = rembg.new_session(providers=['CPUExecutionProvider'])",
        "except Exception:",
        "    session = rembg.new_session()",
        "",
        "def simple_preprocess(input_image, rembg_session=None, background_color=255):",
        "    if 'remove_backgroud' in globals():",
        "        return remove_backgroud(input_image, rembg_session=rembg_session or session, background_color=background_color)",
        "    return input_image"
    ]
    if "def simple_preprocess" not in u_kod:
        u_kod = u_kod + "\n" + "\n".join(u_ek) + "\n"
    open("scripts/utils.py", "w", encoding="utf-8").write(u_kod)

if os.path.exists("scripts/project_mesh.py"):
    pm_satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_pm = []
    for s in pm_satirlar:
        if "assert " in s and "cos_angles" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_pm.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_pm.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_pm)

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
mvc = mvc.replace("_ = multiview_color_projection(", "# _ = multiview_color_projection(")
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("🚀 8. STUDYO TAM ZIRHLI OLARAK BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, re, subprocess, time

os.chdir("/kaggle/working/Unique3D")

print("1. Gradio WebSocket Queue (Zaman Asimi Korumasi) ve Otomatik Kayit Isleniyor...")
os.makedirs("/kaggle/working/uretilen_3d_modeller", exist_ok=True)

# 1. Üretilen modelleri bağlantı kopsa bile /kaggle/working/uretilen_3d_modeller içine yedekle:
if os.path.exists("app/gradio_3dgen.py"):
    g3d = open("app/gradio_3dgen.py", "r", encoding="utf-8").read()
    kayit_kodu = (
        "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)\n"
        "    try:\n"
        "        import time as _t\n"
        "        _cikti = f'/kaggle/working/uretilen_3d_modeller/model_{int(_t.time())}.glb'\n"
        "        new_meshes.export(_cikti)\n"
        "        print(f'\\n✅ 3D MODEL DISKE YEDEKLENDI: {_cikti}\\n')\n"
        "    except Exception as _e:\n"
        "        print('Yedekleme uyarisi:', _e)"
    )
    if "uretilen_3d_modeller" not in g3d:
        g3d = g3d.replace(
            "    new_meshes = geo_reconstruct(rgb_pils, None, front_pil, do_refine=do_refine, predict_normal=True, expansion_weight=expansion_weight, init_type=init_type)",
            kayit_kodu
        )
        open("app/gradio_3dgen.py", "w", encoding="utf-8").write(g3d)

# 2. gradio_app.py içine .queue() ve sabit 7860 portu ekle (WebSocket keep-alive):
if os.path.exists("gradio_app.py"):
    ga = open("gradio_app.py", "r", encoding="utf-8").read()
    # Eski .launch(...) çağrısını bulup .queue().launch(...) ile değiştir
    ga = re.sub(
        r"demo\.launch\([^\)]*\)",
        "demo.queue().launch(share=True, server_name='0.0.0.0', server_port=7860, max_threads=40)",
        ga
    )
    open("gradio_app.py", "w", encoding="utf-8").write(ga)

print("2. Kopmayan Cloudflare Tunnel Kuruluyor...")
if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/kaggle/working/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/kaggle/working/cloudflared"], check=True)

os.system("pkill -f cloudflared")
cf_log = open("/kaggle/working/cf.log", "w")
subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)

time.sleep(5)
cf_icerik = open("/kaggle/working/cf.log", "r").read()
linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
if linkler:
    print("\n" + "="*65)
    print(f"🌐 KOPMAYAN CLOUDFLARE LINKIN (BUNA TIKLA): {linkler[0]}")
    print("="*65 + "\n")

print("🚀 3. STUDYO BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("💥 Çakışmaya sebep olan gereksiz JAX kütüphanesi sistemden tamamen siliniyor...")
os.system(f"{sys.executable} -m pip uninstall -y jax jaxlib >/dev/null 2>&1")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu hatasız başlatılıyor (JAX çakışması yok, modeller manuel olarak yerinde!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=open(gradio_log_path, "w"),
    stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in log_icerik or "OSError" in log_icerik or "SyntaxError" in log_icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, shutil

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧹 Kaggle diski temizleniyor...")
os.system("rm -rf /root/.cache/huggingface/hub/* ~/.cache/huggingface/hub/*")

print("📥 Doğru yoldan (unet alt klasöründen) model ağırlığı indiriliyor...")
from huggingface_hub import hf_hub_download

os.makedirs("ckpt/img2mvimg/unet", exist_ok=True)

try:
    # Doğru dosya yolundan indiriyoruz
    file_path = hf_hub_download(
        repo_id="Wuvin/Unique3D",
        filename="ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors",
        repo_type="space"
    )
    if file_path and os.path.exists(file_path):
        shutil.copy(file_path, "ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")
        print("✅ Ağırlık dosyası unet klasörüne başarıyla yerleştirildi!")
except Exception as e:
    print(f"⚠️ İndirme detayı: {e}")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🔍 Kod içindeki harici model çağrıları yerel yola (ckpt/unet_4ch) sabitleniyor...")

# Proje içerisindeki tüm .py dosyalarında lambdalabs çağrısını yerel unet_4ch klasörüne yönlendiriyoruz
unet_4ch_yolu = os.path.abspath("ckpt/unet_4ch")
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    txt = f.read()
                if "lambdalabs/sd-image-variations-diffusers" in txt:
                    new_txt = txt.replace("lambdalabs/sd-image-variations-diffusers", unet_4ch_yolu)
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(new_txt)
                    print(f"✅ Güncellenen dosya: {file}")
            except Exception:
                pass

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu nihai olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🛡️ Harici model çağrılarını anında yerele yönlendiren global yama (Monkey-Patch) yazılıyor...")

# gradio_app.py dosyasının en başına global yönlendiriciyi ekliyoruz
app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    patch_code = """
# --- GLOBAL LAMBDALABS REDIRECT PATCH ---
import os
import diffusers
from diffusers import DiffusionPipeline

_old_from_pretrained = DiffusionPipeline.from_pretrained
def _new_from_pretrained(pretrained_model_name_or_path, *args, **kwargs):
    if pretrained_model_name_or_path == "lambdalabs/sd-image-variations-diffusers" or str(pretrained_model_name_or_path).endswith("sd-image-variations-diffusers"):
        pretrained_model_name_or_path = os.path.abspath("ckpt/unet_4ch")
    return _old_from_pretrained(pretrained_model_name_or_path, *args, **kwargs)

DiffusionPipeline.from_pretrained = _new_from_pretrained
# ----------------------------------------
"""
    if "GLOBAL LAMBDALABS REDIRECT PATCH" not in content:
        with open(app_path, "w", encoding="utf-8") as f:
            f.write(patch_code + "\n" + content)
        print("✅ Global yama gradio_app.py dosyasına başarıyla entegre edildi!")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu hatasız olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("")
print("======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================")
print("")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🛡️ Boru hattı kayıt yaması (NameError düzeltilmiş haliyle) uygulanıyor...")

app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Eski yamayı güvenli bir şekilde temizle
    if "# --- PIPELINE REGISTER PATCH ---" in content:
        parts = content.split("# -------------------------------------------------")
        if len(parts) > 1:
            content = parts[-1]
    
    patch_code = """# --- PIPELINE REGISTER PATCH ---
import sys
import os

sys.path.insert(0, os.path.abspath("."))
try:
    import diffusers
    from custum_3d_diffusion.custum_pipeline.unifield_pipeline_img2mvimg import StableDiffusionImage2MVCustomPipeline
    from custum_3d_diffusion.custum_pipeline.unifield_pipeline_img2img import StableDiffusionImageCustomPipeline
    
    setattr(diffusers, "StableDiffusionImage2MVCustomPipeline", StableDiffusionImage2MVCustomPipeline)
    setattr(diffusers, "StableDiffusionImageCustomPipeline", StableDiffusionImageCustomPipeline)
except Exception as e:
    print(f"Pipeline register warning: {e}")

import huggingface_hub
# -------------------------------------------------
"""
    with open(app_path, "w", encoding="utf-8") as f:
        f.write(patch_code + "\n" + content)
    print("✅ Boru hattı kayıt yaması başarıyla eklendi!")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu nihai olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🛡️ Diffusers config yükleyicisi (load_config) global olarak yamalanıyor...")

app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    if "# --- CONFIG REDIRECT PATCH ---" in content:
        parts = content.split("# -------------------------------------------------")
        if len(parts) > 1:
            content = parts[-1]
    
    patch_code = """# --- CONFIG REDIRECT PATCH ---
import sys
import os

sys.path.insert(0, os.path.abspath("."))
try:
    import diffusers
    from custum_3d_diffusion.custum_pipeline.unifield_pipeline_img2mvimg import StableDiffusionImage2MVCustomPipeline
    from custum_3d_diffusion.custum_pipeline.unifield_pipeline_img2img import StableDiffusionImageCustomPipeline
    
    setattr(diffusers, "StableDiffusionImage2MVCustomPipeline", StableDiffusionImage2MVCustomPipeline)
    setattr(diffusers, "StableDiffusionImageCustomPipeline", StableDiffusionImageCustomPipeline)
    
    # load_config fonksiyonunu yamalayarak lambdalabs çağrılarını yerel yola kilitliyoruz
    from diffusers.configuration_utils import ConfigMixin
    _old_load_config = ConfigMixin.load_config
    
    @classmethod
    def _new_load_config(cls, pretrained_model_name_or_path, *args, **kwargs):
        if pretrained_model_name_or_path == "lambdalabs/sd-image-variations-diffusers" or str(pretrained_model_name_or_path).endswith("sd-image-variations-diffusers"):
            pretrained_model_name_or_path = os.path.abspath("ckpt/unet_4ch")
        return _old_load_config(pretrained_model_name_or_path, *args, **kwargs)
        
    ConfigMixin.load_config = _new_load_config
except Exception as e:
    print(f"Config patch warning: {e}")
# -------------------------------------------------
"""
    with open(app_path, "w", encoding="utf-8") as f:
        f.write(patch_code + "\n" + content)
    print("✅ Config yönlendirme yaması başarıyla eklendi!")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu nihai olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🔍 Kod tabanındaki tüm harici lambdalabs referansları fiziksel olarak temizleniyor...")

# Projedeki tüm Python ve yapılandırma dosyalarında kelime araması yapıp değiştiriyoruz
hedef_klasor = os.path.abspath("ckpt/unet_4ch")
degisen_sayisi = 0

for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith((".py", ".json", ".yaml", ".yml", ".txt")):
            path = os.path.join(root, file)
            # Kendi eklediğimiz yama dosyasını ve git klasörünü es geç
            if "studio_2d_patch" in path or ".git" in path:
                continue
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                
                if "lambdalabs/sd-image-variations-diffusers" in content or "sd-image-variations-diffusers" in content:
                    new_content = content.replace("lambdalabs/sd-image-variations-diffusers", hedef_klasor)
                    new_content = new_content.replace("sd-image-variations-diffusers", hedef_klasor)
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(new_content)
                    degisen_sayisi += 1
            except Exception:
                pass

print(f"✅ Toplam {degisen_sayisi} dosya içindeki harici model yolu yerel yola (ckpt/unet_4ch) sabitlendi.")

# gradio_app.py başındaki hatalı yamaları temizleyip sadece tertemiz sunucu başlatıcıyı bırakıyoruz
app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()
    # Önceki yamaları temizle
    code = re.sub(r'# ---.*?PATCH ---.*?# ---------------------------------', '', code, flags=re.DOTALL)
    # weights_only ve diğer güvenli importlar kalsın
    patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
    if "weights_only" not in code:
        code = patch + code
    with open(app_path, "w", encoding="utf-8") as f:
        f.write(code)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo nihai ve hatasız olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("")
print("======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR VE ÇALIŞIYOR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================")
print("")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 gradio_app.py dosyası orijinal haline getirilip temizleniyor...")
os.system("git checkout gradio_app.py >/dev/null 2>&1")

# PyTorch 2.4+ uyumluluğu için güvenli torch.load yamasını ekliyoruz
app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()
    patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
    if "weights_only" not in code:
        with open(app_path, "w", encoding="utf-8") as f:
            f.write(patch + code)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Unique3D Stüdyo hatasız olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("")
print("======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR VE ÇALIŞIYOR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================")
print("")

In [ ]:
import os, sys, subprocess, time, re, shutil

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🔧 Hata veren unet_4ch klasörüne eksik olan ağırlık dosyaları kopyalanıp adlandırılıyor...")

# unet_4ch içine unet klasörünü garantileyelim
os.makedirs("ckpt/unet_4ch/unet", exist_ok=True)

# Elimizde nerede olursa olsun sağlam bir ağırlık dosyasını (safetensors veya bin) bulup unet_4ch içine kopyalıyoruz
kaynak_dosya = None
for aranan in [
    "ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors",
    "ckpt/img2mvimg/diffusion_pytorch_model.safetensors",
    "ckpt/img2mvimg/unet/diffusion_pytorch_model.bin",
    "ckpt/img2mvimg/diffusion_pytorch_model.bin"
]:
    if os.path.exists(aranan):
        kaynak_dosya = aranan
        break

if kaynak_dosya:
    # Hem .bin hem .safetensors olarak kopyalayalım ki Diffusers hangisini ararsa arasın bulsun!
    shutil.copy(kaynak_dosya, "ckpt/unet_4ch/diffusion_pytorch_model.bin")
    shutil.copy(kaynak_dosya, "ckpt/unet_4ch/diffusion_pytorch_model.safetensors")
    shutil.copy(kaynak_dosya, "ckpt/unet_4ch/unet/diffusion_pytorch_model.bin")
    shutil.copy(kaynak_dosya, "ckpt/unet_4ch/unet/diffusion_pytorch_model.safetensors")
    print(f"✅ '{kaynak_dosya}' dosyası unet_4ch içine başarıyla çoğaltıldı!")
else:
    print("⚠️ Hiçbir ağırlık dosyası bulunamadı, indiriliyor...")
    os.system("wget -q -O ckpt/unet_4ch/diffusion_pytorch_model.safetensors https://huggingface.co/spaces/Wuvin/Unique3D/resolve/main/ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")
    shutil.copy("ckpt/unet_4ch/diffusion_pytorch_model.safetensors", "ckpt/unet_4ch/unet/diffusion_pytorch_model.safetensors")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧹 Diski şişiren kopyalar temizleniyor...")
os.system("rm -f ckpt/unet_4ch/diffusion_pytorch_model.* ckpt/unet_4ch/unet/diffusion_pytorch_model.*")

print("🔗 Fiziksel kopyalama yerine Sıfır Maliyetli Kısayol (Symlink) bağlanıyor...")
os.makedirs("ckpt/unet_4ch/unet", exist_ok=True)

kaynak_dosya = None
for aranan in [
    "ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors",
    "ckpt/img2mvimg/diffusion_pytorch_model.safetensors",
    "ckpt/img2mvimg/unet/diffusion_pytorch_model.bin",
    "ckpt/img2mvimg/diffusion_pytorch_model.bin"
]:
    if os.path.exists(aranan):
        kaynak_dosya = os.path.abspath(aranan)
        break

if kaynak_dosya:
    # Symlink (kısayol) oluşturuyoruz - diskte yer kaplamaz, anında bağlanır
    for hedef in [
        "ckpt/unet_4ch/diffusion_pytorch_model.bin",
        "ckpt/unet_4ch/diffusion_pytorch_model.safetensors",
        "ckpt/unet_4ch/unet/diffusion_pytorch_model.bin",
        "ckpt/unet_4ch/unet/diffusion_pytorch_model.safetensors"
    ]:
        if os.path.lexists(hedef):
            os.remove(hedef)
        os.symlink(kaynak_dosya, hedef)
    print("✅ Ağırlık dosyaları symlink (kısayol) ile bağlandı!")
else:
    print("❌ Kaynak dosya bulunamadı!")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🩹 Diffusers yükleyicisine ignore_mismatched_sizes=True yaması ekleniyor...")

# unifield_pipeline veya sd_model_zoo içindeki from_pretrained çağrılarına ignore_mismatched_sizes=True ekliyoruz
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                
                if "from_pretrained(" in content and "ignore_mismatched_sizes" not in content:
                    new_content = content.replace("from_pretrained(", "from_pretrained(ignore_mismatched_sizes=True, ")
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(new_content)
            except Exception:
                pass

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🔧 Bozuk python dosyası git ile orijinal haline getiriliyor...")
os.system("git checkout custum_3d_diffusion/trainings/image2mvimage_trainer.py >/dev/null 2>&1")

print("⚙️ unet_4ch ve img2mvimg config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu hatasız olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Tüm proje dosyaları git reset ile tamamen orijinal ve hatasız haline getiriliyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

print("⚙️ unet_4ch ve img2mvimg config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu tertemiz bir şekilde başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Tüm proje dosyaları git reset ile tamamen orijinal ve hatasız haline getiriliyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: rembg sürümünü sabitle (Unique3D uyumlu sürüm)
# ============================================================
print("📦 rembg sürümü kontrol ediliyor ve sabitleniyor...")
os.system("pip install -q 'rembg==2.0.50' 'onnxruntime-gpu>=1.16.0' 2>&1 | tail -3")
# Alternatif: os.system("pip install -q 'rembg[cpu]==2.0.50'")

# ============================================================
# 🔧 DÜZELTME 2: scripts/utils.py içindeki new_session çağrısını yamala
# ============================================================
print("🩹 scripts/utils.py içindeki new_session çağrısı yamalanıyor...")
utils_path = "scripts/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Hatalı satırı güvenli hale getir
    # Eski: session = new_session(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    # Yeni: providers argümanı yoksa veya tuple ise düzelt
    if "new_session" in content:
        # Basit regex ile new_session çağrısını güvenli formata çevir
        # Örnek: new_session(providers=("CUDAExecutionProvider", "CPUExecutionProvider"))
        #    -> new_session()
        content = re.sub(
            r"new_session\s*\(\s*providers\s*=\s*\([^)]*\)\s*\)",
            "new_session()",
            content
        )
        content = re.sub(
            r"new_session\s*\(\s*providers\s*=\s*\[[^\]]*\]\s*\)",
            "new_session()",
            content
        )
        # Eğer hala tuple/liste argümanı varsa temizle
        content = re.sub(
            r"new_session\s*\(\s*[\(\[]",
            "new_session(",
            content
        )
        
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(content)
        print("   ✅ utils.py güncellendi")

# ============================================================
print("⚙️ unet_4ch ve img2mvimg config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu tertemiz bir şekilde başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CUDAExecutionProvider"  # 🔧 ek güvenlik

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
# ============================================================
# 🔧 DÜZELTME 1: Python 3.13 uyumlu rembg sürümünü sabitle
# ============================================================
print("📦 rembg sürümü kontrol ediliyor ve sabitleniyor...")
# Python 3.13'ü destekleyen minimum sürüm 2.0.62'dir
os.system("pip install -q 'rembg==2.0.62' 2>&1 | tail -3")

# ============================================================
# 🔧 DÜZELTME 2: scripts/utils.py dosyasındaki providers parametresini düzelt
# ============================================================
print("🩹 scripts/utils.py içindeki new_session çağrısı yamalanıyor...")
utils_path = "scripts/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # providers değişkeninin tanımını bul ve düz bir liste olduğundan emin ol
    # Yaygın hatalı yazım: providers = (('CUDAExecutionProvider', 'CPUExecutionProvider'),)
    # veya providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] (doğru ama iç içe olabilir)
    
    # Tüm providers tanımlarını düz listeye dönüştür
    # Önce tuple içeren atamaları bul
    import ast
    
    # Basit ve etkili yöntem: providers tanım satırını doğrudan değiştir
    lines = content.split('\n')
    new_lines = []
    for line in lines:
        if 'providers' in line and '=' in line and ('new_session' not in line):
            # providers atama satırını düz listeye zorla
            if 'CUDAExecutionProvider' in line or 'CPUExecutionProvider' in line:
                indent = line[:len(line) - len(line.lstrip())]
                new_lines.append(f"{indent}providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']")
                continue
        if 'new_session(providers=providers)' in line:
            # new_session çağrısını doğrudan listeye çevir
            indent = line[:len(line) - len(line.lstrip())]
            new_lines.append(f"{indent}session = new_session(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])")
            continue
        new_lines.append(line)
    
    content = '\n'.join(new_lines)
    
    # Ek güvence: eğer hala tuple new_session içinde varsa temizle
    content = re.sub(
        r"new_session\s*\(\s*providers\s*=\s*\([^)]*\)\s*\)",
        "new_session(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])",
        content
    )
    
    with open(utils_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("   ✅ utils.py güncellendi")

In [ ]:
!tail -50 /kaggle/working/gradio.log

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: rembg sürümü (Python 3.13 uyumlu)
# ============================================================
print("📦 rembg sürümü sabitleniyor...")
os.system("pip install -q 'rembg>=2.0.62' 2>&1 | tail -2")

# ============================================================
# 🔧 DÜZELTME 2: utils.py içindeki providers tanımını ve new_session'ı zorla düzelt
# ============================================================
print("🩹 scripts/utils.py zorla düzeltiliyor...")
utils_path = "scripts/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        stripped = line.strip()
        
        # providers değişkeninin HERHANGİ bir tanımını düz listeye zorla
        if re.match(r"^providers\s*=", stripped):
            indent = line[:len(line) - len(line.lstrip())]
            new_lines.append(f"{indent}providers = ['CPUExecutionProvider']\n")
            print(f"   🔧 providers satırı düzeltildi: {stripped[:60]}")
            continue
        
        # new_session çağrısını doğrudan düz listeye çevir
        if "new_session" in stripped and "providers" in stripped:
            indent = line[:len(line) - len(line.lstrip())]
            new_lines.append(f"{indent}session = new_session(providers=['CPUExecutionProvider'])\n")
            print(f"   🔧 new_session satırı düzeltildi")
            continue
        
        new_lines.append(line)
    
    content = ''.join(new_lines)
    
    # Regex ile kalan tüm hatalı çağrıları temizle
    content = re.sub(
        r"new_session\s*\(\s*providers\s*=\s*\([^)]*\)\s*\)",
        "new_session(providers=['CPUExecutionProvider'])",
        content
    )
    content = re.sub(
        r"new_session\s*\(\s*providers\s*=\s*providers\s*\)",
        "new_session(providers=['CPUExecutionProvider'])",
        content
    )
    
    with open(utils_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("   ✅ utils.py güncellendi")

# Doğrulama: 20-30. satırları yazdır
print("\n📄 Doğrulama - utils.py ilk 30 satır:")
with open(utils_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f.readlines()[:30], 1):
        if i >= 18 and i <= 28:
            print(f"   {i:3d}: {line.rstrip()}")

# ============================================================
# 🔧 DÜZELTME 3: onnxruntime-gpu dene, olmazsa CPU
# ============================================================
print("\n📦 onnxruntime kontrol ediliyor...")
import onnxruntime as ort
available = ort.get_available_providers()
print(f"   Mevcut sağlayıcılar: {available}")

# ============================================================
print("\n⚙️ config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
providers = [
    ('CUDAExecutionProvider', {
        'device_id': 0,
        'arena_extend_strategy': 'kSameAsRequested',
        'gpu_mem_limit': 8 * 1024 * 1024 * 1024,
        'cudnn_conv_algo_search': 'HEURISTIC',
    })
]

session = new_session(providers=providers)

In [ ]:
!pip install --upgrade rembg

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("⚙️ unet_4ch konfigürasyonu 4 kanallı yapıya (in_channels=4) tam uyumlu hale getiriliyor...")

# config.json dosyasını bulup in_channels değerini 4 yapıyoruz (bu sayede kanal boyutu hatası tamamen biter)
config_yollari = [
    "ckpt/unet_4ch/unet/config.json",
    "ckpt/img2mvimg/unet/config.json"
]

for cfg_path in config_yollari:
    if os.path.exists(cfg_path):
        try:
            with open(cfg_path, "r", encoding="utf-8") as f:
                cfg = json.load(f)
            cfg["in_channels"] = 4
            cfg.pop("num_class_embeds", None)
            cfg["addition_embed_type"] = None
            with open(cfg_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
            print(f"✅ Güncellenen config: {cfg_path}")
        except Exception as e:
            print(f"⚠️ Config güncellenemedi ({cfg_path}): {e}")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu hatasız olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

print("1. Unique3D Dosyalari Indiriliyor...")
if not os.path.exists(f"{WORK_DIR}/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir(f"{WORK_DIR}/Unique3D")

print("2. Bozuk Paketler Silinip Orijinal GPU PyTorch (CUDA 12.4) Kuruluyor...")
os.system("pip uninstall -y xformers torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu124"
])

print("3. Gerekli Kutuphaneler Kuruluyor...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ninja", "torch-scatter", "setuptools", "wheel"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "fire", "rembg", "onnxruntime", "trimesh", "pymeshlab",
    "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2",
    "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"
])

print("4. C++ 3D Motorlari (Nvdiffrast ve PyTorch3D) Derleniyor (3-4 dk)...")
for kls in ["nvdiffrast", "nvdiffrast_repo"]:
    if os.path.exists(kls):
        shutil.rmtree(kls)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"
])

print("5. Yapay Zeka Model Agirliklari Indiriliyor...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system(
        "huggingface-cli download Wuvin/Unique3D --repo-type space "
        "--include 'ckpt/*' --local-dir . --local-dir-use-symlinks False"
    )

print("6. 4 Kanalli U-Net Kisayollari Olusturuluyor...")
img2mvimg_yolu = f"{WORK_DIR}/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = f"{WORK_DIR}/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)
if os.path.exists(img2mvimg_yolu):
    for d in os.listdir(img2mvimg_yolu):
        hedef_d = os.path.join(unet_4ch_yolu, d)
        if d != "unet" and not os.path.exists(hedef_d):
            try: os.symlink(os.path.join(img2mvimg_yolu, d), hedef_d)
            except: pass
    for d in os.listdir(os.path.join(img2mvimg_yolu, "unet")):
        hedef_u = os.path.join(unet_4ch_yolu, "unet", d)
        if d != "config.json" and not os.path.exists(hedef_u):
            try: os.symlink(os.path.join(img2mvimg_yolu, "unet", d), hedef_u)
            except: pass
    cfg_path = os.path.join(img2mvimg_yolu, "unet", "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet", "config.json"), "w") as f:
            json.dump(cfg, f, indent=2)

print("7. Tum Yamalar (Rembg, Tensor ve 4GB->32MB Bellek Zirhi) Uygulaniyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py")
os.system("git checkout app/custom_models/utils.py")
if os.path.exists("scripts/utils.py"):
    os.system("git checkout scripts/utils.py")
if os.path.exists("scripts/multiview_inference.py"):
    os.system("git checkout scripts/multiview_inference.py")

# 4.00 GB OOM Hatasini 32 MB'a Dusuren Cekirdek Matris Dilimleyici
yama_kodu = """import os, gc, torch
import torch.nn.functional as _F

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

if not hasattr(_F, "_orig_sdpa"):
    _F._orig_sdpa = _F.scaled_dot_product_attention

def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):
    if query.dim() == 4:
        B, H, Lq, _ = query.shape
        Lk = key.shape[2]
        if B * H * Lq * Lk > 2097152 and attn_mask is None:
            out = torch.empty_like(query)
            step = 1024
            for b in range(B):
                for h in range(H):
                    kb = key[b:b+1, h:h+1]
                    vb = value[b:b+1, h:h+1]
                    for i in range(0, Lq, step):
                        q_sl = query[b:b+1, h:h+1, i:i+step]
                        out[b:b+1, h:h+1, i:i+step] = _F._orig_sdpa(
                            q_sl, kb, vb, attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs
                        )
            return out
    elif query.dim() == 3:
        BH, Lq, _ = query.shape
        Lk = key.shape[1]
        if BH * Lq * Lk > 2097152 and attn_mask is None:
            out = torch.empty_like(query)
            step = 1024
            for bh in range(BH):
                kb = key[bh:bh+1]
                vb = value[bh:bh+1]
                for i in range(0, Lq, step):
                    q_sl = query[bh:bh+1, i:i+step]
                    out[bh:bh+1, i:i+step] = _F._orig_sdpa(
                        q_sl, kb, vb, attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs
                    )
            return out
    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)

_F.scaled_dot_product_attention = _sliced_sdpa
torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa
"""
with open("oom_patch.py", "w", encoding="utf-8") as f:
    f.write(yama_kodu)

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w") as f: f.write(c)
            except: pass

try:
    with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "r") as f:
        kod1 = f.read()
    kod1 = kod1.replace(
        'subfolder="unet", torch_dtype=self.unet.dtype',
        'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True'
    )
    kod1 = kod1.replace(
        '**unet_kwargs',
        "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False"
    )
    with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "w") as f:
        f.write(kod1)
except: pass

try:
    with open("app/custom_models/utils.py", "r") as f:
        kod2 = f.readlines()
    for i, satir in enumerate(kod2):
        if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in satir:
            b = satir[:len(satir) - len(satir.lstrip())]
            yeni = [
                "try:\n",
                b + "    configurable_unet.unet.to_empty(device='cpu')\n",
                b + "except Exception:\n",
                b + "    pass\n",
                b + "model_sd = configurable_unet.unet.state_dict()\n",
                b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n",
                b + "    if k in state_dict and k in model_sd:\n",
                b + "        s1 = state_dict[k].shape\n",
                b + "        s2 = model_sd[k].shape\n",
                b + "        if s1 != s2:\n",
                b + "            new_w = torch.zeros_like(model_sd[k])\n",
                b + "            c_min = min(s1[1], s2[1])\n",
                b + "            new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]\n",
                b + "            state_dict[k] = new_w\n",
                b + "try:\n",
                b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n",
                b + "except Exception:\n",
                b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
            ]
            kod2[i] = b + "".join(yeni)
        elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in satir:
            b = satir[:len(satir) - len(satir.lstrip())]
            yeni = [
                "try:\n",
                b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n",
                b + "except NotImplementedError:\n",
                b + "    configurable_unet.unet.to_empty(device=device)\n",
                b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n"
            ]
            kod2[i] = b + "".join(yeni)
        elif "return trainer, pipeline" in satir:
            b = satir[:len(satir) - len(satir.lstrip())]
            yeni = [
                "try:\n",
                b + "    pipeline.enable_attention_slicing(1)\n",
                b + "    pipeline.enable_vae_slicing()\n",
                b + "except:\n",
                b + "    pass\n",
                b + "return trainer, pipeline\n"
            ]
            kod2[i] = b + "".join(yeni)
    with open("app/custom_models/utils.py", "w") as f:
        f.writelines(kod2)
except: pass

try:
    if os.path.exists("scripts/utils.py"):
        with open("scripts/utils.py", "r") as f: k_rembg = f.read()
        k_rembg = k_rembg.replace("providers=providers", "providers=['CPUExecutionProvider']")
        with open("scripts/utils.py", "w") as f: f.write(k_rembg)
except: pass

try:
    if os.path.exists("scripts/multiview_inference.py"):
        with open("scripts/multiview_inference.py", "r", encoding="utf-8") as f:
            k_mv = f.read()
        if "import gc" not in k_mv:
            k_mv = "import torch, gc\n" + k_mv
        if "gc.collect()" not in k_mv:
            k_mv = k_mv.replace(
                "rm_normals = predict_normals",
                "gc.collect(); torch.cuda.empty_cache()\n    rm_normals = predict_normals"
            )
        with open("scripts/multiview_inference.py", "w", encoding="utf-8") as f:
            f.write(k_mv)
except: pass

for hedef in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(hedef):
        with open(hedef, "r") as f: k = f.read()
        k = k.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in k:
            k = k.replace(".launch(", ".launch(share=True, ")
        if "cached_download" not in k:
            k = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + k
        if hedef == "gradio_app.py" and "import oom_patch" not in k:
            k = "import oom_patch\n" + k
        with open(hedef, "w") as f: f.write(k)

print("8. STUDYO TAM ZIRHLI OLARAK BASLATILIYOR (Mavi link birazdan gelecek)...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. multiview_inference.py Dosyası Temizleniyor...")
os.system("git checkout scripts/multiview_inference.py")

with open("scripts/multiview_inference.py", "r", encoding="utf-8") as f:
    kmv = f.read()

if "import gc" not in kmv:
    kmv = "import torch, gc\n" + kmv

# Doğru fonksiyon çağrısını hedef alarak güvenli temizlik enjeksiyonu:
hedef_metin = "vertices, faces = run_mesh_refine("
temizlik_kodu = (
    "gc.collect(); torch.cuda.empty_cache(); "
    "[getattr(obj, 'to')('cpu') for obj in list(globals().values()) + list(locals().values()) if hasattr(obj, 'to') and 'cuda' in str(getattr(obj, 'device', ''))]; "
    "gc.collect(); torch.cuda.empty_cache(); "
    "vertices, faces = run_mesh_refine("
)

if hedef_metin in kmv:
    kmv = kmv.replace(hedef_metin, temizlik_kodu)

with open("scripts/multiview_inference.py", "w", encoding="utf-8") as f:
    f.write(kmv)

print("🚀 2. Stüdyo 12GB Boş VRAM Garantisiyle Yeniden Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. multiview_inference.py Dosyası Temizleniyor...")
os.system("git checkout scripts/multiview_inference.py")

with open("scripts/multiview_inference.py", "r", encoding="utf-8") as f:
    kmv = f.read()

if "import gc" not in kmv:
    kmv = "import torch, gc\n" + kmv

# Doğru fonksiyon çağrısını hedef alarak güvenli temizlik enjeksiyonu:
hedef_metin = "vertices, faces = run_mesh_refine("
temizlik_kodu = (
    "gc.collect(); torch.cuda.empty_cache(); "
    "[getattr(obj, 'to')('cpu') for obj in list(globals().values()) + list(locals().values()) if hasattr(obj, 'to') and 'cuda' in str(getattr(obj, 'device', ''))]; "
    "gc.collect(); torch.cuda.empty_cache(); "
    "vertices, faces = run_mesh_refine("
)

if hedef_metin in kmv:
    kmv = kmv.replace(hedef_metin, temizlik_kodu)

with open("scripts/multiview_inference.py", "w", encoding="utf-8") as f:
    f.write(kmv)

print("🚀 2. Stüdyo 12GB Boş VRAM Garantisiyle Yeniden Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. multiview_inference.py Dosyası Tamamen Sıfırlanıyor...")
os.system("git checkout scripts/multiview_inference.py")

with open("scripts/multiview_inference.py", "r", encoding="utf-8") as f:
    kmv = f.read()

if "import gc" not in kmv:
    kmv = "import torch, gc\n" + kmv

# Kesin ve net bellek boşaltma komutu (hiçbir liste karmaşası yok):
kesin_kod = (
    "gc.collect(); torch.cuda.empty_cache(); "
    "from torch.cuda import empty_cache; empty_cache(); "
    "vertices, faces = run_mesh_refine("
)

kmv = kmv.replace("vertices, faces = run_mesh_refine(", kesin_kod)

with open("scripts/multiview_inference.py", "w", encoding="utf-8") as f:
    f.write(kmv)

print("🚀 2. Stüdyo Yeniden Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import onnxruntime as ort
from rembg import new_session, remove   # ← import burada

providers = [
    ('CUDAExecutionProvider', {...})
]

session = new_session(providers=providers)

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME: numpy, scipy, numba ve rembg sürümlerini uyumlu hale getir
# ============================================================
print("📦 Bağımlılık sürümleri uyumlu hale getiriliyor (numpy/scipy/numba/rembg)...")
# Python 3.13 ile uyumlu bilinen kombinasyon:
# numpy 2.1.x, scipy 1.14.x, numba 0.61.x, rembg 2.0.62+
os.system(
    "pip install -q "
    "'numpy==2.1.3' "
    "'scipy==1.14.1' "
    "'numba==0.61.0' "
    "'rembg==2.0.62' "
    "2>&1 | tail -5"
)

# ============================================================
# 🔧 utils.py: providers bloğunu temizle, new_session'ı düzelt, import'u garantile
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"

with open(utils_path, "r", encoding="utf-8") as f:
    original = f.read()

lines = original.split('\n')

# 1) providers bloğunu tamamen kaldır
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break

if start_idx is not None:
    bracket_depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        bracket_depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if bracket_depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]
    print(f"   🔧 providers bloğu silindi ({start_idx+1}-{end_idx+1}. satırlar)")

content = '\n'.join(lines)

# 2) new_session çağrısını güvenli forma getir
content = re.sub(
    r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
    "new_session(providers=['CPUExecutionProvider'])",
    content
)

# 3) rembg import'unu garantile
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
    print("   🔧 'from rembg import new_session, remove' eklendi")
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(
        r"from rembg import [^\n]+",
        "from rembg import new_session, remove",
        content,
        count=1
    )
    print("   🔧 rembg import satırı 'new_session, remove' olarak güncellendi")

# 4) session tanımını garantile
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1
    )
    print("   🔧 session tanımı eklendi")

with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# Doğrulama
print("\n📄 Doğrulama - utils.py ilk 15 satır:")
with open(utils_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f.readlines()[:15], 1):
        print(f"   {i:3d}: {line.rstrip()}")

# ============================================================
print("\n⚙️ config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if any(x in log_icerik for x in ["Traceback", "OSError", "SyntaxError", "IndentationError", "NameError", "ImportError"]) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-20:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n======================================================================")
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("======================================================================\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: rembg + bağımlılıklar
# ============================================================
print("📦 Bağımlılıklar uyumlu hale getiriliyor...")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -3"
)

# ============================================================
# 🔧 DÜZELTME 2: utils.py patch (önceki düzeltme, korunuyor)
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# providers bloğunu sil
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 🔧 DÜZELTME 3: HuggingFace modelini önceden indir + doğrula
# ============================================================
print("📥 HuggingFace modeli önceden indiriliyor ve doğrulanıyor...")
HF_MODEL = "lambdalabs/sd-image-variations-diffusers"
try:
    from huggingface_hub import hf_hub_download, snapshot_download
    try:
        # Önce erişim testi
        p = hf_hub_download(HF_MODEL, "model_index.json")
        print(f"   ✅ Erişim tamam: {p}")
        # Tam indirme (diffusers formatı)
        snapshot_download(
            HF_MODEL,
            allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt", "*.jsonl"],
            ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
        )
        print("   ✅ Model cache'e indirildi")
    except Exception as e:
        print(f"   ⚠️  HF indirme hatası: {type(e).__name__}: {e}")
        print(f"   💡 Çözüm: `!huggingface-cli login` çalıştırın veya HF_TOKEN ayarlayın")
except ImportError:
    print("   ⚠️  huggingface_hub yüklü değil")

# ============================================================
# config.json 4 kanala sabitle
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# Sunucuyu başlat
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

# ============================================================
# 🔍 KAPSAMLI HATA TESPİT DÖNGÜSÜ
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file",
]

server_started = False
error_detected = False
max_wait = 600  # 10 dakika max
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    # BAŞARI
    if "Running on local URL" in log_icerik or "http://0.0.0.0:7860" in log_icerik:
        server_started = True
        break

    # Süreç öldü mü?
    if proc.poll() is not None:
        time.sleep(2)
        error_detected = True
        break

    # Hata var mı?
    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        error_detected = True
        break

# ============================================================
# 🎯 SONUÇ RAPORU
# ============================================================
print()  # newline

if server_started:
    print("\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    time.sleep(3)
    cf_log_path = f"{WORK_DIR}/cf.log"
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("======================================================================\n")
else:
    # HATA DURUMU - detaylı rapor
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI! (ya hata var ya da zaman aşımı)")
    print("="*70)

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    # Traceback bloğunu bul
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i

    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    # Hata türlerini listele
    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    if found:
        for t in set(found):
            print(f"   ❌ {t}")
    else:
        print("   (spesifik hata türü yok — çıktı yukarıda)")

    # Öneriler
    print("\n💡 ÖNERİLEN ÇÖZÜMLER:")
    log_low = log_icerik.lower()
    if "sd-image-variations-diffusers" in log_low or "does not appear to have a file" in log_low:
        print("   • HF erişim sorunu tespit edildi:")
        print("     → `!pip install -U huggingface_hub`")
        print("     → `!huggingface-cli login` (token girin)")
        print("     → Kaggle Secrets'a HF_TOKEN ekleyin")
        print("     → Model sayfasında 'Agree and access' tıklayın")
    if "cuda out of memory" in log_low:
        print("   • GPU belleği yetersiz → batch size / çözünürlük düşürün")
    if "no module named" in log_low:
        print("   • Eksik modül → `pip install <modül>`")
    if "file not found" in log_low or "no such file" in log_low:
        print("   • Dosya eksik → checkpoint'lerin indirildiğini kontrol edin")
    if "connection" in log_low or "timeout" in log_low:
        print("   • Ağ sorunu → tekrar deneyin veya proxy kullanın")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: Model revizyonunu yapılandırma dosyalarına ekle
# ============================================================
print("🩹 Model revizyonu ('v2.0') yapılandırma dosyalarına ekleniyor...")
yaml_files = glob.glob("app/custom_models/*.yaml") + glob.glob("gradio_app/custom_models/*.yaml")

for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Model adının geçtiği satırları bul ve revizyon ekle
    # Örnek: pretrained_model_name_or_path: "lambdalabs/sd-image-variations-diffusers"
    # Yeni:  pretrained_model_name_or_path: "lambdalabs/sd-image-variations-diffusers"
    #        revision: "v2.0"
    if "sd-image-variations-diffusers" in content:
        # Zaten revision varsa atla
        if "revision:" not in content:
            # Model adı satırından sonra revision ekle (girintiyi koru)
            lines = content.split('\n')
            new_lines = []
            for line in lines:
                new_lines.append(line)
                if "sd-image-variations-diffusers" in line and ":" in line:
                    # Girintiyi al
                    indent = line[:len(line) - len(line.lstrip())]
                    # Aynı girinti seviyesinde revision ekle
                    new_lines.append(f"{indent}revision: \"v2.0\"")
            content = '\n'.join(new_lines)
            print(f"   🔧 {os.path.basename(yaml_path)}: revision eklendi")
        else:
            print(f"   ✓ {os.path.basename(yaml_path)}: revision zaten mevcut")
    
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(content)

# ============================================================
# 🔧 DÜZELTME 2: Modeli doğru revizyonla önceden indir
# ============================================================
print("📥 Model doğru revizyonla indiriliyor (v2.0)...")
HF_MODEL = "lambdalabs/sd-image-variations-diffusers"
REVISION = "v2.0"
try:
    from huggingface_hub import hf_hub_download, snapshot_download
    # Önce v2.0 revizyonunda model_index.json var mı kontrol et
    try:
        p = hf_hub_download(HF_MODEL, "model_index.json", revision=REVISION)
        print(f"   ✅ v2.0 revizyonuna erişim tamam: {p}")
        # Tam indirme (sadece gerekli dosyalar)
        snapshot_download(
            HF_MODEL,
            revision=REVISION,
            allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
            ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
        )
        print("   ✅ Model cache'e indirildi")
    except Exception as e:
        print(f"   ⚠️  v2.0 indirme hatası: {type(e).__name__}: {e}")
        # Alternatif: matttrent kopyasını dene
        print("   🔄 Alternatif model (matttrent/sd-image-variations-diffusers) deneniyor...")
        try:
            snapshot_download(
                "matttrent/sd-image-variations-diffusers",
                allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
                ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
            )
            print("   ✅ Alternatif model indirildi (matttrent/sd-image-variations-diffusers)")
            # Yapılandırmayı alternatif modele yönlendir
            for yaml_path in yaml_files:
                if os.path.exists(yaml_path):
                    with open(yaml_path, "r", encoding="utf-8") as f:
                        c = f.read()
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", "matttrent/sd-image-variations-diffusers")
                    with open(yaml_path, "w", encoding="utf-8") as f:
                        f.write(c)
            print("   🔧 Yapılandırma alternatif modele yönlendirildi")
        except Exception as e2:
            print(f"   ❌ Alternatif model de başarısız: {e2}")
except ImportError:
    print("   ⚠️  huggingface_hub yüklü değil")

# ============================================================
# (İsteğe bağlı) config.json 4 kanala sabitleme
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor (varsa)...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# Sunucuyu başlat
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

# ============================================================
# 🔍 KAPSAMLI HATA TESPİT DÖNGÜSÜ
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file",
]

server_started = False
error_detected = False
max_wait = 600  # 10 dakika
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    if "Running on local URL" in log_icerik or "http://0.0.0.0:7860" in log_icerik:
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        error_detected = True
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        error_detected = True
        break

# ============================================================
# 🎯 SONUÇ RAPORU
# ============================================================
print()

if server_started:
    print("\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    time.sleep(3)
    cf_log_path = f"{WORK_DIR}/cf.log"
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI! (hata veya zaman aşımı)")
    print("="*70)

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i

    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    if found:
        for t in set(found):
            print(f"   ❌ {t}")
    else:
        print("   (spesifik hata türü yok)")

    print("\n💡 ÖNERİLEN ÇÖZÜMLER:")
    log_low = log_icerik.lower()
    if "sd-image-variations-diffusers" in log_low or "does not appear to have a file" in log_low:
        print("   • Model erişim sorunu devam ediyor:")
        print("     → `huggingface-cli login` ile giriş yapın")
        print("     → Model sayfasında 'Agree and access' tıklayın")
        print("     → Alternatif olarak `matttrent/sd-image-variations-diffusers` kullanılabilir")
    if "cuda out of memory" in log_low:
        print("   • GPU belleği yetersiz → batch size düşürün")
    if "no module named" in log_low:
        print("   • Eksik modül → `pip install <modül>`")
    if "connection" in log_low or "timeout" in log_low:
        print("   • Ağ sorunu → tekrar deneyin")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 rembg + bağımlılıklar (korunuyor)
# ============================================================
print("📦 Bağımlılıklar uyumlu hale getiriliyor...")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -3"
)

# ============================================================
# 🔧 utils.py patch (korunuyor)
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 🔧 YENİ DÜZELTME: YAML - revision satırını sil, matttrent kullan
# ============================================================
print("🩹 Yapılandırma dosyaları düzeltiliyor (revision kaldırılıyor)...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))

for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        content = f.read()

    original = content

    # 1. Bizim eklediğimiz `revision: "v2.0"` satırını KALDIR (AttnConfig hatası)
    content = re.sub(r'^\s*revision:\s*["\']v2\.0["\']\s*\n', '', content, flags=re.MULTILINE)
    content = re.sub(r'^\s*revision:\s*["\']main["\']\s*\n', '', content, flags=re.MULTILINE)

    # 2. Model yolunu matttrent'e çevir (güvenli, revizyon gerektirmez)
    content = content.replace(
        "lambdalabs/sd-image-variations-diffusers",
        "matttrent/sd-image-variations-diffusers"
    )

    if content != original:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"   🔧 {yaml_path}: düzeltildi (matttrent, revision yok)")

# ============================================================
# 🔧 matttrent'in doğru yapıda olduğunu doğrula
# ============================================================
print("📥 matttrent modeli doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download

MODEL = "matttrent/sd-image-variations-diffusers"
try:
    p = hf_hub_download(MODEL, "unet/config.json")
    print(f"   ✅ unet/config.json mevcut: {p}")
    # Tam indirme (subfolder'lı yapı)
    snapshot_download(
        MODEL,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model cache'e indirildi")
except Exception as e:
    print(f"   ❌ matttrent unet/config.json erişimi başarısız: {e}")
    # Alternatif: bdpc
    print("   🔄 Alternatif: bdpc/sd-image-variations-diffusers deneniyor...")
    try:
        p = hf_hub_download("bdpc/sd-image-variations-diffusers", "unet/config.json")
        print(f"   ✅ bdpc çalışıyor")
        for yaml_path in yaml_files:
            if os.path.exists(yaml_path):
                with open(yaml_path, "r") as f:
                    c = f.read()
                c = c.replace(MODEL, "bdpc/sd-image-variations-diffusers")
                with open(yaml_path, "w") as f:
                    f.write(c)
        print("   🔧 Yapılandırma bdpc'ye yönlendirildi")
    except Exception as e2:
        print(f"   ❌ bdpc de başarısız: {e2}")

# ============================================================
# config.json 4ch
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor (varsa)...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# Sunucuyu başlat
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

# ============================================================
# Kapsamlı hata tespit döngüsü (korunuyor)
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file", "unexpected keyword argument",
]

server_started = False
max_wait = 600
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    if "Running on local URL" in log_icerik or "http://0.0.0.0:7860" in log_icerik:
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        break

# ============================================================
# Sonuç raporu
# ============================================================
print()

if server_started:
    print("\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    time.sleep(3)
    cf_log_path = f"{WORK_DIR}/cf.log"
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI!")
    print("="*70)
    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i
    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    if found:
        for t in set(found):
            print(f"   ❌ {t}")
    else:
        print("   (spesifik hata türü yok)")

    print("\n💡 ÖNERİLEN ÇÖZÜMLER:")
    log_low = log_icerik.lower()
    if "unexpected keyword argument 'revision'" in log_low:
        print("   • YAML'deki revision satırı kaldırılmalı (bu script yapıyor)")
    if "sd-image-variations-diffusers" in log_low:
        print("   • Model erişim sorunu → matttrent/bdpc kullanılıyor")
    if "cuda out of memory" in log_low:
        print("   • GPU belleği yetersiz → batch size düşürün")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: rembg + bağımlılıklar (korunuyor)
# ============================================================
print("📦 Bağımlılıklar uyumlu hale getiriliyor...")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -3"
)

# ============================================================
# 🔧 DÜZELTME 2: nvdiffrast kurulumu (YENİ)
# ============================================================
print("📦 nvdiffrast kuruluyor (kaynaktan derleme)...")
# Derleme araçları
os.system("pip install -q setuptools wheel ninja 2>&1 | tail -2")
# OpenGL bağımlılığı (bazı durumlarda gerekli)
os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
# nvdiffrast'ı GitHub'dan derle (Python 3.13 uyumlu)
os.system(
    "pip install -q --no-build-isolation "
    "git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -5"
)
# Doğrulama
verify = os.system("python -c 'import nvdiffrast.torch as dr; print(\"nvdiffrast OK\")' 2>&1")
if verify != 0:
    print("   ⚠️  nvdiffrast derleme başarısız, Unique3D deposundaki wheel deneniyor...")
    # Unique3D deposunda önceden derlenmiş wheel olabilir
    for wheel in glob.glob("package/nvdiffrast*.whl"):
        print(f"   🔄 Deneniyor: {wheel}")
        os.system(f"pip install -q --force-reinstall --no-deps '{wheel}' 2>&1 | tail -3")
        break
    verify2 = os.system("python -c 'import nvdiffrast.torch as dr' 2>&1")
    if verify2 != 0:
        print("   ❌ nvdiffrast kurulumu başarısız! Manuel kontrol gerekli.")
    else:
        print("   ✅ nvdiffrast wheel ile kuruldu")
else:
    print("   ✅ nvdiffrast başarıyla kuruldu")

# ============================================================
# 🔧 DÜZELTME 3: utils.py patch (korunuyor)
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 🔧 DÜZELTME 4: YAML - revision satırını sil, matttrent kullan (korunuyor)
# ============================================================
print("🩹 Yapılandırma dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))

for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        content = f.read()
    original = content
    # revision satırını kaldır
    content = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', content, flags=re.MULTILINE)
    # model yolunu matttrent'e çevir
    content = content.replace(
        "lambdalabs/sd-image-variations-diffusers",
        "matttrent/sd-image-variations-diffusers"
    )
    if content != original:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"   🔧 {yaml_path}: düzeltildi")

# ============================================================
# Model doğrulama (korunuyor)
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download

MODEL = "matttrent/sd-image-variations-diffusers"
try:
    hf_hub_download(MODEL, "unet/config.json")
    snapshot_download(
        MODEL,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model hatası: {e}")

# ============================================================
# config.json 4ch
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor (varsa)...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# Sunucuyu başlat
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

# ============================================================
# Kapsamlı hata tespit döngüsü
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file", "unexpected keyword argument",
]

server_started = False
max_wait = 900  # nvdiffrast derlemesi zaman alabilir, 15 dakika
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    if "Running on local URL" in log_icerik or "http://0.0.0.0:7860" in log_icerik:
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        break

# ============================================================
# Sonuç raporu
# ============================================================
print()

if server_started:
    print("\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    time.sleep(3)
    cf_log_path = f"{WORK_DIR}/cf.log"
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI!")
    print("="*70)
    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i
    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    if found:
        for t in set(found):
            print(f"   ❌ {t}")
    else:
        print("   (spesifik hata türü yok)")

    print("\n💡 ÖNERİLEN ÇÖZÜMLER:")
    log_low = log_icerik.lower()
    if "nvdiffrast" in log_low:
        print("   • nvdiffrast kurulumu başarısız → Manuel derleme gerekli:")
        print("     `git clone https://github.com/NVlabs/nvdiffrast && cd nvdiffrast && pip install .`")
    if "unexpected keyword argument 'revision'" in log_low:
        print("   • YAML'deki revision satırı kaldırılmalı")
    if "cuda out of memory" in log_low:
        print("   • GPU belleği yetersiz → batch size düşürün")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: rembg + bağımlılıklar (korunuyor)
# ============================================================
print("📦 Bağımlılıklar uyumlu hale getiriliyor...")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -3"
)

# ============================================================
# 🔧 DÜZELTME 2: nvdiffrast (korunuyor)
# ============================================================
print("📦 nvdiffrast kontrol ediliyor...")
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system(
        "pip install -q --no-build-isolation "
        "git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -3"
    )
    print("   ✅ nvdiffrast kuruldu")
else:
    print("   ✓ nvdiffrast zaten kurulu")

# ============================================================
# 🔧 DÜZELTME 3: torch_scatter kurulumu (YENİ)
# ============================================================
print("📦 torch_scatter kuruluyor...")
import torch
TORCH_VERSION = torch.__version__.split('+')[0]  # örn: "2.7.0"
CUDA_TAG = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"  # örn: "cu126"
print(f"   🔍 PyTorch: {TORCH_VERSION}, CUDA: {CUDA_TAG}")

# 1. Yöntem: PyTorch Geometric resmi wheel indeksi (en hızlı ve güvenilir)
install_cmd = (
    f"pip install -q torch-scatter "
    f"-f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html "
    f"2>&1 | tail -5"
)
print(f"   ⏳ Deneniyor: {install_cmd}")
os.system(install_cmd)

# Doğrulama
if os.system("python -c 'import torch_scatter; print(\"torch_scatter OK\")' 2>&1") == 0:
    print("   ✅ torch_scatter wheel ile kuruldu")
else:
    # 2. Yöntem: Kaynaktan derleme (wheel yoksa)
    print("   ⚠️  Wheel bulunamadı, kaynaktan derleniyor (bu birkaç dakika sürebilir)...")
    os.system("pip install -q pybind11 cmake 2>&1 | tail -1")
    os.system(
        "pip install -q --no-build-isolation "
        "git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -5"
    )
    if os.system("python -c 'import torch_scatter' 2>/dev/null") == 0:
        print("   ✅ torch_scatter kaynaktan derlendi")
    else:
        print("   ❌ torch_scatter kurulumu başarısız! Manuel kontrol gerekli.")

# ============================================================
# 🔧 DÜZELTME 4: utils.py patch (korunuyor)
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 🔧 DÜZELTME 5: YAML - revision sil, matttrent kullan (korunuyor)
# ============================================================
print("🩹 Yapılandırma dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))

for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        content = f.read()
    original = content
    content = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', content, flags=re.MULTILINE)
    content = content.replace(
        "lambdalabs/sd-image-variations-diffusers",
        "matttrent/sd-image-variations-diffusers"
    )
    if content != original:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"   🔧 {yaml_path}: düzeltildi")

# ============================================================
# Model doğrulama (korunuyor)
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download

MODEL = "matttrent/sd-image-variations-diffusers"
try:
    hf_hub_download(MODEL, "unet/config.json")
    snapshot_download(
        MODEL,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model hatası: {e}")

# ============================================================
# config.json 4ch
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor (varsa)...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# Sunucuyu başlat
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

# ============================================================
# Kapsamlı hata tespit döngüsü
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file", "unexpected keyword argument",
]

server_started = False
max_wait = 900  # 15 dakika (derleme uzun sürebilir)
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    if "Running on local URL" in log_icerik or "http://0.0.0.0:7860" in log_icerik:
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        break

# ============================================================
# Sonuç raporu
# ============================================================
print()

if server_started:
    print("\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    time.sleep(3)
    cf_log_path = f"{WORK_DIR}/cf.log"
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI!")
    print("="*70)
    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i
    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    if found:
        for t in set(found):
            print(f"   ❌ {t}")
    else:
        print("   (spesifik hata türü yok)")

    print("\n💡 ÖNERİLEN ÇÖZÜMLER:")
    log_low = log_icerik.lower()
    if "torch_scatter" in log_low:
        print("   • torch_scatter kurulumu başarısız → Manuel olarak şu komutu deneyin:")
        print(f"     pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html")
    if "cuda out of memory" in log_low:
        print("   • GPU belleği yetersiz → batch size düşürün")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: hf_transfer paketini kur (YENİ)
# ============================================================
print("📦 hf_transfer kuruluyor...")
os.system("pip install -q hf_transfer 2>&1 | tail -2")
# Doğrulama
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    print("   ✅ hf_transfer kuruldu")
else:
    print("   ⚠️  hf_transfer kurulumu başarısız — env değişkeni kaldırılacak")

# ============================================================
# 🔧 DÜZELTME 2: rembg + bağımlılıklar (korunuyor)
# ============================================================
print("📦 Bağımlılıklar uyumlu hale getiriliyor...")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -2"
)

# ============================================================
# 🔧 DÜZELTME 3: nvdiffrast (korunuyor)
# ============================================================
print("📦 nvdiffrast kontrol ediliyor...")
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system(
        "pip install -q --no-build-isolation "
        "git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2"
    )
    print("   ✅ nvdiffrast kuruldu")
else:
    print("   ✓ nvdiffrast zaten kurulu")

# ============================================================
# 🔧 DÜZELTME 4: torch_scatter (korunuyor)
# ============================================================
print("📦 torch_scatter kontrol ediliyor...")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TORCH_VERSION = torch.__version__.split('+')[0]
    CUDA_TAG = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    print(f"   🔍 PyTorch: {TORCH_VERSION}, CUDA: {CUDA_TAG}")
    os.system(
        f"pip install -q torch-scatter "
        f"-f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html "
        f"2>&1 | tail -2"
    )
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")
    print("   ✅ torch_scatter hazır")
else:
    print("   ✓ torch_scatter zaten kurulu")

# ============================================================
# 🔧 DÜZELTME 5: utils.py patch (korunuyor)
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 🔧 DÜZELTME 6: YAML - revision sil, matttrent kullan (korunuyor)
# ============================================================
print("🩹 Yapılandırma dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))

for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        content = f.read()
    original = content
    content = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', content, flags=re.MULTILINE)
    content = content.replace(
        "lambdalabs/sd-image-variations-diffusers",
        "matttrent/sd-image-variations-diffusers"
    )
    if content != original:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"   🔧 {yaml_path}: düzeltildi")

# ============================================================
# Model doğrulama
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download

MODEL = "matttrent/sd-image-variations-diffusers"
try:
    hf_hub_download(MODEL, "unet/config.json")
    snapshot_download(
        MODEL,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model hatası: {e}")

# ============================================================
# config.json 4ch
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor (varsa)...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# Sunucuyu başlat
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"

# 🔧 HF_TRANSFER: paket varsa aktif, yoksa KAPALI
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("   ✓ hf_transfer aktif")
else:
    env_vars.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
    print("   ⚠️  hf_transfer KAPALI (paket yok)")

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

# ============================================================
# Kapsamlı hata tespit döngüsü
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file", "unexpected keyword argument",
]

server_started = False
max_wait = 900
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    if "Running on local URL" in log_icerik or "http://0.0.0.0:7860" in log_icerik:
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        break

# ============================================================
# Sonuç raporu
# ============================================================
print()

if server_started:
    print("\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    time.sleep(3)
    cf_log_path = f"{WORK_DIR}/cf.log"
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI!")
    print("="*70)
    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i
    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    if found:
        for t in set(found):
            print(f"   ❌ {t}")
    else:
        print("   (spesifik hata türü yok)")

    print("\n💡 ÖNERİLEN ÇÖZÜMLER:")
    log_low = log_icerik.lower()
    if "hf_transfer" in log_low:
        print("   • hf_transfer sorunu → pip install hf_transfer VEYA env kaldır (bu script yapar)")
    if "cuda out of memory" in log_low:
        print("   • GPU belleği yetersiz → batch size düşürün")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ============================================================
# 1) GIT RESET
# ============================================================
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 2) BAĞIMLILIKLAR
# ============================================================
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -1"
)

# nvdiffrast
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    print("📦 nvdiffrast kuruluyor...")
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system(
        "pip install -q --no-build-isolation "
        "git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2"
    )

# torch_scatter
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    print("📦 torch_scatter kuruluyor...")
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(
        f"pip install -q torch-scatter "
        f"-f https://data.pyg.org/whl/torch-{TV}+{CT}.html "
        f"2>&1 | tail -2"
    )
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ============================================================
# 3) utils.py PATCH (rembg providers)
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 4) YAML PATCH (matttrent, revision yok)
# ============================================================
print("🩹 YAML dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))
for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers",
                  "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(c)
        print(f"   🔧 {yaml_path}")

# ============================================================
# 5) gradio_app.py PATCH — server_name="0.0.0.0" (KRİTİK!)
# ============================================================
print("🩹 gradio_app.py dış erişime açılıyor (0.0.0.0)...")
gradio_app_path = "gradio_app.py"
with open(gradio_app_path, "r", encoding="utf-8") as f:
    gcontent = f.read()

# Mevcut launch çağrısını yakala ve değiştir
# Örnek eski: demo.queue().launch(share=True) veya demo.launch(share=True)
# Yeni:       demo.queue().launch(server_name="0.0.0.0", server_port=7860, share=False)
new_launch = (
    'demo.queue().launch('
    'server_name="0.0.0.0", '
    'server_port=7860, '
    'share=False, '
    'show_error=True'
    ')'
)

# .launch(...) içeren tüm çağrıları tek satırla değiştir (basit ve etkili)
gcontent = re.sub(
    r"(demo(?:\.queue\(\))?\.launch\([^)]*\))",
    new_launch,
    gcontent
)

with open(gradio_app_path, "w", encoding="utf-8") as f:
    f.write(gcontent)
print("   ✅ gradio_app.py güncellendi (server_name=0.0.0.0, share=False)")

# Doğrulama
with open(gradio_app_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f.readlines(), 1):
        if ".launch(" in line:
            print(f"   📄 gradio_app.py:{i}: {line.strip()[:100]}")
            break

# ============================================================
# 6) MODEL DOĞRULAMA
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download
MODEL = "matttrent/sd-image-variations-diffusers"
try:
    hf_hub_download(MODEL, "unet/config.json")
    snapshot_download(
        MODEL,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model hatası: {e}")

# ============================================================
# 7) config.json 4 KANAL
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# 8) CLOUDFLARE TÜNELİNİ YENİDEN BAŞLAT (eski tünel hedefi değişmiş olabilir)
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
time.sleep(2)

# cloudflared var mı?
if os.system("which cloudflared >/dev/null 2>&1") != 0:
    print("📦 cloudflared indiriliyor...")
    os.system(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && "
        "chmod +x /usr/local/bin/cloudflared"
    )

cf_log_path = f"{WORK_DIR}/cf.log"
with open(cf_log_path, "w") as f:
    f.write("")

# Cloudflare tünelini ARKA PLANDA başlat (127.0.0.1:7860'a yönlensin)
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=open(cf_log_path, "w"),
    stderr=subprocess.STDOUT,
)
print(f"   ✅ Cloudflare tüneli başlatıldı (PID={cf_proc.pid})")

# ============================================================
# 9) SUNUCUYU BAŞLAT
# ============================================================
print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
env_vars["GRADIO_SERVER_NAME"] = "0.0.0.0"
env_vars["GRADIO_SERVER_PORT"] = "7860"

# hf_transfer koşullu
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("   ✓ hf_transfer aktif")
else:
    env_vars.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
    print("   ⚠️  hf_transfer KAPALI")

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f:
    f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(gradio_log_path, "w"),
    stderr=subprocess.STDOUT,
)

# ============================================================
# 10) BEKLEME DÖNGÜSÜ
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file", "unexpected keyword argument",
]

server_started = False
max_wait = 900
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    if ("Running on local URL" in log_icerik
            or "http://0.0.0.0:7860" in log_icerik
            or "Running on public URL" in log_icerik):
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        break

# ============================================================
# 11) CLOUDFLARE LİNKİ İÇİN BİRAZ BEKLE
# ============================================================
if server_started:
    print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    print("⏳ Cloudflare linki bekleniyor (10sn)...")
    for _ in range(10):
        time.sleep(1)
        cf_icerik = open(cf_log_path, "r", errors="ignore").read()
        if "trycloudflare.com" in cf_icerik:
            break

# ============================================================
# 12) SONUÇ RAPORU
# ============================================================
print()

if server_started:
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler:
        print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler:
        print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print(f"🏠 YEREL ADRES           : http://127.0.0.1:7860")
    print("======================================================================")
    print("\n💡 Cloudflare linki açılmıyorsa:")
    print("   1. 30 saniye bekleyin, tünel yeni kuruldu")
    print("   2. Farklı tarayıcı / gizli mod deneyin")
    print("   3. `!tail -20 /kaggle/working/cf.log` ile tünel durumunu kontrol edin")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI!")
    print("="*70)
    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i
    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    for t in set(found):
        print(f"   ❌ {t}")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ============================================================
# 1) GIT RESET
# ============================================================
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 2) BAĞIMLILIKLAR
# ============================================================
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -1"
)

# nvdiffrast
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    print("📦 nvdiffrast kuruluyor...")
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system(
        "pip install -q --no-build-isolation "
        "git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2"
    )

# torch_scatter
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    print("📦 torch_scatter kuruluyor...")
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(
        f"pip install -q torch-scatter "
        f"-f https://data.pyg.org/whl/torch-{TV}+{CT}.html "
        f"2>&1 | tail -2"
    )
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ============================================================
# 3) utils.py PATCH
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 4) YAML PATCH (matttrent, revision yok)
# ============================================================
print("🩹 YAML dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))
for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers",
                  "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(c)
        print(f"   🔧 {yaml_path}")

# ============================================================
# 5) gradio_app.py PATCH — share=True (KRİTİK!)
# ============================================================
print("🩹 gradio_app.py launch(share=True) yapılıyor...")
gradio_app_path = "gradio_app.py"
with open(gradio_app_path, "r", encoding="utf-8") as f:
    gcontent = f.read()

# Tüm launch(...) çağrılarını yakala ve share=True yap:
#   demo.launch(...)           -> demo.queue().launch(share=True)
#   demo.queue().launch(...)   -> demo.queue().launch(share=True)
gcontent_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True)",
    gcontent
)
if gcontent_new != gcontent:
    with open(gradio_app_path, "w", encoding="utf-8") as f:
        f.write(gcontent_new)
    print("   ✅ gradio_app.py güncellendi (share=True)")
else:
    print("   ⚠️  launch() çağrısı bulunamadı veya zaten güncel")

# Doğrulama
with open(gradio_app_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f.readlines(), 1):
        if ".launch(" in line:
            print(f"   📄 gradio_app.py:{i}: {line.strip()[:100]}")
            break

# ============================================================
# 6) MODEL DOĞRULAMA
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download
MODEL = "matttrent/sd-image-variations-diffusers"
try:
    hf_hub_download(MODEL, "unet/config.json")
    snapshot_download(
        MODEL,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model hatası: {e}")

# ============================================================
# 7) config.json 4 KANAL
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# 8) ESKİ SÜREÇLERİ KAPAT (Cloudflare dahil)
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
time.sleep(2)

# ============================================================
# 9) SUNUCUYU BAŞLAT (share=True)
# ============================================================
print("🚀 Sunucu başlatılıyor (share=True)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"

# hf_transfer koşullu
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("   ✓ hf_transfer aktif")
else:
    env_vars.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
    print("   ⚠️  hf_transfer KAPALI")

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f:
    f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(gradio_log_path, "w"),
    stderr=subprocess.STDOUT,
)

# ============================================================
# 10) BEKLEME DÖNGÜSÜ
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file", "unexpected keyword argument",
]

server_started = False
max_wait = 900
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    # Başarı: local URL veya public URL veya gradio.live
    if ("Running on local URL" in log_icerik
            or "http://0.0.0.0:7860" in log_icerik
            or "Running on public URL" in log_icerik
            or "gradio.live" in log_icerik):
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        break

# ============================================================
# 11) PUBLIC URL İÇİN EK BEKLEME
# ============================================================
if server_started:
    print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    print("⏳ Gradio Live linki bekleniyor (15sn)...")
    for _ in range(15):
        time.sleep(1)
        gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
        if "gradio.live" in gr_icerik:
            break

# ============================================================
# 12) SONUÇ RAPORU
# ============================================================
print()

if server_started:
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if gr_linkler:
        print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    else:
        print("⚠️  Gradio Live linki henüz loglarda görünmedi.")
        print(f"   10-20 saniye sonra kontrol edin:")
        print(f"   !grep 'gradio.live' {gradio_log_path}")
    print(f"🏠 YEREL ADRES           : http://127.0.0.1:7860 (Kaggle içinden)")
    print("======================================================================")
    print("\n💡 Link açılmazsa:")
    print("   1. 30 saniye bekleyin (Gradio tüneli yeni kuruldu)")
    print("   2. Farklı tarayıcı / gizli mod deneyin")
    print("   3. `!grep 'gradio.live' /kaggle/working/gradio.log` ile linki tekrar alın")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI!")
    print("="*70)
    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i
    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    for t in set(found):
        print(f"   ❌ {t}")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ============================================================
# 1) GIT RESET
# ============================================================
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 2) BAĞIMLILIKLAR
# ============================================================
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -1"
)
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    print("📦 nvdiffrast kuruluyor...")
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    print("📦 torch_scatter kuruluyor...")
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ============================================================
# 3) utils.py PATCH
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    lines = f.readlines()
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]
content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n", 1)
with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 4) YAML PATCH
# ============================================================
print("🩹 YAML dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))
for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers",
                  "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(c)

# ============================================================
# 5) gradio_app.py — share=True + hata görünürlüğü
# ============================================================
print("🩹 gradio_app.py düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gcontent = f.read()

# launch() -> share=True + show_error=True (hataları UI'da göster)
gcontent_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, show_error=True, debug=True)",
    gcontent
)
if gcontent_new != gcontent:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gcontent_new)
    print("   ✅ gradio_app.py güncellendi (share=True, show_error=True, debug=True)")
else:
    print("   ⚠️  launch() çağrısı bulunamadı")

# Doğrulama
with open("gradio_app.py", "r", encoding="utf-8") as f:
    for i, line in enumerate(f.readlines(), 1):
        if ".launch(" in line:
            print(f"   📄 Satır {i}: {line.strip()[:110]}")
            break

# ============================================================
# 6) MODEL + CONFIG
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download
try:
    hf_hub_download("matttrent/sd-image-variations-diffusers", "unet/config.json")
    snapshot_download(
        "matttrent/sd-image-variations-diffusers",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model: {e}")

print("⚙️ config.json 4 kanal...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# 7) ESKİ SÜREÇLER
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
time.sleep(2)

# ============================================================
# 8) SUNUCUYU BAŞLAT
# ============================================================
print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

LOG = os.path.join(WORK_DIR, "gradio.log")
with open(LOG, "w") as f:
    f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
)

# ============================================================
# 9) BEKLEME DÖNGÜSÜ — PUBLIC URL + EK BEKLEME
# ============================================================
print("⏳ Link ve model yüklemesi bekleniyor...")
public_url = None
local_ready = False
max_wait = 900
elapsed = 0
last_log_size = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(LOG):
        continue
    log_icerik = open(LOG, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son: {satirlar[-1][:100]:<100}", end="")

    if "Running on local URL" in log_icerik:
        local_ready = True
    gr_match = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", log_icerik)
    if gr_match:
        public_url = gr_match.group(0)

    # Public URL geldiyse biraz daha bekle (model yükleme için)
    if public_url and local_ready:
        # Model yükleme tamamlandı mı? (Unique3D loglarında "Model loaded" gibi bir mesaj varsa)
        if "Model loaded" in log_icerik or "Running on local URL" in log_icerik:
            # 30 sn daha bekle, sonra çık
            print(f"\n   ✅ Public URL alındı: {public_url}")
            print("   ⏳ Model yüklemesi için 30sn ek bekleme...")
            for i in range(30):
                time.sleep(1)
                print(f"\r   ⏳ {30-i:2d}s ", end="")
            print()
            break

    if proc.poll() is not None:
        print(f"\n   ❌ Süreç öldü (kod: {proc.returncode})")
        break

# ============================================================
# 10) SAĞLIK KONTROLÜ
# ============================================================
print("\n" + "="*70)
print("🩺 SAĞLIK KONTROLÜ")
print("="*70)
time.sleep(2)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 10")

# ============================================================
# 11) SONUÇ
# ============================================================
print()
if public_url:
    print("="*70)
    print("🚀 UNIQUE3D HAZIR!")
    print("="*70)
    print(f"🌐 GRADIO LIVE LİNKİNİZ : {public_url}")
    print("="*70)
    print("\n⚠️  ÖNEMLİ — 'Internal Server Error' alırsanız:")
    print("   1. ⏳ 2-3 DAKİKA bekleyin (modeller arka planda yükleniyor olabilir)")
    print("   2. 🔄 Sayfayı yenileyin (F5)")
    print("   3. 🕵️ Gizli mod / farklı tarayıcı deneyin")
    print("   4. 📋 Hâlâ hata varsa şu komutu çalıştırıp çıktıyı paylaşın:")
    print(f"      !tail -50 {LOG}")
    print("="*70)
else:
    print("="*70)
    print("❌ Public URL alınamadı. Log:")
    print("="*70)
    log_icerik = open(LOG, "r", errors="ignore").read() if os.path.exists(LOG) else ""
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    print('\n'.join(satirlar[-40:]))
    print("="*70)

In [ ]:
!tail -n 60 /kaggle/working/gradio.log

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 🔧 DÜZELTME 1: hf_transfer paketini kur (YENİ)
# ============================================================
print("📦 hf_transfer kuruluyor...")
os.system("pip install -q hf_transfer 2>&1 | tail -2")
# Doğrulama
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    print("   ✅ hf_transfer kuruldu")
else:
    print("   ⚠️  hf_transfer kurulumu başarısız — env değişkeni kaldırılacak")

# ============================================================
# 🔧 DÜZELTME 2: rembg + bağımlılıklar (korunuyor)
# ============================================================
print("📦 Bağımlılıklar uyumlu hale getiriliyor...")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -2"
)

# ============================================================
# 🔧 DÜZELTME 3: nvdiffrast (korunuyor)
# ============================================================
print("📦 nvdiffrast kontrol ediliyor...")
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system(
        "pip install -q --no-build-isolation "
        "git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2"
    )
    print("   ✅ nvdiffrast kuruldu")
else:
    print("   ✓ nvdiffrast zaten kurulu")

# ============================================================
# 🔧 DÜZELTME 4: torch_scatter (korunuyor)
# ============================================================
print("📦 torch_scatter kontrol ediliyor...")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TORCH_VERSION = torch.__version__.split('+')[0]
    CUDA_TAG = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    print(f"   🔍 PyTorch: {TORCH_VERSION}, CUDA: {CUDA_TAG}")
    os.system(
        f"pip install -q torch-scatter "
        f"-f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html "
        f"2>&1 | tail -2"
    )
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")
    print("   ✅ torch_scatter hazır")
else:
    print("   ✓ torch_scatter zaten kurulu")

# ============================================================
# 🔧 DÜZELTME 5: utils.py patch (korunuyor)
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
utils_path = "scripts/utils.py"
with open(utils_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]

content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n",
        1)
with open(utils_path, "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 🔧 DÜZELTME 6: YAML - revision sil, matttrent kullan (korunuyor)
# ============================================================
print("🩹 Yapılandırma dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))

for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        content = f.read()
    original = content
    content = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', content, flags=re.MULTILINE)
    content = content.replace(
        "lambdalabs/sd-image-variations-diffusers",
        "matttrent/sd-image-variations-diffusers"
    )
    if content != original:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"   🔧 {yaml_path}: düzeltildi")

# ============================================================
# Model doğrulama
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download

MODEL = "matttrent/sd-image-variations-diffusers"
try:
    hf_hub_download(MODEL, "unet/config.json")
    snapshot_download(
        MODEL,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model hatası: {e}")

# ============================================================
# config.json 4ch
# ============================================================
print("⚙️ config.json ayarları 4 kanala sabitleniyor (varsa)...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# Sunucuyu başlat
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"

# 🔧 HF_TRANSFER: paket varsa aktif, yoksa KAPALI
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("   ✓ hf_transfer aktif")
else:
    env_vars.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
    print("   ⚠️  hf_transfer KAPALI (paket yok)")

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars, stdout=open(gradio_log_path, "w"), stderr=subprocess.STDOUT
)

# ============================================================
# Kapsamlı hata tespit döngüsü
# ============================================================
ERROR_PATTERNS = [
    "Traceback (most recent call last)",
    "OSError:", "ImportError:", "ModuleNotFoundError:",
    "NameError:", "TypeError:", "ValueError:", "RuntimeError:",
    "AttributeError:", "KeyError:", "IndexError:",
    "FileNotFoundError:", "IndentationError:", "SyntaxError:",
    "CUDA out of memory", "cuda out of memory", "CUDA error",
    "ConnectionError:", "TimeoutError:", "HTTPError:",
    "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",
    "PermissionError:", "AssertionError:",
    "does not appear to have a file", "unexpected keyword argument",
]

server_started = False
max_wait = 900
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(gradio_log_path):
        continue

    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son işlem: {satirlar[-1][:110]:<110}", end="")

    if "Running on local URL" in log_icerik or "http://0.0.0.0:7860" in log_icerik:
        server_started = True
        break

    if proc.poll() is not None:
        time.sleep(2)
        break

    if "Traceback (most recent call last)" in log_icerik:
        time.sleep(2)
        break

# ============================================================
# Sonuç raporu
# ============================================================
print()

if server_started:
    print("\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
    time.sleep(3)
    cf_log_path = f"{WORK_DIR}/cf.log"
    cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
    gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
    cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
    gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

    print("\n======================================================================")
    print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
    if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
    if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
    print("======================================================================\n")
else:
    print("\n" + "="*70)
    print("❌ SUNUCU BAŞLATILAMADI!")
    print("="*70)
    log_icerik = open(gradio_log_path, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    tb_start = None
    for i, line in enumerate(satirlar):
        if "Traceback (most recent call last)" in line:
            tb_start = i
    print("\n📋 HATA DETAYI:")
    print("-"*70)
    if tb_start is not None:
        print('\n'.join(satirlar[tb_start:]))
    else:
        print('\n'.join(satirlar[-30:]))
    print("-"*70)

    print("\n🔍 TESPİT EDİLEN HATA TÜRLERİ:")
    found = [p.rstrip(':') for p in ERROR_PATTERNS if p in log_icerik]
    if found:
        for t in set(found):
            print(f"   ❌ {t}")
    else:
        print("   (spesifik hata türü yok)")

    print("\n💡 ÖNERİLEN ÇÖZÜMLER:")
    log_low = log_icerik.lower()
    if "hf_transfer" in log_low:
        print("   • hf_transfer sorunu → pip install hf_transfer VEYA env kaldır (bu script yapar)")
    if "cuda out of memory" in log_low:
        print("   • GPU belleği yetersiz → batch size düşürün")
    print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ── 1) GIT RESET ──────────────────────────────────────────
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ── 2) BAĞIMLILIKLAR & STARLETTE/FastAPI SABİTLEME ────────
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q 'starlette<1.0' 'fastapi<0.116' 'jinja2>=3.1,<3.2' 2>&1 | tail -2")
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' 2>&1 | tail -1")
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ── 3) utils.py PATCH ─────────────────────────────────────
print("🩹 scripts/utils.py düzenleniyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    lines = f.readlines()
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i; break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0: break
    lines = lines[:start_idx] + lines[end_idx + 1:]
content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+", "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n", 1)
with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(content)

# ── 4) YAML PATCH ─────────────────────────────────────────
print("🩹 YAML dosyaları düzeltiliyor...")
for yaml_path in set(glob.glob("**/*.yaml", recursive=True)):
    if not os.path.exists(yaml_path): continue
    with open(yaml_path, "r", encoding="utf-8") as f: c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f: f.write(c)

# ── 5) gradio_app.py launch(share=True) ───────────────────
print("🩹 gradio_app.py düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f: gc = f.read()
gc_new = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
                r"\1.queue().launch(share=True, show_error=True, debug=True)", gc)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_new)

# ── 6) MODEL + CONFIG ─────────────────────────────────────
from huggingface_hub import hf_hub_download, snapshot_download
try:
    hf_hub_download("matttrent/sd-image-variations-diffusers", "unet/config.json")
    snapshot_download("matttrent/sd-image-variations-diffusers",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"])
except Exception as e: print(f"   ⚠️  Model: {e}")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = 4; cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ── 7) ESKİ SÜREÇLER ──────────────────────────────────────
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)

# ── 8) SUNUCUYU BAŞLAT ────────────────────────────────────
print("🚀 Sunucu başlatılıyor (arka planda kalıcı)...")
env_vars = os.environ.copy()
env_vars.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0", "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

LOG = os.path.join(WORK_DIR, "gradio.log")
open(LOG, "w").close()

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,   # 🔧 KRİTİK SATIR — hücre bitince ölmez!
)

# ── 9) BEKLEME DÖNGÜSÜ ────────────────────────────────────
print("⏳ Link bekleniyor...")
public_url = None
for elapsed in range(0, 900, 3):
    time.sleep(3)
    if not os.path.exists(LOG): continue
    log_icerik = open(LOG, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son: {satirlar[-1][:100]:<100}", end="")
    gr = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", log_icerik)
    if gr:
        public_url = gr.group(0)
        print(f"\n   ✅ Public URL: {public_url}")
        print("   ⏳ Model yüklemesi için 20sn ek bekleme...")
        time.sleep(20)
        break
    if proc.poll() is not None:
        print(f"\n   ❌ Süreç öldü (kod: {proc.returncode})")
        break

# ── 10) SAĞLIK KONTROLÜ ───────────────────────────────────
print("\n" + "="*70)
print("🩺 SAĞLIK KONTROLÜ")
print("="*70)
time.sleep(2)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 10)

# ── 11) SÜRECİN HÂLÂ CANLI OLDUĞUNU DOĞRULA ───────────────
print("\n" + "="*70)
print("🔍 SÜREÇ DURUMU (hücre bittikten sonra)")
print("="*70)
os.system(f"ps -p {proc.pid} -o pid,ppid,sess,cmd 2>/dev/null || echo '❌ Süreç öldü'")

# ── 12) SONUÇ ─────────────────────────────────────────────
print()
if public_url:
    print("="*70)
    print("🚀 UNIQUE3D HAZIR (arka planda çalışıyor)")
    print("="*70)
    print(f"🌐 LİNK: {public_url}")
    print(f"🆔 PID : {proc.pid}")
    print("="*70)
    print("\n✅ Bu hücre bittikten sonra da sunucu çalışmaya devam edecek.")
    print("   Kontrol:  !ps aux | grep gradio_app")
    print("   Log    :  !tail -30 /kaggle/working/gradio.log")
    print("   Durdur :  !kill -9 " + str(proc.pid))
    print("="*70)
else:
    print("❌ Link alınamadı. Log son 40 satır:")
    print("="*70)
    os.system(f"tail -40 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ── 1) GIT RESET ──────────────────────────────────────────
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ── 2) BAĞIMLILIKLAR & STARLETTE/FastAPI SABİTLEME ────────
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q 'starlette<1.0' 'fastapi<0.116' 'jinja2>=3.1,<3.2' 2>&1 | tail -2")
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' 2>&1 | tail -1")
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ── 3) utils.py PATCH ─────────────────────────────────────
print("🩹 scripts/utils.py düzenleniyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    lines = f.readlines()
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i; break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0: break
    lines = lines[:start_idx] + lines[end_idx + 1:]
content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+", "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n", 1)
with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(content)

# ── 4) YAML PATCH ─────────────────────────────────────────
print("🩹 YAML dosyaları düzeltiliyor...")
for yaml_path in set(glob.glob("**/*.yaml", recursive=True)):
    if not os.path.exists(yaml_path): continue
    with open(yaml_path, "r", encoding="utf-8") as f: c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f: f.write(c)

# ── 5) gradio_app.py launch(share=True) ───────────────────
print("🩹 gradio_app.py düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f: gc = f.read()
gc_new = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
                r"\1.queue().launch(share=True, show_error=True, debug=True)", gc)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_new)

# ── 6) MODEL + CONFIG ─────────────────────────────────────
from huggingface_hub import hf_hub_download, snapshot_download
try:
    hf_hub_download("matttrent/sd-image-variations-diffusers", "unet/config.json")
    snapshot_download("matttrent/sd-image-variations-diffusers",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"])
except Exception as e: print(f"   ⚠️  Model: {e}")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = 4; cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ── 7) ESKİ SÜREÇLER ──────────────────────────────────────
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)

# ── 8) SUNUCUYU BAŞLAT ────────────────────────────────────
print("🚀 Sunucu başlatılıyor (arka planda kalıcı)...")
env_vars = os.environ.copy()
env_vars.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0", "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

LOG = os.path.join(WORK_DIR, "gradio.log")
open(LOG, "w").close()

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,   # 🔧 KRİTİK SATIR
)

# ── 9) BEKLEME DÖNGÜSÜ ────────────────────────────────────
print("⏳ Link bekleniyor...")
public_url = None
for elapsed in range(0, 900, 3):
    time.sleep(3)
    if not os.path.exists(LOG): continue
    log_icerik = open(LOG, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son: {satirlar[-1][:100]:<100}", end="")
    gr = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", log_icerik)
    if gr:
        public_url = gr.group(0)
        print(f"\n   ✅ Public URL: {public_url}")
        print("   ⏳ Model yüklemesi için 20sn ek bekleme...")
        time.sleep(20)
        break
    if proc.poll() is not None:
        print(f"\n   ❌ Süreç öldü (kod: {proc.returncode})")
        break

# ── 10) SAĞLIK KONTROLÜ ───────────────────────────────────
print("\n" + "="*70)
print("🩺 SAĞLIK KONTROLÜ")
print("="*70)
time.sleep(2)

# ✅ DÜZELTİLMİŞ SATIR — sonda " var
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 10")

# ── 11) SÜREÇ DURUMU ──────────────────────────────────────
print("\n" + "="*70)
print("🔍 SÜREÇ DURUMU")
print("="*70)
os.system(f"ps -p {proc.pid} -o pid,ppid,sess,cmd 2>/dev/null || echo '❌ Süreç öldü'")

# ── 12) SONUÇ ─────────────────────────────────────────────
print()
if public_url:
    print("="*70)
    print("🚀 UNIQUE3D HAZIR (arka planda çalışıyor)")
    print("="*70)
    print(f"🌐 LİNK: {public_url}")
    print(f"🆔 PID : {proc.pid}")
    print("="*70)
    print("\n✅ Bu hücre bittikten sonra da sunucu çalışmaya devam edecek.")
    print("   Kontrol:  !ps aux | grep gradio_app")
    print("   Log    :  !tail -30 /kaggle/working/gradio.log")
    print("   Durdur :  !kill -9 " + str(proc.pid))
    print("="*70)
else:
    print("❌ Link alınamadı. Log son 40 satır:")
    print("="*70)
    os.system(f"tail -40 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json, glob

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ============================================================
# 1) GIT RESET
# ============================================================
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ============================================================
# 2) BAĞIMLILIKLAR & GRADIO DÜZELTMESİ (KRİTİK)
# ============================================================
print("📦 Gradio/Starlette uyumsuzluğu gideriliyor...")
# Seçenek A: Gradio'yu güncelle (en temiz çözüm)
os.system("pip install -q 'gradio>=5.29.0' 2>&1 | tail -3")

# Eğer Gradio 5 Unique3D ile uyumsuzluk çıkarırsa, aşağıdaki satırları açın ve
# yukarıdaki satırı yorumlayın:
# os.system("pip install -q 'starlette<1.0' 'fastapi<0.116' 'jinja2==3.1.2' 2>&1 | tail -3")

print("   ✅ Gradio güncellendi (>=5.29.0)")

# Diğer bağımlılıklar
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system(
    "pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' "
    "2>&1 | tail -1"
)
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    print("📦 nvdiffrast kuruluyor...")
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    print("📦 torch_scatter kuruluyor...")
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ============================================================
# 3) utils.py PATCH
# ============================================================
print("🩹 scripts/utils.py düzenleniyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    lines = f.readlines()
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i
        break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0:
            break
    lines = lines[:start_idx] + lines[end_idx + 1:]
content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+",
                     "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n", 1)
with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi")

# ============================================================
# 4) YAML PATCH
# ============================================================
print("🩹 YAML dosyaları düzeltiliyor...")
yaml_files = list(set(
    glob.glob("app/custom_models/*.yaml")
    + glob.glob("gradio_app/custom_models/*.yaml")
    + glob.glob("**/*.yaml", recursive=True)
))
for yaml_path in yaml_files:
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, "r", encoding="utf-8") as f:
        c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers",
                  "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(c)

# ============================================================
# 5) gradio_app.py — launch ayarı
# ============================================================
print("🩹 gradio_app.py düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gcontent = f.read()

# Gradio 5 ile uyumlu: share=True + show_error=True
gcontent_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, show_error=True, debug=True)",
    gcontent
)
if gcontent_new != gcontent:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gcontent_new)
    print("   ✅ gradio_app.py güncellendi (share=True, show_error=True)")
else:
    print("   ⚠️  launch() çağrısı bulunamadı")

# ============================================================
# 6) MODEL + CONFIG
# ============================================================
print("📥 Model doğrulanıyor...")
from huggingface_hub import hf_hub_download, snapshot_download
try:
    hf_hub_download("matttrent/sd-image-variations-diffusers", "unet/config.json")
    snapshot_download(
        "matttrent/sd-image-variations-diffusers",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print("   ✅ Model hazır")
except Exception as e:
    print(f"   ⚠️  Model: {e}")

print("⚙️ config.json 4 kanal...")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)

# ============================================================
# 7) ESKİ SÜREÇLER
# ============================================================
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
time.sleep(2)

# ============================================================
# 8) SUNUCUYU BAŞLAT
# ============================================================
print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["REMBG_PROVIDERS"] = "CPUExecutionProvider"
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

LOG = os.path.join(WORK_DIR, "gradio.log")
with open(LOG, "w") as f:
    f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
)

# ============================================================
# 9) BEKLEME DÖNGÜSÜ
# ============================================================
print("⏳ Link ve model yüklemesi bekleniyor...")
public_url = None
max_wait = 900
elapsed = 0

while elapsed < max_wait:
    time.sleep(3)
    elapsed += 3
    if not os.path.exists(LOG):
        continue
    log_icerik = open(LOG, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]

    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son: {satirlar[-1][:100]:<100}", end="")

    gr_match = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", log_icerik)
    if gr_match:
        public_url = gr_match.group(0)
        # Link geldikten sonra model yüklemesi için 20 sn daha bekle
        print(f"\n   ✅ Public URL alındı: {public_url}")
        print("   ⏳ Model yüklemesi için 20sn ek bekleme...")
        time.sleep(20)
        break

    if proc.poll() is not None:
        print(f"\n   ❌ Süreç öldü (kod: {proc.returncode})")
        break

# ============================================================
# 10) SAĞLIK KONTROLÜ
# ============================================================
print("\n" + "="*70)
print("🩺 SAĞLIK KONTROLÜ")
print("="*70)
time.sleep(2)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 10")

# ============================================================
# 11) SONUÇ
# ============================================================
print()
if public_url:
    print("="*70)
    print("🚀 UNIQUE3D HAZIR!")
    print("="*70)
    print(f"🌐 GRADIO LIVE LİNKİNİZ : {public_url}")
    print("="*70)
    print("\n💡 'Internal Server Error' alırsanız:")
    print("   1. Sayfayı yenileyin (F5) — Gradio 5 güncellemesi sonrası ilk yükleme uzun sürebilir")
    print("   2. Hâlâ hata varsa: `!tail -50 /kaggle/working/gradio.log` çıktısını paylaşın")
    print("="*70)
else:
    print("="*70)
    print("❌ Public URL alınamadı. Log:")
    print("="*70)
    log_icerik = open(LOG, "r", errors="ignore").read() if os.path.exists(LOG) else ""
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    print('\n'.join(satirlar[-40:]))
    print("="*70)

In [ ]:
import os, sys, subprocess, time, re, json, glob, types

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ── 1) GIT RESET ──────────────────────────────────────────
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ── 2) KRİTİK: rembg SAM import'unu EN BAŞTA blokla ──────
# Bu, rembg.sessions.sam -> jsonschema -> isoduration -> arrow -> dateutil
# zincirini tamamen atlar. Import süresi 20sn -> 3sn'ye düşer.
_fake_sam = types.ModuleType("rembg.sessions.sam")
class _FakeSamSession:
    def __init__(self, *a, **k): pass
_fake_sam.SamSession = _FakeSamSession
sys.modules["rembg.sessions.sam"] = _fake_sam
print("✅ rembg SAM modülü bloklandı (import hızlandırıldı)")

# ── 3) BAĞIMLILIKLAR (Starlette/FastAPI sabitleme dahil) ──
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q 'starlette<1.0' 'fastapi<0.116' 'jinja2>=3.1,<3.2' 2>&1 | tail -2")
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' 2>&1 | tail -1")
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ── 4) utils.py PATCH (rembg providers) ───────────────────
print("🩹 scripts/utils.py düzenleniyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    lines = f.readlines()
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i; break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0: break
    lines = lines[:start_idx] + lines[end_idx + 1:]
content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+", "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n", 1)

# 🔧 KRİTİK: utils.py'nin EN BAŞINA sys.modules hack'ini ekle
# Böylece gradio_app.py ayrı süreçte çalışsa bile SAM import'u bloklanır
hack_header = (
    "import sys, types as _t\n"
    "_fake = _t.ModuleType('rembg.sessions.sam')\n"
    "class _FakeSamSession:\n"
    "    def __init__(self, *a, **k): pass\n"
    "_fake.SamSession = _FakeSamSession\n"
    "sys.modules['rembg.sessions.sam'] = _fake\n"
    "\n"
)
if "sys.modules['rembg.sessions.sam']" not in content:
    content = hack_header + content

with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi (SAM blocker eklendi)")

# ── 5) YAML PATCH ─────────────────────────────────────────
print("🩹 YAML dosyaları düzeltiliyor...")
for yaml_path in set(glob.glob("**/*.yaml", recursive=True)):
    if not os.path.exists(yaml_path): continue
    with open(yaml_path, "r", encoding="utf-8") as f: c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f: f.write(c)

# ── 6) gradio_app.py launch(share=True) ───────────────────
print("🩹 gradio_app.py düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f: gc = f.read()
gc_new = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
                r"\1.queue().launch(share=True, show_error=True, debug=True)", gc)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_new)

# ── 7) MODEL + CONFIG ─────────────────────────────────────
from huggingface_hub import hf_hub_download, snapshot_download
try:
    hf_hub_download("matttrent/sd-image-variations-diffusers", "unet/config.json")
    snapshot_download("matttrent/sd-image-variations-diffusers",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"])
except Exception as e: print(f"   ⚠️  Model: {e}")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = 4; cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ── 8) ESKİ SÜREÇLER ──────────────────────────────────────
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)

# ── 9) SUNUCUYU BAŞLAT (kalıcı) ──────────────────────────
print("🚀 Sunucu başlatılıyor (arka planda kalıcı)...")
env_vars = os.environ.copy()
env_vars.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0", "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

LOG = os.path.join(WORK_DIR, "gradio.log")
open(LOG, "w").close()

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,   # 🔧 hücre bitince ölmez
)

# ── 10) BEKLEME DÖNGÜSÜ ──────────────────────────────────
print("⏳ Link bekleniyor...")
public_url = None
for elapsed in range(0, 900, 3):
    time.sleep(3)
    if not os.path.exists(LOG): continue
    log_icerik = open(LOG, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son: {satirlar[-1][:100]:<100}", end="")
    gr = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", log_icerik)
    if gr:
        public_url = gr.group(0)
        print(f"\n   ✅ Public URL: {public_url}")
        print("   ⏳ Model yüklemesi için 20sn ek bekleme...")
        time.sleep(20)
        break
    if proc.poll() is not None:
        print(f"\n   ❌ Süreç öldü (kod: {proc.returncode})")
        break

# ── 11) SAĞLIK KONTROLÜ ──────────────────────────────────
print("\n" + "="*70)
print("🩺 SAĞLIK KONTROLÜ")
print("="*70)
time.sleep(2)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 10")

# ── 12) SÜREÇ DURUMU ─────────────────────────────────────
print("\n" + "="*70)
print("🔍 SÜREÇ DURUMU")
print("="*70)
os.system(f"ps -p {proc.pid} -o pid,ppid,sess,cmd 2>/dev/null || echo '❌ Süreç öldü'")

# ── 13) SONUÇ ────────────────────────────────────────────
print()
if public_url:
    print("="*70)
    print("🚀 UNIQUE3D HAZIR (arka planda çalışıyor)")
    print("="*70)
    print(f"🌐 LİNK: {public_url}")
    print(f"🆔 PID : {proc.pid}")
    print("="*70)
    print("\n✅ Bu hücre bittikten sonra da sunucu çalışmaya devam eder.")
    print("   Kontrol:  !ps aux | grep gradio_app")
    print("   Log    :  !tail -30 /kaggle/working/gradio.log")
    print("   Durdur :  !kill -9 " + str(proc.pid))
    print("="*70)
else:
    print("❌ Link alınamadı. Log son 40 satır:")
    print("="*70)
    os.system(f"tail -40 {LOG}")

In [ ]:
import os, subprocess

print("="*70)
print("🔍 1) SÜREÇ DURUMU")
print("="*70)
os.system("ps aux | grep gradio_app | grep -v grep || echo '❌ gradio_app.py SÜRECİ YOK — çökmüş'")

print("\n" + "="*70)
print("🔍 2) PORT 7860")
print("="*70)
os.system("ss -tlnp 2>/dev/null | grep 7860 || echo '❌ Port dinlenmiyor'")

print("\n" + "="*70)
print("🔍 3) YEREL HTTP")
print("="*70)
os.system("curl -s -o /dev/null -w '   HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

print("\n" + "="*70)
print("🔍 4) GRADIO LOG SON 60 SATIR")
print("="*70)
os.system("tail -60 /kaggle/working/gradio.log")

In [ ]:
import os, sys, subprocess, time, re, json, glob, types

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

# ── 1) GIT RESET ──────────────────────────────────────────
print("🧨 Git reset yapılıyor...")
os.system("git reset --hard HEAD >/dev/null 2>&1")
os.system("git clean -fd >/dev/null 2>&1")

# ── 2) KRİTİK: rembg SAM import'unu EN BAŞTA blokla ──────
# Bu, rembg.sessions.sam -> jsonschema -> isoduration -> arrow -> dateutil
# zincirini tamamen atlar. Import süresi 20sn -> 3sn'ye düşer.
_fake_sam = types.ModuleType("rembg.sessions.sam")
class _FakeSamSession:
    def __init__(self, *a, **k): pass
_fake_sam.SamSession = _FakeSamSession
sys.modules["rembg.sessions.sam"] = _fake_sam
print("✅ rembg SAM modülü bloklandı (import hızlandırıldı)")

# ── 3) BAĞIMLILIKLAR (Starlette/FastAPI sabitleme dahil) ──
print("📦 Bağımlılıklar kuruluyor...")
os.system("pip install -q 'starlette<1.0' 'fastapi<0.116' 'jinja2>=3.1,<3.2' 2>&1 | tail -2")
os.system("pip install -q hf_transfer 2>&1 | tail -1")
os.system("pip install -q 'numpy==2.1.3' 'scipy==1.14.1' 'numba==0.61.0' 'rembg==2.0.62' 2>&1 | tail -1")
if os.system("python -c 'import nvdiffrast.torch' 2>/dev/null") != 0:
    os.system("pip install -q setuptools wheel ninja 2>&1 | tail -1")
    os.system("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev >/dev/null 2>&1")
    os.system("pip install -q --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -2")
if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
    import torch
    TV = torch.__version__.split('+')[0]
    CT = "cpu" if not torch.cuda.is_available() else f"cu{torch.version.cuda.replace('.', '')}"
    os.system(f"pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+{CT}.html 2>&1 | tail -2")
    if os.system("python -c 'import torch_scatter' 2>/dev/null") != 0:
        os.system("pip install -q --no-build-isolation git+https://github.com/rusty1s/pytorch_scatter.git 2>&1 | tail -2")

# ── 4) utils.py PATCH (rembg providers) ───────────────────
print("🩹 scripts/utils.py düzenleniyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    lines = f.readlines()
start_idx = None
for i, line in enumerate(lines):
    if re.match(r"^\s*providers\s*=", line):
        start_idx = i; break
if start_idx is not None:
    depth = lines[start_idx].count('[') - lines[start_idx].count(']')
    end_idx = start_idx
    for j in range(start_idx + 1, len(lines)):
        depth += lines[j].count('[') - lines[j].count(']')
        end_idx = j
        if depth <= 0: break
    lines = lines[:start_idx] + lines[end_idx + 1:]
content = ''.join(lines)
content = re.sub(r"new_session\s*\(\s*providers\s*=\s*[^)]+\)",
                 "new_session(providers=['CPUExecutionProvider'])", content)
if "from rembg import" not in content:
    content = "from rembg import new_session, remove\n" + content
elif "new_session" not in content.split("from rembg import")[1].split('\n')[0]:
    content = re.sub(r"from rembg import [^\n]+", "from rembg import new_session, remove", content, count=1)
if "session = new_session(" not in content:
    content = content.replace(
        "from rembg import new_session, remove\n",
        "from rembg import new_session, remove\n\nsession = new_session(providers=['CPUExecutionProvider'])\n", 1)

# 🔧 KRİTİK: utils.py'nin EN BAŞINA sys.modules hack'ini ekle
# Böylece gradio_app.py ayrı süreçte çalışsa bile SAM import'u bloklanır
hack_header = (
    "import sys, types as _t\n"
    "_fake = _t.ModuleType('rembg.sessions.sam')\n"
    "class _FakeSamSession:\n"
    "    def __init__(self, *a, **k): pass\n"
    "_fake.SamSession = _FakeSamSession\n"
    "sys.modules['rembg.sessions.sam'] = _fake\n"
    "\n"
)
if "sys.modules['rembg.sessions.sam']" not in content:
    content = hack_header + content

with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(content)
print("   ✅ utils.py güncellendi (SAM blocker eklendi)")

# ── 5) YAML PATCH ─────────────────────────────────────────
print("🩹 YAML dosyaları düzeltiliyor...")
for yaml_path in set(glob.glob("**/*.yaml", recursive=True)):
    if not os.path.exists(yaml_path): continue
    with open(yaml_path, "r", encoding="utf-8") as f: c = f.read()
    orig = c
    c = re.sub(r'^\s*revision:\s*["\'].*["\']\s*\n', '', c, flags=re.MULTILINE)
    c = c.replace("lambdalabs/sd-image-variations-diffusers", "matttrent/sd-image-variations-diffusers")
    if c != orig:
        with open(yaml_path, "w", encoding="utf-8") as f: f.write(c)

# ── 6) gradio_app.py launch(share=True) ───────────────────
print("🩹 gradio_app.py düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f: gc = f.read()
gc_new = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
                r"\1.queue().launch(share=True, show_error=True, debug=True)", gc)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f: f.write(gc_new)

# ── 7) MODEL + CONFIG ─────────────────────────────────────
from huggingface_hub import hf_hub_download, snapshot_download
try:
    hf_hub_download("matttrent/sd-image-variations-diffusers", "unet/config.json")
    snapshot_download("matttrent/sd-image-variations-diffusers",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"])
except Exception as e: print(f"   ⚠️  Model: {e}")
for cfg_path in ["ckpt/unet_4ch/unet/config.json", "ckpt/img2mvimg/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = 4; cfg.pop("num_class_embeds", None); cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ── 8) ESKİ SÜREÇLER ──────────────────────────────────────
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)

# ── 9) SUNUCUYU BAŞLAT (kalıcı) ──────────────────────────
print("🚀 Sunucu başlatılıyor (arka planda kalıcı)...")
env_vars = os.environ.copy()
env_vars.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0", "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env_vars["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

LOG = os.path.join(WORK_DIR, "gradio.log")
open(LOG, "w").close()

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env_vars,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,   # 🔧 hücre bitince ölmez
)

# ── 10) BEKLEME DÖNGÜSÜ ──────────────────────────────────
print("⏳ Link bekleniyor...")
public_url = None
for elapsed in range(0, 900, 3):
    time.sleep(3)
    if not os.path.exists(LOG): continue
    log_icerik = open(LOG, "r", errors="ignore").read()
    satirlar = [l for l in log_icerik.strip().split('\n') if l.strip()]
    if satirlar:
        print(f"\r⏳ ({elapsed:3d}s) Son: {satirlar[-1][:100]:<100}", end="")
    gr = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", log_icerik)
    if gr:
        public_url = gr.group(0)
        print(f"\n   ✅ Public URL: {public_url}")
        print("   ⏳ Model yüklemesi için 20sn ek bekleme...")
        time.sleep(20)
        break
    if proc.poll() is not None:
        print(f"\n   ❌ Süreç öldü (kod: {proc.returncode})")
        break

# ── 11) SAĞLIK KONTROLÜ ──────────────────────────────────
print("\n" + "="*70)
print("🩺 SAĞLIK KONTROLÜ")
print("="*70)
time.sleep(2)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 10")

# ── 12) SÜREÇ DURUMU ─────────────────────────────────────
print("\n" + "="*70)
print("🔍 SÜREÇ DURUMU")
print("="*70)
os.system(f"ps -p {proc.pid} -o pid,ppid,sess,cmd 2>/dev/null || echo '❌ Süreç öldü'")

# ── 13) SONUÇ ────────────────────────────────────────────
print()
if public_url:
    print("="*70)
    print("🚀 UNIQUE3D HAZIR (arka planda çalışıyor)")
    print("="*70)
    print(f"🌐 LİNK: {public_url}")
    print(f"🆔 PID : {proc.pid}")
    print("="*70)
    print("\n✅ Bu hücre bittikten sonra da sunucu çalışmaya devam eder.")
    print("   Kontrol:  !ps aux | grep gradio_app")
    print("   Log    :  !tail -30 /kaggle/working/gradio.log")
    print("   Durdur :  !kill -9 " + str(proc.pid))
    print("="*70)
else:
    print("❌ Link alınamadı. Log son 40 satır:")
    print("="*70)
    os.system(f"tail -40 {LOG}")

In [ ]:
import os, subprocess

print("="*70)
print("🔍 1) SÜREÇ DURUMU")
print("="*70)
os.system("ps aux | grep gradio_app | grep -v grep || echo '❌ gradio_app.py SÜRECİ YOK — çökmüş'")

print("\n" + "="*70)
print("🔍 2) PORT 7860")
print("="*70)
os.system("ss -tlnp 2>/dev/null | grep 7860 || echo '❌ Port dinlenmiyor'")

print("\n" + "="*70)
print("🔍 3) YEREL HTTP")
print("="*70)
os.system("curl -s -o /dev/null -w '   HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

print("\n" + "="*70)
print("🔍 4) GRADIO LOG SON 60 SATIR")
print("="*70)
os.system("tail -60 /kaggle/working/gradio.log")

In [ ]:
import os, sys, subprocess, time, re, json, glob, types

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧨 Eski süreçler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

print("🩹 gradio_app.py düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()

# Güvenlik denetleyicisini kapat ve share=True ekle
gc = re.sub(r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
            r"\1.queue().launch(share=True, show_error=True, inbrowser=False)", gc)
if "safety_checker=None" not in gc:
    gc = re.sub(r"(StableDiffusionPipeline\.from_pretrained\([^)]*\))",
                r"\1, safety_checker=None", gc)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc)
print("   ✅ gradio_app.py güncellendi")

# rembg SAM engelleyici (isteğe bağlı, hız için)
_fake_sam = types.ModuleType("rembg.sessions.sam")
class _FakeSamSession:
    def __init__(self, *a, **k): pass
_fake_sam.SamSession = _FakeSamSession
sys.modules["rembg.sessions.sam"] = _fake_sam

print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir, "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0", "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"   PID: {proc.pid}")

# Canlı log takibi
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0
for elapsed in range(0, 900, 5):
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modeller yükleniyor, 60sn bekleyip tıklayın...\n")

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n🌐 LİNK: {public_url}")
    print("✅ Sunucu çalışıyor.")
else:
    print("\n❌ Süreç çöktü veya link alınamadı. Son 40 satır:")
    print("="*70)
    os.system(f"tail -40 {LOG}")
    print("="*70)

In [ ]:
!tail -50 /kaggle/working/gradio.log

In [ ]:
!pip install -q --upgrade "gradio>=4.44.1"

In [ ]:
import os, sys, subprocess, time, re, json, glob, types

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

# ══════════════════════════════════════════════════════════
# 2) CONFIG.JSON 4 KANAL SABİTLEME (conv_in.weight hatası için)
# ══════════════════════════════════════════════════════════
print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

# ══════════════════════════════════════════════════════════
# 3) REMBG SAM ENGELLEYİCİ (Hız için)
# ══════════════════════════════════════════════════════════
_fake_sam = types.ModuleType("rembg.sessions.sam")
class _FakeSamSession:
    def __init__(self, *a, **k): pass
_fake_sam.SamSession = _FakeSamSession
sys.modules["rembg.sessions.sam"] = _fake_sam

# utils.py içine de ekle (alt süreç için)
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    content = f.read()
if "rembg.sessions.sam" not in content:
    hack = (
        "import sys as _s, types as _t\n"
        "_f=_t.ModuleType('rembg.sessions.sam')\n"
        "class _FS:\n"
        "    def __init__(self,*a,**k):pass\n"
        "_f.SamSession=_FS\n"
        "_s.modules['rembg.sessions.sam']=_f\n\n"
    )
    content = hack + content
    with open("scripts/utils.py", "w", encoding="utf-8") as f:
        f.write(content)

# ══════════════════════════════════════════════════════════
# 4) GRADIO_APP.PY DÜZENLEME (show_api=False ve share=True)
# ══════════════════════════════════════════════════════════
print("🩹 gradio_app.py düzenleniyor (show_api=False, share=True)...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()

# Regex ile launch() kısmını bul ve değiştir
gc_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, show_error=True, show_api=False, inbrowser=False)",
    gc
)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc_new)
    print("   ✅ gradio_app.py güncellendi")

# ══════════════════════════════════════════════════════════
# 5) SUNUCUYU BAŞLAT (Arka planda kalıcı)
# ══════════════════════════════════════════════════════════
print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,  # Hücre bitince ölmez
)
print(f"   PID: {proc.pid}")

# ══════════════════════════════════════════════════════════
# 6) CANLI LOG TAKİBİ VE LİNK YAKALAMA
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0

for elapsed in range(0, 600, 5):  # Maksimum 10 dakika bekle
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        # Son 2 satırı ekrana bas
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modellerin yüklenmesi için 30-60sn bekleyip linke tıklayın...\n")

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

# ══════════════════════════════════════════════════════════
# 7) FİNAL RAPOR VE SAĞLIK KONTROLÜ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n✅ SUNUCU ARKA PLANDA ÇALIŞIYOR")
    print(f"🌐 LİNK: {public_url}")
    print("💡 Eğer link açılmazsa 30 saniye bekleyip F5 yapın.")
elif proc.poll() is not None:
    print("\n❌ SÜREÇ ÇÖKTÜ. Log'un son 40 satırı:")
    print("-"*70)
    os.system(f"tail -40 {LOG}")
    print("-"*70)
else:
    print("\n⚠️  Link henüz alınamadı. Log'u kontrol edin:")
    os.system(f"tail -30 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json, types

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ORTAMI TEMİZLE VE SÜRÜMLERİ SABİTLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

print("📦 Sürüm çakışmaları gideriliyor (transformers & huggingface-hub)...")
# transformers 5.x sürümleri huggingface-hub<2.0 ister. 
# Ortamda 2.3.0 olduğu için ikisini de uyumlu sürümlere sabitliyoruz.
os.system("pip install -q 'transformers>=4.41.0,<5.0' 'huggingface-hub>=0.30.0,<1.0' 'accelerate>=0.30.0' 2>&1 | tail -3")
print("   ✅ Sürümler sabitlendi.")

# ══════════════════════════════════════════════════════════
# 2) MODEL YAPILANDIRMASI (4 KANAL SABİTLEME)
# ══════════════════════════════════════════════════════════
print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

# ══════════════════════════════════════════════════════════
# 3) REMBG SAM ENGELLEYİCİ
# ══════════════════════════════════════════════════════════
_fake_sam = types.ModuleType("rembg.sessions.sam")
class _FakeSamSession:
    def __init__(self, *a, **k): pass
_fake_sam.SamSession = _FakeSamSession
sys.modules["rembg.sessions.sam"] = _fake_sam

# utils.py içine de ekle
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    content = f.read()
if "rembg.sessions.sam" not in content:
    hack = (
        "import sys as _s, types as _t\n"
        "_f=_t.ModuleType('rembg.sessions.sam')\n"
        "class _FS:\n"
        "    def __init__(self,*a,**k):pass\n"
        "_f.SamSession=_FS\n"
        "_s.modules['rembg.sessions.sam']=_f\n\n"
    )
    content = hack + content
    with open("scripts/utils.py", "w", encoding="utf-8") as f:
        f.write(content)

# ══════════════════════════════════════════════════════════
# 4) GRADIO_APP.PY DÜZENLEME (API HATASINI ÖNLEME VE SHARE)
# ══════════════════════════════════════════════════════════
print("🩹 gradio_app.py düzenleniyor (show_api=False, share=True)...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()

gc_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, show_error=True, show_api=False, inbrowser=False)",
    gc
)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc_new)
    print("   ✅ gradio_app.py güncellendi")

# ══════════════════════════════════════════════════════════
# 5) SUNUCUYU BAŞLAT (Arka planda kalıcı)
# ══════════════════════════════════════════════════════════
print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,  # Hücre bitince ölmez
)
print(f"   PID: {proc.pid}")

# ══════════════════════════════════════════════════════════
# 6) CANLI LOG TAKİBİ VE LİNK YAKALAMA
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0

for elapsed in range(0, 600, 5):
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modellerin yüklenmesi için 30-60sn bekleyip linke tıklayın...\n")

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

# ══════════════════════════════════════════════════════════
# 7) FİNAL RAPOR VE SAĞLIK KONTROLÜ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n✅ SUNUCU ARKA PLANDA ÇALIŞIYOR")
    print(f"🌐 LİNK: {public_url}")
    print("💡 Eğer link açılmazsa 30 saniye bekleyip F5 yapın.")
elif proc.poll() is not None:
    print("\n❌ SÜREÇ ÇÖKTÜ. Log'un son 40 satırı:")
    print("-"*70)
    os.system(f"tail -40 {LOG}")
    print("-"*70)
else:
    print("\n⚠️  Link henüz alınamadı. Log'u kontrol edin:")
    os.system(f"tail -30 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json, types

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

# ══════════════════════════════════════════════════════════
# 2) SÜRÜM ÇAKIŞMALARINI GİDER (Gradio 6 + Diffusers + Transformers)
# ══════════════════════════════════════════════════════════
print("📦 Sürümler uyumlu hale getiriliyor...")
# Gradio 6.x için huggingface-hub >=1.16 gerekir. 
# Ancak Unique3D'nin diffusers sürümü cached_download arıyor. 
# En temiz çözüm: diffusers'ı güncellemek ve gerekli paketleri kurmak.
os.system("pip install -q 'diffusers>=0.36.0' 'transformers>=4.41.0,<5.0' 'accelerate>=0.30.0' 'huggingface-hub>=1.16.0,<3.0' 2>&1 | tail -3")
print("   ✅ Sürümler güncellendi.")

# ══════════════════════════════════════════════════════════
# 3) KRİTİK YAMA: cached_download SHIM'İ
# ══════════════════════════════════════════════════════════
# Eğer güncelleme yetmezse, huggingface_hub'a cached_download'u geri ekle.
print("🩹 huggingface_hub'a cached_download shim'i ekleniyor...")
shim_code = """
import huggingface_hub
if not hasattr(huggingface_hub, 'cached_download'):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
    print("   ✅ cached_download shim'i aktif edildi.")
"""

# Bu shim'i hem ana sürece hem de utils.py'ye ekle
# (utils.py alt süreçte çalıştığı için oraya da yazmalıyız)
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    content = f.read()
if "huggingface_hub.cached_download = huggingface_hub.hf_hub_download" not in content:
    # Mevcut SAM hack'inin üstüne veya en başa ekle
    hack = (
        "import huggingface_hub as _hf\n"
        "if not hasattr(_hf, 'cached_download'):\n"
        "    _hf.cached_download = _hf.hf_hub_download\n"
    )
    content = hack + content
    with open("scripts/utils.py", "w", encoding="utf-8") as f:
        f.write(content)
    print("   ✅ utils.py içine shim eklendi.")

# Ana süreçte de aktif et
exec(shim_code)

# ══════════════════════════════════════════════════════════
# 4) MODEL YAPILANDIRMASI (4 KANAL SABİTLEME)
# ══════════════════════════════════════════════════════════
print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

# ══════════════════════════════════════════════════════════
# 5) GRADIO_APP.PY DÜZENLEME (API HATASINI ÖNLEME VE SHARE)
# ══════════════════════════════════════════════════════════
print("🩹 gradio_app.py düzenleniyor (show_api=False, share=True)...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
gc_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, show_error=True, show_api=False, inbrowser=False)",
    gc
)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc_new)
    print("   ✅ gradio_app.py güncellendi")

# ══════════════════════════════════════════════════════════
# 6) SUNUCUYU BAŞLAT (Arka planda kalıcı)
# ══════════════════════════════════════════════════════════
print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,  # Hücre bitince ölmez
)
print(f"   PID: {proc.pid}")

# ══════════════════════════════════════════════════════════
# 7) CANLI LOG TAKİBİ VE LİNK YAKALAMA
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0

for elapsed in range(0, 600, 5):
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modellerin yüklenmesi için 30-60sn bekleyip linke tıklayın...\n")

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

# ══════════════════════════════════════════════════════════
# 8) FİNAL RAPOR VE SAĞLIK KONTROLÜ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n✅ SUNUCU ARKA PLANDA ÇALIŞIYOR")
    print(f"🌐 LİNK: {public_url}")
    print("💡 Eğer link açılmazsa 30 saniye bekleyip F5 yapın.")
elif proc.poll() is not None:
    print("\n❌ SÜREÇ ÇÖKTÜ. Log'un son 40 satırı:")
    print("-"*70)
    os.system(f"tail -40 {LOG}")
    print("-"*70)
else:
    print("\n⚠️  Link henüz alınamadı. Log'u kontrol edin:")
    os.system(f"tail -30 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

# ══════════════════════════════════════════════════════════
# 2) HUGGINGFACE_HUB YAMASI (cached_download)
# ══════════════════════════════════════════════════════════
print("🩹 huggingface_hub 'cached_download' yaması uygulanıyor...")
shim_code = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download
"""

# Bu yamayı gradio_app.py dosyasının en başına ekle
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
if "cached_download" not in gc:
    gc = shim_code + "\n" + gc
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc)
    print("   ✅ gradio_app.py içine yama eklendi")

# Ana süreçte de aktif et
exec(shim_code)

# ══════════════════════════════════════════════════════════
# 3) MODEL YAPILANDIRMASI (4 KANAL SABİTLEME)
# ══════════════════════════════════════════════════════════
print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

# ══════════════════════════════════════════════════════════
# 4) GRADIO_APP.PY DÜZENLEME (API HATASINI ÖNLEME VE SHARE)
# ══════════════════════════════════════════════════════════
print("🩹 gradio_app.py launch ayarları düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
gc_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, show_error=True, show_api=False, inbrowser=False)",
    gc
)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc_new)
    print("   ✅ gradio_app.py güncellendi")

# ══════════════════════════════════════════════════════════
# 5) SUNUCUYU BAŞLAT (Arka planda kalıcı)
# ══════════════════════════════════════════════════════════
print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"   PID: {proc.pid}")

# ══════════════════════════════════════════════════════════
# 6) CANLI LOG TAKİBİ VE LİNK YAKALAMA
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0

for elapsed in range(0, 600, 5):
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modellerin yüklenmesi için 30-60sn bekleyip linke tıklayın...\n")

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

# ══════════════════════════════════════════════════════════
# 7) FİNAL RAPOR VE SAĞLIK KONTROLÜ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n✅ SUNUCU ARKA PLANDA ÇALIŞIYOR")
    print(f"🌐 LİNK: {public_url}")
    print("💡 Eğer link açılmazsa 30 saniye bekleyip F5 yapın.")
elif proc.poll() is not None:
    print("\n❌ SÜREÇ ÇÖKTÜ. Log'un son 40 satırı:")
    print("-"*70)
    os.system(f"tail -40 {LOG}")
    print("-"*70)
else:
    print("\n⚠️  Link henüz alınamadı. Log'u kontrol edin:")
    os.system(f"tail -30 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

# ══════════════════════════════════════════════════════════
# 2) GRADIO_3DGEN.PY DÜZELTME (show_share_button kaldır)
# ══════════════════════════════════════════════════════════
print("🩹 app/gradio_3dgen.py düzeltiliyor (show_share_button kaldırılıyor)...")
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # show_share_button=True veya show_share_button=False parametrelerini temizle
    # Örnek: gr.Video(..., show_share_button=True, ...) -> gr.Video(..., ...)
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    # Eğer parametre başta ise (nadir): show_share_button=True, ... -> ...
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("   ✅ app/gradio_3dgen.py güncellendi")

# ══════════════════════════════════════════════════════════
# 3) HUGGINGFACE_HUB YAMASI (cached_download)
# ══════════════════════════════════════════════════════════
print("🩹 huggingface_hub 'cached_download' yaması uygulanıyor...")
shim_code = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download
"""
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
if "cached_download" not in gc:
    gc = shim_code + "\n" + gc
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc)
    print("   ✅ gradio_app.py içine yama eklendi")
exec(shim_code)

# ══════════════════════════════════════════════════════════
# 4) MODEL YAPILANDIRMASI (4 KANAL SABİTLEME)
# ══════════════════════════════════════════════════════════
print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

# ══════════════════════════════════════════════════════════
# 5) GRADIO_APP.PY LAUNCH AYARLARI
# ══════════════════════════════════════════════════════════
print("🩹 gradio_app.py launch ayarları düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
gc_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, show_error=True, show_api=False, inbrowser=False)",
    gc
)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc_new)
    print("   ✅ gradio_app.py güncellendi")

# ══════════════════════════════════════════════════════════
# 6) SUNUCUYU BAŞLAT (Arka planda kalıcı)
# ══════════════════════════════════════════════════════════
print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"   PID: {proc.pid}")

# ══════════════════════════════════════════════════════════
# 7) CANLI LOG TAKİBİ VE LİNK YAKALAMA
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0

for elapsed in range(0, 600, 5):
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modellerin yüklenmesi için 30-60sn bekleyip linke tıklayın...\n")

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

# ══════════════════════════════════════════════════════════
# 8) FİNAL RAPOR VE SAĞLIK KONTROLÜ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n✅ SUNUCU ARKA PLANDA ÇALIŞIYOR")
    print(f"🌐 LİNK: {public_url}")
    print("💡 Eğer link açılmazsa 30 saniye bekleyip F5 yapın.")
elif proc.poll() is not None:
    print("\n❌ SÜREÇ ÇÖKTÜ. Log'un son 40 satırı:")
    print("-"*70)
    os.system(f"tail -40 {LOG}")
    print("-"*70)
else:
    print("\n⚠️  Link henüz alınamadı. Log'u kontrol edin:")
    os.system(f"tail -30 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

# ══════════════════════════════════════════════════════════
# 2) GRADIO_3DGEN.PY DÜZELTME (show_share_button kaldır)
# ══════════════════════════════════════════════════════════
print("🩹 app/gradio_3dgen.py düzeltiliyor (show_share_button kaldırılıyor)...")
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("   ✅ app/gradio_3dgen.py güncellendi")

# ══════════════════════════════════════════════════════════
# 3) HUGGINGFACE_HUB YAMASI (cached_download)
# ══════════════════════════════════════════════════════════
print("🩹 huggingface_hub 'cached_download' yaması uygulanıyor...")
shim_code = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download
"""
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
if "cached_download" not in gc:
    gc = shim_code + "\n" + gc
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc)
    print("   ✅ gradio_app.py içine yama eklendi")
exec(shim_code)

# ══════════════════════════════════════════════════════════
# 4) MODEL YAPILANDIRMASI (4 KANAL SABİTLEME)
# ══════════════════════════════════════════════════════════
print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

# ══════════════════════════════════════════════════════════
# 5) GRADIO_APP.PY LAUNCH AYARLARI (show_api HATASI GİDERİLDİ)
# ══════════════════════════════════════════════════════════
print("🩹 gradio_app.py launch ayarları düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
gc_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc
)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc_new)
    print("   ✅ gradio_app.py güncellendi")

# ══════════════════════════════════════════════════════════
# 6) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"   PID: {proc.pid}")

# ══════════════════════════════════════════════════════════
# 7) CANLI LOG TAKİBİ VE LİNK YAKALAMA
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0

for elapsed in range(0, 600, 5):
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modellerin yüklenmesi için 30-60sn bekleyip linke tıklayın...\n")
            break

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

# ══════════════════════════════════════════════════════════
# 8) FİNAL RAPOR VE SAĞLIK KONTROLÜ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n✅ SUNUCU ARKA PLANDA ÇALIŞIYOR")
    print(f"🌐 LİNK: {public_url}")
    print("💡 Eğer link açılmazsa 30 saniye bekleyip F5 yapın.")
elif proc.poll() is not None:
    print("\n❌ SÜREÇ ÇÖKTÜ. Log'un son 40 satırı:")
    print("-"*70)
    os.system(f"tail -40 {LOG}")
    print("-"*70)
else:
    print("\n⚠️  Link henüz alınamadı. Log'u kontrol edin:")
    os.system(f"tail -30 {LOG}")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)

print("⚙️ Model kanalları doğru değerlere sabitleniyor (img2mvimg=8, unet_4ch=4)...")
# Multiview UNet 8 kanal ister
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# Standard UNet 4 kanal ister
cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0"
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

print("\n" + "="*70)
print("📺 CANLI İŞLEM VE ÜRETİM LOGLARI (Sitede Generate'e bastığınızda burayı izleyin)")
print("="*70)

last_size = 0
while proc.poll() is None:
    time.sleep(2)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir}")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)

# ══════════════════════════════════════════════════════════
# 2) MODEL KANALLARINI SABİTLE (img2mvimg=8, unet_4ch=4)
# ══════════════════════════════════════════════════════════
print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 3) VRAM VE GRADIO UYUMLULUK YAMALARI (OOM ENGELLEME)
# ══════════════════════════════════════════════════════════
print("🩹 VRAM Bellek Temizleme ve Gradio yamaları uygulanıyor...")

# normal_prediction.py içine VRAM temizleme ve attention slicing yaması ekliyoruz
norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f:
        norm_code = f.read()
    if "torch.cuda.empty_cache()" not in norm_code:
        patch_vram = """
import torch, gc
gc.collect()
torch.cuda.empty_cache()
"""
        norm_code = norm_code.replace("def predict_normals(", patch_vram + "\ndef predict_normals(")
        # pipeline çağrısından önce attention slicing aktif edelim
        norm_code = norm_code.replace("pipeline.to(", "if hasattr(pipeline, 'enable_attention_slicing'): pipeline.enable_attention_slicing()\n    pipeline.to(")
        with open(norm_path, "w", encoding="utf-8") as f:
            f.write(norm_code)
        print("   ✅ VRAM temizleme yaması normal_prediction.py dosyasına uygulandı")

# gradio_3dgen.py parametre temizliği
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

# gradio_app.py launch ve torch shim
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

# ══════════════════════════════════════════════════════════
# 5) CANLI LOG VE ÜRETİM TAKİBİ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)

# ══════════════════════════════════════════════════════════
# 2) MODEL KANALLARINI SABİTLE
# ══════════════════════════════════════════════════════════
print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 3) DERİN VRAM VE ATTENTION SLICING YAMALARI
# ══════════════════════════════════════════════════════════
print("🩹 Derin VRAM temizliği ve Attention Slicing yamaları uygulanıyor...")

# Multiview inference aşamasında normal tahmininden önce VRAM'i sıfırla
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    if "gc.collect()" not in mvi_code:
        mvi_code = mvi_code.replace(
            "rm_normals = predict_normals(",
            "import gc, torch\ngc.collect()\ntorch.cuda.empty_cache()\nrm_normals = predict_normals("
        )
        with open(mvi_path, "w", encoding="utf-8") as f:
            f.write(mvi_code)
        print("   ✅ Normal tahmini öncesine VRAM temizleme eklendi")

# Model yükleme yardımcısına Attention Slicing aktif etme eklendi
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code += """

def apply_vram_opt(pipe):
    if hasattr(pipe, 'enable_attention_slicing'):
        pipe.enable_attention_slicing(1)
    if hasattr(pipe, 'enable_vae_slicing'):
        pipe.enable_vae_slicing()
    return pipe
"""
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
        print("   ✅ Pipeline yükleyicilere Attention Slicing entegre edildi")

# Gradio arayüz parametre temizlikleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:64",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

# ══════════════════════════════════════════════════════════
# 5) CANLI LOG VE ÜRETİM TAKİBİ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ VE BOZUK DOSYALARI TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve girintisi bozulan dosyalar sıfırlanıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
os.system("git checkout scripts/multiview_inference.py app/custom_models/normal_prediction.py app/custom_models/utils.py >/dev/null 2>&1")
time.sleep(2)

# ══════════════════════════════════════════════════════════
# 2) MODEL KANALLARINI SABİTLE
# ══════════════════════════════════════════════════════════
print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 3) DÜZGÜN GİRİNTİLİ VRAM & ATTENTION SLICING YAMALARI
# ══════════════════════════════════════════════════════════
print("🩹 VRAM temizleme ve Attention Slicing yamaları (girinti korumalı) uygulanıyor...")

# normal_prediction.py fonksiyonunun içine tek satırlık VRAM temizliği
norm_path = "app/custom_models/normal_prediction.py"
if os.path.exists(norm_path):
    with open(norm_path, "r", encoding="utf-8") as f:
        norm_code = f.read()
    clean_stmt = "    import gc, torch; gc.collect(); torch.cuda.empty_cache()\n"
    norm_code = norm_code.replace("def predict_normals(", "def predict_normals(\n" + clean_stmt)
    with open(norm_path, "w", encoding="utf-8") as f:
        f.write(norm_code)
    print("   ✅ normal_prediction.py için VRAM temizleyici eklendi")

# utils.py içine Attention Slicing entegrasyonu
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    if "enable_attention_slicing" not in u_code:
        u_code += "\ndef apply_vram_opt(p):\n    if hasattr(p, 'enable_attention_slicing'): p.enable_attention_slicing(1)\n    if hasattr(p, 'enable_vae_slicing'): p.enable_vae_slicing()\n    return p\n"
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
        print("   ✅ utils.py içine VRAM optimizasyonları eklendi")

# Gradio arayüz parametre temizlikleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:64",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

# ══════════════════════════════════════════════════════════
# 5) CANLI LOG VE ÜRETİM TAKİBİ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE VE GPU'YU BOŞALT
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor ve VRAM temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(2)
os.system("python -c 'import torch; torch.cuda.empty_cache()' >/dev/null 2>&1")

# ══════════════════════════════════════════════════════════
# 2) MODEL KANALLARINI SABİTLE (Hata veren kısım)
# ══════════════════════════════════════════════════════════
print("⚙️ Model kanal yapılandırmaları kontrol ediliyor...")
cfg_mv = "ckpt/img2mvimg/unet/config.json"
if os.path.exists(cfg_mv):
    with open(cfg_mv, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 8
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_mv, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

cfg_4ch = "ckpt/unet_4ch/unet/config.json"
if os.path.exists(cfg_4ch):
    with open(cfg_4ch, "r", encoding="utf-8") as f: cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_4ch, "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

# ══════════════════════════════════════════════════════════
# 3) DERİN VRAM VE ATTENTION SLICING YAMALARI (Geliştirildi)
# ══════════════════════════════════════════════════════════
print("🩹 Derin VRAM temizliği, CPU Offload ve Attention Slicing yamaları uygulanıyor...")

# Multiview inference aşamasında normal tahmininden önce VRAM'i sıfırla
mvi_path = "scripts/multiview_inference.py"
if os.path.exists(mvi_path):
    with open(mvi_path, "r", encoding="utf-8") as f:
        mvi_code = f.read()
    if "gc.collect()" not in mvi_code:
        mvi_code = mvi_code.replace(
            "rm_normals = predict_normals(",
            "import gc, torch\ngc.collect()\ntorch.cuda.empty_cache()\nrm_normals = predict_normals("
        )
        with open(mvi_path, "w", encoding="utf-8") as f:
            f.write(mvi_code)
        print("   ✅ Normal tahmini öncesine VRAM temizleme eklendi")

# Model yükleme yardımcısına GELİŞMİŞ VRAM Optimizasyonları ekle
utils_path = "app/custom_models/utils.py"
if os.path.exists(utils_path):
    with open(utils_path, "r", encoding="utf-8") as f:
        u_code = f.read()
    
    if "enable_attention_slicing" not in u_code:
        vram_opt_code = """

def apply_vram_opt(pipe):
    try:
        # 1. Attention Slicing: Dikkat katmanlarını parçalara bölerek VRAM'i düşürür
        if hasattr(pipe, 'enable_attention_slicing'):
            pipe.enable_attention_slicing("max")
        
        # 2. VAE Slicing ve Tiling: Görüntü çözünürlüğünü parçalara bölerek VRAM'i düşürür
        if hasattr(pipe, 'enable_vae_slicing'):
            pipe.enable_vae_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()

        # 3. Model CPU Offload: Kullanılmayan model ağırlıklarını RAM'de tutar (EN ÖNEMLİSİ)
        if hasattr(pipe, 'enable_sequential_cpu_offload'):
            pipe.enable_sequential_cpu_offload()
        elif hasattr(pipe, 'enable_model_cpu_offload'):
            pipe.enable_model_cpu_offload()
            
    except Exception as e:
        print(f"VRAM Optimizasyon Uyarısı: {e}")
    return pipe
"""
        u_code += vram_opt_code
        # Pipeline return edilmeden hemen önce fonksiyonu çağır
        u_code = u_code.replace("return pipeline", "pipeline = apply_vram_opt(pipeline)\n    return pipeline")
        with open(utils_path, "w", encoding="utf-8") as f:
            f.write(u_code)
        print("   ✅ Pipeline yükleyicilere Gelişmiş VRAM Optimizasyonu entegre edildi")

# Gradio arayüz parametre temizlikleri
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc_code = f.read()

# Gradio ana dosyasına yamaları ekle
patches = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download

import torch, gc
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""
if "cached_download" not in gc_code:
    gc_code = patches + "\n" + gc_code

gc_code = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc_code
)

with open("gradio_app.py", "w", encoding="utf-8") as f:
    f.write(gc_code)

# ══════════════════════════════════════════════════════════
# 4) SUNUCUYU BAŞLAT (Gelişmiş Bellek Yönetimi İle)
# ══════════════════════════════════════════════════════════
print("🚀 Unique3D Stüdyo başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    # Bellek parçalanmasını önlemek için gelişmiş ayar
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128", 
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True
)

# ══════════════════════════════════════════════════════════
# 5) CANLI LOG VE ÜRETİM TAKİBİ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI İŞLEM VE LİNK TAKİBİ")
print("="*70)

public_url = None
last_size = 0

while proc.poll() is None:
    time.sleep(3)
    if os.path.exists(LOG):
        boyut = os.path.getsize(LOG)
        if boyut > last_size:
            with open(LOG, errors="ignore") as f:
                f.seek(last_size)
                yeni = f.read()
                last_size = boyut
                for satir in yeni.strip().split('\n'):
                    if satir.strip():
                        print(f"👉 {satir[:140]}")

        if not public_url:
            icerik = open(LOG, errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
            if m:
                public_url = m.group(0)
                print("\n" + "="*70)
                print(f"🌐 LİNKİNİZ HAZIR: {public_url}")
                print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)
LOG = os.path.join(WORK_DIR, "gradio.log")

# ══════════════════════════════════════════════════════════
# 1) ESKİ SÜREÇLERİ TEMİZLE
# ══════════════════════════════════════════════════════════
print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
time.sleep(3)

# ══════════════════════════════════════════════════════════
# 2) GRADIO_3DGEN.PY DÜZELTME (show_share_button kaldır)
# ══════════════════════════════════════════════════════════
print("🩹 app/gradio_3dgen.py düzeltiliyor (show_share_button kaldırılıyor)...")
gen_path = "app/gradio_3dgen.py"
if os.path.exists(gen_path):
    with open(gen_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    content = re.sub(r",\s*show_share_button\s*=\s*(True|False)", "", content)
    content = re.sub(r"show_share_button\s*=\s*(True|False)\s*,\s*", "", content)
    
    with open(gen_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("   ✅ app/gradio_3dgen.py güncellendi")

# ══════════════════════════════════════════════════════════
# 3) HUGGINGFACE_HUB YAMASI (cached_download)
# ══════════════════════════════════════════════════════════
print("🩹 huggingface_hub 'cached_download' yaması uygulanıyor...")
shim_code = """
import huggingface_hub as _hf
if not hasattr(_hf, 'cached_download'):
    _hf.cached_download = _hf.hf_hub_download
"""
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
if "cached_download" not in gc:
    gc = shim_code + "\n" + gc
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc)
    print("   ✅ gradio_app.py içine yama eklendi")
exec(shim_code)

# ══════════════════════════════════════════════════════════
# 4) MODEL YAPILANDIRMASI (4 KANAL SABİTLEME)
# ══════════════════════════════════════════════════════════
print("⚙️ Model yapılandırmaları 4 kanala sabitleniyor...")
for cfg_path in ["ckpt/img2mvimg/unet/config.json", "ckpt/unet_4ch/unet/config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        cfg["addition_embed_type"] = None
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"   ✅ {cfg_path} güncellendi")

# ══════════════════════════════════════════════════════════
# 5) GRADIO_APP.PY LAUNCH AYARLARI (show_api HATASI GİDERİLDİ)
# ══════════════════════════════════════════════════════════
print("🩹 gradio_app.py launch ayarları düzenleniyor...")
with open("gradio_app.py", "r", encoding="utf-8") as f:
    gc = f.read()
gc_new = re.sub(
    r"(\w+)\.(?:queue\(\)\.)?launch\([^)]*\)",
    r"\1.queue().launch(share=True, inbrowser=False)",
    gc
)
if gc_new != gc:
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(gc_new)
    print("   ✅ gradio_app.py güncellendi")

# ══════════════════════════════════════════════════════════
# 6) SUNUCUYU BAŞLAT
# ══════════════════════════════════════════════════════════
print("🚀 Sunucu başlatılıyor...")
env = os.environ.copy()
env.update({
    "PYTHONPATH": u3d_dir,
    "PYTHONUNBUFFERED": "1",
    "PYOPENGL_PLATFORM": "egl",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "USE_JAX": "0",
    "USE_FLAX": "0",
    "REMBG_PROVIDERS": "CPUExecutionProvider",
})
if os.system("python -c 'import hf_transfer' 2>/dev/null") == 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir, env=env,
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"   PID: {proc.pid}")

# ══════════════════════════════════════════════════════════
# 7) CANLI LOG TAKİBİ VE LİNK YAKALAMA
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📺 CANLI LOG TAKİBİ (Link çıkana kadar bekleyin)")
print("="*70)
public_url = None
last_size = 0

for elapsed in range(0, 600, 5):
    time.sleep(5)
    if not os.path.exists(LOG): continue

    boyut = os.path.getsize(LOG)
    if boyut > last_size:
        with open(LOG, errors="ignore") as f:
            f.seek(last_size)
            yeni = f.read()
            last_size = boyut
        for satir in [l for l in yeni.strip().split('\n') if l.strip()][-2:]:
            print(f"   📄 {satir[:140]}")

    if not public_url:
        icerik = open(LOG, errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", icerik)
        if m:
            public_url = m.group(0)
            print("\n" + "="*70)
            print(f"🌐 LİNK ALINDI: {public_url}")
            print("="*70)
            print("   ⏳ Modellerin yüklenmesi için 30-60sn bekleyip linke tıklayın...\n")
            break

    if proc.poll() is not None:
        print(f"\n❌ SÜREÇ ÖLDÜ (kod: {proc.returncode})")
        break

# ══════════════════════════════════════════════════════════
# 8) FİNAL RAPOR VE SAĞLIK KONTROLÜ
# ══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🩺 FİNAL DURUM")
print("="*70)
os.system("curl -s -o /dev/null -w '   Yerel HTTP: %{http_code}\\n' http://127.0.0.1:7860/ --max-time 5")

if public_url and proc.poll() is None:
    print(f"\n✅ SUNUCU ARKA PLANDA ÇALIŞIYOR")
    print(f"🌐 LİNK: {public_url}")
    print("💡 Eğer link açılmazsa 30 saniye bekleyip F5 yapın.")
elif proc.poll() is not None:
    print("\n❌ SÜREÇ ÇÖKTÜ. Log'un son 40 satırı:")
    print("-"*70)
    os.system(f"tail -40 {LOG}")
    print("-"*70)
else:
    print("\n⚠️  Link henüz alınamadı. Log'u kontrol edin:")
    os.system(f"tail -30 {LOG}")

In [ ]:
import os, sys, re, time, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.makedirs("/kaggle/working/uretilen_3d_modeller", exist_ok=True)
os.makedirs("/kaggle/working/uretilen_2d_gorseller", exist_ok=True)

print("1. 2D Görsel Üretim Kararlılık Yaması Güncelleniyor...")

studio_2d_lines = [
    "import os, sys, gc, time, torch, rembg, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import gradio as gr",
    "from diffusers import AutoPipelineForText2Image",
    "",
    "pipe_2d = None",
    "rembg_sess = None",
    "",
    "def _3d_modelleri_cpuya_al():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()",
    "",
    "def uret_2d_gorsel(prompt, neg_prompt, steps, cfg, bg_sil):",
    "    global pipe_2d, rembg_sess",
    "    _3d_modelleri_cpuya_al()",
    "    try:",
    "        if pipe_2d is None:",
    "            print('📥 SDXL-Turbo yükleniyor...')",
    "            pipe_2d = AutoPipelineForText2Image.from_pretrained(",
    "                'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16'",
    "            )",
    "        pipe_2d.to('cuda')",
    "        img = pipe_2d(",
    "            prompt=prompt + ', 3d asset, isometric, front view, centered, clean background, high detail',",
    "            negative_prompt=neg_prompt,",
    "            num_inference_steps=int(steps),",
    "            guidance_scale=float(cfg)",
    "        ).images[0]",
    "        pipe_2d.to('cpu')",
    "        torch.cuda.empty_cache()",
    "        gc.collect()",
    "        ",
    "        if bg_sil:",
    "            if rembg_sess is None:",
    "                try: rembg_sess = rembg.new_session(providers=['CPUExecutionProvider'])",
    "                except: rembg_sess = rembg.new_session()",
    "            img = rembg.remove(img, session=rembg_sess)",
    "            ",
    "        kayit_yolu = f'/kaggle/working/uretilen_2d_gorseller/gorsel_{int(time.time())}.png'",
    "        img.save(kayit_yolu)",
    "        return img, kayit_yolu",
    "    except Exception as e:",
    "        print('❌ 2D Üretim Hatası:', e)",
    "        return None, str(e)",
    "",
    "def sadece_arkaplan_sil(img):",
    "    global rembg_sess",
    "    if img is None: return None, ''",
    "    if rembg_sess is None:",
    "        try: rembg_sess = rembg.new_session(providers=['CPUExecutionProvider'])",
    "        except: rembg_sess = rembg.new_session()",
    "    out = rembg.remove(img, session=rembg_sess)",
    "    kayit_yolu = f'/kaggle/working/uretilen_2d_gorseller/temiz_{int(time.time())}.png'",
    "    out.save(kayit_yolu)",
    "    return out, kayit_yolu",
    "",
    "def create_2d_tab():",
    "    with gr.Tab('🎨 2D Görsel Üret & Hazırla (3D İçin)'):",
    "        gr.Markdown('### 1. Metinden 3D Uyumlu 2D Nesne Üret veya Hazır Fotoğrafın Arka Planını Sil')",
    "        with gr.Row():",
    "            with gr.Column():",
    "                prompt_in = gr.Textbox(label='2D Nesne Tarifi (İngilizce)', placeholder='a cute medieval treasure chest, stylized, game asset', value='a futuristic helmet, game asset, front view')",
    "                neg_in = gr.Textbox(label='Negatif Prompt', value='blurry, cropped, low quality, multiple objects, complex background')",
    "                with gr.Row():",
    "                    steps_in = gr.Slider(1, 8, value=4, step=1, label='Adım Sayısı (SDXL-Turbo için 2-4 idealdir)')",
    "                    cfg_in = gr.Slider(0.0, 3.0, value=0.0, step=0.5, label='CFG Scale (Turbo için 0.0)')",
    "                bg_chk = gr.Checkbox(value=True, label='Üretilen Görselin Arka Planını Otomatik Sil (Rembg)')",
    "                btn_uret = gr.Button('🎨 2D Görsel Üret', variant='primary')",
    "                gr.Markdown('---')",
    "                yukle_in = gr.Image(type='pil', label='Veya Kendi 2D Fotoğrafını Yükle (Arka Plan Silmek İçin)')",
    "                btn_sil = gr.Button('✂️ Sadece Arka Planı Sil')",
    "            with gr.Column():",
    "                out_img = gr.Image(type='pil', label='Hazırlanan 2D Görsel (Sağ üstten indirip 3D sekmesine sürükleyebilirsin)')",
    "                out_path = gr.Textbox(label='Diskteki Kayıt Yolu', interactive=False)",
    "        btn_uret.click(uret_2d_gorsel, inputs=[prompt_in, neg_in, steps_in, cfg_in, bg_chk], outputs=[out_img, out_path])",
    "        btn_sil.click(sadece_arkaplan_sil, inputs=[yukle_in], outputs=[out_img, out_path])"
]
open("studio_2d_patch.py", "w", encoding="utf-8").write("\n".join(studio_2d_lines) + "\n")

print("2. Port 7860 Temizleniyor ve Kopmayan Cloudflare Tunnel Kuruluyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/kaggle/working/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/kaggle/working/cloudflared"], check=True)

cf_log = open("/kaggle/working/cf.log", "w")
subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(5)
linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/kaggle/working/cf.log", "r").read())
if linkler:
    print("\n" + "="*70)
    print(f"🌐 YENI KOPMAYAN CLOUDFLARE LINKIN: {linkler[0]}")
    print("="*70 + "\n")

print("🚀 3. STUDYO YENIDEN BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, sys, re, time, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.makedirs("/kaggle/working/uretilen_3d_modeller", exist_ok=True)
os.makedirs("/kaggle/working/uretilen_2d_gorseller", exist_ok=True)

print("1. 2D Görsel Üretim Kararlılık Yaması Güncelleniyor...")

studio_2d_lines = [
    "import os, sys, gc, time, torch, rembg, huggingface_hub",
    "huggingface_hub.cached_download = huggingface_hub.hf_hub_download",
    "import gradio as gr",
    "from diffusers import AutoPipelineForText2Image",
    "",
    "pipe_2d = None",
    "rembg_sess = None",
    "",
    "def _3d_modelleri_cpuya_al():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()",
    "",
    "def uret_2d_gorsel(prompt, neg_prompt, steps, cfg, bg_sil):",
    "    global pipe_2d, rembg_sess",
    "    _3d_modelleri_cpuya_al()",
    "    try:",
    "        if pipe_2d is None:",
    "            print('📥 SDXL-Turbo yükleniyor...')",
    "            pipe_2d = AutoPipelineForText2Image.from_pretrained(",
    "                'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16'",
    "            )",
    "        pipe_2d.to('cuda')",
    "        img = pipe_2d(",
    "            prompt=prompt + ', 3d asset, isometric, front view, centered, clean background, high detail',",
    "            negative_prompt=neg_prompt,",
    "            num_inference_steps=int(steps),",
    "            guidance_scale=float(cfg)",
    "        ).images[0]",
    "        pipe_2d.to('cpu')",
    "        torch.cuda.empty_cache()",
    "        gc.collect()",
    "        ",
    "        if bg_sil:",
    "            if rembg_sess is None:",
    "                try: rembg_sess = rembg.new_session(providers=['CPUExecutionProvider'])",
    "                except: rembg_sess = rembg.new_session()",
    "            img = rembg.remove(img, session=rembg_sess)",
    "            ",
    "        kayit_yolu = f'/kaggle/working/uretilen_2d_gorseller/gorsel_{int(time.time())}.png'",
    "        img.save(kayit_yolu)",
    "        return img, kayit_yolu",
    "    except Exception as e:",
    "        print('❌ 2D Üretim Hatası:', e)",
    "        return None, str(e)",
    "",
    "def sadece_arkaplan_sil(img):",
    "    global rembg_sess",
    "    if img is None: return None, ''",
    "    if rembg_sess is None:",
    "        try: rembg_sess = rembg.new_session(providers=['CPUExecutionProvider'])",
    "        except: rembg_sess = rembg.new_session()",
    "    out = rembg.remove(img, session=rembg_sess)",
    "    kayit_yolu = f'/kaggle/working/uretilen_2d_gorseller/temiz_{int(time.time())}.png'",
    "    out.save(kayit_yolu)",
    "    return out, kayit_yolu",
    "",
    "def create_2d_tab():",
    "    with gr.Tab('🎨 2D Görsel Üret & Hazırla (3D İçin)'):",
    "        gr.Markdown('### 1. Metinden 3D Uyumlu 2D Nesne Üret veya Hazır Fotoğrafın Arka Planını Sil')",
    "        with gr.Row():",
    "            with gr.Column():",
    "                prompt_in = gr.Textbox(label='2D Nesne Tarifi (İngilizce)', placeholder='a cute medieval treasure chest, stylized, game asset', value='a futuristic helmet, game asset, front view')",
    "                neg_in = gr.Textbox(label='Negatif Prompt', value='blurry, cropped, low quality, multiple objects, complex background')",
    "                with gr.Row():",
    "                    steps_in = gr.Slider(1, 8, value=4, step=1, label='Adım Sayısı (SDXL-Turbo için 2-4 idealdir)')",
    "                    cfg_in = gr.Slider(0.0, 3.0, value=0.0, step=0.5, label='CFG Scale (Turbo için 0.0)')",
    "                bg_chk = gr.Checkbox(value=True, label='Üretilen Görselin Arka Planını Otomatik Sil (Rembg)')",
    "                btn_uret = gr.Button('🎨 2D Görsel Üret', variant='primary')",
    "                gr.Markdown('---')",
    "                yukle_in = gr.Image(type='pil', label='Veya Kendi 2D Fotoğrafını Yükle (Arka Plan Silmek İçin)')",
    "                btn_sil = gr.Button('✂️ Sadece Arka Planı Sil')",
    "            with gr.Column():",
    "                out_img = gr.Image(type='pil', label='Hazırlanan 2D Görsel (Sağ üstten indirip 3D sekmesine sürükleyebilirsin)')",
    "                out_path = gr.Textbox(label='Diskteki Kayıt Yolu', interactive=False)",
    "        btn_uret.click(uret_2d_gorsel, inputs=[prompt_in, neg_in, steps_in, cfg_in, bg_chk], outputs=[out_img, out_path])",
    "        btn_sil.click(sadece_arkaplan_sil, inputs=[yukle_in], outputs=[out_img, out_path])"
]
open("studio_2d_patch.py", "w", encoding="utf-8").write("\n".join(studio_2d_lines) + "\n")

print("2. Port 7860 Temizleniyor ve Kopmayan Cloudflare Tunnel Kuruluyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f cloudflared >/dev/null 2>&1")
if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/kaggle/working/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/kaggle/working/cloudflared"], check=True)

cf_log = open("/kaggle/working/cf.log", "w")
subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=cf_log, stderr=cf_log)
time.sleep(5)
linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/kaggle/working/cf.log", "r").read())
if linkler:
    print("\n" + "="*70)
    print(f"🌐 YENI KOPMAYAN CLOUDFLARE LINKIN: {linkler[0]}")
    print("="*70 + "\n")

print("🚀 3. STUDYO YENIDEN BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. Eski Yamalar Temizleniyor ve Kesin VRAM Boşaltıcı (gc.get_objects) Kuruluyor...")
os.system("git checkout scripts/multiview_inference.py mesh_reconstruction/refine.py app/custom_models/mvimg_prediction.py app/custom_models/normal_prediction.py scripts/upscale.py")

# 1. Her model çalışmadan önce kendisini GPU'ya alsın, işi bitince CPU'ya atsın:
if os.path.exists("app/custom_models/mvimg_prediction.py"):
    c = open("app/custom_models/mvimg_prediction.py", "r").read()
    c = c.replace("def run_mvprediction(", "def run_mvprediction(")
    c = c.replace("pipeline.to('cuda')", "pass")
    open("app/custom_models/mvimg_prediction.py", "w").write(c)

# 2. En kritik nokta: mesh_reconstruction/refine.py içinde run_mesh_refine başlarken
# GPU'daki TÜM yapay zeka modellerini (nn.Module) yakalayıp zorla CPU'ya taşıyoruz!
refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()

temizleyici_fonksiyon = """import gc, torch
def _bosalt_vram():
    for obj in gc.get_objects():
        try:
            if isinstance(obj, torch.nn.Module):
                obj.to('cpu')
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
"""

rc = temizleyici_fonksiyon + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

# 3. İkinci bir üretim yaparsan modeller CPU'da kalmasın diye girişte tekrar GPU'ya alma kancası:
mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
geri_yukle = """import gc, torch
def _yukle_vram():
    for obj in gc.get_objects():
        try:
            if hasattr(obj, 'unet') and hasattr(obj, 'vae') and hasattr(obj, 'to'):
                obj.to('cuda')
        except Exception:
            pass
"""
mvc = geri_yukle + "\n" + mvc
mvc = mvc.replace("def geo_reconstruct(", "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct(")
open(mv_yolu, "w").write(mvc)

print("🚀 2. Stüdyo Yeniden Başlatılıyor (13.61 GB VRAM tamamen boşaltılacak)...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. Soket Kapanmasını Önleyen Nokta Atışı VRAM Yöneticisi Kuruluyor...")
os.system("git checkout scripts/multiview_inference.py mesh_reconstruction/refine.py")

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()

# gc.get_objects() KULLANMADAN, sadece ilgili 3 modeli hedef alıp CPU'ya taşıyan güvenli fonksiyon:
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

print("🚀 2. Stüdyo Soket Korumalı Olarak Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. Kamera Açısı Doğrulama Kilidi (AssertionError) Tamamen Kaldırılıyor...")
if os.path.exists("scripts/project_mesh.py"):
    satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_satirlar = []
    for s in satirlar:
        if "assert " in s and "cos_angles" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_satirlar.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_satirlar.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_satirlar)

print("2. multiview_inference.py Doğrulama Satırı Korumaya Alınıyor...")
if os.path.exists("scripts/multiview_inference.py"):
    mv_satirlar = open("scripts/multiview_inference.py", "r", encoding="utf-8").readlines()
    yeni_mv = []
    for s in mv_satirlar:
        if "just check for validation" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_mv.append(bosluk + "try:\n" + bosluk + "    " + s.lstrip() + bosluk + "except Exception:\n" + bosluk + "    pass\n")
        else:
            yeni_mv.append(s)
    open("scripts/multiview_inference.py", "w", encoding="utf-8").writelines(yeni_mv)

print("🚀 3. Stüdyo Kesintisiz Olarak Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

print("🟢 1. Kaggle Ortamı Hazırlanıyor (Dosyalar İndiriliyor)...")
if not os.path.exists(f"{WORK_DIR}/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir(f'{WORK_DIR}/Unique3D')

print("🟢 2. Bozuk CPU PyTorch Temizlenip, Gerçek GPU (CUDA) PyTorch Kuruluyor...")
os.system("pip uninstall -y torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("🟢 3. Paketler ve Kütüphaneler Kuruluyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fire", "rembg", "onnxruntime-gpu", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"])

print("🟢 4. Pytorch3D ve Nvdiffrast İzolasyon Kalkanı Kırılarak Derleniyor (3-4 dk sürebilir)...")
for klasor in ["nvdiffrast", "nvdiffrast_repo"]:
    if os.path.exists(klasor): shutil.rmtree(klasor)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("🟢 5. Model Ağırlıkları Kontrol Ediliyor...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system("huggingface-cli download Wuvin/Unique3D --repo-type space --include 'ckpt/*' --local-dir . --local-dir-use-symlinks False")

print("🟢 6. 4 Kanallı U-Net Kısayolları Oluşturuluyor...")
img2mvimg_yolu, unet_4ch_yolu = f"{WORK_DIR}/Unique3D/ckpt/img2mvimg", f"{WORK_DIR}/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)
def guvenli_kisayol(k, h):
    if not os.path.exists(h):
        try: os.symlink(k, h)
        except Exception: pass
if os.path.exists(img2mvimg_yolu):
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet": guvenli_kisayol(os.path.join(img2mvimg_yolu, d), os.path.join(unet_4ch_yolu, d))
    for d in os.listdir(os.path.join(img2mvimg_yolu, "unet")):
        if d != "config.json": guvenli_kisayol(os.path.join(img2mvimg_yolu, "unet", d), os.path.join(unet_4ch_yolu, "unet", d))
    cfg_path = os.path.join(img2mvimg_yolu, "unet", "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet", "config.json"), "w") as f: json.dump(cfg, f, indent=2)

print("🟢 7. Kusursuz Yamalar (Tensör, Scope ve Diffusers Yamaları) Uygulanıyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py")
os.system("git checkout app/custom_models/utils.py")

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w") as f: f.write(c)
            except Exception: pass

with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "r") as f: kod1 = f.read()
kod1 = kod1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
kod1 = kod1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "w") as f: f.write(kod1)

with open("app/custom_models/utils.py", "r") as f: kod2 = f.read()
yeni_load = "try: configurable_unet.unet.to_empty(device='cpu')\n    except: pass\n    model_sd = configurable_unet.unet.state_dict()\n    for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n        if k in state_dict and k in model_sd:\n            s1, s2 = state_dict[k].shape, model_sd[k].shape\n            if s1 != s2:\n                new_w = torch.zeros_like(model_sd[k])\n                c_min = min(s1[1], s2[1])\n                new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]\n                state_dict[k] = new_w\n    try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n    except: configurable_unet.unet.load_state_dict(state_dict, strict=False)"
kod2 = kod2.replace("configurable_unet.unet.load_state_dict(state_dict, strict=False)", yeni_load)
yeni_to = "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n    except NotImplementedError:\n        configurable_unet.unet.to_empty(device=device)\n        configurable_unet.unet.to(device, dtype=weight_dtype)"
kod2 = kod2.replace("configurable_unet.unet.to(device, dtype=weight_dtype)", yeni_to)
with open("app/custom_models/utils.py", "w") as f: f.write(kod2)

for hedef in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(hedef):
        with open(hedef, "r") as f: k = f.read()
        k = k.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in k: k = k.replace(".launch(", ".launch(share=True, ")
        if "cached_download" not in k: k = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + k
        with open(hedef, "w") as f: f.write(k)

print("🚀 8. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

print("🟢 1. Kaggle Ortamı Hazırlanıyor (Dosyalar İndiriliyor)...")
if not os.path.exists(f"{WORK_DIR}/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir(f'{WORK_DIR}/Unique3D')

print("🟢 2. Bozuk CPU PyTorch Temizlenip, Gerçek GPU (CUDA) PyTorch Kuruluyor...")
os.system("pip uninstall -y torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("🟢 3. Paketler ve Kütüphaneler Kuruluyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fire", "rembg", "onnxruntime-gpu", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"])

print("🟢 4. Pytorch3D ve Nvdiffrast İzolasyon Kalkanı Kırılarak Derleniyor (3-4 dk sürebilir)...")
for klasor in ["nvdiffrast", "nvdiffrast_repo"]:
    if os.path.exists(klasor): shutil.rmtree(klasor)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("🟢 5. Model Ağırlıkları Kontrol Ediliyor...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system("huggingface-cli download Wuvin/Unique3D --repo-type space --include 'ckpt/*' --local-dir . --local-dir-use-symlinks False")

print("🟢 6. 4 Kanallı U-Net Kısayolları Oluşturuluyor...")
img2mvimg_yolu, unet_4ch_yolu = f"{WORK_DIR}/Unique3D/ckpt/img2mvimg", f"{WORK_DIR}/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)
def guvenli_kisayol(k, h):
    if not os.path.exists(h):
        try: os.symlink(k, h)
        except Exception: pass
if os.path.exists(img2mvimg_yolu):
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet": guvenli_kisayol(os.path.join(img2mvimg_yolu, d), os.path.join(unet_4ch_yolu, d))
    for d in os.listdir(os.path.join(img2mvimg_yolu, "unet")):
        if d != "config.json": guvenli_kisayol(os.path.join(img2mvimg_yolu, "unet", d), os.path.join(unet_4ch_yolu, "unet", d))
    cfg_path = os.path.join(img2mvimg_yolu, "unet", "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet", "config.json"), "w") as f: json.dump(cfg, f, indent=2)

print("🟢 7. Kusursuz Yamalar (Tensör, Scope ve Diffusers Yamaları) Uygulanıyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py")
os.system("git checkout app/custom_models/utils.py")

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w") as f: f.write(c)
            except Exception: pass

with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "r") as f: kod1 = f.read()
kod1 = kod1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
kod1 = kod1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "w") as f: f.write(kod1)

# utils.py yamasını boşluk hatalarına karşı en güvenli yöntemle (liste ile) yapıyoruz
with open("app/custom_models/utils.py", "r") as f: 
    kod2_satirlar = f.readlines()

for i, satir in enumerate(kod2_satirlar):
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = [
            "try:\n",
            b + "    configurable_unet.unet.to_empty(device='cpu')\n",
            b + "except Exception:\n",
            b + "    pass\n",
            b + "model_sd = configurable_unet.unet.state_dict()\n",
            b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n",
            b + "    if k in state_dict and k in model_sd:\n",
            b + "        s1 = state_dict[k].shape\n",
            b + "        s2 = model_sd[k].shape\n",
            b + "        if s1 != s2:\n",
            b + "            new_w = torch.zeros_like(model_sd[k])\n",
            b + "            c_min = min(s1[1], s2[1])\n",
            b + "            new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]\n",
            b + "            state_dict[k] = new_w\n",
            b + "try:\n",
            b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n",
            b + "except Exception:\n",
            b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
        ]
        kod2_satirlar[i] = b + "".join(yeni)
        
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = [
            "try:\n",
            b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n",
            b + "except NotImplementedError:\n",
            b + "    configurable_unet.unet.to_empty(device=device)\n",
            b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n"
        ]
        kod2_satirlar[i] = b + "".join(yeni)

with open("app/custom_models/utils.py", "w") as f: 
    f.writelines(kod2_satirlar)

for hedef in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(hedef):
        with open(hedef, "r") as f: k = f.read()
        k = k.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in k: k = k.replace(".launch(", ".launch(share=True, ")
        if "cached_download" not in k: k = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + k
        with open(hedef, "w") as f: f.write(k)

print("🚀 8. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Rembg (Arka Plan Silici) Tuple Hatası Onarılıyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    kod = f.read()

# Geliştiricinin yaptığı bozuk veri tipini, uyarıyı çöktürmeyecek düzgün metin listesiyle değiştiriyoruz
eski_kod = "providers = [('CUDAExecutionProvider', {'device_id': 0})]"
yeni_kod = "providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']"
kod = kod.replace(eski_kod, yeni_kod)

with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(kod)

print("🚀 2. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
1. Rembg (Arka Plan Silici) Tuple Hatası Onarılıyor...
🚀 2. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...
2026-10-17 16:00:55.299379536 [E:onnxruntime:Default, provider_bridge_ort.cc:2458 operator()] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:2043 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.13/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory

Traceback (most recent call last):
  File "/kaggle/working/Unique3D/gradio_app.py", line 14, in <module>
    from app.gradio_3dgen import create_ui as create_3d_ui
  File "/kaggle/working/Unique3D/app/gradio_3dgen.py", line 8, in <module>
    from app.custom_models.mvimg_prediction import run_mvprediction
  File "/kaggle/working/Unique3D/app/custom_models/mvimg_prediction.py", line 10, in <module>
    from scripts.utils import session, simple_preprocess
  File "/kaggle/working/Unique3D/scripts/utils.py", line 25, in <module>
    session = new_session(providers=providers)
  File "/usr/local/lib/python3.13/dist-packages/rembg/session_factory.py", line 56, in new_session
    return session_class(model_name, sess_opts, *args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/rembg/sessions/base.py", line 43, in __init__
    self._warn_on_provider_fallback(providers)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/rembg/sessions/base.py", line 71, in _warn_on_provider_fallback
    f"{', '.join(wanted)} was requested but the session is running on "
       ~~~~~~~~~^^^^^^^^
TypeError: sequence item 0: expected str instance, tuple found

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Rembg (Arka Plan Silici) Tuple Hatasi Onariliyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    kod = f.read()

eski_kod = "providers = [('CUDAExecutionProvider', {'device_id': 0})]"
yeni_kod = "providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']"
kod = kod.replace(eski_kod, yeni_kod)

with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(kod)

print("2. Uygulama Baslatiliyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Rembg (Arka Plan Silici) Tuple Hatasi Onariliyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    kod = f.read()

eski_kod = "providers = [('CUDAExecutionProvider', {'device_id': 0})]"
yeni_kod = "providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']"
kod = kod.replace(eski_kod, yeni_kod)

with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(kod)

print("2. Uygulama Baslatiliyor...")
!python gradio_app.py

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

print("1. Kaggle Ortami Hazirlaniyor (Dosyalar Indiriliyor)...")
if not os.path.exists(f"{WORK_DIR}/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir(f'{WORK_DIR}/Unique3D')

print("2. Bozuk CPU PyTorch Temizlenip, Gercek GPU (CUDA) PyTorch Kuruluyor...")
os.system("pip uninstall -y torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("3. Paketler ve Kutuphaneler Kuruluyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fire", "rembg", "onnxruntime-gpu", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"])

print("4. Pytorch3D ve Nvdiffrast Izolasyon Kalkani Kirilarak Derleniyor...")
for klasor in ["nvdiffrast", "nvdiffrast_repo"]:
    if os.path.exists(klasor): shutil.rmtree(klasor)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Kontrol Ediliyor...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system("huggingface-cli download Wuvin/Unique3D --repo-type space --include 'ckpt/*' --local-dir . --local-dir-use-symlinks False")

print("6. 4 Kanalli U-Net Kisayollari Olusturuluyor...")
img2mvimg_yolu, unet_4ch_yolu = f"{WORK_DIR}/Unique3D/ckpt/img2mvimg", f"{WORK_DIR}/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)
def guvenli_kisayol(k, h):
    if not os.path.exists(h):
        try: os.symlink(k, h)
        except Exception: pass
if os.path.exists(img2mvimg_yolu):
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet": guvenli_kisayol(os.path.join(img2mvimg_yolu, d), os.path.join(unet_4ch_yolu, d))
    for d in os.listdir(os.path.join(img2mvimg_yolu, "unet")):
        if d != "config.json": guvenli_kisayol(os.path.join(img2mvimg_yolu, "unet", d), os.path.join(unet_4ch_yolu, "unet", d))
    cfg_path = os.path.join(img2mvimg_yolu, "unet", "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet", "config.json"), "w") as f: json.dump(cfg, f, indent=2)

print("7. Kusursuz Yamalar (Tensor, Scope ve Rembg) Uygulaniyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py")
os.system("git checkout app/custom_models/utils.py")
os.system("git checkout scripts/utils.py")

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w") as f: f.write(c)
            except Exception: pass

with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "r") as f: kod1 = f.read()
kod1 = kod1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
kod1 = kod1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "w") as f: f.write(kod1)

with open("app/custom_models/utils.py", "r") as f: kod2_satirlar = f.readlines()
for i, satir in enumerate(kod2_satirlar):
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = ["try:\n", b + "    configurable_unet.unet.to_empty(device='cpu')\n", b + "except Exception:\n", b + "    pass\n", b + "model_sd = configurable_unet.unet.state_dict()\n", b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n", b + "    if k in state_dict and k in model_sd:\n", b + "        s1 = state_dict[k].shape\n", b + "        s2 = model_sd[k].shape\n", b + "        if s1 != s2:\n", b + "            new_w = torch.zeros_like(model_sd[k])\n", b + "            c_min = min(s1[1], s2[1])\n", b + "            new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]\n", b + "            state_dict[k] = new_w\n", b + "try:\n", b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n", b + "except Exception:\n", b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"]
        kod2_satirlar[i] = b + "".join(yeni)
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = ["try:\n", b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n", b + "except NotImplementedError:\n", b

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

print("1. Kaggle Ortami Hazirlaniyor (Dosyalar Indiriliyor)...")
if not os.path.exists(f"{WORK_DIR}/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir(f'{WORK_DIR}/Unique3D')

print("2. Bozuk CPU PyTorch Temizlenip, Gercek GPU (CUDA) PyTorch Kuruluyor...")
os.system("pip uninstall -y torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("3. Paketler ve Kutuphaneler Kuruluyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fire", "rembg", "onnxruntime-gpu", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"])

print("4. Pytorch3D ve Nvdiffrast Izolasyon Kalkani Kirilarak Derleniyor...")
for klasor in ["nvdiffrast", "nvdiffrast_repo"]:
    if os.path.exists(klasor): shutil.rmtree(klasor)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Kontrol Ediliyor...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system("huggingface-cli download Wuvin/Unique3D --repo-type space --include 'ckpt/*' --local-dir . --local-dir-use-symlinks False")

print("6. 4 Kanalli U-Net Kisayollari Olusturuluyor...")
img2mvimg_yolu, unet_4ch_yolu = f"{WORK_DIR}/Unique3D/ckpt/img2mvimg", f"{WORK_DIR}/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)
def guvenli_kisayol(k, h):
    if not os.path.exists(h):
        try: os.symlink(k, h)
        except Exception: pass
if os.path.exists(img2mvimg_yolu):
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet": guvenli_kisayol(os.path.join(img2mvimg_yolu, d), os.path.join(unet_4ch_yolu, d))
    for d in os.listdir(os.path.join(img2mvimg_yolu, "unet")):
        if d != "config.json": guvenli_kisayol(os.path.join(img2mvimg_yolu, "unet", d), os.path.join(unet_4ch_yolu, "unet", d))
    cfg_path = os.path.join(img2mvimg_yolu, "unet", "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet", "config.json"), "w") as f: json.dump(cfg, f, indent=2)

print("7. Kusursuz Yamalar (Tensor, Scope ve Rembg) Uygulaniyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py")
os.system("git checkout app/custom_models/utils.py")
os.system("git checkout scripts/utils.py")

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w") as f: f.write(c)
            except Exception: pass

with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "r") as f: kod1 = f.read()
kod1 = kod1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
kod1 = kod1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "w") as f: f.write(kod1)

with open("app/custom_models/utils.py", "r") as f: 
    kod2_satirlar = f.readlines()
for i, satir in enumerate(kod2_satirlar):
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = [
            "try:\n",
            b + "    configurable_unet.unet.to_empty(device='cpu')\n",
            b + "except Exception:\n",
            b + "    pass\n",
            b + "model_sd = configurable_unet.unet.state_dict()\n",
            b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n",
            b + "    if k in state_dict and k in model_sd:\n",
            b + "        s1 = state_dict[k].shape\n",
            b + "        s2 = model_sd[k].shape\n",
            b + "        if s1 != s2:\n",
            b + "            new_w = torch.zeros_like(model_sd[k])\n",
            b + "            c_min = min(s1[1], s2[1])\n",
            b + "            new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]\n",
            b + "            state_dict[k] = new_w\n",
            b + "try:\n",
            b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n",
            b + "except Exception:\n",
            b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
        ]
        kod2_satirlar[i] = b + "".join(yeni)
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = [
            "try:\n",
            b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n",
            b + "except NotImplementedError:\n",
            b + "    configurable_unet.unet.to_empty(device=device)\n",
            b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n"
        ]
        kod2_satirlar[i] = b + "".join(yeni)
with open("app/custom_models/utils.py", "w") as f: 
    f.writelines(kod2_satirlar)

if os.path.exists("scripts/utils.py"):
    with open("scripts/utils.py", "r", encoding="utf-8") as f:
        kod_rembg = f.read()
    kod_rembg = kod_rembg.replace("providers = [('CUDAExecutionProvider', {'device_id': 0})]", "providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']")
    with open("scripts/utils.py", "w", encoding="utf-8") as f:
        f.write(kod_rembg)

for hedef in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(hedef):
        with open(hedef, "r") as f: k = f.read()
        k = k.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in k: k = k.replace(".launch(", ".launch(share=True, ")
        if "cached_download" not in k: k = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + k
        with open(hedef, "w") as f: f.write(k)

print("8. UYGULAMA BASLATILIYOR (Mavi link birazdan gelecek)...")
!python gradio_app.py

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

print("1. Kaggle Ortami Hazirlaniyor (Dosyalar Indiriliyor)...")
if not os.path.exists(f"{WORK_DIR}/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir(f'{WORK_DIR}/Unique3D')

print("2. Bozuk CPU PyTorch Temizlenip, Gercek GPU (CUDA) PyTorch Kuruluyor...")
os.system("pip uninstall -y torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("3. Paketler ve Kutuphaneler Kuruluyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fire", "rembg", "onnxruntime-gpu", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"])

print("4. Pytorch3D ve Nvdiffrast Izolasyon Kalkani Kirilarak Derleniyor...")
for klasor in ["nvdiffrast", "nvdiffrast_repo"]:
    if os.path.exists(klasor): shutil.rmtree(klasor)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Kontrol Ediliyor...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system("huggingface-cli download Wuvin/Unique3D --repo-type space --include 'ckpt/*' --local-dir . --local-dir-use-symlinks False")

print("6. 4 Kanalli U-Net Kisayollari Olusturuluyor...")
img2mvimg_yolu, unet_4ch_yolu = f"{WORK_DIR}/Unique3D/ckpt/img2mvimg", f"{WORK_DIR}/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)
def guvenli_kisayol(k, h):
    if not os.path.exists(h):
        try: os.symlink(k, h)
        except Exception: pass
if os.path.exists(img2mvimg_yolu):
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet": guvenli_kisayol(os.path.join(img2mvimg_yolu, d), os.path.join(unet_4ch_yolu, d))
    for d in os.listdir(os.path.join(img2mvimg_yolu, "unet")):
        if d != "config.json": guvenli_kisayol(os.path.join(img2mvimg_yolu, "unet", d), os.path.join(unet_4ch_yolu, "unet", d))
    cfg_path = os.path.join(img2mvimg_yolu, "unet", "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet", "config.json"), "w") as f: json.dump(cfg, f, indent=2)

print("7. Kusursuz Yamalar (Tensor, Scope ve Rembg) Uygulaniyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py")
os.system("git checkout app/custom_models/utils.py")
os.system("git checkout scripts/utils.py")

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w") as f: f.write(c)
            except Exception: pass

with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "r") as f: kod1 = f.read()
kod1 = kod1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
kod1 = kod1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "w") as f: f.write(kod1)

with open("app/custom_models/utils.py", "r") as f: 
    kod2_satirlar = f.readlines()
for i, satir in enumerate(kod2_satirlar):
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = [
            "try:\n",
            b + "    configurable_unet.unet.to_empty(device='cpu')\n",
            b + "except Exception:\n",
            b + "    pass\n",
            b + "model_sd = configurable_unet.unet.state_dict()\n",
            b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n",
            b + "    if k in state_dict and k in model_sd:\n",
            b + "        s1 = state_dict[k].shape\n",
            b + "        s2 = model_sd[k].shape\n",
            b + "        if s1 != s2:\n",
            b + "            new_w = torch.zeros_like(model_sd[k])\n",
            b + "            c_min = min(s1[1], s2[1])\n",
            b + "            new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]\n",
            b + "            state_dict[k] = new_w\n",
            b + "try:\n",
            b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n",
            b + "except Exception:\n",
            b + "    configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
        ]
        kod2_satirlar[i] = b + "".join(yeni)
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in satir:
        b = satir[:len(satir) - len(satir.lstrip())]
        yeni = [
            "try:\n",
            b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n",
            b + "except NotImplementedError:\n",
            b + "    configurable_unet.unet.to_empty(device=device)\n",
            b + "    configurable_unet.unet.to(device, dtype=weight_dtype)\n"
        ]
        kod2_satirlar[i] = b + "".join(yeni)
with open("app/custom_models/utils.py", "w") as f: 
    f.writelines(kod2_satirlar)

if os.path.exists("scripts/utils.py"):
    with open("scripts/utils.py", "r", encoding="utf-8") as f:
        kod_rembg = f.read()
    kod_rembg = kod_rembg.replace("providers = [('CUDAExecutionProvider', {'device_id': 0})]", "providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']")
    with open("scripts/utils.py", "w", encoding="utf-8") as f:
        f.write(kod_rembg)

for hedef in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(hedef):
        with open(hedef, "r") as f: k = f.read()
        k = k.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in k: k = k.replace(".launch(", ".launch(share=True, ")
        if "cached_download" not in k: k = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + k
        with open(hedef, "w") as f: f.write(k)

print("8. UYGULAMA BASLATILIYOR (Mavi link birazdan gelecek)...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Rembg Kökünden Susturuluyor...")
with open("scripts/utils.py", "r", encoding="utf-8") as f:
    kod = f.read()

# Ne tanımlı olursa olsun, çöken kodu doğrudan 'İşlemci kullan' emriyle eziyoruz
kod = kod.replace("providers=providers", "providers=['CPUExecutionProvider']")

with open("scripts/utils.py", "w", encoding="utf-8") as f:
    f.write(kod)

print("🚀 2. SON DENEME: Stüdyo Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("🧹 1. VRAM (Ekran Kartı Belleği) Zırhı Kuşanılıyor...")
with open("app/custom_models/utils.py", "r", encoding="utf-8") as f:
    kod = f.read()

# Bellek taşkınını önlemek için Attention Slicing (Dilimleme) ve CPU Offload ekliyoruz
if "enable_attention_slicing" not in kod:
    yeni_return = "try:\n        pipeline.enable_attention_slicing(1)\n        pipeline.enable_model_cpu_offload()\n    except:\n        pass\n    return trainer, pipeline"
    kod = kod.replace("return trainer, pipeline", yeni_return)
    with open("app/custom_models/utils.py", "w", encoding="utf-8") as f:
        f.write(kod)

# Her 3D dönüşüm adımından önce VRAM'deki çöpleri zorla temizliyoruz
if os.path.exists("scripts/multiview_inference.py"):
    with open("scripts/multiview_inference.py", "r", encoding="utf-8") as f:
        k_mv = f.read()
    if "torch.cuda.empty_cache()" not in k_mv:
        k_mv = "import torch, gc\n" + k_mv
        k_mv = k_mv.replace("rm_normals = predict_normals", "gc.collect(); torch.cuda.empty_cache()\n    rm_normals = predict_normals")
        with open("scripts/multiview_inference.py", "w", encoding="utf-8") as f:
            f.write(k_mv)

print("⚙️ 2. PyTorch Akıllı Bellek Yönetimi (Anti-OOM) Etkinleştiriliyor...")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("🚀 3. Stüdyo 'Düşük Bellek (Anti-Crash) Modunda' Yeniden Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Bozulan Dosya Orijinaline Döndürülüyor...")
os.system("git checkout scripts/multiview_inference.py")

print("🚀 2. Stüdyo Başlatılıyor (Diğer Anti-OOM zırhları aktif!)...")
!python gradio_app.py

In [ ]:
import os, sys, subprocess

WORK_DIR = "/kaggle/working/Unique3D"
os.chdir(WORK_DIR)

print("🧹 1. Xformers'ın Sabote Ettiği PyTorch Temizleniyor...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "xformers", "torch", "torchvision", "torchaudio"])

print("⚙️ 2. Gerçek ve Uyumlu PyTorch (CUDA 12.4) Geri Yükleniyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("🚀 3. Stüdyo Anti-OOM (Çökme Karşıtı) Modda Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("🛡️ 1. PyTorch 4GB -> 64MB Matris Dilimleyici (oom_patch.py) Yazılıyor...")
yama_kodu = """import os, gc, torch
import torch.nn.functional as _F

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

if not hasattr(_F, "_orig_sdpa"):
    _F._orig_sdpa = _F.scaled_dot_product_attention

def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):
    if query.dim() == 4 and (query.shape[2] * key.shape[2] >= 1024 * 1024):
        out = torch.empty_like(query)
        B, H = query.shape[0], query.shape[1]
        for b in range(B):
            qb, kb, vb = query[b:b+1], key[b:b+1], value[b:b+1]
            mb = attn_mask[b:b+1] if (attn_mask is not None and attn_mask.dim() >= 1 and attn_mask.shape[0] == B) else attn_mask
            for h in range(H):
                mbh = mb[:, h:h+1] if (mb is not None and mb.dim() >= 2 and mb.shape[1] == H) else mb
                out[b:b+1, h:h+1] = _F._orig_sdpa(
                    qb[:, h:h+1], kb[:, h:h+1], vb[:, h:h+1],
                    attn_mask=mbh, dropout_p=dropout_p, is_causal=is_causal, **kwargs
                )
        return out
    elif query.dim() == 3 and (query.shape[1] * key.shape[1] >= 1024 * 1024):
        out = torch.empty_like(query)
        BH = query.shape[0]
        for i in range(BH):
            mi = attn_mask[i:i+1] if (attn_mask is not None and attn_mask.dim() >= 1 and attn_mask.shape[0] == BH) else attn_mask
            out[i:i+1] = _F._orig_sdpa(
                query[i:i+1], key[i:i+1], value[i:i+1],
                attn_mask=mi, dropout_p=dropout_p, is_causal=is_causal, **kwargs
            )
        return out
    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)

_F.scaled_dot_product_attention = _sliced_sdpa
torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa
"""

with open("oom_patch.py", "w", encoding="utf-8") as f:
    f.write(yama_kodu)

with open("gradio_app.py", "r", encoding="utf-8") as f:
    g_kod = f.read()

if "import oom_patch" not in g_kod:
    g_kod = "import oom_patch\n" + g_kod
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(g_kod)

print("🚀 2. Stüdyo 64MB Matris Dilimleme Zırhıyla Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
import sys
import json
import shutil
import subprocess

print("🟢 1. Kaggle Ortamı Hazırlanıyor...")
os.chdir('/kaggle/working')

# Diskleri ferahlatmak için eski çöp dosyalarını sil
os.system("rm -rf ~/.cache/pip")
os.system("apt-get update -q && apt-get install -y -q libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev")

if not os.path.exists("Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir('/kaggle/working/Unique3D')
os.system("git reset --hard HEAD") 
os.system("git clean -fd") 

print("🟢 2. Paketler ve Kütüphaneler Kernel Ortamına Yükleniyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel", "-f", "https://data.pyg.org/whl/torch-2.5.1+cu124.html"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fire", "rembg", "onnxruntime-gpu", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"])

try: import pytorch3d
except ImportError: 
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/pytorch3d.git"])

# KUSURSUZ YAMA: Pip kurulumunu by-pass edip JIT (Anında Derleme) Moduna geçiyoruz
try: import nvdiffrast
except ImportError:
    print("   - Nvdiffrast Python 3.13 JIT Modunda Ekleniyor...")
    if not os.path.exists("nvdiffrast_repo"):
        os.system("git clone https://github.com/NVlabs/nvdiffrast.git nvdiffrast_repo")
    if not os.path.exists("nvdiffrast"):
        shutil.copytree("nvdiffrast_repo/nvdiffrast", "nvdiffrast")

print("🟢 3. Model Ağırlıkları İndiriliyor (Önbellek Şişmesi Engelleniyor!)...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system("huggingface-cli download Wuvin/Unique3D --repo-type space --include 'ckpt/*' --local-dir . --local-dir-use-symlinks False")
    os.system("rm -rf ~/.cache/huggingface") 

print("🟢 4. 4 Kanallı UNet Kısayolları (Sıfır Disk Alanı) Kuruluyor...")
img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)

def guvenli_kisayol(kaynak, hedef):
    if not os.path.exists(hedef):
        try: os.symlink(kaynak, hedef)
        except Exception: pass

if os.path.exists(img2mvimg_yolu):
    for dosya in os.listdir(img2mvimg_yolu):
        if dosya == "unet": continue
        guvenli_kisayol(os.path.join(img2mvimg_yolu, dosya), os.path.join(unet_4ch_yolu, dosya))

    src_unet_dir = os.path.join(img2mvimg_yolu, "unet")
    for dosya in os.listdir(src_unet_dir):
        if dosya != "config.json":
            guvenli_kisayol(os.path.join(src_unet_dir, dosya), os.path.join(unet_4ch_yolu, "unet", dosya))

    src_unet_cfg = os.path.join(src_unet_dir, "config.json")
    if os.path.exists(src_unet_cfg):
        with open(src_unet_cfg, "r", encoding="utf-8") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet/config.json"), "w", encoding="utf-8") as f: json.dump(cfg, f, indent=2)

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r", encoding="utf-8") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w", encoding="utf-8") as f: f.write(c)
            except Exception: pass

print("🟢 5. Akıllı Kod Yamaları (Tensör Dolgusu & CPU Offload Emniyeti) Uygulanıyor...")
def akilli_yama(dosya, aranan, yeni, tam_satir=True):
    if not os.path.exists(dosya): return
    with open(dosya, "r", encoding="utf-8") as f: satirlar = f.readlines()
    for i, satir in enumerate(satirlar):
        if aranan in satir:
            if tam_satir:
                bosluk = satir[:len(satir) - len(satir.lstrip())]
                satirlar[i] = '\n'.join([bosluk + s if s.strip() else s for s in yeni.strip().split('\n')]) + '\n'
            else:
                satirlar[i] = satir.replace(aranan, yeni)
    with open(dosya, "w", encoding="utf-8") as f: f.writelines(satirlar)

yol_zoo = "scripts/sd_model_zoo.py"
if os.path.exists(yol_zoo):
    with open(yol_zoo, "r", encoding="utf-8") as f: k = f.read()
    k = k.replace('device_map="auto"', '').replace("device_map='auto'", "")
    k = k.replace('pipe = pipe.to("cuda")', '# pipe = pipe.to("cuda")')
    k = k.replace("pipe = pipe.to('cuda')", "# pipe = pipe.to('cuda')")
    k = k.replace('pipe.to("cuda")', '# pipe.to("cuda")')
    with open(yol_zoo, "w", encoding="utf-8") as f: f.write(k)
    
eski_zoo = "pipe = model_cls.from_pretrained(ckpt_or_pretrained, torch_dtype=torch_dtype, **kwargs)"
yeni_zoo = "kwargs.pop('device_map', None)\n" + eski_zoo
akilli_yama("scripts/sd_model_zoo.py", eski_zoo, yeni_zoo)

yeni_init = '''model_zoo.init_models()
for attr in dir(model_zoo):
    if attr.startswith('pipe'):
        try: getattr(model_zoo, attr).enable_model_cpu_offload()
        except: pass'''
akilli_yama("gradio_app.py", "model_zoo.init_models()", yeni_init)

eski_norm = "trainer, pipeline = load_pipeline(training_config, checkpoint_path)"
yeni_norm = eski_norm + "\npipeline.enable_model_cpu_offload()"
akilli_yama("app/custom_models/normal_prediction.py", eski_norm, yeni_norm)

akilli_yama("custum_3d_diffusion/custum_modules/unifield_processor.py", "**unet_kwargs", "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False", tam_satir=False)
akilli_yama("custum_3d_diffusion/custum_modules/unifield_processor.py", "torch_dtype=self.unet.dtype", "torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True", tam_satir=False)

# İŞTE KUSURSUZ YAMA: Tensör boyut uyuşmazlığını "sıfır (0)" ile doldurarak çözme algortiması
yeni_load = '''try:
    configurable_unet.unet.to_empty(device="cpu")
except Exception:
    pass
import torch
model_sd = configurable_unet.unet.state_dict()
for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:
    if k in state_dict and k in model_sd:
        s1 = state_dict[k].shape
        s2 = model_sd[k].shape
        if s1 != s2:
            new_w = torch.zeros_like(model_sd[k])
            c_min = min(s1[1], s2[1])
            new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]
            state_dict[k] = new_w
try:
    configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)
except Exception:
    configurable_unet.unet.load_state_dict(state_dict, strict=False)'''
akilli_yama("app/custom_models/utils.py", "configurable_unet.unet.load_state_dict(state_dict, strict=False)", yeni_load)

yeni_to = '''try:
    configurable_unet.unet.to(device, dtype=weight_dtype)
except NotImplementedError:
    configurable_unet.unet.to_empty(device=device)
    configurable_unet.unet.to(device, dtype=weight_dtype)'''
akilli_yama("app/custom_models/utils.py", "configurable_unet.unet.to(device, dtype=weight_dtype)", yeni_to)

# KOPYALAMA KORUMALI DEĞİŞKENLER (Kesilmeyi Engeller)
eski_rm = "rm_normals = predict_normals([img.resize((512, 512), resample=Image.LANCZOS) for img in img_list], guidance_scale=1.5)"
yeni_rm = "import torch, gc\ntorch.cuda.empty_cache()\ngc.collect()\n" + eski_rm
akilli_yama("scripts/multiview_inference.py", eski_rm, yeni_rm)

eski_gen = "def generate3dv2(rgb_pils, front_pil, init_type, do_refine=True, expansion_weight=0.1):"
yeni_gen = eski_gen + "\n    import torch, gc\n    torch.cuda.empty_cache()\n    gc.collect()"
akilli_yama("app/gradio_3dgen.py", eski_gen, yeni_gen)

akilli_yama("scripts/utils.py", "providers=providers", "providers=['CPUExecutionProvider']", tam_satir=False)

for hedef in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(hedef):
        with open(hedef, "r", encoding="utf-8") as f: k = f.read()
        k = k.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in k: k = k.replace(".launch(", ".launch(share=True, ")
        if "cached_download" not in k: k = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + k
        with open(hedef, "w", encoding="utf-8") as f: f.write(k)

print("🚀 6. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Hatalı (CPU) PyTorch Sürümü Zorla Siliniyor...")
!pip uninstall -y torch torchvision torchaudio

print("2. Gerçek Ekran Kartı (CUDA) PyTorch Kuruluyor (Kaggle'da 1-2 dakika sürebilir)...")
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

print("🚀 3. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Eski Uyumsuz Pytorch3D Parçası Temizleniyor...")
!pip uninstall -y pytorch3d

print("2. Pytorch3D, Yeni CUDA Motoru İçin Baştan Derleniyor (2-3 dakika sürebilir)...")
!pip install --no-cache-dir -q "git+https://github.com/facebookresearch/pytorch3d.git"

print("🚀 3. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Hatalı (CPU) PyTorch Sürümü Zorla Siliniyor...")
!pip uninstall -y torch torchvision torchaudio

print("2. Gerçek Ekran Kartı (CUDA) PyTorch Kuruluyor (1-2 dakika sürebilir)...")
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

print("🚀 3. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Eski Uyumsuz Pytorch3D Parçası Temizleniyor...")
!pip uninstall -y pytorch3d

print("2. Pytorch3D, Yeni CUDA Motoru İçin Baştan Derleniyor (Kaggle'da 2-3 dakika sürebilir)...")
!pip install --no-cache-dir -q "git+https://github.com/facebookresearch/pytorch3d.git"

print("🚀 3. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

print("🟢 1. Kaggle Ortamı Hazırlanıyor...")
if not os.path.exists(f"{WORK_DIR}/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir(f'{WORK_DIR}/Unique3D')

print("🟢 2. Bozuk CPU PyTorch Temizlenip, Gerçek GPU (CUDA) PyTorch Kuruluyor...")
os.system("pip uninstall -y torch torchvision torchaudio pytorch3d")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("🟢 3. Paketler ve Kütüphaneler Kuruluyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fire", "rembg", "onnxruntime-gpu", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard"])

print("🟢 4. Pytorch3D ve Nvdiffrast C++ Motorları Yeni Ekran Kartına Göre Derleniyor (3-4 dk sürebilir)...")
for klasor in ["nvdiffrast", "nvdiffrast_repo"]:
    if os.path.exists(klasor): shutil.rmtree(klasor)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("🟢 5. Model Ağırlıkları Kontrol Ediliyor...")
if not os.path.exists("ckpt/img2mvimg"):
    os.system("huggingface-cli download Wuvin/Unique3D --repo-type space --include 'ckpt/*' --local-dir . --local-dir-use-symlinks False")

print("🟢 6. 4 Kanallı U-Net Kısayolları Oluşturuluyor...")
img2mvimg_yolu, unet_4ch_yolu = f"{WORK_DIR}/Unique3D/ckpt/img2mvimg", f"{WORK_DIR}/Unique3D/ckpt/unet_4ch"
os.makedirs(os.path.join(unet_4ch_yolu, "unet"), exist_ok=True)
def guvenli_kisayol(k, h):
    if not os.path.exists(h):
        try: os.symlink(k, h)
        except Exception: pass
if os.path.exists(img2mvimg_yolu):
    for d in os.listdir(img2mvimg_yolu):
        if d != "unet": guvenli_kisayol(os.path.join(img2mvimg_yolu, d), os.path.join(unet_4ch_yolu, d))
    for d in os.listdir(os.path.join(img2mvimg_yolu, "unet")):
        if d != "config.json": guvenli_kisayol(os.path.join(img2mvimg_yolu, "unet", d), os.path.join(unet_4ch_yolu, "unet", d))
    cfg_path = os.path.join(img2mvimg_yolu, "unet", "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: cfg = json.load(f)
        cfg["in_channels"] = 4
        cfg.pop("num_class_embeds", None)
        if "addition_embed_type" in cfg: cfg["addition_embed_type"] = None
        with open(os.path.join(unet_4ch_yolu, "unet", "config.json"), "w") as f: json.dump(cfg, f, indent=2)

print("🟢 7. Kusursuz Yamalar (Tensör, Scope ve Diffusers Yamaları) Uygulanıyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py")
os.system("git checkout app/custom_models/utils.py")

for kok, _, dosyalar in os.walk("."):
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            try:
                with open(yol, "r") as f: c = f.read()
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu)
                    degisti = True
                if "normal" in dosya.lower() and img2mvimg_yolu in c:
                    c = c.replace(img2mvimg_yolu, unet_4ch_yolu)
                    degisti = True
                if degisti:
                    with open(yol, "w") as f: f.write(c)
            except Exception: pass

with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "r") as f: kod1 = f.read()
kod1 = kod1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
kod1 = kod1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
with open("custum_3d_diffusion/custum_modules/unifield_processor.py", "w") as f: f.write(kod1)

with open("app/custom_models/utils.py", "r") as f: kod2 = f.read()
yeni_load = "try: configurable_unet.unet.to_empty(device='cpu')\n    except: pass\n    model_sd = configurable_unet.unet.state_dict()\n    for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n        if k in state_dict and k in model_sd:\n            s1, s2 = state_dict[k].shape, model_sd[k].shape\n            if s1 != s2:\n                new_w = torch.zeros_like(model_sd[k])\n                c_min = min(s1[1], s2[1])\n                new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]\n                state_dict[k] = new_w\n    try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n    except: configurable_unet.unet.load_state_dict(state_dict, strict=False)"
kod2 = kod2.replace("configurable_unet.unet.load_state_dict(state_dict, strict=False)", yeni_load)
yeni_to = "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n    except NotImplementedError:\n        configurable_unet.unet.to_empty(device=device)\n        configurable_unet.unet.to(device, dtype=weight_dtype)"
kod2 = kod2.replace("configurable_unet.unet.to(device, dtype=weight_dtype)", yeni_to)
with open("app/custom_models/utils.py", "w") as f: f.write(kod2)

for hedef in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(hedef):
        with open(hedef, "r") as f: k = f.read()
        k = k.replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in k: k = k.replace(".launch(", ".launch(share=True, ")
        if "cached_download" not in k: k = "import huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + k
        with open(hedef, "w") as f: f.write(k)

print("🚀 8. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
!pip install --no-cache-dir git+https://github.com/NVlabs/nvdiffrast.git

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Eski Motora Ait Uyumsuz Pytorch3D Parçası Siliniyor...")
!pip uninstall -y pytorch3d

print("2. Pytorch3D, Yeni Ekran Kartı Motoruna (CUDA) Göre Yeniden Derleniyor (Bu 2-3 dakika sürebilir)...")
!pip install --no-cache-dir -q "git+https://github.com/facebookresearch/pytorch3d.git"

print("🚀 3. Füzeyi Ateşle! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Bozulan PyTorch sürümü Orijinal GPU (CUDA) versiyonu ile değiştiriliyor...")
# Sadece ekran kartı uyumlu orijinal PyTorch sürümünü zorla yüklüyoruz
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --upgrade

print("2. Füzeyi Ateşle! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Hatalı (CPU) PyTorch Sürümü Zorla Siliniyor...")
!pip uninstall -y torch torchvision torchaudio

print("2. Gerçek Ekran Kartı (CUDA) PyTorch Kuruluyor...")
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

print("3. Füzeyi Ateşle! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Bozulan Dosya Fabrika Ayarlarına Döndürülüyor...")
os.system("git checkout app/custom_models/utils.py")

print("2. Kusursuz Boşluk (Indentation) Korumalı Yama Uygulanıyor...")
with open("app/custom_models/utils.py", "r", encoding="utf-8") as f:
    satirlar = f.readlines()
    
for i, satir in enumerate(satirlar):
    # Model Yükleme ve Tensör Dolgusu (Boşlukları otomatik hizalar)
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in satir:
        bosluk = satir[:len(satir) - len(satir.lstrip())]
        yeni_kod = f'''try:
{bosluk}    configurable_unet.unet.to_empty(device="cpu")
{bosluk}except Exception:
{bosluk}    pass
{bosluk}import torch
{bosluk}model_sd = configurable_unet.unet.state_dict()
{bosluk}for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:
{bosluk}    if k in state_dict and k in model_sd:
{bosluk}        s1 = state_dict[k].shape
{bosluk}        s2 = model_sd[k].shape
{bosluk}        if s1 != s2:
{bosluk}            new_w = torch.zeros_like(model_sd[k])
{bosluk}            c_min = min(s1[1], s2[1])
{bosluk}            new_w[:, :c_min, :, :] = state_dict[k][:, :c_min, :, :]
{bosluk}            state_dict[k] = new_w
{bosluk}try:
{bosluk}    configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)
{bosluk}except Exception:
{bosluk}    configurable_unet.unet.load_state_dict(state_dict, strict=False)'''
        satirlar[i] = bosluk + yeni_kod.strip() + '\n'
        
    # Meta-Tensör Onarımı
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in satir:
        bosluk = satir[:len(satir) - len(satir.lstrip())]
        yeni_kod = f'''try:
{bosluk}    configurable_unet.unet.to(device, dtype=weight_dtype)
{bosluk}except NotImplementedError:
{bosluk}    configurable_unet.unet.to_empty(device=device)
{bosluk}    configurable_unet.unet.to(device, dtype=weight_dtype)'''
        satirlar[i] = bosluk + yeni_kod.strip() + '\n'

with open("app/custom_models/utils.py", "w", encoding="utf-8") as f:
    f.writelines(satirlar)

print("🚀 3. FÜZEYİ ATEŞLE! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import os
os.chdir('/kaggle/working/Unique3D')

print("1. Eski Motora Ait Uyumsuz Pytorch3D Parçası Siliniyor...")
!pip uninstall -y pytorch3d

print("2. Pytorch3D, Yeni Ekran Kartı Motoruna (CUDA) Göre Yeniden Derleniyor (Bu 2-3 dakika sürebilir)...")
# --no-cache-dir komutu eski bozuk sürümü hafızadan çağırmasını engeller, sıfırdan inşa eder
!pip install --no-cache-dir -q "git+https://github.com/facebookresearch/pytorch3d.git"

print("🚀 3. Füzeyi Ateşle! Unique3D Stüdyosu Başlatılıyor...")
!python gradio_app.py

In [ ]:
import gradio as gr
import os

def dosyalari_bul():
    bulunanlar = []
    # Kaggle'daki tüm çalışma ve geçici klasörleri tarar
    for klasor in ['/kaggle/working', '/tmp']:
        for root, dirs, files in os.walk(klasor):
            for dosya in files:
                if dosya.endswith('.obj') or dosya.endswith('.glb') or dosya.endswith('.zip'):
                    bulunanlar.append(os.path.join(root, dosya))
    return bulunanlar

# İndirme Sitesinin Arayüzü
with gr.Blocks() as demo:
    gr.Markdown("## 📦 Sanal Kabin - 3D Model İndirme Merkezi")
    gr.Markdown("Arka planda üretilen 3D model dosyalarını aşağıdan bilgisayarına indirebilirsin.")
    
    bul_butonu = gr.Button("Üretilen Modelleri Getir", variant="primary")
    indirme_alani = gr.File(label="Dosyalar", file_count="multiple")
    
    bul_butonu.click(fn=dosyalari_bul, inputs=[], outputs=indirme_alani)

# Siteyi yayına al
demo.launch(share=True)

In [ ]:
import os
import sys
import torch

print("1. Yeni 3D Motorumuz (TripoSR) İndiriliyor...")
os.chdir('/kaggle/working')

if not os.path.exists("TripoSR"):
    os.system("git clone https://github.com/VAST-AI-Research/TripoSR.git")

os.chdir('/kaggle/working/TripoSR')

print("2. TripoSR Kütüphaneleri Kuruluyor...")
os.system("pip install -q -r requirements.txt")

print("3. Ürettiğimiz Resim Kontrol Ediliyor...")
resim_yolu = "/kaggle/working/Sonuc_Aci_0.png"

if not os.path.exists(resim_yolu):
    print(f"\n❌ HATA: {resim_yolu} bulunamadı!")
    print("Önceki adımdaki resimlerin Kaggle 'working' klasörüne kaydedildiğinden emin olun.")
else:
    print("✅ Resim bulundu! 3D İskelet (Mesh) örme işlemi başlıyor...")
    
    # TripoSR motorunu çalıştırıyoruz
    print("4. TripoSR Yapay Zekası Çalışıyor (Bu işlem sadece 5-10 saniye sürecek!)...")
    
    # Kodu arka planda çalıştırıp çıktıyı Tripo3D_Sonuc klasörüne kaydediyoruz
    os.system(f"python run.py {resim_yolu} --output-dir /kaggle/working/Tripo3D_Sonuc")
    
    print("\n🎉 ZAFER! 3D Model başarıyla üretildi.")
    print("👉 Kaggle'ın sağ menüsündeki Output (/kaggle/working) klasörüne gidin.")
    print("👉 'Tripo3D_Sonuc' klasörünün içindeki 'mesh.obj' dosyasını bilgisayarınıza indirebilirsiniz!")

In [ ]:
import os
import trimesh
from IPython.display import FileLink, display

# 1. /kaggle/working/ klasöründeki .obj dosyasını OTOMATİK buluyoruz
klasor = '/kaggle/working/'
obj_dosyalari = [f for f in os.listdir(klasor) if f.endswith('.obj')]

if not obj_dosyalari:
    print("❌ HATA: Klasörde hiç .obj dosyası bulunamadı!")
    print("👉 Modelin Unique3D tarafından üretildiğinden emin ol.")
else:
    gercek_obj_adi = obj_dosyalari[0] # Bulduğu ilk obj dosyasını alır
    obj_yolu = os.path.join(klasor, gercek_obj_adi)
    glb_yolu = '/kaggle/working/Sanal_Kabin_Modeli.glb'

    print(f"✅ Model bulundu: {gercek_obj_adi}")
    print("⏳ GLB formatına dönüştürülüyor, lütfen bekle...")
    
    # 2. Modeli yüklüyoruz
    mesh = trimesh.load(obj_yolu)

    # 3. React projesi için .glb olarak dışa aktarıyoruz
    mesh.export(glb_yolu)

    print("🎉 Dönüştürme Başarılı! Modelini indirebilirsin:")
    display(FileLink('Sanal_Kabin_Modeli.glb'))

In [ ]:
# 1. Eksik olan renk kaplama kütüphanesini kuruyoruz
!pip install -q xatlas

# 2. TripoSR klasörüne geçiş yapıyoruz
%cd /kaggle/working/TripoSR

# 3. Ve TripoSR'ı gerçek anlamda çalıştırıyoruz!
!python run.py /kaggle/working/Sonuc_Aci_0.png --output-dir /kaggle/working/Tripo3D_Sonuc

In [ ]:
# 1. Eksik olan 3D iskelet örme (Marching Cubes) kütüphanesini manuel kuruyoruz
!pip install -q torchmcubes

# 2. TripoSR klasörüne geçiş yapıyoruz
%cd /kaggle/working/TripoSR

# 3. Ve TripoSR'ı tekrar çalıştırıyoruz! Hadi bu kez tam isabet!
!python run.py /kaggle/working/Sonuc_Aci_0.png --output-dir /kaggle/working/Tripo3D_Sonuc

In [ ]:
# 1. Kütüphaneyi marketten değil, doğrudan GitHub'ın kalbinden çekip kuruyoruz!
!pip install -q git+https://github.com/tatsy/torchmcubes.git

# 2. TripoSR klasörüne geçiş yapıyoruz
%cd /kaggle/working/TripoSR

# 3. Ve şalteri tekrar indiriyoruz!
!python run.py /kaggle/working/Sonuc_Aci_0.png --output-dir /kaggle/working/Tripo3D_Sonuc

In [ ]:
# 1. Derleme için gereken temel malzemeleri hazırlıyoruz
!pip install -q setuptools wheel cython numpy

# 2. İZOLASYONU KIRIYORUZ! (--no-build-isolation ile zorla kurulum)
!pip install -q git+https://github.com/tatsy/torchmcubes.git --no-build-isolation

# 3. TripoSR klasörüne geçiş yapıyoruz
import os
os.chdir('/kaggle/working/TripoSR')

# 4. Ve TripoSR'ı çalıştırıyoruz!
!python run.py /kaggle/working/Sonuc_Aci_0.png --output-dir /kaggle/working/Tripo3D_Sonuc

In [ ]:
# 1. Önce inşaat ustalarını (derleme araçlarını) Kaggle'a getiriyoruz
!pip install -q scikit-build-core cmake ninja

# 2. Şimdi o izolasyonu kırıp motoru zorla kuruyoruz!
!pip install -q git+https://github.com/tatsy/torchmcubes.git --no-build-isolation

# 3. TripoSR klasörüne geçiş yapıyoruz
import os
os.chdir('/kaggle/working/TripoSR')

# 4. Ve final vuruşu... TripoSR çalışıyor!
!python run.py /kaggle/working/Sonuc_Aci_0.png --output-dir /kaggle/working/Tripo3D_Sonuc

In [ ]:
import os
import shutil
from IPython.display import FileLink, display

print("1. Klasörler kontrol ediliyor...")
if not os.path.exists('/kaggle/working/TripoSR'):
    os.system("git clone https://github.com/VAST-AI-Research/TripoSR.git /kaggle/working/TripoSR")

print("2. Kütüphaneler, TripoSR'ın anlayacağı DOĞRU VERSİYONLARLA kuruluyor...")
os.system("pip install -q -r /kaggle/working/TripoSR/requirements.txt")
# İŞTE BURASI: 'transformers==4.38.2' diyerek beyni okuyamama sorununu kökten çözüyoruz!
os.system("pip install -q rembg onnxruntime xatlas PyMCubes trimesh moderngl transformers==4.38.2")

print("3. TripoSR'ı kandıracak Truva Atı yerleştiriliyor...")
sahte_kutuphane = """import mcubes
import torch

def marching_cubes(volume, isovalue):
    vol_np = volume.detach().cpu().float().numpy()
    verts, faces = mcubes.marching_cubes(vol_np, float(isovalue))
    
    verts_tensor = torch.tensor(verts.copy(), dtype=torch.float32, device=volume.device)
    faces_tensor = torch.tensor(faces.copy().astype('int64'), dtype=torch.int64, device=volume.device)
    return verts_tensor, faces_tensor
"""
with open("/kaggle/working/TripoSR/torchmcubes.py", "w", encoding="utf-8") as f:
    f.write(sahte_kutuphane)

print("✅ Kurulumlar tamam! 3D Model Üretimi BAŞLIYOR...\n")

os.chdir('/kaggle/working/TripoSR')
!python run.py /kaggle/working/Sonuc_Aci_0.png --output-dir /kaggle/working/Tripo3D_Sonuc

if os.path.exists('/kaggle/working/Tripo3D_Sonuc/mesh.obj'):
    shutil.make_archive('/kaggle/working/Benim_3D_Modelim', 'zip', '/kaggle/working/Tripo3D_Sonuc')
    print("\n🎉 İŞTE BU KADAR! MODEL KAGGLE'DA KENDİ BİLGİSAYARIMIZ GİBİ ÜRETİLDİ!")
    display(FileLink('/kaggle/working/Benim_3D_Modelim.zip'))
else:
    print("\n❌ Bir şeyler ters gitti, lütfen yukarıdaki kırmızı hata loglarını bana kopyala.")

In [ ]:
import os
import shutil
from IPython.display import FileLink, display

# 0. GÜVENLİ BÖLGEYE ÇIKIŞ: İçinde bulunduğumuz oda silinmesin diye ana klasöre dönüyoruz!
os.chdir('/kaggle/working')

print("🧹 1. Eski ve bozuk kalıntılar temizleniyor...")
if os.path.exists('/kaggle/working/TripoSR'):
    shutil.rmtree('/kaggle/working/TripoSR') 

print("⬇️ 2. TripoSR kodları GitHub'dan tertemiz indiriliyor...")
os.system("git clone https://github.com/VAST-AI-Research/TripoSR.git /kaggle/working/TripoSR")

if not os.path.exists('/kaggle/working/TripoSR/run.py'):
    print("❌ HATA: Kodlar inemedi! Lütfen işlemi tekrar başlatın.")
else:
    print("🛠️ 3. Bağımlılıklar ve eksik kütüphaneler kuruluyor (Bu biraz sürebilir)...")
    os.system("pip install -q -r /kaggle/working/TripoSR/requirements.txt")
    os.system("pip install -q rembg onnxruntime xatlas PyMCubes trimesh moderngl transformers==4.38.2")

    print("🐎 4. Truva Atı yerleştiriliyor...")
    sahte_kutuphane = """import mcubes\nimport torch\n\ndef marching_cubes(volume, isovalue):\n    vol_np = volume.detach().cpu().float().numpy()\n    verts, faces = mcubes.marching_cubes(vol_np, float(isovalue))\n    verts_tensor = torch.tensor(verts.copy(), dtype=torch.float32, device=volume.device)\n    faces_tensor = torch.tensor(faces.copy().astype('int64'), dtype=torch.int64, device=volume.device)\n    return verts_tensor, faces_tensor\n"""
    with open("/kaggle/working/TripoSR/torchmcubes.py", "w", encoding="utf-8") as f:
        f.write(sahte_kutuphane)

    print("🔍 5. Resim kontrol ediliyor...")
    resim_yolu = "/kaggle/working/Sonuc_Aci_0.png"
    hedef_klasor = "/kaggle/working/Tripo3D_Sonuc"

    if not os.path.exists(resim_yolu):
        print(f"\n❌ DUR: {resim_yolu} bulunamadı!")
        print("👉 Lütfen sağ menüden 'working' klasörünün içine 'Sonuc_Aci_0.png' isimli resmini sürükleyip bırak.")
    else:
        print("🚀 6. HER ŞEY HAZIR! 3D Model Örülüyor (10-15 saniye bekleyin)...")
        os.chdir('/kaggle/working/TripoSR')
        
        # O muhteşem son vuruş
        os.system(f"python run.py {resim_yolu} --output-dir {hedef_klasor}")

        if os.path.exists(f"{hedef_klasor}/mesh.obj"):
            zip_adi = '/kaggle/working/Benim_3D_Modelim'
            # İşimiz bitince ana klasöre geri dönüp öyle zipleme yapıyoruz (garanti olsun diye)
            os.chdir('/kaggle/working') 
            shutil.make_archive(zip_adi, 'zip', hedef_klasor)
            print("\n🎉 ZAFER! MODEL KUSURSUZCA ÜRETİLDİ VE PAKETLENDİ!")
            display(FileLink('Benim_3D_Modelim.zip'))
        else:
            print("\n❌ İşlem tamamlanamadı. Bir hata oluştu.")

In [ ]:
%cd /kaggle/working/TripoSR
!python run.py /kaggle/working/Sonuc_Aci_0.png --output-dir /kaggle/working/Tripo3D_Sonuc

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

print("1. Unique3D Projesi Hazirlaniyor...")
if not os.path.exists("/kaggle/working/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir("/kaggle/working/Unique3D")

print("2. Orijinal GPU PyTorch (CUDA 12.4) Kuruluyor...")
os.system("pip uninstall -y xformers torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("3. Gerekli Kutuphaneler Yukleniyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel", "fire", "rembg", "onnxruntime", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard", "huggingface_hub"])

print("4. C++ 3D Motorlari (Nvdiffrast ve PyTorch3D) Derleniyor...")
shutil.rmtree("nvdiffrast", ignore_errors=True)
shutil.rmtree("nvdiffrast_repo", ignore_errors=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Indiriliyor...")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    shutil.rmtree("ckpt/img2mvimg", ignore_errors=True)
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

print("6. 4 Kanalli U-Net (unet_4ch) Mimarisi Kuruluyor...")
img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"
shutil.rmtree(unet_4ch_yolu, ignore_errors=True)
os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)

ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
if os.path.exists(ana_cfg_path):
    ana_cfg = json.loads(open(ana_cfg_path, "r").read())
    ana_cfg["in_channels"] = 4
    ana_cfg.pop("num_class_embeds", None)
    ana_cfg["addition_embed_type"] = None
    open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))

for d in os.listdir(img2mvimg_yolu):
    if d != "unet" and not os.path.exists(unet_4ch_yolu + "/" + d):
        os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)

for d in os.listdir(img2mvimg_yolu + "/unet"):
    if d != "config.json" and not os.path.exists(unet_4ch_yolu + "/unet/" + d):
        os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("7. Tum Zirhlar ve Soket Dostu VRAM Bosaltici Isleniyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py app/custom_models/utils.py scripts/utils.py scripts/multiview_inference.py scripts/project_mesh.py mesh_reconstruction/refine.py app/gradio_3dgen.py gradio_app.py")

oom_lines = [
    "import os, gc, torch",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    open("scripts/utils.py", "w").write(open("scripts/utils.py", "r").read().replace("providers=providers", "providers=['CPUExecutionProvider']"))

if os.path.exists("scripts/project_mesh.py"):
    open("scripts/project_mesh.py", "w").write(open("scripts/project_mesh.py", "r").read().replace("assert cos_angles.mean() < 0", "# assert cos_angles.mean() < 0"))

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("🚀 8. STUDYO TAM ZIRHLI OLARAK BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. scripts/utils.py icindeki eksik 'session' degiskeni onariliyor...")
if os.path.exists("scripts/utils.py"):
    u_kod = open("scripts/utils.py", "r", encoding="utf-8").read()
    u_kod = u_kod.replace("providers=providers", "providers=['CPUExecutionProvider']")
    if "session =" not in u_kod and "session=" not in u_kod:
        u_kod += "\nimport rembg\ntry:\n    session = rembg.new_session(providers=['CPUExecutionProvider'])\nexcept Exception:\n    session = None\n"
    open("scripts/utils.py", "w", encoding="utf-8").write(u_kod)

print("2. Kamera acisi kilidi (AssertionError) kontrol ediliyor...")
if os.path.exists("scripts/project_mesh.py"):
    pm_satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_pm = []
    for s in pm_satirlar:
        if "assert " in s and "cos_angles" in s and not s.lstrip().startswith("pass"):
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_pm.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_pm.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_pm)

print("🚀 3. Studyo Baslatiliyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.chdir("/kaggle/working/Unique3D")

print("1. Güncel scripts/utils.py Hugging Face'ten indiriliyor ve onarılıyor...")
try:
    hf_hub_download(
        repo_id="Wuvin/Unique3D",
        repo_type="space",
        filename="scripts/utils.py",
        local_dir=".",
        local_dir_use_symlinks=False
    )
except Exception as e:
    print("HF indirme uyarısı:", e)

u_kod = open("scripts/utils.py", "r", encoding="utf-8").read()
u_kod = u_kod.replace("providers=providers", "providers=['CPUExecutionProvider']")

# Eğer hala session veya simple_preprocess eksikse doğrudan ekle:
ek_kod = """
import rembg
from PIL import Image

if 'session' not in globals():
    try:
        session = rembg.new_session(providers=['CPUExecutionProvider'])
    except Exception:
        session = rembg.new_session()

if 'simple_preprocess' not in globals():
    def simple_preprocess(input_image, rembg_session=None, background_color=255):
        if rembg_session is None:
            rembg_session = session
        if input_image.mode != 'RGBA':
            input_image = rembg.remove(input_image, session=rembg_session)
        arr = np.array(input_image)
        alpha = arr[:, :, 3:4].astype(np.float32) / 255.0
        rgb = arr[:, :, :3].astype(np.float32)
        bg = np.ones_like(rgb) * background_color
        out = (rgb * alpha + bg * (1.0 - alpha)).astype(np.uint8)
        return Image.fromarray(out)
"""
if "def simple_preprocess" not in u_kod:
    u_kod = "import numpy as np\n" + u_kod + "\n" + ek_kod

open("scripts/utils.py", "w", encoding="utf-8").write(u_kod)

print("🚀 2. Stüdyo Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
from huggingface_hub import snapshot_download

os.chdir("/kaggle/working/Unique3D")

print("1. GitHub'in eski dosyalari yerine Hugging Face'teki guncel kod dosyalari indiriliyor...")
snapshot_download(
    repo_id="Wuvin/Unique3D",
    repo_type="space",
    allow_patterns=["app/*", "scripts/*", "mesh_reconstruction/*", "custum_3d_diffusion/*", "gradio_app.py"],
    local_dir=".",
    local_dir_use_symlinks=False
)

img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"

print("2. Dosya yollari ve yamalar guncel kodlara isleniyor...")
for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    open("scripts/utils.py", "w").write(open("scripts/utils.py", "r").read().replace("providers=providers", "providers=['CPUExecutionProvider']"))

if os.path.exists("scripts/project_mesh.py"):
    pm_satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_pm = []
    for s in pm_satirlar:
        if "assert " in s and "cos_angles" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_pm.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_pm.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_pm)

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mv_satirlar = open(mv_yolu, "r", encoding="utf-8").readlines()
yeni_mv = []
for s in mv_satirlar:
    if "just check for validation" in s:
        bosluk = s[:len(s) - len(s.lstrip())]
        yeni_mv.append(bosluk + "try:\n" + bosluk + "    " + s.lstrip() + bosluk + "except Exception:\n" + bosluk + "    pass\n")
    else:
        yeni_mv.append(s)
mvc = "".join(yeni_mv)
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w", encoding="utf-8").write(mvc)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("🚀 3. STUDYO BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. app/custom_models/utils.py içindeki SyntaxError düzeltiliyor...")
p2 = "app/custom_models/utils.py"
if os.path.exists(p2):
    kod = open(p2, "r", encoding="utf-8").read()
    
    # Hatalı tek satırlık try bloklarını temizleyip düzgün bloklar ekliyoruz:
    eski_parca = "configurable_unet.unet.load_state_dict(state_dict, strict=False)"
    
    yeni_parca = """
    try:
        configurable_unet.unet.to_empty(device='cpu')
    except:
        pass
    
    msd = configurable_unet.unet.state_dict()
    for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:
        if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:
            nw = torch.zeros_like(msd[k])
            cm = min(state_dict[k].shape[1], msd[k].shape[1])
            nw[:, :cm] = state_dict[k][:, :cm]
            state_dict[k] = nw
            
    try:
        configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)
    except:
        configurable_unet.unet.load_state_dict(state_dict, strict=False)
    """
    
    if eski_parca in kod:
        kod = kod.replace(eski_parca, yeniparca if 'yeniparca' in locals() else yeni_parca)
        open(p2, "w", encoding="utf-8").write(kod)
        print("✅ utils.py başarıyla düzeltildi!")
    else:
        print("⚠️ Eşleşme bulunamadı, dosya zaten güncel olabilir.")

print("🚀 2. Stüdyo Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. app/custom_models/utils.py içindeki hatalı tek satırlık try blokları temizleniyor...")
p2 = "app/custom_models/utils.py"

if os.path.exists(p2):
    with open(p2, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Hatalı/tek satırlık try kalıplarını temiz ve düzgün bloklarla değiştiriyoruz:
    content = content.replace("try: configurable_unet.unet.to_empty(device='cpu')", "try:\n        configurable_unet.unet.to_empty(device='cpu')\n    except Exception:\n        pass")
    content = content.replace("try: configurable_unet.unet.to(device, dtype=weight_dtype)", "try:\n        configurable_unet.unet.to(device, dtype=weight_dtype)\n    except NotImplementedError:\n        configurable_unet.unet.to_empty(device=device)\n        configurable_unet.unet.to(device, dtype=weight_dtype)")
    content = content.replace("try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()", "try:\n        pipeline.enable_attention_slicing(1)\n        pipeline.enable_vae_slicing()\n    except Exception:\n        pass")

    with open(p2, "w", encoding="utf-8") as f:
        f.write(content)
    print("✅ utils.py blok yapısı başarıyla düzeltildi!")

print("🚀 2. Stüdyo Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

print("1. app/custom_models/utils.py girinti (Indentation) hatalari onariliyor...")
p2 = "app/custom_models/utils.py"

if os.path.exists(p2):
    with open(p2, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    fixed_lines = []
    skip_next = False
    for i, line in enumerate(lines):
        # Tek satırlık hatalı try bloklarını temiz ve doğru girintili bloklarla değiştiriyoruz
        if "try:" in line and i + 1 < len(lines) and not lines[i+1].startswith("    "):
            fixed_lines.append(line)
            # Bir sonraki satıra doğru girintiyi ekle
            indent = "    " + line[:len(line) - len(line.lstrip())]
            next_line = lines[i+1].strip()
            fixed_lines.append(indent + next_line + "\n")
            fixed_lines.append(line[:len(line) - len(line.lstrip())] + "except Exception:\n")
            fixed_lines.append(indent + "pass\n")
            skip_next = True
        elif skip_next:
            skip_next = False
            continue
        else:
            fixed_lines.append(line)

    with open(p2, "w", encoding="utf-8") as f:
        f.writelines(fixed_lines)
    print("✅ utils.py girintileri başarıyla düzeltildi!")

print("🚀 2. Stüdyo Başlatılıyor...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, sys, json, shutil, subprocess

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

print("1. Unique3D Projesi Hazirlaniyor...")
if not os.path.exists("/kaggle/working/Unique3D"):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git"], check=True)
os.chdir("/kaggle/working/Unique3D")

print("2. Orijinal GPU PyTorch (CUDA 12.4) Kuruluyor...")
os.system("pip uninstall -y xformers torch torchvision torchaudio pytorch3d nvdiffrast")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("3. Gerekli Kutuphaneler Yukleniyor...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "torch-scatter", "setuptools", "wheel", "fire", "rembg", "onnxruntime", "trimesh", "pymeshlab", "einops", "omegaconf", "transformers==4.38.2", "diffusers==0.27.2", "peft==0.10.0", "accelerate", "pydantic==2.8.2", "jaxtyping", "typeguard", "huggingface_hub"])

print("4. C++ 3D Motorlari (Nvdiffrast ve PyTorch3D) Derleniyor...")
shutil.rmtree("nvdiffrast", ignore_errors=True)
shutil.rmtree("nvdiffrast_repo", ignore_errors=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/facebookresearch/pytorch3d.git"])

print("5. Model Agirliklari Indiriliyor...")
from huggingface_hub import snapshot_download
if not (os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.bin") or os.path.exists("ckpt/img2mvimg/unet/diffusion_pytorch_model.safetensors")):
    shutil.rmtree("ckpt/img2mvimg", ignore_errors=True)
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

print("6. 4 Kanalli U-Net (unet_4ch) Mimarisi Kuruluyor...")
img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"
shutil.rmtree(unet_4ch_yolu, ignore_errors=True)
os.makedirs(unet_4ch_yolu + "/unet", exist_ok=True)

ana_cfg_path = img2mvimg_yolu + "/unet/config.json"
if os.path.exists(ana_cfg_path):
    ana_cfg = json.loads(open(ana_cfg_path, "r").read())
    ana_cfg["in_channels"] = 4
    ana_cfg.pop("num_class_embeds", None)
    ana_cfg["addition_embed_type"] = None
    open(unet_4ch_yolu + "/unet/config.json", "w").write(json.dumps(ana_cfg, indent=2))

for d in os.listdir(img2mvimg_yolu):
    if d != "unet" and not os.path.exists(unet_4ch_yolu + "/" + d):
        os.symlink(img2mvimg_yolu + "/" + d, unet_4ch_yolu + "/" + d)

for d in os.listdir(img2mvimg_yolu + "/unet"):
    if d != "config.json" and not os.path.exists(unet_4ch_yolu + "/unet/" + d):
        os.symlink(img2mvimg_yolu + "/unet/" + d, unet_4ch_yolu + "/unet/" + d)

print("7. Tum Zirhlar ve Soket Dostu VRAM Bosaltici Isleniyor...")
os.system("git checkout custum_3d_diffusion/custum_modules/unifield_processor.py app/custom_models/utils.py scripts/utils.py scripts/multiview_inference.py scripts/project_mesh.py mesh_reconstruction/refine.py app/gradio_3dgen.py gradio_app.py")

oom_lines = [
    "import os, gc, torch",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    open("scripts/utils.py", "w").write(open("scripts/utils.py", "r").read().replace("providers=providers", "providers=['CPUExecutionProvider']"))

if os.path.exists("scripts/project_mesh.py"):
    open("scripts/project_mesh.py", "w").write(open("scripts/project_mesh.py", "r").read().replace("assert cos_angles.mean() < 0", "# assert cos_angles.mean() < 0"))

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("🚀 8. STUDYO TAM ZIRHLI OLARAK BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, json

os.chdir("/kaggle/working/Unique3D")

print("1. Bozulan tüm Python dosyaları orijinal çalışan GitHub sürümüne sıfırlanıyor...")
os.system("git checkout -- .")

img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"

print("2. Çalışan yamalar, VRAM boşaltıcı ve Kamera Açısı kilidi işleniyor...")
oom_lines = [
    "import os, gc, torch",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

# Senin çalışan kodundaki birebir utils.py yaması (hiçbir IndentationError vermez)
p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    open("scripts/utils.py", "w").write(open("scripts/utils.py", "r").read().replace("providers=providers", "providers=['CPUExecutionProvider']"))

# Kamera açısı (AssertionError: cos_angles) kilidi iptali
if os.path.exists("scripts/project_mesh.py"):
    pm_kod = open("scripts/project_mesh.py", "r").read()
    pm_kod = pm_kod.replace("assert cos_angles.mean() < 0", "# assert cos_angles.mean() < 0")
    pm_kod = pm_kod.replace("assert (cos_angles.mean() < 0)", "# assert (cos_angles.mean() < 0)")
    open("scripts/project_mesh.py", "w").write(pm_kod)

# Soket dostu VRAM boşaltıcı
refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

# İkinci üretimde GPU'ya geri yükleme + Doğrulama kontrolü iptali
mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
mvc = mvc.replace("_ = multiview_color_projection(", "# _ = multiview_color_projection(")
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("🚀 3. STÜDYO TAM ZIRHLI OLARAK BAŞLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, shutil, subprocess

os.chdir("/kaggle/working/Unique3D")

print("1. Karışan kod klasörleri temizleniyor (ckpt ağırlıklarına DOKUNULMAZ)...")
for klasor in ["app", "scripts", "mesh_reconstruction", "custum_3d_diffusion"]:
    shutil.rmtree(klasor, ignore_errors=True)

# Git geçmişini ve çalışma ağacını tamamen orijinal GitHubコミtine zorla sıfırla:
os.system("git reset --hard HEAD")
os.system("git clean -fd -e ckpt -e ckpt/*")

img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"

print("2. Senin çalışan yamaların + Kamera açısı kilidi iptali uygulanıyor...")
oom_lines = [
    "import os, gc, torch",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

if os.path.exists("scripts/utils.py"):
    open("scripts/utils.py", "w").write(open("scripts/utils.py", "r").read().replace("providers=providers", "providers=['CPUExecutionProvider']"))

if os.path.exists("scripts/project_mesh.py"):
    open("scripts/project_mesh.py", "w").write(open("scripts/project_mesh.py", "r").read().replace("assert cos_angles.mean() < 0", "# assert cos_angles.mean() < 0"))

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
mvc = mvc.replace("_ = multiview_color_projection(", "# _ = multiview_color_projection(")
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("🚀 3. STUDYO BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os, shutil

os.chdir("/kaggle/working/Unique3D")

print("1. Kod agaci temizleniyor (IndentationError onlenir, ckpt korunur)...")
for klasor in ["app", "scripts", "mesh_reconstruction", "custum_3d_diffusion"]:
    shutil.rmtree(klasor, ignore_errors=True)
os.system("git reset --hard HEAD")
os.system("git clean -fd -e ckpt -e ckpt/*")

img2mvimg_yolu = "/kaggle/working/Unique3D/ckpt/img2mvimg"
unet_4ch_yolu = "/kaggle/working/Unique3D/ckpt/unet_4ch"

print("2. Senin calisan yamalarin + session/simple_preprocess + Kamera kilidi isleniyor...")
oom_lines = [
    "import os, gc, torch",
    "import torch.nn.functional as _F",
    "os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'",
    "if not hasattr(_F, '_orig_sdpa'): _F._orig_sdpa = _F.scaled_dot_product_attention",
    "def _sliced_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):",
    "    if query.dim() == 4 and query.shape[0]*query.shape[1]*query.shape[2]*key.shape[2] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for b in range(query.shape[0]):",
    "            for h in range(query.shape[1]):",
    "                for i in range(0, query.shape[2], 1024):",
    "                    out[b:b+1, h:h+1, i:i+1024] = _F._orig_sdpa(query[b:b+1, h:h+1, i:i+1024], key[b:b+1, h:h+1], value[b:b+1, h:h+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    elif query.dim() == 3 and query.shape[0]*query.shape[1]*key.shape[1] > 2097152 and attn_mask is None:",
    "        out = torch.empty_like(query)",
    "        for bh in range(query.shape[0]):",
    "            for i in range(0, query.shape[1], 1024):",
    "                out[bh:bh+1, i:i+1024] = _F._orig_sdpa(query[bh:bh+1, i:i+1024], key[bh:bh+1], value[bh:bh+1], attn_mask=None, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "        return out",
    "    return _F._orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)",
    "_F.scaled_dot_product_attention = _sliced_sdpa",
    "torch.nn.functional.scaled_dot_product_attention = _sliced_sdpa"
]
open("oom_patch.py", "w", encoding="utf-8").write("\n".join(oom_lines) + "\n")

for kok, _, dosyalar in os.walk("."):
    if "ckpt" in kok: continue
    for dosya in dosyalar:
        if dosya.endswith((".py", ".yaml", ".yml", ".json")):
            yol = os.path.join(kok, dosya)
            c = open(yol, "r", errors="ignore").read()
            c_yeni = c.replace("lambdalabs/sd-image-variations-diffusers", img2mvimg_yolu).replace("./ckpt/img2mvimg", img2mvimg_yolu)
            if "normal" in dosya.lower(): c_yeni = c_yeni.replace(img2mvimg_yolu, unet_4ch_yolu)
            if c != c_yeni: open(yol, "w").write(c_yeni)

p1 = "custum_3d_diffusion/custum_modules/unifield_processor.py"
k1 = open(p1, "r").read()
k1 = k1.replace('subfolder="unet", torch_dtype=self.unet.dtype', 'subfolder="unet", torch_dtype=self.unet.dtype, low_cpu_mem_usage=False, ignore_mismatched_sizes=True')
k1 = k1.replace('**unet_kwargs', "**{k: v for k, v in unet_kwargs.items() if k != 'low_cpu_mem_usage'}, low_cpu_mem_usage=False")
open(p1, "w").write(k1)

p2 = "app/custom_models/utils.py"
kod2 = open(p2, "r").readlines()
for i, s in enumerate(kod2):
    b = s[:len(s) - len(s.lstrip())]
    if "configurable_unet.unet.load_state_dict(state_dict, strict=False)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to_empty(device='cpu')\n" + b + "except: pass\n" + b + "msd = configurable_unet.unet.state_dict()\n" + b + "for k in ['conv_in.weight', 'ref_unet.conv_in.weight']:\n" + b + "    if k in state_dict and k in msd and state_dict[k].shape != msd[k].shape:\n" + b + "        nw = torch.zeros_like(msd[k]); cm = min(state_dict[k].shape[1], msd[k].shape[1]); nw[:, :cm] = state_dict[k][:, :cm]; state_dict[k] = nw\n" + b + "try: configurable_unet.unet.load_state_dict(state_dict, strict=False, assign=True)\n" + b + "except: configurable_unet.unet.load_state_dict(state_dict, strict=False)\n"
    elif "configurable_unet.unet.to(device, dtype=weight_dtype)" in s:
        kod2[i] = b + "try: configurable_unet.unet.to(device, dtype=weight_dtype)\n" + b + "except NotImplementedError: configurable_unet.unet.to_empty(device=device); configurable_unet.unet.to(device, dtype=weight_dtype)\n"
    elif "return trainer, pipeline" in s:
        kod2[i] = b + "try: pipeline.enable_attention_slicing(1); pipeline.enable_vae_slicing()\n" + b + "except: pass\n" + b + "return trainer, pipeline\n"
open(p2, "w").writelines(kod2)

# GitHub'daki scripts/utils.py icinde eksik olan session ve simple_preprocess eklentisi:
if os.path.exists("scripts/utils.py"):
    u_kod = open("scripts/utils.py", "r", encoding="utf-8").read()
    u_kod = u_kod.replace("providers=providers", "providers=['CPUExecutionProvider']")
    u_ek = [
        "",
        "import rembg",
        "try:",
        "    session = rembg.new_session(providers=['CPUExecutionProvider'])",
        "except Exception:",
        "    session = rembg.new_session()",
        "",
        "def simple_preprocess(input_image, rembg_session=None, background_color=255):",
        "    if 'remove_backgroud' in globals():",
        "        return remove_backgroud(input_image, rembg_session=rembg_session or session, background_color=background_color)",
        "    return input_image"
    ]
    if "def simple_preprocess" not in u_kod:
        u_kod = u_kod + "\n" + "\n".join(u_ek) + "\n"
    open("scripts/utils.py", "w", encoding="utf-8").write(u_kod)

# Kamera acisi (AssertionError: cos_angles) kilidi iptali
if os.path.exists("scripts/project_mesh.py"):
    pm_satirlar = open("scripts/project_mesh.py", "r", encoding="utf-8").readlines()
    yeni_pm = []
    for s in pm_satirlar:
        if "assert " in s and "cos_angles" in s:
            bosluk = s[:len(s) - len(s.lstrip())]
            yeni_pm.append(bosluk + "pass # " + s.lstrip())
        else:
            yeni_pm.append(s)
    open("scripts/project_mesh.py", "w", encoding="utf-8").writelines(yeni_pm)

refine_yolu = "mesh_reconstruction/refine.py"
rc = open(refine_yolu, "r").read()
guvenli_bosalt = [
    "import sys, gc, torch",
    "def _bosalt_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cpu')",
    "                    except: pass",
    "    gc.collect()",
    "    torch.cuda.empty_cache()"
]
rc = "\n".join(guvenli_bosalt) + "\n" + rc
rc = rc.replace(
    "def run_mesh_refine(",
    "def run_mesh_refine(*args, **kwargs):\n    _bosalt_vram()\n    return _orig_run_mesh_refine(*args, **kwargs)\ndef _orig_run_mesh_refine("
)
open(refine_yolu, "w").write(rc)

mv_yolu = "scripts/multiview_inference.py"
mvc = open(mv_yolu, "r").read()
mvc = mvc.replace("_ = multiview_color_projection(", "# _ = multiview_color_projection(")
guvenli_yukle = [
    "import sys, gc, torch",
    "def _yukle_vram():",
    "    for m_adi in ['app.custom_models.mvimg_prediction', 'app.custom_models.normal_prediction', 'scripts.upscale']:",
    "        mod = sys.modules.get(m_adi)",
    "        if mod:",
    "            for k in ['pipeline', 'trainer']:",
    "                obj = getattr(mod, k, None)",
    "                if obj is not None and hasattr(obj, 'to'):",
    "                    try: obj.to('cuda')",
    "                    except: pass"
]
mvc = "\n".join(guvenli_yukle) + "\n" + mvc
mvc = mvc.replace(
    "def geo_reconstruct(",
    "def geo_reconstruct(*args, **kwargs):\n    _yukle_vram()\n    return _orig_geo_reconstruct(*args, **kwargs)\ndef _orig_geo_reconstruct("
)
open(mv_yolu, "w").write(mvc)

for h in ["gradio_app.py", "app/gradio_3dgen.py"]:
    if os.path.exists(h):
        kh = open(h, "r").read().replace("show_share_button=True, ", "").replace(", show_share_button=True", "")
        if "share=True" not in kh: kh = kh.replace(".launch(", ".launch(share=True, ")
        kh = "import oom_patch, huggingface_hub\nhuggingface_hub.cached_download = huggingface_hub.hf_hub_download\n" + kh
        open(h, "w").write(kh)

print("🚀 3. STUDYO BASLATILIYOR...")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128 python gradio_app.py

In [ ]:
import os
os.chdir("/kaggle/working/Unique3D")

# Mevcut çalışan Gradio uygulamasını tetikleyip yeni tünel linki alıyoruz
import gradio as gr
from gradio_app import create_3d_ui # ya da arayüzü başlatan fonksiyon

# Eğer gradio_app.py doğrudan çalıştırılabiliyorsa:
os.system("python gradio_app.py")

In [ ]:
import os, sys, shutil, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(WORK_DIR)

print("🚀 [1/8] Kaggle Ortamı Hazırlanıyor (Paketler güvenli gruplar halinde kuruluyor)...")
paket_gruplari = [
    ["numpy>=2.0.0"],
    ["gradio==4.31.5", "gradio-client==0.16.4"],
    ["huggingface-hub==0.23.2", "diffusers==0.27.2", "transformers==4.39.3", "peft==0.10.0", "accelerate==0.28.0", "safetensors"],
    ["rembg", "onnxruntime", "trimesh", "einops", "omegaconf", "pydantic", "jaxtyping", "typeguard"],
    ["imageio", "scikit-image", "opencv-python-headless", "matplotlib", "plyfile", "pymeshlab"]
]

for i, grup in enumerate(paket_gruplari, 1):
    print(f"📦 Kuruluyor ({i}/{len(paket_gruplari)}): {', '.join(grup[:3])}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + grup, check=False)

print("🚀 [2/8] Proje Klonlanıyor ve Modeller İndiriliyor...")
if not os.path.exists(u3d_dir):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git", u3d_dir], check=False)
os.chdir(u3d_dir)

from huggingface_hub import snapshot_download
if not os.path.exists("ckpt/img2mvimg/unet/config.json"):
    print("⏳ Unique3D Modelleri Çekiliyor...")
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

lambda_dir = f"{u3d_dir}/lambdalabs_local"
if not os.path.exists(lambda_dir):
    print("⏳ Lambdalabs Temel Yapay Zeka Modeli Çekiliyor...")
    snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir=lambda_dir)

print("🚀 [3/8] Manuel Model Yolları Düzeltiliyor (.bin & .safetensors)...")
for hedef_klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef_yol = os.path.join(u3d_dir, hedef_klasor)
    unet_yol = os.path.join(hedef_yol, "unet")
    os.makedirs(hedef_yol, exist_ok=True)
    os.makedirs(unet_yol, exist_ok=True)
    if os.path.exists(unet_yol):
        for dosya in os.listdir(unet_yol):
            src = os.path.join(unet_yol, dosya)
            dst = os.path.join(hedef_yol, dosya)
            if os.path.isfile(src) and not os.path.exists(dst): shutil.copy2(src, dst)
    if os.path.exists(hedef_yol):
        for dosya in os.listdir(hedef_yol):
            if dosya != "unet":
                src = os.path.join(hedef_yol, dosya)
                dst = os.path.join(unet_yol, dosya)
                if os.path.isfile(src) and not os.path.exists(dst): shutil.copy2(src, dst)

print("🚀 [4/8] utils.py ve unifield_processor Yamaları Uygulanıyor...")
utils_kodu = """import os, torch\nimport onnxruntime as ort\nfrom rembg import new_session\nproviders = ["CPUExecutionProvider"]\ntry:\n    session = new_session(providers=providers)\nexcept:\n    session = None\ndef simple_preprocess(img): return img\ndef rotate_normals_torch(normals, rot_mat):\n    original_shape = normals.shape\n    normals = normals.reshape(-1, 3)\n    normals = torch.matmul(normals, rot_mat.T)\n    return normals.reshape(original_shape)\n"""
open("scripts/utils.py", "w", encoding="utf-8").write(utils_kodu)

proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(proc_file):
    c = open(proc_file, "r").read()
    c = c.replace("self.set_config(init_config)", "try:\n            self.set_config(init_config)\n        except:\n            pass")
    open(proc_file, "w").write(c)

print("🚀 [5/8] Diffusers ve Hub Monkey Patch'leri Enjekte Ediliyor...")
monkey_patch = f"""
import huggingface_hub
_old_hf_hub_download = huggingface_hub.hf_hub_download
def _new_hf_hub_download(repo_id, *args, **kwargs):
    if repo_id and "lambdalabs" in str(repo_id):
        filename = kwargs.get("filename", "")
        if filename: return r"{lambda_dir}/" + filename
        return r"{lambda_dir}/config.json"
    return _old_hf_hub_download(repo_id, *args, **kwargs)
huggingface_hub.hf_hub_download = _new_hf_hub_download

import torch, diffusers
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{{**k, 'weights_only': False}})

_old_get_model_file = diffusers.utils.hub_utils._get_model_file
def _new_get_model_file(*args, **kwargs):
    try: return _old_get_model_file(*args, **kwargs)
    except:
        if "weights_name" in kwargs and "safetensors" in kwargs["weights_name"]:
            kwargs["weights_name"] = kwargs["weights_name"].replace("safetensors", "bin")
        elif len(args) > 3 and "safetensors" in str(args[3]):
            args = list(args); args[3] = args[3].replace("safetensors", "bin"); args = tuple(args)
        return _old_get_model_file(*args, **kwargs)
diffusers.utils.hub_utils._get_model_file = _new_get_model_file
"""
app_path = "gradio_app.py"
if os.path.exists(app_path):
    acode = open(app_path, "r").read()
    if "_new_hf_hub_download" not in acode:
        open(app_path, "w").write(monkey_patch + "\n" + acode)

print("🚀 [6/8] 3D Motorları (nvdiffrast & PyTorch3D) Kuruluyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NVlabs/nvdiffrast.git"], check=False)
py_v = f"cp{sys.version_info.major}{sys.version_info.minor}"
whl_url = f"https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/torch2.3.0_cu121/pytorch3d-0.7.8-{py_v}-{py_v}-linux_x86_64.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", whl_url], check=False)

print("🚀 [7/8] Cloudflare Tüneli Hazırlanıyor...")
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin])
    os.system(f"chmod +x {cf_bin}")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=open(f"{WORK_DIR}/cf.log", "w"), stderr=subprocess.STDOUT)

print("🚀 [8/8] Sunucu Başlatılıyor! (Tüm Modeller Yükleniyor)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log = open(f"{WORK_DIR}/gradio.log", "w")
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env_vars, stdout=gradio_log, stderr=subprocess.STDOUT)

while True:
    if os.path.exists(f"{WORK_DIR}/gradio.log"):
        log_str = open(f"{WORK_DIR}/gradio.log", "r", errors="ignore").read()
        lines = log_str.strip().split('\n')
        if lines: print(f"\r⏳ Son işlem: {lines[-1][:120]:<120}", end="")
        if "Running on local URL" in log_str or "7860" in log_str:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
        if ("Traceback" in log_str or "SyntaxError" in log_str or "TypeError" in log_str or "ImportError" in log_str or "RuntimeError" in log_str or "ModuleNotFoundError" in log_str) and "FutureWarning" not in lines[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(lines[-15:]))
            break
    time.sleep(3)

time.sleep(3)
cf_log = open(f"{WORK_DIR}/cf.log", "r").read() if os.path.exists(f"{WORK_DIR}/cf.log") else ""
gr_log = open(f"{WORK_DIR}/gradio.log", "r").read()
cf_link = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_log)
gr_link = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_log)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_link: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_link[-1]}")
if gr_link: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_link[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, shutil, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(WORK_DIR)

print("🧹 [0/8] Disk Alanı Temizleniyor (Yarım kalan ağır kopyalar silinip yer açılıyor)...")
for hedef_klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef_yol = os.path.join(u3d_dir, hedef_klasor)
    if os.path.exists(hedef_yol):
        for dosya in os.listdir(hedef_yol):
            tam_yol = os.path.join(hedef_yol, dosya)
            if os.path.isfile(tam_yol) and (tam_yol.endswith(".safetensors") or tam_yol.endswith(".bin") or tam_yol.endswith(".json")):
                try: os.remove(tam_yol)
                except: pass

print("🚀 [1/8] Kaggle Ortamı Hazırlanıyor (Paketler güvenli gruplar halinde kuruluyor)...")
paket_gruplari = [
    ["numpy>=2.0.0"],
    ["gradio==4.31.5", "gradio-client==0.16.4"],
    ["huggingface-hub==0.23.2", "diffusers==0.27.2", "transformers==4.39.3", "peft==0.10.0", "accelerate==0.28.0", "safetensors"],
    ["rembg", "onnxruntime", "trimesh", "einops", "omegaconf", "pydantic", "jaxtyping", "typeguard"],
    ["imageio", "scikit-image", "opencv-python-headless", "matplotlib", "plyfile", "pymeshlab"]
]
for i, grup in enumerate(paket_gruplari, 1):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + grup, check=False)

print("🚀 [2/8] Proje Klonlanıyor ve Modeller İndiriliyor...")
if not os.path.exists(u3d_dir):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git", u3d_dir], check=False)
os.chdir(u3d_dir)

from huggingface_hub import snapshot_download
if not os.path.exists("ckpt/img2mvimg/unet/config.json"):
    print("⏳ Unique3D Modelleri Çekiliyor...")
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

lambda_dir = f"{u3d_dir}/lambdalabs_local"
if not os.path.exists(lambda_dir):
    print("⏳ Lambdalabs Temel Yapay Zeka Modeli Çekiliyor...")
    snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir=lambda_dir)

print("🚀 [3/8] Model Yolları Düzeltiliyor (Sıfır Alan Tüketen Kısayollarla)...")
for hedef_klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef_yol = os.path.join(u3d_dir, hedef_klasor)
    unet_yol = os.path.join(hedef_yol, "unet")
    os.makedirs(hedef_yol, exist_ok=True)
    os.makedirs(unet_yol, exist_ok=True)
    # Kopyalamak yerine symlink (kısayol) oluşturuyoruz -> Disk dolmuyor!
    if os.path.exists(unet_yol):
        for dosya in os.listdir(unet_yol):
            src = os.path.join(unet_yol, dosya)
            dst = os.path.join(hedef_yol, dosya)
            if os.path.isfile(src) and not os.path.exists(dst): 
                os.symlink(os.path.abspath(src), os.path.abspath(dst))
    if os.path.exists(hedef_yol):
        for dosya in os.listdir(hedef_yol):
            if dosya != "unet":
                src = os.path.join(hedef_yol, dosya)
                dst = os.path.join(unet_yol, dosya)
                if os.path.isfile(src) and not os.path.exists(dst): 
                    os.symlink(os.path.abspath(src), os.path.abspath(dst))

print("🚀 [4/8] utils.py ve unifield_processor Yamaları Uygulanıyor...")
utils_kodu = """import os, torch\nimport onnxruntime as ort\nfrom rembg import new_session\nproviders = ["CPUExecutionProvider"]\ntry:\n    session = new_session(providers=providers)\nexcept:\n    session = None\ndef simple_preprocess(img): return img\ndef rotate_normals_torch(normals, rot_mat):\n    original_shape = normals.shape\n    normals = normals.reshape(-1, 3)\n    normals = torch.matmul(normals, rot_mat.T)\n    return normals.reshape(original_shape)\n"""
open("scripts/utils.py", "w", encoding="utf-8").write(utils_kodu)

proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(proc_file):
    c = open(proc_file, "r").read()
    c = c.replace("self.set_config(init_config)", "try:\n            self.set_config(init_config)\n        except:\n            pass")
    open(proc_file, "w").write(c)

print("🚀 [5/8] Diffusers ve Hub Monkey Patch'leri Enjekte Ediliyor...")
monkey_patch = f"""
import huggingface_hub
_old_hf_hub_download = huggingface_hub.hf_hub_download
def _new_hf_hub_download(repo_id, *args, **kwargs):
    if repo_id and "lambdalabs" in str(repo_id):
        filename = kwargs.get("filename", "")
        if filename: return r"{lambda_dir}/" + filename
        return r"{lambda_dir}/config.json"
    return _old_hf_hub_download(repo_id, *args, **kwargs)
huggingface_hub.hf_hub_download = _new_hf_hub_download

import torch, diffusers
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{{**k, 'weights_only': False}})

_old_get_model_file = diffusers.utils.hub_utils._get_model_file
def _new_get_model_file(*args, **kwargs):
    try: return _old_get_model_file(*args, **kwargs)
    except:
        if "weights_name" in kwargs and "safetensors" in kwargs["weights_name"]:
            kwargs["weights_name"] = kwargs["weights_name"].replace("safetensors", "bin")
        elif len(args) > 3 and "safetensors" in str(args[3]):
            args = list(args); args[3] = args[3].replace("safetensors", "bin"); args = tuple(args)
        return _old_get_model_file(*args, **kwargs)
diffusers.utils.hub_utils._get_model_file = _new_get_model_file
"""
app_path = "gradio_app.py"
if os.path.exists(app_path):
    acode = open(app_path, "r").read()
    if "_new_hf_hub_download" not in acode:
        open(app_path, "w").write(monkey_patch + "\n" + acode)

print("🚀 [6/8] 3D Motorları (nvdiffrast & PyTorch3D) Kuruluyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NVlabs/nvdiffrast.git"], check=False)
py_v = f"cp{sys.version_info.major}{sys.version_info.minor}"
whl_url = f"https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/torch2.3.0_cu121/pytorch3d-0.7.8-{py_v}-{py_v}-linux_x86_64.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", whl_url], check=False)

print("🚀 [7/8] Cloudflare Tüneli Hazırlanıyor...")
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin])
    os.system(f"chmod +x {cf_bin}")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=open(f"{WORK_DIR}/cf.log", "w"), stderr=subprocess.STDOUT)

print("🚀 [8/8] Sunucu Başlatılıyor! (Tüm Modeller Yükleniyor)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log = open(f"{WORK_DIR}/gradio.log", "w")
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env_vars, stdout=gradio_log, stderr=subprocess.STDOUT)

while True:
    if os.path.exists(f"{WORK_DIR}/gradio.log"):
        log_str = open(f"{WORK_DIR}/gradio.log", "r", errors="ignore").read()
        lines = log_str.strip().split('\n')
        if lines: print(f"\r⏳ Son işlem: {lines[-1][:120]:<120}", end="")
        if "Running on local URL" in log_str or "7860" in log_str:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
        if ("Traceback" in log_str or "SyntaxError" in log_str or "TypeError" in log_str or "ImportError" in log_str or "RuntimeError" in log_str or "ModuleNotFoundError" in log_str) and "FutureWarning" not in lines[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(lines[-15:]))
            break
    time.sleep(3)

time.sleep(3)
cf_log = open(f"{WORK_DIR}/cf.log", "r").read() if os.path.exists(f"{WORK_DIR}/cf.log") else ""
gr_log = open(f"{WORK_DIR}/gradio.log", "r").read()
cf_link = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_log)
gr_link = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_log)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_link: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_link[-1]}")
if gr_link: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_link[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 Eksik 'fire' kütüphanesi yükleniyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fire"], check=True)

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Son eksik de tamamlandı!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 PyTorch3D kaynak koddan Kaggle donanımına özel derleniyor...")
print("⏳ DİKKAT: Bu işlem 3-5 dakika sürebilir. Lütfen tamamlanmasını bekleyin...")

# Derleme hızlandırıcıları ve temel gereksinimler
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja", "setuptools", "iopath", "fvcore"])

# PyTorch3D'nin en güncel halini GitHub'dan çekip derliyoruz
subprocess.run([sys.executable, "-m", "pip", "install", "git+https://github.com/facebookresearch/pytorch3d.git"], check=True)

print("✅ PyTorch3D başarıyla derlendi ve kuruldu!")

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Sonunda arayüze kavuşuyoruz!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: Dosya yönlendirme (Hub) yaması alt klasörleri (subfolder) tanıyacak şekilde güncelleniyor...")

app_path = "gradio_app.py"
if os.path.exists(app_path):
    code = open(app_path, "r", encoding="utf-8").read()
    
    # Eski eksik yönlendirme yamalarını tamamen temizliyoruz
    eski_yama_deseni = r'import huggingface_hub\n_old_hf_hub_download = huggingface_hub\.hf_hub_download\ndef _new_hf_hub_download\(repo_id, \*args, \*\*kwargs\):.*?huggingface_hub\.hf_hub_download = _new_hf_hub_download'
    code = re.sub(eski_yama_deseni, '', code, flags=re.DOTALL)
    
    # Alt klasör (subfolder) ve dosya adlarını (filename) doğru birleştiren KUSURSUZ YAMA
    kusursuz_yama = """
import huggingface_hub, os
_old_hf_hub_download = huggingface_hub.hf_hub_download

def _new_hf_hub_download(repo_id, *args, **kwargs):
    if repo_id and "lambdalabs" in str(repo_id):
        # Dosya adını args (pozisyonel) veya kwargs (isimli) argümanlardan yakala
        filename = kwargs.get("filename")
        if not filename and len(args) > 0:
            filename = args[0]
            
        subfolder = kwargs.get("subfolder")
        base = "/kaggle/working/Unique3D/lambdalabs_local"
        
        # Yolu doğru şekilde birleştir
        if subfolder and filename:
            return os.path.join(base, subfolder, filename)
        elif filename:
            return os.path.join(base, filename)
        return os.path.join(base, "config.json")
        
    return _old_hf_hub_download(repo_id, *args, **kwargs)

huggingface_hub.hf_hub_download = _new_hf_hub_download
"""
    
    code = kusursuz_yama + "\n" + code.strip()
    open(app_path, "w", encoding="utf-8").write(code)
    print("✅ Alt klasör (subfolder) desteği eklendi. Artık VAE doğru config dosyasını okuyacak!")

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Arayüz artık ekranda olacak!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, urllib.request

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: Karmaşık yamalar temizleniyor, ayarlar doğrudan YAML ve Python içinden değiştiriliyor...")

# 1. gradio_app.py içindeki bozuk/agresif yamalardan kurtulmak için orijinal dosyayı geri çekiyoruz
try:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/AiuniAI/Unique3D/main/gradio_app.py", "gradio_app.py")
except:
    pass # İnternet anlık koparsa mevcut kalsın ama yamaları silelim
    
if os.path.exists("gradio_app.py"):
    c = open("gradio_app.py", "r", encoding="utf-8").read()
    c = re.sub(r'import huggingface_hub.*?(?=import os|import sys|import gradio|import argparse)', '', c, flags=re.DOTALL)
    open("gradio_app.py", "w", encoding="utf-8").write(c.strip())

# 2. Tüm dosyalarda derin arama ve değiştirme (PY, YAML, JSON). Hatanın asıl kaynağı burasıydı!
lambda_dir = f"{u3d_dir}/lambdalabs_local"
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith((".py", ".yaml", ".yml", ".json")):
            p = os.path.join(root, file)
            try:
                with open(p, "r", encoding="utf-8") as f:
                    c = f.read()
                
                changed = False
                # İnternet indirmesini tamamen engelle ve yerel klasöre yönlendir
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", lambda_dir)
                    changed = True
                
                # Model okuma hatalarını önlemek için safetensors zorunluluğunu kaldır
                if "use_safetensors=True" in c:
                    c = c.replace("use_safetensors=True", "use_safetensors=False")
                    changed = True

                if changed:
                    with open(p, "w", encoding="utf-8") as f:
                        f.write(c)
            except Exception:
                pass

# 3. Diffusers kütüphanesi sızlanmasın diye .bin dosyalarına sahte bir .safetensors kısayolu ekliyoruz
for root, _, files in os.walk(f"{u3d_dir}/ckpt"):
    for file in files:
        if file.endswith(".bin"):
            bin_path = os.path.join(root, file)
            safe_path = bin_path.replace(".bin", ".safetensors")
            if not os.path.exists(safe_path):
                try: os.symlink(bin_path, safe_path)
                except: pass

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (VAE yapılandırması artık kusursuz çalışacak!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: Kütüphane sürümleri Unique3D'nin orijinal yapısına (Diffusers 0.24.0) uyarlanıyor...")

# Sürüm düşürme işlemi (Sadece birkaç saniye sürer)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.24.0", "transformers==4.36.2"], check=True)

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Sürüm uyumsuzluğu kalıcı olarak çözüldü!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: Kütüphaneler orijinal modern sürümlerine geri alınıyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.27.2", "transformers==4.39.3"], check=True)

print("🧹 Eski agresif yamalar temizleniyor ve saf klasör yönlendirmesi yapılıyor...")
# 1. gradio_app.py içindeki hub yamalarını temizle
app_path = "gradio_app.py"
if os.path.exists(app_path):
    c = open(app_path, "r", encoding="utf-8").read()
    c = re.sub(r'import huggingface_hub.*?(?=import os|import sys|import gradio|import argparse)', '', c, flags=re.DOTALL)
    open(app_path, "w", encoding="utf-8").write(c.strip())

# 2. Kod içindeki tüm "lambdalabs" metinlerini doğrudan yerel klasörün tam yoluyla değiştir (En temiz yöntem)
lambda_dir = f"{u3d_dir}/lambdalabs_local"
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            p = os.path.join(root, file)
            try:
                with open(p, "r", encoding="utf-8") as f:
                    content = f.read()
                if "lambdalabs/sd-image-variations-diffusers" in content:
                    content = content.replace("lambdalabs/sd-image-variations-diffusers", lambda_dir)
                    with open(p, "w", encoding="utf-8") as f:
                        f.write(content)
            except:
                pass

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Klasör yapısı hatası çözüldü!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 KESİN ÇÖZÜM: VAE ve UNet yapılandırmaları zorla düzeltiliyor ve Diffusers kısıtlaması kaldırılıyor...")

# 1. Diffusers kütüphanesi içindeki hata fırlatan kısıtlamayı kalıcı olarak yamalıyoruz
patch_code = """
import diffusers.models.unets.unet_2d_blocks
_orig_get_down_block = diffusers.models.unets.unet_2d_blocks.get_down_block

def _safe_get_down_block(down_block_type, *args, **kwargs):
    # Eğer sistem CrossAttention istiyor ama boyut belirtilmemişse, otomatik 768 atayarak çökmesini engelle
    if "CrossAttn" in down_block_type and kwargs.get("cross_attention_dim") is None:
        kwargs["cross_attention_dim"] = 768
    return _orig_get_down_block(down_block_type, *args, **kwargs)

diffusers.models.unets.unet_2d_blocks.get_down_block = _safe_get_down_block
"""
app_path = "gradio_app.py"
if os.path.exists(app_path):
    code = open(app_path, "r", encoding="utf-8").read()
    if "_safe_get_down_block" not in code:
        code = patch_code + "\n" + code
        open(app_path, "w", encoding="utf-8").write(code)

# 2. JSON Config dosyalarını tarayıp eksik 'cross_attention_dim' değerlerini zorla ekliyoruz
duzeltilen_dosya_sayisi = 0
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file == "config.json":
            p = os.path.join(root, file)
            try:
                with open(p, "r", encoding="utf-8") as f:
                    data = json.load(f)
                
                changed = False
                
                # A) VAE Config Düzeltmesi (Yanlışlıkla UNet katmanları atanmışsa düzelt)
                if "AutoencoderKL" in data.get("_class_name", "") or "vae" in root.lower():
                    if "down_block_types" in data and any("CrossAttn" in b for b in data["down_block_types"]):
                        data["down_block_types"] = ["DownEncoderBlock2D", "DownEncoderBlock2D", "DownEncoderBlock2D", "DownEncoderBlock2D"]
                        data["up_block_types"] = ["UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D"]
                        changed = True
                        
                # B) UNet Config Düzeltmesi (Eksik boyutu ekle)
                if "UNet" in data.get("_class_name", "") or ("down_block_types" in data and any("CrossAttn" in b for b in data["down_block_types"])):
                    if "cross_attention_dim" not in data or data["cross_attention_dim"] is None:
                        data["cross_attention_dim"] = 768
                        changed = True
                        
                if changed:
                    with open(p, "w", encoding="utf-8") as f:
                        json.dump(data, f, indent=4)
                    duzeltilen_dosya_sayisi += 1
            except:
                pass

print(f"✅ {duzeltilen_dosya_sayisi} adet hatalı ayar dosyası onarıldı!")

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (Arayüz artık KESİN olarak ekranda olacak!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, urllib.request

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔥 NİHAİ ÇÖZÜM: Zararlı yamalar siliniyor ve sistem orijinal haline döndürülüyor...")

# 1. Yamalarla bozulan gradio_app.py dosyasını tamamen silip orijinalini temizce indiriyoruz
app_path = "gradio_app.py"
if os.path.exists(app_path):
    os.remove(app_path)
urllib.request.urlretrieve("https://raw.githubusercontent.com/AiuniAI/Unique3D/main/gradio_app.py", app_path)

# 2. Sadece zorunlu olan ve güvenli 'ağırlık uzantısı' yamasını ekliyoruz
guvenli_yamalar = """
import torch, diffusers
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})

_old_get_model_file = diffusers.utils.hub_utils._get_model_file
def _new_get_model_file(*args, **kwargs):
    try: return _old_get_model_file(*args, **kwargs)
    except:
        if "weights_name" in kwargs and "safetensors" in kwargs["weights_name"]:
            kwargs["weights_name"] = kwargs["weights_name"].replace("safetensors", "bin")
        elif len(args) > 3 and "safetensors" in str(args[3]):
            args = list(args); args[3] = args[3].replace("safetensors", "bin"); args = tuple(args)
        return _old_get_model_file(*args, **kwargs)
diffusers.utils.hub_utils._get_model_file = _new_get_model_file
"""

# 3. Modelin internete çıkmasını metin değiştirme ile (Monkey Patch OLMADAN) engelliyoruz
lambda_dir = f"{u3d_dir}/lambdalabs_local"
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            p = os.path.join(root, file)
            try:
                with open(p, "r", encoding="utf-8") as f:
                    c = f.read()
                
                degisti = False
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", lambda_dir)
                    degisti = True
                
                if degisti:
                    with open(p, "w", encoding="utf-8") as f:
                        f.write(c)
            except:
                pass

# Orijinal gradio_app.py'nin başına güvenli yamamızı ekliyoruz
c = open(app_path, "r", encoding="utf-8").read()
open(app_path, "w", encoding="utf-8").write(guvenli_yamalar + "\n" + c)

# 4. Sistemi çökerten sahte ".safetensors" kısayollarını siliyoruz
print("🧹 Çökmeye sebep olan sahte dosya uzantıları temizleniyor...")
for root, _, files in os.walk(f"{u3d_dir}/ckpt"):
    for file in files:
        if file.endswith(".safetensors"):
            p = os.path.join(root, file)
            if os.path.islink(p):
                try: os.remove(p)
                except: pass

print("🧹 Takılı kalan eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu yeniden başlatılıyor (VAE ve UNet artık kendi doğru dosyalarını okuyacak!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, shutil, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(WORK_DIR)

print("🚀 [1/9] Kaggle Ortamı Hazırlanıyor (Paketler güvenli gruplarla kuruluyor)...")
paket_gruplari = [
    ["numpy>=2.0.0"],
    ["gradio==4.31.5", "gradio-client==0.16.4"],
    ["huggingface-hub==0.23.2", "diffusers==0.27.2", "transformers==4.39.3", "peft==0.10.0", "accelerate==0.28.0", "safetensors"],
    ["rembg", "onnxruntime", "trimesh", "einops", "omegaconf", "pydantic", "jaxtyping", "typeguard", "fire"],
    ["imageio", "scikit-image", "opencv-python-headless", "matplotlib", "plyfile", "pymeshlab", "ninja", "setuptools", "iopath", "fvcore"]
]
for grup in paket_gruplari:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + grup, check=False)

print("🚀 [2/9] Proje Klonlanıyor ve Modeller İndiriliyor...")
if not os.path.exists(u3d_dir):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git", u3d_dir], check=False)
os.chdir(u3d_dir)

from huggingface_hub import snapshot_download
if not os.path.exists("ckpt/img2mvimg/unet/config.json"):
    print("⏳ Unique3D Modelleri Çekiliyor...")
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

lambda_dir = f"{u3d_dir}/lambdalabs_local"
if not os.path.exists(os.path.join(lambda_dir, "config.json")):
    print("⏳ Lambdalabs Temel Yapay Zeka Modeli Çekiliyor...")
    snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir=lambda_dir)

print("🚀 [3/9] Model Yolları ve Disk Optimizasyonu (Sıfır Alan Tüketen Symlink)...")
for hedef_klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef_yol = os.path.join(u3d_dir, hedef_klasor)
    unet_yol = os.path.join(hedef_yol, "unet")
    os.makedirs(unet_yol, exist_ok=True)

    if os.path.exists(unet_yol):
        for dosya in os.listdir(unet_yol):
            src = os.path.join(unet_yol, dosya)
            dst = os.path.join(hedef_yol, dosya)
            if os.path.isfile(src) and not os.path.exists(dst): os.symlink(os.path.abspath(src), os.path.abspath(dst))
    if os.path.exists(hedef_yol):
        for dosya in os.listdir(hedef_yol):
            if dosya != "unet":
                src = os.path.join(hedef_yol, dosya)
                dst = os.path.join(unet_yol, dosya)
                if os.path.isfile(src) and not os.path.exists(dst): os.symlink(os.path.abspath(src), os.path.abspath(dst))

# Diffusers çökmelerini engellemek için .bin dosyalarına sahte .safetensors kısayolu
for root, _, files in os.walk(f"{u3d_dir}/ckpt"):
    for file in files:
        if file.endswith(".bin"):
            bin_path = os.path.join(root, file)
            safe_path = bin_path.replace(".bin", ".safetensors")
            if not os.path.exists(safe_path):
                try: os.symlink(os.path.abspath(bin_path), os.path.abspath(safe_path))
                except: pass

print("🚀 [4/9] Güvenli Kod Yamaları Uygulanıyor (Karmaşık Hub yamaları İPTAL)...")
utils_kodu = """import os, torch\nimport onnxruntime as ort\nfrom rembg import new_session\nproviders = ["CPUExecutionProvider"]\ntry:\n    session = new_session(providers=providers)\nexcept:\n    session = None\ndef simple_preprocess(img): return img\ndef rotate_normals_torch(normals, rot_mat):\n    original_shape = normals.shape\n    normals = normals.reshape(-1, 3)\n    normals = torch.matmul(normals, rot_mat.T)\n    return normals.reshape(original_shape)\n"""
open("scripts/utils.py", "w", encoding="utf-8").write(utils_kodu)

proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(proc_file):
    c = open(proc_file, "r").read()
    c = c.replace("self.set_config(init_config)", "try:\n            self.set_config(init_config)\n        except:\n            pass")
    open(proc_file, "w").write(c)

# Modellerin internete çıkmasını salt metin değişikliğiyle (monkey patch olmadan) engelliyoruz
for root, _, files in os.walk(u3d_dir):
    for file in files:
        if file.endswith(".py"):
            p = os.path.join(root, file)
            try:
                with open(p, "r", encoding="utf-8") as f: c = f.read()
                if "lambdalabs/sd-image-variations-diffusers" in c:
                    c = c.replace("lambdalabs/sd-image-variations-diffusers", lambda_dir)
                    with open(p, "w", encoding="utf-8") as f: f.write(c)
            except: pass

# Sadece PyTorch 2.3+ için zorunlu ağırlık (weights_only) yamasını ekliyoruz
app_path = "gradio_app.py"
if os.path.exists(app_path):
    c = open(app_path, "r", encoding="utf-8").read()
    patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
    if "_old_load" not in c:
        open(app_path, "w", encoding="utf-8").write(patch + c)

print("🚀 [5/9] Nvdiffrast Motoru Kuruluyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NVlabs/nvdiffrast.git"], check=False)

print("🚀 [6/9] PyTorch3D Derleniyor (⚠️ DİKKAT: 3-5 Dakika Sürebilir)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/pytorch3d.git"], check=False)

print("🚀 [7/9] Cloudflare Tüneli Hazırlanıyor...")
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin])
    os.system(f"chmod +x {cf_bin}")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=open(f"{WORK_DIR}/cf.log", "w"), stderr=subprocess.STDOUT)

print("🚀 [8/9] Sunucu Başlatılıyor! (Tüm Modeller Yükleniyor)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log = open(f"{WORK_DIR}/gradio.log", "w")
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env_vars, stdout=gradio_log, stderr=subprocess.STDOUT)

while True:
    if os.path.exists(f"{WORK_DIR}/gradio.log"):
        log_str = open(f"{WORK_DIR}/gradio.log", "r", errors="ignore").read()
        lines = log_str.strip().split('\n')
        if lines: print(f"\r⏳ Son işlem: {lines[-1][:120]:<120}", end="")
        if "Running on local URL" in log_str or "7860" in log_str:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
        if ("Traceback" in log_str or "SyntaxError" in log_str or "TypeError" in log_str or "ImportError" in log_str or "RuntimeError" in log_str or "ModuleNotFoundError" in log_str) and "FutureWarning" not in lines[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(lines[-15:]))
            break
    time.sleep(3)

time.sleep(3)
cf_log = open(f"{WORK_DIR}/cf.log", "r").read() if os.path.exists(f"{WORK_DIR}/cf.log") else ""
gr_log = open(f"{WORK_DIR}/gradio.log", "r").read()
cf_link = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_log)
gr_link = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_log)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_link: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_link[-1]}")
if gr_link: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_link[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🛡️ NİHAİ KORUMA KALKANI AKTİF EDİLİYOR...")
patch_code = """
# --- NIHAI KORUMA KALKANI ---
from diffusers.models.autoencoders.vae import AutoencoderKL
from diffusers.models.unets.unet_2d_condition import UNet2DConditionModel

# 1. VAE Koruması (Yanlış katman istenirse zorla standart VAE katmanına çevirir)
_orig_vae_init = AutoencoderKL.__init__
def _safe_vae_init(self, *args, **kwargs):
    if "down_block_types" in kwargs:
        kwargs["down_block_types"] = tuple("DownEncoderBlock2D" if "CrossAttn" in b else b for b in kwargs["down_block_types"])
    if "up_block_types" in kwargs:
        kwargs["up_block_types"] = tuple("UpDecoderBlock2D" if "CrossAttn" in b else b for b in kwargs["up_block_types"])
    return _orig_vae_init(self, *args, **kwargs)
AutoencoderKL.__init__ = _safe_vae_init

# 2. UNet Koruması (Eksik boyut parametresini otomatik 768 olarak tamamlar)
_orig_unet_init = UNet2DConditionModel.__init__
def _safe_unet_init(self, *args, **kwargs):
    if kwargs.get("cross_attention_dim") is None:
        kwargs["cross_attention_dim"] = 768
    return _orig_unet_init(self, *args, **kwargs)
UNet2DConditionModel.__init__ = _safe_unet_init
# -----------------------------
"""

app_path = "gradio_app.py"
with open(app_path, "r", encoding="utf-8") as f:
    code = f.read()

if "NIHAI KORUMA KALKANI" not in code:
    with open(app_path, "w", encoding="utf-8") as f:
        f.write(patch_code + "\n" + code)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Koruma kalkanı devrede, arayüz KESİN açılacak!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

print("🔍 Loglar izleniyor...\n")
while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 İçe aktarma hatası (ImportError) gideriliyor...")
app_path = "gradio_app.py"

with open(app_path, "r", encoding="utf-8") as f:
    code = f.read()

# Benim yazdığım hatalı yamayı temizliyoruz
code = re.sub(r'# --- NIHAI KORUMA KALKANI ---.*?# -----------------------------', '', code, flags=re.DOTALL)

# Kütüphane yollarının (import) doğru olduğu kusursuz yamayı ekliyoruz
patch_code = """
# --- NIHAI KORUMA KALKANI ---
from diffusers import AutoencoderKL, UNet2DConditionModel

_orig_vae_init = AutoencoderKL.__init__
def _safe_vae_init(self, *args, **kwargs):
    if "down_block_types" in kwargs:
        kwargs["down_block_types"] = tuple("DownEncoderBlock2D" if "CrossAttn" in b else b for b in kwargs["down_block_types"])
    if "up_block_types" in kwargs:
        kwargs["up_block_types"] = tuple("UpDecoderBlock2D" if "CrossAttn" in b else b for b in kwargs["up_block_types"])
    return _orig_vae_init(self, *args, **kwargs)
AutoencoderKL.__init__ = _safe_vae_init

_orig_unet_init = UNet2DConditionModel.__init__
def _safe_unet_init(self, *args, **kwargs):
    if kwargs.get("cross_attention_dim") is None:
        kwargs["cross_attention_dim"] = 768
    return _orig_unet_init(self, *args, **kwargs)
UNet2DConditionModel.__init__ = _safe_unet_init
# -----------------------------
"""

with open(app_path, "w", encoding="utf-8") as f:
    f.write(patch_code.strip() + "\n\n" + code.strip())

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Hata tamamen çözüldü!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🔧 Xformers eklentisi devre dışı bırakılıyor (PyTorch'un kendi yerleşik hızlandırıcısı kullanılacak)...")
target_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"

if os.path.exists(target_file):
    with open(target_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # xformers zorunluluğunu pas geçiyoruz
    content = content.replace(
        "module.set_use_memory_efficient_attention_xformers(True)",
        "pass  # xformers iptal edildi, PyTorch SDPA devrede"
    )
    
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Tüm engeller aşıldı!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🗑️ Bozulan ağırlık dosyaları temizleniyor...")
# Sorunlu klasörleri tamamen siliyoruz
for klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef = os.path.join(u3d_dir, klasor)
    if os.path.exists(hedef):
        import shutil
        shutil.rmtree(hedef)
        
print("⏳ Modeller en saf haliyle (yamasız) tekrar indiriliyor...")
from huggingface_hub import snapshot_download
# Modelleri symlink (kısayol) OLMADAN, tamamen fiziksel dosyalar olarak indiriyoruz
snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

# Kodu safetensors dosyalarını okumaya zorluyoruz
app_path = "gradio_app.py"
with open(app_path, "r", encoding="utf-8") as f:
    code = f.read()

# Daha önce yazdığım kalkan vb. varsa kalsın, sadece weights_only yamasını ekle/güncelle
patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
if "weights_only" not in code:
    with open(app_path, "w", encoding="utf-8") as f:
        f.write(patch + "\n" + code)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Fiziksel model dosyaları yenilendi!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, shutil

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🗑️ Klasörler temizleniyor...")
for klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef = os.path.join(u3d_dir, klasor)
    if os.path.exists(hedef):
        shutil.rmtree(hedef)
        
print("⏳ Modeller sırayla (çakışma olmadan) güvenle indiriliyor...")
from huggingface_hub import snapshot_download

# max_workers=1 ekleyerek HuggingFace'in dosya taşıma sırasındaki çökme hatasını kalıcı olarak engelliyoruz
snapshot_download(
    repo_id="Wuvin/Unique3D", 
    repo_type="space", 
    allow_patterns=["ckpt/*"], 
    local_dir=".", 
    local_dir_use_symlinks=False,
    max_workers=1
)

app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()
    
    patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
    if "weights_only" not in code:
        with open(app_path, "w", encoding="utf-8") as f:
            f.write(patch + "\n" + code)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Fiziksel model dosyaları yenilendi!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, shutil

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🗑️ Bozuk indirme önbelleği (416 Hatası) temizleniyor...")
# Gizli HuggingFace önbelleğini ve bozuk geçici dosyaları siliyoruz
hf_cache_dir = os.path.join(u3d_dir, ".huggingface")
if os.path.exists(hf_cache_dir):
    shutil.rmtree(hf_cache_dir)

for root, dirs, files in os.walk(u3d_dir):
    for f in files:
        if f.endswith(".incomplete") or f.startswith("tmp_"):
            try: os.remove(os.path.join(root, f))
            except: pass

print("⏳ Modeller sıfırdan ve zorunlu olarak tekrar indiriliyor...")
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Wuvin/Unique3D", 
    repo_type="space", 
    allow_patterns=["ckpt/*"], 
    local_dir=".", 
    local_dir_use_symlinks=False,
    max_workers=1,
    force_download=True  # 416 hatasını engellemek için zorunlu yeniden indirme
)

app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()
    
    patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
    if "weights_only" not in code:
        with open(app_path, "w", encoding="utf-8") as f:
            f.write(patch + "\n" + code)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, shutil

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🛠️ Hugging Face kütüphane hatası (bug) atlatılıyor...")
for klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef = os.path.join(u3d_dir, klasor)
    if os.path.exists(hedef):
        shutil.rmtree(hedef)

print("⏳ Modeller sorunsuz varsayılan havuza indiriliyor...")
from huggingface_hub import snapshot_download

# local_dir paramatresi SİLİNDİ (Çökme sebebi buydu)
cache_path = snapshot_download(
    repo_id="Wuvin/Unique3D", 
    repo_type="space", 
    allow_patterns=["ckpt/*"]
)

print("🚚 Dosyalar Unique3D klasörüne güvenle kopyalanıyor...")
source_ckpt = os.path.join(cache_path, "ckpt")
dest_ckpt = os.path.join(u3d_dir, "ckpt")
os.makedirs(dest_ckpt, exist_ok=True)
shutil.copytree(source_ckpt, dest_ckpt, dirs_exist_ok=True)
print("✅ Kopyalama işlemi kusursuz tamamlandı!")

app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()
    
    patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
    if "weights_only" not in code:
        with open(app_path, "w", encoding="utf-8") as f:
            f.write(patch + "\n" + code)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Tüm modeller fiziksel olarak yerli yerinde!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, shutil

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🛠️ Hugging Face kütüphane hatası (bug) atlatılıyor...")
for klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef = os.path.join(u3d_dir, klasor)
    if os.path.exists(hedef):
        shutil.rmtree(hedef)

print("⏳ Modeller sorunsuz varsayılan havuza indiriliyor...")
from huggingface_hub import snapshot_download

# local_dir paramatresi SİLİNDİ (Çökme sebebi buydu)
cache_path = snapshot_download(
    repo_id="Wuvin/Unique3D", 
    repo_type="space", 
    allow_patterns=["ckpt/*"]
)

print("🚚 Dosyalar Unique3D klasörüne güvenle kopyalanıyor...")
source_ckpt = os.path.join(cache_path, "ckpt")
dest_ckpt = os.path.join(u3d_dir, "ckpt")
os.makedirs(dest_ckpt, exist_ok=True)
shutil.copytree(source_ckpt, dest_ckpt, dirs_exist_ok=True)
print("✅ Kopyalama işlemi kusursuz tamamlandı!")

app_path = "gradio_app.py"
if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()
    
    patch = "import torch\n_old_load = torch.load\ntorch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})\n"
    if "weights_only" not in code:
        with open(app_path, "w", encoding="utf-8") as f:
            f.write(patch + "\n" + code)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Tüm modeller fiziksel olarak yerli yerinde!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🧠 Gizli Hata Yutan Yama Temizleniyor (Sonsuz Döngü Kırılıyor)...")
proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"

if os.path.exists(proc_file):
    with open(proc_file, "r", encoding="utf-8") as f:
        c = f.read()
    
    # 1. Eski hatayı yutan yamayı tamamen siliyoruz (Modelin UNet'i başarıyla başlatabilmesi için)
    c = re.sub(r'try:\n\s+self\.set_config\(init_config\)\n\s+except:\n\s+pass', 'self.set_config(init_config)', c)
    c = c.replace("try:\n            self.set_config(init_config)\n        except:\n            pass", "self.set_config(init_config)")
    
    # 2. Sonsuz döngü (RecursionError) kilidini güvenli hale getiriyoruz
    if "if name == 'unet':" not in c:
        c = c.replace(
            "def __getattr__(self, name):\n        return getattr(self.unet, name)", 
            "def __getattr__(self, name):\n        if name == 'unet': raise AttributeError('UNet baslatilamadi')\n        return getattr(self.unet, name)"
        )
    
    # 3. Belleği yoran eski xformers eklenti çağrılarını kalıcı olarak etkisiz bırakıyoruz
    c = c.replace("self.unet.enable_xformers_memory_efficient_attention", "pass #")
    
    with open(proc_file, "w", encoding="utf-8") as f:
        f.write(c)

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (Sonsuz döngü kırıldı!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re, json

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(u3d_dir)

print("🛡️ Eksik config.json dosyaları fiziksel olarak oluşturuluyor...")

# Lambdalabs modelinin arandığı yerel yola eksik olan config.json dosyalarını ekliyoruz
lambda_dir = f"{u3d_dir}/lambdalabs_local"
os.makedirs(lambda_dir, exist_ok=True)

# Diffusers'ın aradığı ana config yapısı
base_config = {
    "_class_name": "StableDiffusionPipeline",
    "architectures": ["StableDiffusionPipeline"],
    "subfolder": "",
    "vae": ["diffusers", "AutoencoderKL"],
    "unet": ["diffusers", "UNet2DConditionModel"],
    "image_encoder": ["transformers", "CLIPVisionModelWithProjection"]
}

for sub in ["vae", "unet", "image_encoder", "scheduler"]:
    sub_path = os.path.join(lambda_dir, sub)
    os.makedirs(sub_path, exist_ok=True)
    cfg_file = os.path.join(sub_path, "config.json")
    if not os.path.exists(cfg_file):
        with open(cfg_file, "w") as f:
            json.dump(base_config, f)

# Tüm hatalı/bozuk yamaları gradio_app.py dosyasından tamamen temizliyoruz
app_path = "gradio_app.py"
with open(app_path, "r", encoding="utf-8") as f:
    code = f.read()

# Eski yamaları temizle
code = re.sub(r'# --- CONFIG KORUMA YAMASI ---.*?# -----------------------------', '', code, flags=re.DOTALL)

# Sadece güvenli torch yükleme yamasını bırakıyoruz
clean_patch = """
import torch
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{**k, 'weights_only': False})
"""

with open(app_path, "w", encoding="utf-8") as f:
    f.write(clean_patch.strip() + "\n\n" + code.strip())

print("🧹 Eski işlemler temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu nihai olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = f"{WORK_DIR}/gradio.log"
cf_log_path = f"{WORK_DIR}/cf.log"
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🔧 Sonsuz döngüye sebep olan üst üste binen yamalar temizleniyor...")
app_path = "gradio_app.py"

if os.path.exists(app_path):
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()

    # Tüm eski torch.load yamalarını tamamen temizliyoruz
    code = re.sub(r'torch\.load\s*=.*?\n', '', code)
    code = re.sub(r'# --- CONFIG KORUMA YAMASI ---.*?# -----------------------------', '', code, flags=re.DOTALL)

    # Güvenli ve tek seferlik çalışacak torch.load sarmalayıcısını ekliyoruz
    safe_torch_patch = """
import torch
if not getattr(torch.load, "_is_safe_load", False):
    _orig_torch_load = torch.load
    def _safe_torch_load(*a, **k):
        k.setdefault('weights_only', False)
        return _orig_torch_load(*a, **k)
    _safe_torch_load._is_safe_load = True
    torch.load = _safe_torch_load
"""

    with open(app_path, "w", encoding="utf-8") as f:
        f.write(safe_torch_patch.strip() + "\n\n" + code.strip())

print("🧹 Eski arka plan süreçleri temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu hatasız olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", gr_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
!pip install "numpy<2.0.0" "cupy-cuda12x" --force-reinstall

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧹 Eski arka plan süreçleri temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu hatasız olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", gr_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧹 Eski arka plan süreçleri temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor (JAX pasifize edildi, PyTorch çakışmasız çalışacak)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- JAX ÇAKIŞMASINI ENGELLEYEN KRİTİK ORTAM DEĞİŞKENLERİ ---
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["USE_TF"] = "0"
# -------------------------------------------------------------

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", gr_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🔧 unifield_processor.py dosyasındaki özyineleme kilitlenmesi çözülüyor...")
proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"

if os.path.exists(proc_file):
    with open(proc_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Kendi kendini çağıran hatalı getattr ve xformers satırlarını güvenli yapıya çekiyoruz
    safe_getattr = """    def __getattr__(self, name):
        if name == 'unet' or not hasattr(self, 'unet') or self.unet is None:
            raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")
        return getattr(self.unet, name)"""
    
    # __getattr__ metodunu güvenli olanla değiştir
    content = re.sub(r'def __getattr__\(self, name\):.*?\n\s+return getattr\(self\.unet, name\)', safe_getattr, content, flags=re.DOTALL)
    
    # enable_xformers_memory_efficient_attention metodunu güvenli hale getir
    safe_xformers = """    def enable_xformers_memory_efficient_attention(self, attention_op=None):
        try:
            if hasattr(self, 'unet') and self.unet is not None:
                self.unet.enable_xformers_memory_efficient_attention(attention_op)
        except Exception:
            pass"""
            
    content = re.sub(r'def enable_xformers_memory_efficient_attention\(self.*?\):.*?(?=\n    def |\Z)', safe_xformers + "\n\n", content, flags=re.DOTALL)

    with open(proc_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("✅ Dosya tamiratı tamamlandı!")

print("🧹 Eski arka plan süreçleri temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["USE_TF"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", gr_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🔧 unifield_processor.py dosyasındaki özyineleme kilitlenmesi çözülüyor...")
proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"

if os.path.exists(proc_file):
    with open(proc_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Kendi kendini çağıran hatalı getattr ve xformers satırlarını güvenli yapıya çekiyoruz
    safe_getattr = """    def __getattr__(self, name):
        if name == 'unet' or not hasattr(self, 'unet') or self.unet is None:
            raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")
        return getattr(self.unet, name)"""
    
    # __getattr__ metodunu güvenli olanla değiştir
    content = re.sub(r'def __getattr__\(self, name\):.*?\n\s+return getattr\(self\.unet, name\)', safe_getattr, content, flags=re.DOTALL)
    
    # enable_xformers_memory_efficient_attention metodunu güvenli hale getir
    safe_xformers = """    def enable_xformers_memory_efficient_attention(self, attention_op=None):
        try:
            if hasattr(self, 'unet') and self.unet is not None:
                self.unet.enable_xformers_memory_efficient_attention(attention_op)
        except Exception:
            pass"""
            
    content = re.sub(r'def enable_xformers_memory_efficient_attention\(self.*?\):.*?(?=\n    def |\Z)', safe_xformers + "\n\n", content, flags=re.DOTALL)

    with open(proc_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("✅ Dosya tamiratı tamamlandı!")

print("🧹 Eski arka plan süreçleri temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["USE_TF"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", gr_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, urllib.request

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("📥 Bozulan dosya orijinaliyle değiştiriliyor...")
proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"

# Dosyanın temiz orijinal halini indiriyoruz
url = "https://huggingface.co/spaces/Wuvin/Unique3D/raw/main/custum_3d_diffusion/custum_modules/unifield_processor.py"
urllib.request.urlretrieve(url, proc_file)

print("🔧 Dosyaya temiz ve güvenli yamalar uygulanıyor...")
with open(proc_file, "r", encoding="utf-8") as f:
    content = f.read()

# 1. Sonsuz döngü (RecursionError) yamasını basit ve hatasız ekliyoruz
eski_getattr = "def __getattr__(self, name):\n        return getattr(self.unet, name)"
yeni_getattr = "def __getattr__(self, name):\n        if name == 'unet': raise AttributeError('UNet baslatilamadi')\n        return getattr(self.unet, name)"
content = content.replace(eski_getattr, yeni_getattr)

# 2. Xformers kilitlenmesini engelliyoruz (çağrıları güvenle pasifleştiriyoruz)
content = content.replace(
    "self.unet.enable_xformers_memory_efficient_attention(attention_op)", 
    "pass # xformers iptal edildi"
)
content = content.replace(
    "module.set_use_memory_efficient_attention_xformers(True)", 
    "pass # xformers iptal edildi"
)

with open(proc_file, "w", encoding="utf-8") as f:
    f.write(content)

print("🧹 Eski arka plan süreçleri temizleniyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu nihai olarak başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["USE_TF"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
gradio_log = open(gradio_log_path, "w")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=gradio_log,
    stderr=subprocess.STDOUT
)

while True:
    if os.path.exists(gradio_log_path):
        icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in icerik or "7860" in icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in icerik or "SyntaxError" in icerik or "TypeError" in icerik or "ImportError" in icerik or "RuntimeError" in icerik or "ModuleNotFoundError" in icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break
            
    time.sleep(3)

time.sleep(3)
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()
cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", gr_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("🧩 Eksik aranan model ağırlıkları ana klasörlere kopyalanıyor...")
# Diffusers'ın körü körüne aradığı dosyaları doğrudan gözünün önüne koyuyoruz
os.system("cp ckpt/img2mvimg/unet/* ckpt/img2mvimg/ >/dev/null 2>&1")
os.system("cp ckpt/unet_4ch/unet/* ckpt/unet_4ch/ >/dev/null 2>&1")
os.system("cp ckpt/normal/unet/* ckpt/normal/ >/dev/null 2>&1")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu başlatılıyor...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"
env_vars["USE_TF"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: 
    f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=open(gradio_log_path, "w"),
    stderr=subprocess.STDOUT
)

import re
while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in log_icerik or "OSError" in log_icerik or "SyntaxError" in log_icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, json

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("📥 Dosyalar kütüphane kullanılmadan TAMAMEN MANUEL (wget) indiriliyor...")

# 1. Sistemin aradığı klasörleri zorla oluşturuyoruz
os.makedirs("ckpt/img2mvimg", exist_ok=True)
os.makedirs("ckpt/unet_4ch", exist_ok=True)
os.makedirs("ckpt/normal", exist_ok=True)

# 2. İnternetten doğrudan ana klasörün içine indiriyoruz (Hata veren yer burasıydı)
base_url = "https://huggingface.co/spaces/Wuvin/Unique3D/resolve/main/ckpt/img2mvimg/unet"

indirme_komutlari = [
    f"wget -q -O ckpt/img2mvimg/config.json {base_url}/config.json",
    f"wget -q -O ckpt/img2mvimg/diffusion_pytorch_model.bin {base_url}/diffusion_pytorch_model.bin",
    f"wget -q -O ckpt/img2mvimg/diffusion_pytorch_model.safetensors {base_url}/diffusion_pytorch_model.safetensors",
]

for cmd in indirme_komutlari:
    os.system(cmd)

# 3. İndirilen sağlam dosyaları diğer zorunlu klasörlere kopyalıyoruz
os.system("cp ckpt/img2mvimg/* ckpt/unet_4ch/ >/dev/null 2>&1")
os.system("cp ckpt/img2mvimg/* ckpt/normal/ >/dev/null 2>&1")

# 4. unet_4ch için özel ayarı elle yapıyoruz
cfg_path = "ckpt/unet_4ch/config.json"
if os.path.exists(cfg_path):
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    cfg["in_channels"] = 4
    cfg.pop("num_class_embeds", None)
    cfg["addition_embed_type"] = None
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu nihai olarak başlatılıyor (Dosyalar manuel olarak yerinde!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_vars["USE_JAX"] = "0"
env_vars["USE_FLAX"] = "0"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=open(gradio_log_path, "w"),
    stderr=subprocess.STDOUT
)

import re
while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in log_icerik or "OSError" in log_icerik or "SyntaxError" in log_icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, subprocess, time, re

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
u3d_dir = f"{WORK_DIR}/Unique3D" if os.path.exists(f"{WORK_DIR}/Unique3D") else "."
os.chdir(u3d_dir)

print("💥 Çakışmaya sebep olan gereksiz JAX kütüphanesi sistemden tamamen siliniyor...")
os.system(f"{sys.executable} -m pip uninstall -y jax jaxlib >/dev/null 2>&1")

print("🧹 Eski süreçler kapatılıyor...")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")

print("🚀 Sunucu hatasız başlatılıyor (JAX çakışması yok, modeller manuel olarak yerinde!)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log_path = os.path.join(WORK_DIR, "gradio.log")
with open(gradio_log_path, "w") as f: f.write("")

proc = subprocess.Popen(
    [sys.executable, "-u", "gradio_app.py"],
    cwd=u3d_dir,
    env=env_vars,
    stdout=open(gradio_log_path, "w"),
    stderr=subprocess.STDOUT
)

while True:
    time.sleep(3)
    if os.path.exists(gradio_log_path):
        log_icerik = open(gradio_log_path, "r", errors="ignore").read()
        satirlar = log_icerik.strip().split('\n')
        
        if len(satirlar) > 0:
            print(f"\r⏳ Son işlem: {satirlar[-1][:120]:<120}", end="")
            
        if "Running on local URL" in log_icerik or "7860" in log_icerik:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
            
        if ("Traceback" in log_icerik or "OSError" in log_icerik or "SyntaxError" in log_icerik) and "FutureWarning" not in satirlar[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(satirlar[-15:]))
            break

time.sleep(3)
cf_log_path = f"{WORK_DIR}/cf.log"
cf_icerik = open(cf_log_path, "r", errors="ignore").read() if os.path.exists(cf_log_path) else ""
gr_icerik = open(gradio_log_path, "r", errors="ignore").read()

cf_linkler = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_icerik)
gr_linkler = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_icerik)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_linkler: print(f"🌐 CLOUDFLARE LİNKİNİZ : {cf_linkler[-1]}")
if gr_linkler: print(f"🌐 GRADIO LIVE LİNKİNİZ : {gr_linkler[-1]}")
print("="*70 + "\n")

In [ ]:
import os, sys, shutil, subprocess, time, re

WORK_DIR = "/kaggle/working"
u3d_dir = f"{WORK_DIR}/Unique3D"
os.chdir(WORK_DIR)

print("🚀 [1/9] Kaggle Ortamı Hazırlanıyor (Paketler güvenli gruplar halinde kuruluyor)...")
paket_gruplari = [
    ["numpy>=2.0.0"],
    ["gradio==4.31.5", "gradio-client==0.16.4"],
    ["huggingface-hub==0.23.2", "diffusers==0.27.2", "transformers==4.39.3", "peft==0.10.0", "accelerate==0.28.0", "safetensors"],
    ["rembg", "onnxruntime", "trimesh", "einops", "omegaconf", "pydantic", "jaxtyping", "typeguard", "fire"],
    ["imageio", "scikit-image", "opencv-python-headless", "matplotlib", "plyfile", "pymeshlab", "ninja", "setuptools", "iopath", "fvcore"]
]
for i, grup in enumerate(paket_gruplari, 1):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + grup, check=False)

print("🚀 [2/9] Proje Klonlanıyor ve Modeller İndiriliyor...")
if not os.path.exists(u3d_dir):
    subprocess.run(["git", "clone", "https://github.com/AiuniAI/Unique3D.git", u3d_dir], check=False)
os.chdir(u3d_dir)

from huggingface_hub import snapshot_download
if not os.path.exists("ckpt/img2mvimg/unet/config.json"):
    print("⏳ Unique3D Modelleri Çekiliyor (Symlink ile)...")
    snapshot_download(repo_id="Wuvin/Unique3D", repo_type="space", allow_patterns=["ckpt/*"], local_dir=".", local_dir_use_symlinks=False)

lambda_dir = f"{u3d_dir}/lambdalabs_local"
if not os.path.exists(os.path.join(lambda_dir, "config.json")):
    print("⏳ Lambdalabs Temel Yapay Zeka Modeli Çekiliyor...")
    snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir=lambda_dir)

print("🚀 [3/9] Model Yolları Düzeltiliyor (Sıfır Alan Tüketen Kısayollarla)...")
for hedef_klasor in ["ckpt/img2mvimg", "ckpt/unet_4ch", "ckpt/normal"]:
    hedef_yol = os.path.join(u3d_dir, hedef_klasor)
    unet_yol = os.path.join(hedef_yol, "unet")
    os.makedirs(hedef_yol, exist_ok=True)
    os.makedirs(unet_yol, exist_ok=True)
    
    # Fiziksel dosyaları silip yer aç (eğer önceki denemeden kaldıysa)
    for k in [hedef_yol, unet_yol]:
        for d in os.listdir(k):
            tam = os.path.join(k, d)
            if os.path.isfile(tam) and not os.path.islink(tam) and (tam.endswith(".safetensors") or tam.endswith(".bin")):
                try: os.remove(tam)
                except: pass

    if os.path.exists(unet_yol):
        for dosya in os.listdir(unet_yol):
            src = os.path.join(unet_yol, dosya)
            dst = os.path.join(hedef_yol, dosya)
            if os.path.isfile(src) and not os.path.exists(dst): os.symlink(os.path.abspath(src), os.path.abspath(dst))
    if os.path.exists(hedef_yol):
        for dosya in os.listdir(hedef_yol):
            if dosya != "unet":
                src = os.path.join(hedef_yol, dosya)
                dst = os.path.join(unet_yol, dosya)
                if os.path.isfile(src) and not os.path.exists(dst): os.symlink(os.path.abspath(src), os.path.abspath(dst))

print("🚀 [4/9] utils.py ve unifield_processor Yamaları Uygulanıyor...")
utils_kodu = """import os, torch\nimport onnxruntime as ort\nfrom rembg import new_session\nproviders = ["CPUExecutionProvider"]\ntry:\n    session = new_session(providers=providers)\nexcept:\n    session = None\ndef simple_preprocess(img): return img\ndef rotate_normals_torch(normals, rot_mat):\n    original_shape = normals.shape\n    normals = normals.reshape(-1, 3)\n    normals = torch.matmul(normals, rot_mat.T)\n    return normals.reshape(original_shape)\n"""
open("scripts/utils.py", "w", encoding="utf-8").write(utils_kodu)

proc_file = "custum_3d_diffusion/custum_modules/unifield_processor.py"
if os.path.exists(proc_file):
    c = open(proc_file, "r").read()
    c = c.replace("self.set_config(init_config)", "try:\n            self.set_config(init_config)\n        except:\n            pass")
    open(proc_file, "w").write(c)

print("🚀 [5/9] Diffusers ve Hub Monkey Patch'leri Enjekte Ediliyor...")
monkey_patch = f"""
import huggingface_hub
_old_hf_hub_download = huggingface_hub.hf_hub_download
def _new_hf_hub_download(repo_id, *args, **kwargs):
    if repo_id and "lambdalabs" in str(repo_id):
        filename = kwargs.get("filename", "")
        if filename: return r"{lambda_dir}/" + filename
        return r"{lambda_dir}/config.json"
    return _old_hf_hub_download(repo_id, *args, **kwargs)
huggingface_hub.hf_hub_download = _new_hf_hub_download

import torch, diffusers
_old_load = torch.load
torch.load = lambda *a, **k: _old_load(*a, **{{**k, 'weights_only': False}})

_old_get_model_file = diffusers.utils.hub_utils._get_model_file
def _new_get_model_file(*args, **kwargs):
    try: return _old_get_model_file(*args, **kwargs)
    except:
        if "weights_name" in kwargs and "safetensors" in kwargs["weights_name"]:
            kwargs["weights_name"] = kwargs["weights_name"].replace("safetensors", "bin")
        elif len(args) > 3 and "safetensors" in str(args[3]):
            args = list(args); args[3] = args[3].replace("safetensors", "bin"); args = tuple(args)
        return _old_get_model_file(*args, **kwargs)
diffusers.utils.hub_utils._get_model_file = _new_get_model_file
"""
app_path = "gradio_app.py"
if os.path.exists(app_path):
    acode = open(app_path, "r").read()
    if "_new_hf_hub_download" not in acode:
        open(app_path, "w").write(monkey_patch + "\n" + acode)

print("🚀 [6/9] Nvdiffrast 3D Motoru Kuruluyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NVlabs/nvdiffrast.git"], check=False)

print("🚀 [7/9] PyTorch3D Kaynak Koddan Derleniyor (⚠️ DİKKAT: 3-5 DAKİKA SÜREBİLİR!)...")
# Önceden kurulmadıysa kaynak koddan zorla derle
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/pytorch3d.git"], check=False)

print("🚀 [8/9] Cloudflare Tüneli Hazırlanıyor...")
cf_bin = f"{WORK_DIR}/cloudflared"
if not os.path.exists(cf_bin):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin])
    os.system(f"chmod +x {cf_bin}")
os.system("fuser -k 7860/tcp >/dev/null 2>&1")
os.system("pkill -9 -f gradio_app.py >/dev/null 2>&1")
subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=open(f"{WORK_DIR}/cf.log", "w"), stderr=subprocess.STDOUT)

print("🚀 [9/9] Sunucu Başlatılıyor! (Tüm Modeller Yükleniyor)...")
env_vars = os.environ.copy()
env_vars["PYTHONPATH"] = u3d_dir
env_vars["PYTHONUNBUFFERED"] = "1"
env_vars["PYOPENGL_PLATFORM"] = "egl"
env_vars["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gradio_log = open(f"{WORK_DIR}/gradio.log", "w")
proc = subprocess.Popen([sys.executable, "-u", "gradio_app.py"], cwd=u3d_dir, env=env_vars, stdout=gradio_log, stderr=subprocess.STDOUT)

while True:
    if os.path.exists(f"{WORK_DIR}/gradio.log"):
        log_str = open(f"{WORK_DIR}/gradio.log", "r", errors="ignore").read()
        lines = log_str.strip().split('\n')
        if lines: print(f"\r⏳ Son işlem: {lines[-1][:120]:<120}", end="")
        if "Running on local URL" in log_str or "7860" in log_str:
            print("\n\n✅ SUNUCU BAŞARIYLA BAŞLATILDI!")
            break
        if ("Traceback" in log_str or "SyntaxError" in log_str or "TypeError" in log_str or "ImportError" in log_str or "RuntimeError" in log_str or "ModuleNotFoundError" in log_str) and "FutureWarning" not in lines[-1]:
            print("\n\n❌ BİR HATA OLUŞTU! Hata detayı:")
            print('\n'.join(lines[-15:]))
            break
    time.sleep(3)

time.sleep(3)
cf_log = open(f"{WORK_DIR}/cf.log", "r").read() if os.path.exists(f"{WORK_DIR}/cf.log") else ""
gr_log = open(f"{WORK_DIR}/gradio.log", "r").read()
cf_link = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_log)
gr_link = re.findall(r"https://[a-zA-Z0-9]+\.gradio\.live", gr_log)

print("\n" + "="*70)
print("🚀 UNIQUE3D STÜDYO TAMAMEN HAZIR!")
if cf_link: print(f"🌐 1. CLOUDFLARE LİNKİNİZ : {cf_link[-1]}")
if gr_link: print(f"🌐 2. GRADIO LIVE LİNKİNİZ : {gr_link[-1]}")
print("="*70 + "\n")